# CrowdControl — Complete Fresh Rebuild (Google Colab / MOT20)

One clean notebook integrating the original project and the optimization work. **Everything is regenerated**; the **only existing project files required are in `datasets/`**.

**Save this notebook in MyDrive root or open it locally in Google Colab (not inside the project folder).**

**Execution order:** Run from CELL 0 onward. CELL 0 **does not delete anything** because you already removed the earlier project outputs. It verifies the dataset files, then CELLS 1+ create (or safely resume) the same new run in a directory tree in `MyDrive/Crowd_AI_Project/runs_fresh/`.

1. Regenerate MOT20 train/VAL/TEST datasets from ZIPs, verify split and image-label alignment.
2. **Retrain yolov8n, yolov8s, yolov8m, yolo11n** from official public pretrained initializations (no old project checkpoints).
3. Evaluate and select the best detector **using VAL only**, then generate a fresh detection cache.
4. Compare ByteTrack vs BoT-SORT, with fixed candidate settings, on **the same VAL detections**. Select using official HOTA/IDF1.
5. Evaluate detectors and trackers on held-out **TEST**, saving tables, predictions, metadata, and timing.
6. Recalculate trajectory CSV and videos; zones, transitions, feature tables, per-zone accuracy, visualizations and videos.

**Datasets:** training uses MOT20-01/02/03; VAL = frames 1–1657 of MOT20-05; TEST = frames 1658 onward. VAL and TEST are time-disjoint **but belong to the same camera/scene**, so they are not an unseen-scene generalization test.

**Important:** "Fresh training" means fresh fine-tuning from pretrained YOLO weights, not training with random initialization. No GNN training exists in the source notebooks; this notebook saves graph-ready zone features instead of inventing a GNN result.

**Storage:** The model weights and all important CSV/Parquet/figures/videos go to Drive immediately under organized folders, not only `/content` temp storage.

**Colab disconnect:** You can remount Drive and rerun setup/training; completed checkpoints from this *new run* are retained and unfinished training can resume from `last.pt`. To start another entirely new experiment, first archive or delete `runs_fresh/` yourself.


In [ ]:
# CELL 0 — MOUNT DRIVE + VERIFY PRESERVED DATASETS (NON-DESTRUCTIVE)
from pathlib import Path
import zipfile
from google.colab import drive

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/Crowd_AI_Project')
DATASETS = PROJECT / 'datasets'
assert DATASETS.is_dir() and not DATASETS.is_symlink(), f'Expected preserved folder: {DATASETS}'
for fname in ('MOT20.zip', 'MOT20Labels.zip'):
    item = DATASETS / fname
    assert item.is_file() and zipfile.is_zipfile(item), f'Missing or invalid dataset ZIP: {item}'
    print(f'PRESERVED: {item.name} ({item.stat().st_size / 1024**3:.2f} GB)')

old_run = PROJECT / 'runs_fresh'
if old_run.exists():
    manifest = old_run / 'metadata' / 'run_config.json'
    assert manifest.is_file(), (
        f'{old_run} exists without a fresh-run manifest; STOP to prevent reusing unrelated results.')
    print('Found THIS fresh-rebuild run. You can resume it after a Colab disconnect.')
else:
    print('No existing run results. This will be a completely new rebuild.')
others = [p.name for p in PROJECT.iterdir() if p.name != 'datasets']
print('Project entries (none deleted):', others or 'none')
print('Results directory:', old_run)


Mounted at /content/drive
PRESERVED: MOT20.zip (4.68 GB)
PRESERVED: MOT20Labels.zip (0.02 GB)
Found THIS fresh-rebuild run. You can resume it after a Colab disconnect.
Project entries (none deleted): ['runs_fresh']
Results directory: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh


In [ ]:
# ============================================================
# CELL 1 — FRESH CONFIG  (run first, every session)
# ============================================================

import os, json, time, shutil, zipfile, subprocess
import numpy as np
import pandas as pd



# ---------- Google Drive ----------
PROJECT_DRIVE  = "/content/drive/MyDrive/Crowd_AI_Project"
DATASETS_DRIVE = f"{PROJECT_DRIVE}/datasets"
RUNS_DRIVE     = f"{PROJECT_DRIVE}/runs_fresh"       # newly created by this notebook
MODEL_DIR      = f"{RUNS_DRIVE}/models"
DETECTION_DIR  = f"{RUNS_DRIVE}/detection"
TRACKING_DIR   = f"{RUNS_DRIVE}/tracking"
TRAJECTORY_DIR = f"{RUNS_DRIVE}/trajectories"
VIDEOS_DIR     = f"{RUNS_DRIVE}/videos"
FIGURES_DIR    = f"{RUNS_DRIVE}/figures"
METADATA_DIR   = f"{RUNS_DRIVE}/metadata"
for folder in (MODEL_DIR, DETECTION_DIR, TRACKING_DIR, TRAJECTORY_DIR, VIDEOS_DIR, FIGURES_DIR, METADATA_DIR):
    os.makedirs(folder, exist_ok=True)

MOT20_ZIP_DRIVE  = f"{DATASETS_DRIVE}/MOT20.zip"
LABELS_ZIP_DRIVE = f"{DATASETS_DRIVE}/MOT20Labels.zip"

# ---------- Local Colab paths ----------
MOT20_EXTRACT  = "/content/MOT20"
LABELS_EXTRACT = "/content/MOT20Labels"
YOLO_ROOT      = "/content/MOT20_YOLO_v4"
DATA_YAML      = f"{YOLO_ROOT}/data.yaml"
TE_ROOT        = f"{TRACKING_DIR}/trackeval"           # persist TrackEval predictions + evaluations on Drive

# ---------- Split ----------
# Only MOT20-01/02/03/05 have ground truth (04/06/07/08 are unlabeled test sequences).
TRAIN_SEQUENCES = ["MOT20-01", "MOT20-02", "MOT20-03"]
EVAL_SEQUENCE   = "MOT20-05"
SPLIT_FRAME     = 1657      # VAL = frames 1..1657 | TEST = frames 1658..3315
FPS             = 25        # MOT20 frame rate

# ---------- Detection ----------
DETECTORS      = ["yolov8n", "yolov8s", "yolov8m", "yolo11n"]
IMG_SIZE       = 1280       # MOT20 people are tiny; 512 was too small
EPOCHS         = 50
PATIENCE       = 15
BATCH          = -1         # auto batch size (~60% of GPU memory)
SEED           = 42
MAX_DET        = 1000       # MOT20 frames have up to ~220 people
MIN_VISIBILITY = 0.0        # 0.0 = keep every annotated pedestrian (MOT standard)
MIN_BOX_PX     = 2          # drop boxes smaller than this after clipping

# None = automatically use the detector with the highest VAL mAP50 (ties by VAL mAP50-95).
# Set e.g. "yolov8s" to force a faster/slower model for tracking.
TRACK_DETECTOR = None

# ---------- Output files on Google Drive ----------
DET_VAL_CSV    = f"{DETECTION_DIR}/detector_validation.csv"
DET_COMPARE_CSV = f"{DETECTION_DIR}/detector_comparison_VAL_TEST.csv"
FPS_CSV         = f"{DETECTION_DIR}/detector_fps_validation.csv"
SELECTION_JSON  = f"{METADATA_DIR}/selection.json"
TRACK_VAL_CSV   = f"{TRACKING_DIR}/tracking_VAL_comparison.csv"
BEST_TRACK_JSON = f"{METADATA_DIR}/best_tracker.json"
FINAL_CSV       = f"{TRACKING_DIR}/tracking_TEST_trackeval.csv"
TRAJ_CSV        = f"{TRAJECTORY_DIR}/trajectories_{EVAL_SEQUENCE}.csv"
TRACK_VIDEO     = f"{VIDEOS_DIR}/tracking_{EVAL_SEQUENCE}.mp4"
TRAJ_VIDEO      = f"{VIDEOS_DIR}/trajectories_{EVAL_SEQUENCE}.mp4"
REPORT_ZIP      = f"{FIGURES_DIR}/report_figures.zip"

# ---------- Zones (stage 4) ----------
ZONES_DIR   = f"{RUNS_DRIVE}/zones"
FIG_DIR     = f"{ZONES_DIR}/figures"
ZONES_VIDEO = f"{VIDEOS_DIR}/zones_{EVAL_SEQUENCE}.mp4"
ZONES_ZIP   = f"{ZONES_DIR}/zones_figures.zip"
os.makedirs(FIG_DIR, exist_ok=True)
GRID_ROWS, GRID_COLS = 3, 4             # 12 zones
CUSTOM_ZONES = None                     # optional: {"Gate A": [[x, y], ...], ...} polygons in pixels
STEP       = 25                         # 1 second per step
MIN_RUN    = 5                          # zone visits shorter than 5 frames (0.2 s) are border jitter
SPEED_LAG  = 5                          # speed from foot displacement over 5 frames (0.2 s)
GT_MIN_VIS = 0.25                       # second reference: GT people at least 25% visible

# Crowding levels — all thresholds are LEARNED ON THE VAL HALF ONLY (frames 1..SPLIT_FRAME),
# then applied unchanged to TEST, so later prediction results stay honest.
#   "own"   level: compares a zone with its OWN history.
#           LOW <= zone's median count < MEDIUM < zone's 90th percentile <= HIGH
#   "floor" level: physical crowding, comparable BETWEEN zones.
#           floor coverage = people x typical person size in that zone / zone area
#           (far-away people look smaller, so this corrects for camera perspective);
#           LOW / HIGH cut-offs = median / 90th percentile of coverage pooled over ALL zones.
LEVEL_Q     = (0.50, 0.90)
VIDEO_LEVEL = "floor"                   # which level colours the zone video: "floor" or "own"

def level3(x, lo, hi):
    """LOW if x <= lo, HIGH if x >= hi, else MEDIUM (works on numbers and arrays)."""
    return np.where(np.asarray(x) >= hi, "HIGH", np.where(np.asarray(x) <= lo, "LOW", "MEDIUM"))

GT_COLS = ["frame", "id", "x", "y", "w", "h", "mark", "cls", "vis"]

# ---------- Fresh-session safety ----------
# CELL 0 verified an empty output folder. This cell creates all new directories.
# No reuse/skip function is allowed in this notebook.
def saved(step, *paths):
    return False

# Run manifest: a fresh ID is created only once. Reopening this notebook continues
# THAT run, without importing the historical models or results you deleted.
from datetime import datetime, timezone
from uuid import uuid4
run_config = dict(
    dataset_zip=MOT20_ZIP_DRIVE, labels_zip=LABELS_ZIP_DRIVE,
    train_sequences=TRAIN_SEQUENCES, evaluation_sequence=EVAL_SEQUENCE,
    val_frames=[1, SPLIT_FRAME], test_start_frame=SPLIT_FRAME + 1,
    detectors=DETECTORS, initial_checkpoint_type='public_COCO_pretrained',
    epochs=EPOCHS, patience=PATIENCE, image_size=IMG_SIZE,
    max_detections=MAX_DET, seed=SEED,
    fresh_training=True, old_model_reuse=False,
)
manifest_path = f'{METADATA_DIR}/run_config.json'
if os.path.isfile(manifest_path):
    with open(manifest_path) as f:
        existing = json.load(f)
    for k, v in run_config.items():
        assert existing.get(k) == v, f'Mismatch in previous fresh-run config: {k}. Stop to prevent mixed results.'
    print('Continuing fresh run:', existing.get('run_id'))
else:
    run_config.update(run_id=str(uuid4()), created_at_utc=datetime.now(timezone.utc).isoformat())
    with open(manifest_path, 'w') as f:
        json.dump(run_config, f, indent=2)
    print('Created NEW fresh run:', run_config['run_id'])

# ============================================================
# Helpers used by every later cell
# ============================================================

def run_name(model_name):
    return f"{model_name}_MOT20_{IMG_SIZE}"

def best_weights(model_name):
    return f"{MODEL_DIR}/{run_name(model_name)}/weights/best.pt"

_seq_dir_cache = {}

def find_seq_dir(seq):
    """Return .../<seq>/img1 inside the extracted MOT20 folder."""
    if seq not in _seq_dir_cache:
        for root, dirs, files in os.walk(MOT20_EXTRACT):
            if os.path.basename(root) == "img1" and os.path.basename(os.path.dirname(root)) == seq:
                _seq_dir_cache[seq] = root
                break
        else:
            raise FileNotFoundError(f"{seq}/img1 not found under {MOT20_EXTRACT} — run SETUP first")
    return _seq_dir_cache[seq]

def find_gt(seq):
    for root, dirs, files in os.walk(LABELS_EXTRACT):
        if os.path.basename(root) == "gt" and os.path.basename(os.path.dirname(root)) == seq:
            path = os.path.join(root, "gt.txt")
            if os.path.isfile(path):
                return path
    raise FileNotFoundError(f"gt.txt for {seq} not found under {LABELS_EXTRACT}")

def list_frames(img_dir):
    return sorted(f for f in os.listdir(img_dir) if f.lower().endswith((".jpg", ".jpeg", ".png")))

def load_gt(seq):
    """All MOT20 annotations (all classes). Columns: GT_COLS."""
    return pd.read_csv(find_gt(seq), header=None, names=GT_COLS)

def pedestrians(gt):
    """Valid pedestrians only: mark == 1 and class == 1."""
    return gt[(gt["mark"] == 1) & (gt["cls"] == 1)]

def split_range(split):
    """(first_frame, last_frame) of MOT20-05 for 'val', 'test' or 'full'."""
    n = len(list_frames(find_seq_dir(EVAL_SEQUENCE)))
    return {"val": (1, SPLIT_FRAME), "test": (SPLIT_FRAME + 1, n), "full": (1, n)}[split]

def get_track_detector():
    if TRACK_DETECTOR:
        return TRACK_DETECTOR
    if os.path.isfile(SELECTION_JSON):
        with open(SELECTION_JSON) as f:
            return json.load(f)["track_detector"]
    raise RuntimeError("No detector selected yet — run the DETECTOR COMPARISON cell first")

def all_detectors_done():
    return all(os.path.isfile(f"{MODEL_DIR}/{run_name(d)}/DONE") for d in DETECTORS)

# The YOLO-format dataset (cells 2-3) is only needed to train or compare detectors
NEED_YOLO_DATASET = True  # always convert from original dataset

print("✓ Fresh-run config loaded: no previous model or result will be used")
print("Fresh run folder:", RUNS_DRIVE)
print("Outputs: models, detection, tracking, trajectories, videos, zones, figures, metadata")

Continuing fresh run: 357a4d36-ea10-4cd8-a6e6-0d68f2a890d2
✓ Fresh-run config loaded: no previous model or result will be used
Fresh run folder: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh
Outputs: models, detection, tracking, trajectories, videos, zones, figures, metadata


In [ ]:
# ============================================================
# CELL 2 — SETUP: install, copy and extract MOT20, get TrackEval
# ============================================================

%pip install -q ultralytics deep-sort-realtime pyarrow

import ultralytics, torch
print("Ultralytics:", ultralytics.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "none (fine if every GPU step is already saved; otherwise enable a GPU)")

# ---------- Copy zips Drive → local (much faster to read) ----------
for src in [MOT20_ZIP_DRIVE, LABELS_ZIP_DRIVE]:
    if not os.path.isfile(src):
        raise FileNotFoundError(src)
    dst = f"/content/{os.path.basename(src)}"
    if not os.path.isfile(dst) or os.path.getsize(dst) != os.path.getsize(src):
        print(f"Copying {os.path.basename(src)} ({os.path.getsize(src) / 1024**3:.2f} GB)...")
        shutil.copy2(src, dst)

# ---------- Extract (a marker file makes interrupted extractions re-run) ----------
for zip_path, out_dir in [("/content/MOT20.zip", MOT20_EXTRACT),
                          ("/content/MOT20Labels.zip", LABELS_EXTRACT)]:
    marker = f"{out_dir}/.extracted"
    if not os.path.isfile(marker):
        print(f"Extracting {os.path.basename(zip_path)}...")
        shutil.rmtree(out_dir, ignore_errors=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(out_dir)
        open(marker, "w").close()
print("✓ MOT20 ready")

# ---------- TrackEval (official MOTChallenge evaluation code) ----------
if not os.path.isdir("/content/TrackEval"):
    subprocess.run(["git", "clone", "-q", "https://github.com/JonathonLuiten/TrackEval.git",
                    "/content/TrackEval"], check=True)
print("✓ TrackEval ready")

# ---------- Show sequences ----------
for seq in TRAIN_SEQUENCES + [EVAL_SEQUENCE]:
    print(f"{seq}: {len(list_frames(find_seq_dir(seq)))} frames | GT: {find_gt(seq)}")
print("VAL  frames:", split_range("val"))
print("TEST frames:", split_range("test"))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 133.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 11.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics: 8.4.183
GPU: NVIDIA A100-SXM4-40GB
Copying MOT20.zip (4.68 GB)...
Copying MOT20Labels.zip (0.02 GB)...
Extracting MOT20.zip...
Extracting MOT20Labels.zip...
✓ MOT20 ready
✓ TrackEval ready
MOT20-01: 429 frames | GT: /content/MOT20Labels/MOT20Labels/train/MOT20-01/gt/gt.txt
MOT20-02: 2782 frames | GT: /content/MOT20Labels/MOT20Labels/train/MOT20-02/gt/gt.txt
MOT20-03: 2405 frame

## Detection — retrain from scratch (fresh fine-tuning)
All dataset labels are rebuilt from preserved MOT20 ZIPs. Every YOLO training run starts again from the pretrained `*.pt` initialization, rather than the previous project checkpoint. The four `best.pt` files will be newly produced in `runs_fresh/models/`.

In [ ]:
# ============================================================
# CELL 3 — CONVERT MOT20 → YOLO  (train / val / test)
# FIX: boxes are clipped by their corners (x1,y1,x2,y2), then normalised.
# ============================================================

from collections import defaultdict
from PIL import Image

if not NEED_YOLO_DATASET:
    print("✓ Skipped: all detectors are trained and compared, so the YOLO dataset is not needed (add 'convert' to FORCE to rebuild it)")
else:
    shutil.rmtree(YOLO_ROOT, ignore_errors=True)        # no stale files / label caches
    for split in ["train", "val", "test"]:
        os.makedirs(f"{YOLO_ROOT}/images/{split}", exist_ok=True)
        os.makedirs(f"{YOLO_ROOT}/labels/{split}", exist_ok=True)


    def convert(seq, split, first_frame, last_frame):
        img_dir = find_seq_dir(seq)
        frames = list_frames(img_dir)
        with Image.open(os.path.join(img_dir, frames[0])) as im:
            W, H = im.size

        gt = pedestrians(load_gt(seq))
        gt = gt[(gt["vis"] >= MIN_VISIBILITY) &
                (gt["frame"] >= first_frame) & (gt["frame"] <= last_frame)]

        labels = defaultdict(list)
        dropped = 0
        for fr, x, y, w, h in gt[["frame", "x", "y", "w", "h"]].itertuples(index=False):
            # Clip the CORNERS to the image, then convert
            x1, y1 = max(0.0, x), max(0.0, y)
            x2, y2 = min(float(W), x + w), min(float(H), y + h)
            bw, bh = x2 - x1, y2 - y1
            if bw < MIN_BOX_PX or bh < MIN_BOX_PX:
                dropped += 1
                continue
            labels[int(fr)].append(
                f"0 {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {bw / W:.6f} {bh / H:.6f}")

        n_images = n_boxes = 0
        for name in frames:
            fr = int(os.path.splitext(name)[0])
            if not first_frame <= fr <= last_frame:
                continue
            new_name = f"{seq}_{name}"
            src = os.path.join(img_dir, name)
            dst = f"{YOLO_ROOT}/images/{split}/{new_name}"
            try:
                os.link(src, dst)                # hard link: instant, no extra disk space
            except OSError:
                shutil.copy2(src, dst)
            with open(f"{YOLO_ROOT}/labels/{split}/{os.path.splitext(new_name)[0]}.txt", "w") as f:
                if labels[fr]:
                    f.write("\n".join(labels[fr]) + "\n")
            n_images += 1
            n_boxes += len(labels[fr])

        print(f"{seq} [{first_frame}-{last_frame}] → {split:5s} | {W}x{H} | "
              f"images {n_images} | boxes {n_boxes} | dropped (too small after clipping) {dropped}")


    for seq in TRAIN_SEQUENCES:
        convert(seq, "train", 1, len(list_frames(find_seq_dir(seq))))
    convert(EVAL_SEQUENCE, "val",  *split_range("val"))
    convert(EVAL_SEQUENCE, "test", *split_range("test"))

    with open(DATA_YAML, "w") as f:
        f.write(f"path: {YOLO_ROOT}\ntrain: images/train\nval: images/val\ntest: images/test\n"
                "names:\n  0: person\n")

    shutil.copy2(DATA_YAML, f'{METADATA_DIR}/data.yaml')
    split_rows = []
    for split in ('train', 'val', 'test'):
        img_count = len(os.listdir(f'{YOLO_ROOT}/images/{split}'))
        label_count = len(os.listdir(f'{YOLO_ROOT}/labels/{split}'))
        split_rows.append({'split': split, 'images': img_count, 'label_files': label_count})
    pd.DataFrame(split_rows).to_csv(f'{METADATA_DIR}/dataset_split_counts.csv', index=False)
    print("\n✓ data.yaml:", DATA_YAML)
    print('Dataset metadata saved:', METADATA_DIR)

MOT20-01 [1-429] → train | 1920x1080 | images 429 | boxes 19870 | dropped (too small after clipping) 0
MOT20-02 [1-2782] → train | 1920x1080 | images 2782 | boxes 154742 | dropped (too small after clipping) 0
MOT20-03 [1-2405] → train | 1173x880 | images 2405 | boxes 313658 | dropped (too small after clipping) 0
MOT20-05 [1-1657] → val   | 1654x1080 | images 1657 | boxes 327068 | dropped (too small after clipping) 0
MOT20-05 [1658-3315] → test  | 1654x1080 | images 1658 | boxes 319276 | dropped (too small after clipping) 0

✓ data.yaml: /content/MOT20_YOLO_v4/data.yaml
Dataset metadata saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/metadata


In [ ]:
# ============================================================
# CELL 4 — DATA LEAKAGE CHECK  (train / val / test)
# ============================================================

import hashlib, itertools

if not NEED_YOLO_DATASET:
    print("✓ Skipped: the leakage check is only needed when the training dataset is (re)built")
else:
    def files(split, kind):
        return set(os.listdir(f"{YOLO_ROOT}/{kind}/{split}"))

    def stem(names):
        return {os.path.splitext(n)[0] for n in names}

    def frame_no(name):
        return int(os.path.splitext(name.split("_")[1])[0])

    def sha(path):
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                h.update(chunk)
        return h.hexdigest()

    problems = []
    images = {s: files(s, "images") for s in ["train", "val", "test"]}
    labels = {s: files(s, "labels") for s in ["train", "val", "test"]}

    # 1. Filename overlap between any two splits
    for a, b in itertools.combinations(images, 2):
        n = len(images[a] & images[b])
        print(f"Filename overlap {a}/{b}: {n}")
        if n: problems.append(f"same filenames in {a} and {b}")

    # 2. Sequence overlap: train must not share a sequence with val/test
    seqs = {s: {n.split("_")[0] for n in images[s]} for s in images}
    print("\nSequences:", {s: sorted(v) for s, v in seqs.items()})
    if seqs["train"] & (seqs["val"] | seqs["test"]):
        problems.append("a training sequence is also in val/test")

    # 3. VAL and TEST come from the same sequence → their frame ranges must not overlap
    val_frames  = {frame_no(n) for n in images["val"]}
    test_frames = {frame_no(n) for n in images["test"]}
    print(f"VAL frames {min(val_frames)}-{max(val_frames)} | TEST frames {min(test_frames)}-{max(test_frames)}")
    if max(val_frames) >= min(test_frames):
        problems.append("VAL and TEST frame ranges overlap")

    # 4. Exact duplicate images (SHA-256) between train and val/test
    print("\nHashing images...")
    train_hashes = {sha(f"{YOLO_ROOT}/images/train/{n}") for n in images["train"]}
    dups = sum(sha(f"{YOLO_ROOT}/images/{s}/{n}") in train_hashes
               for s in ["val", "test"] for n in images[s])
    print("Exact duplicates train vs val/test:", dups)
    if dups: problems.append("duplicate images across splits")

    # 5. Every image has a label file
    for s in images:
        missing = len(stem(images[s]) - stem(labels[s]))
        print(f"{s}: images without label = {missing}")
        if missing: problems.append(f"{s} images without labels")

    print("\n" + ("✓ NO DATA LEAKAGE DETECTED" if not problems else "⚠ PROBLEMS: " + "; ".join(problems)))
    with open(f'{METADATA_DIR}/leakage_check.json', 'w') as f:
        json.dump({'problems': problems, 'train_sequences': TRAIN_SEQUENCES,
                   'val_frame_range': list(split_range('val')),
                   'test_frame_range': list(split_range('test'))}, f, indent=2)
    assert not problems, "Split leakage found; STOP before training"

Filename overlap train/val: 0
Filename overlap train/test: 0
Filename overlap val/test: 0

Sequences: {'train': ['MOT20-01', 'MOT20-02', 'MOT20-03'], 'val': ['MOT20-05'], 'test': ['MOT20-05']}
VAL frames 1-1657 | TEST frames 1658-3315

Hashing images...
Exact duplicates train vs val/test: 0
train: images without label = 0
val: images without label = 0
test: images without label = 0

✓ NO DATA LEAKAGE DETECTED


In [ ]:
# CELL 5 — TRAIN ALL FOUR DETECTORS AGAIN (NO RESUME / NO OLD WEIGHTS)
import gc, torch
from ultralytics import YOLO

assert torch.cuda.is_available(), 'Enable GPU in Runtime > Change runtime type before training.'
assert os.path.isdir(DATASETS_DRIVE), 'Dataset folder missing'
assert os.path.isfile(DATA_YAML), 'Run dataset-conversion CELL 3 first.'

for name in DETECTORS:
    dest = Path(MODEL_DIR) / run_name(name)
    done_file = dest / 'DONE'
    latest = dest / 'weights' / 'last.pt'
    complete = done_file.is_file() and Path(best_weights(name)).is_file()

    if complete:
        print(f'Already finished IN THIS fresh run: {name} — retaining new weights.')
        continue

    if dest.exists():
        assert latest.is_file(), (
            f'Incomplete training folder without last.pt: {dest}. '
            'Inspect this fresh run before removing that incomplete model directory.')
        print(f'\n=== RESUMING INTERRUPTED NEW TRAINING: {name} ===')
        model = YOLO(str(latest))
        model.train(resume=True, device=0)
    else:
        print(f'\n=== FIRST-TIME NEW TRAINING: {name} ===')
        model = YOLO(f'{name}.pt')  # COCO initial weights; never earlier project best.pt
        model.train(
            data=DATA_YAML, epochs=EPOCHS, patience=PATIENCE,
            imgsz=IMG_SIZE, batch=BATCH, device=0,
            workers=min(8, os.cpu_count() or 2),
            seed=SEED, deterministic=True, plots=True,
            project=MODEL_DIR, name=run_name(name), exist_ok=False,
        )
    assert Path(best_weights(name)).is_file(), f'{name}: newly trained best.pt missing'
    done_file.write_text('New training complete: this fresh run only.\n')
    print('New model checkpoint:', best_weights(name))
    del model
    gc.collect()
    torch.cuda.empty_cache()
print('\nAll four detectors retrained.')

Already finished IN THIS fresh run: yolov8n — retaining new weights.
Already finished IN THIS fresh run: yolov8s — retaining new weights.
Already finished IN THIS fresh run: yolov8m — retaining new weights.
Already finished IN THIS fresh run: yolo11n — retaining new weights.

All four detectors retrained.


In [ ]:
# CELL 6 — FRESH DETECTOR VALIDATION + SELECT BY VAL ONLY
from ultralytics import YOLO
import pandas as pd, json

rows = []
for name in DETECTORS:
    weight = best_weights(name)
    assert os.path.isfile(weight), f'Fresh trained model missing: {weight}'
    model = YOLO(weight)
    params = sum(p.numel() for p in model.model.parameters()) / 1e6
    r = model.val(
        data=DATA_YAML, split='val', imgsz=IMG_SIZE,
        batch=2, max_det=MAX_DET, device=0,
        plots=False, verbose=False,
        project='/content/crowd_val_tmp', name=name, exist_ok=True,
    )
    rows.append({'Model': name, 'Params (M)': round(params, 2),
                 'VAL P': r.box.mp, 'VAL R': r.box.mr,
                 'VAL mAP50': r.box.map50, 'VAL mAP50-95': r.box.map})
comparison_val = pd.DataFrame(rows).set_index('Model').round(4)
comparison_val.to_csv(DET_VAL_CSV)
# mAP50 is the declared model-selection score; do NOT peek at TEST.
chosen = TRACK_DETECTOR or comparison_val.sort_values(['VAL mAP50', 'VAL mAP50-95'], ascending=False).index[0]
assert chosen in DETECTORS
with open(SELECTION_JSON, 'w') as f:
    json.dump({'track_detector': chosen, 'selection_rule': 'VAL mAP50 then VAL mAP50-95',
               'train_sequences': TRAIN_SEQUENCES, 'val_frames': [1, SPLIT_FRAME]}, f, indent=2)
print('\n=== FRESH DETECTOR VALIDATION (NOT TEST) ===')
print(comparison_val.to_string())
print('\nSelected for initial tracking:', chosen)

Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3270.5±604.8 MB/s, size: 400.0 KB)
val: Scanning /content/MOT20_YOLO_v4/labels/val... 1657 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1657/1657 892.5it/s 1.9s
val: New cache created: /content/MOT20_YOLO_v4/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 829/829 18.1it/s 45.9s
                   all       1657     327068      0.831      0.849      0.872      0.532
Speed: 1.6ms preprocess, 4.0ms inference, 0.0ms loss, 2.8ms postprocess per image
Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3948.8±113

In [ ]:
# CELL 7 — REAL DETECTOR LATENCY ON VALIDATION FRAMES ONLY (BATCH=1)
import cv2, time, numpy as np, torch, pandas as pd
from ultralytics import YOLO

N_WARMUP, N_TIMED = 20, 200
img_dir = find_seq_dir(EVAL_SEQUENCE)
val_files = list_frames(img_dir)[:SPLIT_FRAME]
paths = [os.path.join(img_dir, f) for f in val_files][:N_WARMUP + N_TIMED]
assert len(paths) >= N_WARMUP + N_TIMED
rows=[]
for name in DETECTORS:
    m = YOLO(best_weights(name))
    timings=[]
    for i,p in enumerate(paths):
        torch.cuda.synchronize()
        t0=time.perf_counter()
        frame=cv2.imread(p)
        assert frame is not None, p
        m.predict(frame, imgsz=IMG_SIZE, conf=0.01, max_det=MAX_DET, device=0, verbose=False)
        torch.cuda.synchronize()
        if i >= N_WARMUP:
            timings.append((time.perf_counter()-t0)*1000)
    a=np.asarray(timings)
    rows.append({'Model':name, 'Mean latency (ms)':a.mean(),
                 'P95 latency (ms)':np.percentile(a,95), 'FPS':1000/a.mean()})
fps_table=pd.DataFrame(rows).set_index('Model').round(2)
fps_table.to_csv(FPS_CSV)
print('VAL-only detector latency (image read + YOLO inference/post-processing, not tracker):')
print(fps_table.to_string())
print('\nDetector selection stays based on VAL detection accuracy, not TEST.')

VAL-only detector latency (image read + YOLO inference/post-processing, not tracker):
         Mean latency (ms)  P95 latency (ms)    FPS
Model                                              
yolov8n              26.04             26.98  38.40
yolov8s              25.41             26.01  39.35
yolov8m              27.50             28.02  36.37
yolo11n              27.71             28.38  36.09

Detector selection stays based on VAL detection accuracy, not TEST.


## Tracking — fair ByteTrack and BoT-SORT comparison
A **single fresh detection cache** is generated using the selected freshly trained YOLO model. Both trackers receive the **identical boxes and confidence scores** and are compared on **VAL** before touching TEST. No appearance/ReID or camera-motion compensation is enabled for BoT-SORT in the primary comparison. Original thresholds and one previously useful alternative are candidates, not presumed winners.

In [ ]:
# CELL 8 — GENERATE DETECTION CACHE (SKIP IF EXISTS)

import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

DET_NAME = get_track_detector()
DET_FILE = f'{DETECTION_DIR}/detections_{DET_NAME}_{EVAL_SEQUENCE}.parquet'

if os.path.isfile(DET_FILE) and os.path.getsize(DET_FILE) > 0:
    print("SKIPPED — Detection cache already exists.")
    print("Cache:", DET_FILE)

else:
    print("Generating new detection cache...")

    model = YOLO(best_weights(DET_NAME))
    img_dir = find_seq_dir(EVAL_SEQUENCE)
    frames = list_frames(img_dir)

    chunks = []

    for i, name in enumerate(frames, 1):
        frame = cv2.imread(os.path.join(img_dir, name))
        frame_id = int(Path(name).stem)

        assert frame is not None, name

        result = model.predict(
            frame,
            imgsz=IMG_SIZE,
            conf=0.01,
            iou=0.7,
            max_det=MAX_DET,
            classes=[0],
            device=0,
            verbose=False
        )[0]

        boxes = result.boxes.xyxy.cpu().numpy()
        scores = result.boxes.conf.cpu().numpy()

        chunks.append(pd.DataFrame({
            'frame': np.full(len(scores), frame_id, dtype=int),
            'x1': boxes[:, 0],
            'y1': boxes[:, 1],
            'x2': boxes[:, 2],
            'y2': boxes[:, 3],
            'conf': scores
        }))

        if i % 500 == 0:
            print(f'{i}/{len(frames)} frames')

    detections = pd.concat(chunks, ignore_index=True)

    # Save safely using a temporary file
    temp_file = DET_FILE + ".tmp.parquet"
    detections.to_parquet(temp_file, index=False)
    os.replace(temp_file, DET_FILE)

    print("NEW cache:", DET_FILE)
    print("Detector:", DET_NAME)
    print("Detections:", len(detections))

print("CELL 8 completed. Continue to CELL 9.")

SKIPPED — Detection cache already exists.
Cache: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/detection/detections_yolov8m_MOT20-05.parquet
CELL 8 completed. Continue to CELL 9.


In [ ]:
# ============================================================
# CELL 9 — TRACKER AND OFFICIAL EVALUATION HELPERS
# (run before cells 10-13)
# ============================================================

import sys, cv2
from types import SimpleNamespace
from ultralytics.trackers.byte_tracker import BYTETracker
from ultralytics.trackers.bot_sort import BOTSORT
import inspect

# TrackEval still uses np.float / np.int / np.bool, which NumPy 2 removed
for _n, _t in (("float", float), ("int", int), ("bool", bool)):
    if not hasattr(np, _n):
        setattr(np, _n, _t)
sys.path.insert(0, "/content/TrackEval")
import trackeval

# ---------- Detections grouped by frame ONCE (fast lookup) ----------
DET_NAME = get_track_detector()
DET_FILE = f"{DETECTION_DIR}/detections_{DET_NAME}_{EVAL_SEQUENCE}.parquet"
_det = pd.read_parquet(DET_FILE)
DETS = {int(f): (g[["x1", "y1", "x2", "y2"]].to_numpy(np.float32), g["conf"].to_numpy(np.float32))
        for f, g in _det.groupby("frame")}
EMPTY = (np.empty((0, 4), np.float32), np.empty(0, np.float32))

EVAL_IMG_DIR = find_seq_dir(EVAL_SEQUENCE)
EVAL_FRAMES = list_frames(EVAL_IMG_DIR)
IMG_H, IMG_W = cv2.imread(os.path.join(EVAL_IMG_DIR, EVAL_FRAMES[0])).shape[:2]


# ---------- Object in the format ultralytics' BYTETracker expects ----------
class Detections:
    def __init__(self, boxes, conf):
        self.xyxy = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
        self.conf = np.asarray(conf, dtype=np.float32).reshape(-1)
        self.cls = np.zeros(len(self.xyxy), dtype=np.float32)
        x1, y1, x2, y2 = self.xyxy.T
        self.xywh = np.column_stack([(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1]).astype(np.float32)

    def __getitem__(self, index):
        return Detections(self.xyxy[index], self.conf[index])

    def __len__(self):
        return len(self.xyxy)



def run_tracker(kind, cfg, split):
    """Run ByteTrack or BoT-SORT using identical cached detections (no ReID/GMC)."""
    lo, hi = split_range(split)
    args = SimpleNamespace(
        tracker_type=('bytetrack' if kind=='byte' else 'botsort'),
        track_high_thresh=cfg['high'], track_low_thresh=cfg['low'],
        new_track_thresh=cfg['new'], track_buffer=cfg['buffer'],
        match_thresh=cfg['match'], fuse_score=True,
        gmc_method='none', with_reid=False, proximity_thresh=0.5,
        appearance_thresh=0.25, model='auto'
    )
    cls = BYTETracker if kind=='byte' else BOTSORT
    if 'frame_rate' in inspect.signature(cls).parameters:
        tracker = cls(args, frame_rate=FPS)
    else:
        tracker = cls(args)
    tracker.max_time_lost = int(FPS / 30.0 * cfg['buffer'])
    rows, elapsed = [], 0.0
    for f in range(lo, hi+1):
        boxes, conf = DETS.get(f, EMPTY)
        t0 = time.perf_counter()
        # Keep identical input confidence distribution for both trackers.
        tracks = tracker.update(Detections(boxes, conf), img=None)
        elapsed += time.perf_counter() - t0
        for t in tracks:  # Ultralytics: [x1,y1,x2,y2,id,score,cls,idx]
            rows.append((f, int(t[4]), float(t[0]), float(t[1]),
                         float(t[2]), float(t[3]), float(t[5])))
    return rows, elapsed*1000/(hi-lo+1)
# ---------- TrackEval (official MOTChallenge evaluation) ----------

def _seq_name(split):
    return f"{EVAL_SEQUENCE}-{split}"

def prepare_gt(split):
    """Writes the GT of one split in MOTChallenge layout, frames renumbered from 1."""
    lo, hi = split_range(split)
    seq_dir = f"{TE_ROOT}/gt/{_seq_name(split)}"
    os.makedirs(f"{seq_dir}/gt", exist_ok=True)
    gt = load_gt(EVAL_SEQUENCE)                         # all classes: TrackEval handles distractors
    gt = gt[(gt["frame"] >= lo) & (gt["frame"] <= hi)].copy()
    gt["frame"] -= lo - 1
    gt.to_csv(f"{seq_dir}/gt/gt.txt", header=False, index=False)
    with open(f"{seq_dir}/seqinfo.ini", "w") as f:
        f.write(f"[Sequence]\nname={_seq_name(split)}\nimDir=img1\nframeRate={FPS}\n"
                f"seqLength={hi - lo + 1}\nimWidth={IMG_W}\nimHeight={IMG_H}\nimExt=.jpg\n")

def write_tracker(tracker_name, split, rows):
    lo, _ = split_range(split)
    out_dir = f"{TE_ROOT}/trackers/{split}/{tracker_name}/data"
    os.makedirs(out_dir, exist_ok=True)
    with open(f"{out_dir}/{_seq_name(split)}.txt", "w") as f:
        for fr, tid, x1, y1, x2, y2, conf in sorted(rows):
            f.write(f"{fr - lo + 1},{tid},{x1:.2f},{y1:.2f},{x2 - x1:.2f},{y2 - y1:.2f},{conf:.4f},-1,-1,-1\n")

def run_trackeval(split, tracker_names):
    """Returns a DataFrame with HOTA, MOTA, IDF1, ... for each tracker."""
    prepare_gt(split)
    lo, hi = split_range(split)
    eval_cfg = trackeval.Evaluator.get_default_eval_config()
    eval_cfg.update({"USE_PARALLEL": False, "PRINT_RESULTS": False, "PRINT_CONFIG": False,
                     "TIME_PROGRESS": False, "OUTPUT_SUMMARY": False, "OUTPUT_DETAILED": False,
                     "PLOT_CURVES": False, "DISPLAY_LESS_PROGRESS": True})
    ds_cfg = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
    ds_cfg.update({"GT_FOLDER": f"{TE_ROOT}/gt", "TRACKERS_FOLDER": f"{TE_ROOT}/trackers/{split}",
                   "OUTPUT_FOLDER": f"{TE_ROOT}/out/{split}", "BENCHMARK": "MOT20",
                   "SPLIT_TO_EVAL": "train", "SKIP_SPLIT_FOL": True, "TRACKERS_TO_EVAL": tracker_names,
                   "SEQ_INFO": {_seq_name(split): hi - lo + 1}, "DO_PREPROC": True,
                   "PRINT_CONFIG": False})
    metrics = [trackeval.metrics.HOTA(), trackeval.metrics.CLEAR(), trackeval.metrics.Identity()]
    res, msg = trackeval.Evaluator(eval_cfg).evaluate(
        [trackeval.datasets.MotChallenge2DBox(ds_cfg)], metrics)

    out = []
    for name in tracker_names:
        status = msg["MotChallenge2DBox"][name]
        if status != "Success":
            raise RuntimeError(f"TrackEval failed for {name}: {status}")
        r = res["MotChallenge2DBox"][name]["COMBINED_SEQ"]["pedestrian"]
        out.append({"tracker": name,
                    "HOTA": float(np.mean(r["HOTA"]["HOTA"])) * 100,
                    "DetA": float(np.mean(r["HOTA"]["DetA"])) * 100,
                    "AssA": float(np.mean(r["HOTA"]["AssA"])) * 100,
                    "MOTA": r["CLEAR"]["MOTA"] * 100,
                    "IDF1": r["Identity"]["IDF1"] * 100,
                    "IDSW": int(r["CLEAR"]["IDSW"]),
                    "FP": int(r["CLEAR"]["CLR_FP"]),
                    "FN": int(r["CLEAR"]["CLR_FN"]),
                    "MOTP": r["CLEAR"]["MOTP"] * 100,
                    "MT": int(r["CLEAR"]["MT"]), "ML": int(r["CLEAR"]["ML"])})
    return pd.DataFrame(out).set_index("tracker")

print(f"✓ Helpers ready | detector {DET_NAME} | {len(DETS)} frames of detections | image {IMG_W}x{IMG_H}")

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 179ms
Prepared 1 package in 20ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.5s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

✓ Helpers ready | detector yolov8m | 3315 frames of detections | image 1654x1080


In [ ]:
# CELL 10 — VALIDATION: BYTE VS BOT-SORT, DEFAULT VS TUNED THRESHOLDS
import pandas as pd, json, math

CONFIGS = {
    'ByteTrack_default': ('byte', {'high':0.25,'low':0.10,'new':0.25,'buffer':30,'match':0.8}),
    'ByteTrack_candidate': ('byte', {'high':0.60,'low':0.20,'new':0.70,'buffer':15,'match':0.7}),
    'BoTSORT_candidate': ('bot', {'high':0.60,'low':0.20,'new':0.70,'buffer':15,'match':0.7}),
}
latency={}
for label,(kind,cfg) in CONFIGS.items():
    print('Running on VAL:',label)
    rows,ms=run_tracker(kind,cfg,'val')
    write_tracker(label,'val',rows)
    latency[label]=ms

val_scores=run_trackeval('val',list(CONFIGS))
val_scores['tracker_ms']=pd.Series(latency)
val_scores['tracker_fps']=1000/val_scores.tracker_ms
val_scores=val_scores.sort_values(['HOTA','IDF1'],ascending=False).round(3)
val_scores.to_csv(TRACK_VAL_CSV)
BEST_TRACKER=val_scores.index[0]
with open(BEST_TRACK_JSON,'w') as f:
    json.dump({'tracker':BEST_TRACKER,'kind':CONFIGS[BEST_TRACKER][0],
               'config':CONFIGS[BEST_TRACKER][1],'chosen_using':'VAL HOTA then IDF1'},f,indent=2)
print('\n=== NEW TRACKER COMPARISON — VAL ===')
print(val_scores.to_string())
print('\nSelected tracker for trajectories:',BEST_TRACKER)

Running on VAL: ByteTrack_default
Running on VAL: ByteTrack_candidate
Running on VAL: BoTSORT_candidate

CLEAR Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          

Identity Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          

Evaluating 3 tracker(s) on 1 sequence(s) for 1 class(es) on MotChallenge2DBox dataset using the following metrics: HOTA, CLEAR, Identity, Count


Evaluating ByteTrack_default


Evaluating ByteTrack_candidate


Evaluating BoTSORT_candidate


=== NEW TRACKER COMPARISON — VAL ===
                       HOTA    DetA    AssA    MOTA    IDF1  IDSW     FP     FN    MOTP   MT  ML  tracker_ms  tracker_fps
tracker                                                                                                                  
BoTSORT_candidate    61.272  66.831  56.318  83.082  75.540   791  17266  37275  80.408  551  24      20.952       47.728
By

### Optional preprocessing sanity check (from optimization notebook)
The prior experiments found no worthwhile AP50 improvement from CLAHE, gamma, unsharp, or contrast+unsharp. The clean rebuild **defaults to raw frames**. The next cell is optional: it recomputes a small *VAL-only* diagnostic with the **new weights**, saves a new CSV, and does not affect tracker selection. It is not official detection mAP or a reason to tune on TEST.

In [ ]:
# CELL 11 — OPTIONAL FRESH PREPROCESSING RUNTIME CHECK (VAL; DOES NOT ALTER TRACKING)
RUN_PREPROCESS_CHECK = False
if RUN_PREPROCESS_CHECK:
    import cv2, pandas as pd, time
    from ultralytics import YOLO
    model=YOLO(best_weights(get_track_detector()))
    paths=[os.path.join(EVAL_IMG_DIR,n) for n in EVAL_FRAMES[:200]]
    methods={
        'original': lambda im: im,
        'contrast': lambda im: cv2.convertScaleAbs(im, alpha=1.10, beta=5),
    }
    records=[]
    for label, transform in methods.items():
        prep_ms, model_ms, counts=[],[],[]
        for p in paths:
            frame=cv2.imread(p)
            t0=time.perf_counter(); updated=transform(frame)
            prep_ms.append(1000*(time.perf_counter()-t0))
            r=model.predict(updated,imgsz=IMG_SIZE,conf=0.01,iou=0.7,
                            classes=[0],max_det=MAX_DET,device=0,verbose=False)[0]
            model_ms.append(float(r.speed['inference']))
            counts.append(len(r.boxes))
        records.append({'method':label,'frames':len(paths),
                        'avg_detections':sum(counts)/len(counts),
                        'preprocess_ms':sum(prep_ms)/len(prep_ms),
                        'inference_ms':sum(model_ms)/len(model_ms)})
    diagnostic=pd.DataFrame(records)
    diagnostic.to_csv(f'{DETECTION_DIR}/optional_preprocessing_VAL.csv',index=False)
    print(diagnostic.to_string(index=False))
else:
    print('Skipped optional preprocessing check; original frames selected.')

Skipped optional preprocessing check; original frames selected.


## Final holdout evaluation — TEST used only after selection
Now that detector and tracker have been selected using VAL, report all detector scores on TEST and compare the trackers on exactly the same TEST frames. This is **newly computed**, not copied from previous runs.

In [ ]:
# CELL 12 — FRESH DETECTOR TEST EVALUATION (AFTER VAL SELECTION)
from ultralytics import YOLO
import pandas as pd
comparison_test=[]
for name in DETECTORS:
    model=YOLO(best_weights(name))
    r=model.val(data=DATA_YAML,split='test',imgsz=IMG_SIZE,batch=2,
                max_det=MAX_DET,device=0,plots=False,verbose=False,
                project='/content/crowd_test_tmp',name=name,exist_ok=True)
    comparison_test.append({'Model':name,'TEST P':r.box.mp,'TEST R':r.box.mr,
                            'TEST mAP50':r.box.map50,'TEST mAP50-95':r.box.map})
comparison_test=pd.DataFrame(comparison_test).set_index('Model')
comparison=comparison_val.join(comparison_test).round(4)
comparison.to_csv(DET_COMPARE_CSV)
print('=== DETECTOR TEST RESULTS — first and only holdout evaluation ===')
print(comparison.to_string())
print('Selected earlier using VAL:',get_track_detector())

Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3710.8±812.3 MB/s, size: 411.7 KB)
val: Scanning /content/MOT20_YOLO_v4/labels/test... 1658 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1658/1658 880.3it/s 1.9s
val: New cache created: /content/MOT20_YOLO_v4/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 829/829 18.0it/s 45.9s
                   all       1658     319276       0.84      0.853      0.877      0.544
Speed: 1.6ms preprocess, 4.0ms inference, 0.0ms loss, 2.7ms postprocess per image
Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2783.5±1

In [ ]:
# CELL 13 — FRESH OFFICIAL TRACKING TEST EVALUATION
import pandas as pd
latency_test={}
for label,(kind,cfg) in CONFIGS.items():
    print('TEST:',label)
    rows,ms=run_tracker(kind,cfg,'test')
    write_tracker(label,'test',rows)
    latency_test[label]=ms
final=run_trackeval('test',list(CONFIGS))
final['Tracker ms/frame']=pd.Series(latency_test)
fps_path=Path(FPS_CSV)
if fps_path.exists():
    det_ms=pd.read_csv(fps_path,index_col=0).loc[get_track_detector(),'Mean latency (ms)']
    final['Detector ms/frame']=det_ms
    final['Approx combined FPS']=1000/(final['Tracker ms/frame']+det_ms)
final=final.round(3)
final.to_csv(FINAL_CSV)
print('\n=== OFFICIAL TRACKING TEST RESULTS ===')
print(final.to_string())
print('Chosen before TEST:',BEST_TRACKER)

TEST: ByteTrack_default
TEST: ByteTrack_candidate
TEST: BoTSORT_candidate

CLEAR Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          

Identity Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          

Evaluating 3 tracker(s) on 1 sequence(s) for 1 class(es) on MotChallenge2DBox dataset using the following metrics: HOTA, CLEAR, Identity, Count


Evaluating ByteTrack_default


Evaluating ByteTrack_candidate


Evaluating BoTSORT_candidate


=== OFFICIAL TRACKING TEST RESULTS ===
                       HOTA    DetA    AssA    MOTA    IDF1  IDSW     FP     FN    MOTP   MT  ML  Tracker ms/frame  Detector ms/frame  Approx combined FPS
tracker                                                                                                                                                   
ByteTrack_default    56.516  62.510  51.301  74.860  67.051  2367  58643  19257  79.67

## Trajectories and visualization
Run the **VAL-selected** tracker over the complete MOT20-05 video and generate fresh trajectory CSV and videos. The last 50 frames shown in the visualization are for display only; saved CSV retains every tracked frame.

In [ ]:
# ============================================================
# CELL 14 — TRAJECTORIES: CSV + tracking video + trajectory video
# ============================================================

from collections import defaultdict, deque

# No old trajectories are loaded; create new IDs and points from fresh detections.
selected_kind, selected_cfg = CONFIGS[BEST_TRACKER]
tracker_label = BEST_TRACKER
rows, full_track_ms = run_tracker(selected_kind, selected_cfg, 'full')
print(f"Tracker (selected on VAL): {tracker_label} | {full_track_ms:.2f} ms/frame")

# ---------- CSV ----------
traj = pd.DataFrame(rows, columns=["frame", "track_id", "x1", "y1", "x2", "y2", "conf"])
traj["center_x"] = (traj.x1 + traj.x2) / 2
traj["center_y"] = (traj.y1 + traj.y2) / 2
traj["foot_x"] = traj["center_x"]            # bottom-centre = where the person stands
traj["foot_y"] = traj.y2
traj = traj.sort_values(["frame", "track_id"]).round(2)
traj.to_csv(TRAJ_CSV, index=False)

# ---------- Videos ----------
TRACK_RAW, TRAJ_RAW = "/content/tracking_raw.mp4", "/content/trajectory_raw.mp4"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
w_track = cv2.VideoWriter(TRACK_RAW, fourcc, FPS, (IMG_W, IMG_H))
w_traj  = cv2.VideoWriter(TRAJ_RAW,  fourcc, FPS, (IMG_W, IMG_H))
history = defaultdict(lambda: deque(maxlen=50))
by_frame = {f: g for f, g in traj.groupby("frame")}

for f, name in enumerate(EVAL_FRAMES, 1):
    frame = cv2.imread(os.path.join(EVAL_IMG_DIR, name))
    track_img, traj_img = frame.copy(), frame.copy()
    if f in by_frame:
        for r in by_frame[f].itertuples(index=False):
            cx, cy = int(r.center_x), int(r.center_y)
            fx, fy = int(r.foot_x), int(r.foot_y)
            tid = str(r.track_id)
            history[r.track_id].append((fx, fy))

            cv2.circle(track_img, (cx, cy), 4, (0, 255, 0), -1)
            cv2.putText(track_img, tid, (cx + 6, cy - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1, cv2.LINE_AA)

            pts = list(history[r.track_id])
            for a, b in zip(pts, pts[1:]):
                cv2.line(traj_img, a, b, (0, 255, 255), 2)
            cv2.circle(traj_img, (fx, fy), 4, (0, 255, 0), -1)
            cv2.putText(traj_img, tid, (fx + 6, fy - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1, cv2.LINE_AA)
    w_track.write(track_img)
    w_traj.write(traj_img)
    if f % 500 == 0:
        print(f"{f}/{len(EVAL_FRAMES)}")

w_track.release()
w_traj.release()

# ---------- H.264 for browsers, then copy to Drive ----------
for raw, final_path in [(TRACK_RAW, TRACK_VIDEO), (TRAJ_RAW, TRAJ_VIDEO)]:
    local = raw.replace("_raw", "_h264")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", raw, "-c:v", "libx264",
                    "-pix_fmt", "yuv420p", "-movflags", "+faststart", local], check=True)
    shutil.copy2(local, final_path)

# ---------- Completion check ----------
print("\n" + "=" * 60)
ok = True
for path in [TRAJ_CSV, TRACK_VIDEO, TRAJ_VIDEO]:
    size = os.path.getsize(path) / 1024**2 if os.path.isfile(path) else 0
    ok &= size > 0
    print(f"{'✓' if size > 0 else '✗'} {path} ({size:.1f} MB)")
print("Frames:", len(EVAL_FRAMES), "| trajectory points:", len(traj),
      "| unique IDs:", traj.track_id.nunique())
print("COMPLETE ✓" if ok else "⚠ SOME OUTPUTS ARE MISSING")

Tracker (selected on VAL): BoTSORT_candidate | 21.35 ms/frame
500/3315
1000/3315
1500/3315
2000/3315
2500/3315
3000/3315

✓ /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/trajectories/trajectories_MOT20-05.csv (45.3 MB)
✓ /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/videos/tracking_MOT20-05.mp4 (142.8 MB)
✓ /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/videos/trajectories_MOT20-05.mp4 (187.4 MB)
Frames: 3315 | trajectory points: 674650 | unique IDs: 2482
COMPLETE ✓


In [ ]:
# CELL 15 — GENERATE NEW DETECTOR / TRACKER FIGURES AND VIDEO STILLS
import os, shutil, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

out = Path(FIGURES_DIR) / 'report_figures'
out.mkdir(parents=True, exist_ok=True)
comp = pd.read_csv(DET_COMPARE_CSV, index_col=0)
fs = pd.read_csv(FPS_CSV, index_col=0)
tracking = pd.read_csv(FINAL_CSV, index_col=0)

fig, ax = plt.subplots(figsize=(8, 4))
comp[['VAL mAP50', 'TEST mAP50']].mul(100).plot.bar(ax=ax)
ax.set_ylabel('mAP50 (%)')
ax.set_title('Newly trained YOLO models: VAL vs TEST')
fig.tight_layout(); fig.savefig(out / 'detection_accuracy.png', dpi=170); plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4))
fs['FPS'].plot.bar(ax=ax)
ax.set_ylabel('Detection FPS')
ax.set_title('Batch-1 speed (VAL frames)')
fig.tight_layout(); fig.savefig(out / 'detection_fps.png', dpi=170); plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4))
tracking[['HOTA', 'IDF1', 'MOTA']].plot.bar(ax=ax)
ax.set_ylabel('Metric (%)')
ax.set_title('Fresh official TrackEval results — TEST')
fig.tight_layout(); fig.savefig(out / 'tracking_metrics.png', dpi=170); plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 3.5))
tracking['IDSW'].plot.bar(ax=ax)
ax.set_ylabel('ID switches (lower is better)')
ax.set_title('Identity switches on TEST')
fig.tight_layout(); fig.savefig(out / 'id_switches.png', dpi=170); plt.close(fig)

for frame_number in [500, 1700, 3000]:
    for video, label in [(TRACK_VIDEO, 'tracked'), (TRAJ_VIDEO, 'trajectory')]:
        cap = cv2.VideoCapture(video)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number - 1)
        ok, frame = cap.read()
        cap.release()
        if ok:
            cv2.imwrite(str(out / f'{label}_frame{frame_number}.jpg'), frame)

shutil.make_archive(REPORT_ZIP[:-4], 'zip', out)
print('Fresh report figures:', out)

Fresh report figures: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/figures/report_figures


## Zone-based crowd features and accuracy
Recalculate all zone counts, transitions, speeds, dwell/wait time, and videos from the **new trajectories**. Zone thresholds are calibrated using VAL only. Zone counts are estimates in *image space*, not calibrated people/m². Note that the original code treats class-1 pedestrian GT as the reference; a separate corrected static-person diagnostic is provided at the end.

In [ ]:
# ============================================================
# CELL 16 — ZONES SETUP: load trajectories + ground truth for MOT20-05
# ============================================================

import cv2

SEQ = EVAL_SEQUENCE
IMG_DIR = find_seq_dir(SEQ)
FRAMES = list_frames(IMG_DIR)
N_FRAMES = len(FRAMES)
H, W = cv2.imread(os.path.join(IMG_DIR, FRAMES[0])).shape[:2]
FRAME_IDX = range(1, N_FRAMES + 1)
N_STEPS = (N_FRAMES + STEP - 1) // STEP
FIRST_TEST_STEP = SPLIT_FRAME // STEP + (1 if SPLIT_FRAME % STEP else 0)  # first step that starts in TEST

def foot_table(df, id_col, fx, fy, box_area):
    return pd.DataFrame({"frame": df["frame"].astype(int).values,
                         "track_id": df[id_col].astype(int).values,
                         "foot_x": np.clip(fx, 0, W - 1), "foot_y": np.clip(fy, 0, H - 1),
                         "box_area": np.asarray(box_area, float)})    # person size, for floor coverage

# Tracker trajectories from VAL-selected tracker (may be ByteTrack or BoT-SORT)
traj = pd.read_csv(TRAJ_CSV)
TRK = foot_table(traj, "track_id", traj["foot_x"].values, traj["foot_y"].values,
                 ((traj.x2 - traj.x1) * (traj.y2 - traj.y1)).values)

# Ground truth: valid pedestrians, foot = bottom-centre of the box
gt = pedestrians(load_gt(SEQ))
GT_ALL = foot_table(gt, "id", (gt.x + gt.w / 2).values, (gt.y + gt.h).values, (gt.w * gt.h).values)
gt_vis = gt[gt["vis"] >= GT_MIN_VIS]
GT_VIS = foot_table(gt_vis, "id", (gt_vis.x + gt_vis.w / 2).values, (gt_vis.y + gt_vis.h).values,
                    (gt_vis.w * gt_vis.h).values)

print(f"{SEQ}: {N_FRAMES} frames, {W}x{H}, {N_STEPS} one-second steps (TEST starts at step {FIRST_TEST_STEP})")
print(f"Tracker rows: {len(TRK):,} | tracks: {TRK.track_id.nunique():,} | people/frame: {len(TRK) / N_FRAMES:.1f}")
print(f"GT rows: {len(GT_ALL):,} | people/frame: {len(GT_ALL) / N_FRAMES:.1f} "
      f"| visible >= {GT_MIN_VIS}: {len(GT_VIS) / N_FRAMES:.1f}/frame")

MOT20-05: 3315 frames, 1654x1080, 133 one-second steps (TEST starts at step 67)
Tracker rows: 674,650 | tracks: 2,482 | people/frame: 203.5
GT rows: 646,344 | people/frame: 195.0 | visible >= 0.25: 159.3/frame


In [ ]:
# ============================================================
# CELL 17 — ZONES: grid (or custom polygons) → mask, areas, neighbours, overlay image
# ============================================================

def grid_zones(rows, cols):
    zones = []
    for r in range(rows):
        for c in range(cols):
            x0, x1 = round(c * W / cols), round((c + 1) * W / cols) - 1
            y0, y1 = round(r * H / rows), round((r + 1) * H / rows) - 1
            zones.append({"id": r * cols + c, "name": f"Z{r * cols + c}", "row": r, "col": c,
                          "polygon": [[x0, y0], [x1, y0], [x1, y1], [x0, y1]]})
    return zones

if CUSTOM_ZONES:
    ZONES = [{"id": i, "name": name, "row": None, "col": None, "polygon": poly}
             for i, (name, poly) in enumerate(CUSTOM_ZONES.items())]
else:
    ZONES = grid_zones(GRID_ROWS, GRID_COLS)
ZONE_IDS = [z["id"] for z in ZONES]
ZONE_NAME = {z["id"]: z["name"] for z in ZONES}

# Pixel mask: value = zone id, -1 = outside every zone
ZONE_MASK = np.full((H, W), -1, np.int16)
for z in ZONES:
    cv2.fillPoly(ZONE_MASK, [np.array(z["polygon"], np.int32)], int(z["id"]))

AREA = {z["id"]: int((ZONE_MASK == z["id"]).sum()) for z in ZONES}
CENTROID = {}
for z in ZONES:
    ys, xs = np.nonzero(ZONE_MASK == z["id"])
    CENTROID[z["id"]] = (int(xs.mean()), int(ys.mean()))

# Neighbours = zones sharing a border (used later as graph edges)
kernel = np.ones((5, 5), np.uint8)
NEIGHBOURS = {}
for z in ZONES:
    grown = cv2.dilate((ZONE_MASK == z["id"]).astype(np.uint8), kernel)
    NEIGHBOURS[z["id"]] = sorted(int(v) for v in np.unique(ZONE_MASK[grown > 0]) if v >= 0 and v != z["id"])

for z in ZONES:
    z.update(area_px=AREA[z["id"]], centroid=CENTROID[z["id"]], neighbours=NEIGHBOURS[z["id"]])
with open(f"{ZONES_DIR}/zone_definition.json", "w") as f:
    json.dump({"sequence": SEQ, "image_size": [W, H], "grid": None if CUSTOM_ZONES else [GRID_ROWS, GRID_COLS],
               "zones": ZONES}, f, indent=1)

# Overlay image
overlay = cv2.imread(os.path.join(IMG_DIR, FRAMES[0]))
for z in ZONES:
    cv2.polylines(overlay, [np.array(z["polygon"], np.int32)], True, (255, 255, 255), 2)
    cx, cy = CENTROID[z["id"]]
    cv2.putText(overlay, z["name"], (cx - 25, cy + 12), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 6, cv2.LINE_AA)
    cv2.putText(overlay, z["name"], (cx - 25, cy + 12), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2, cv2.LINE_AA)
cv2.imwrite(f"{FIG_DIR}/zones_overlay.jpg", overlay, [cv2.IMWRITE_JPEG_QUALITY, 90])

pd.DataFrame([{'source': z['name'], 'target': ZONE_NAME[nb]}
              for z in ZONES for nb in z['neighbours']]).to_csv(
                  f'{ZONES_DIR}/zone_graph_edges.csv', index=False)
print(f"✓ {len(ZONES)} zones | zone area {min(AREA.values()):,}–{max(AREA.values()):,} px")
for z in ZONES:
    print(f"  {z['name']}: neighbours {[ZONE_NAME[n] for n in z['neighbours']]}")

✓ 12 zones | zone area 148,680–149,040 px
  Z0: neighbours ['Z1', 'Z4', 'Z5']
  Z1: neighbours ['Z0', 'Z2', 'Z4', 'Z5', 'Z6']
  Z2: neighbours ['Z1', 'Z3', 'Z5', 'Z6', 'Z7']
  Z3: neighbours ['Z2', 'Z6', 'Z7']
  Z4: neighbours ['Z0', 'Z1', 'Z5', 'Z8', 'Z9']
  Z5: neighbours ['Z0', 'Z1', 'Z2', 'Z4', 'Z6', 'Z8', 'Z9', 'Z10']
  Z6: neighbours ['Z1', 'Z2', 'Z3', 'Z5', 'Z7', 'Z9', 'Z10', 'Z11']
  Z7: neighbours ['Z2', 'Z3', 'Z6', 'Z10', 'Z11']
  Z8: neighbours ['Z4', 'Z5', 'Z9']
  Z9: neighbours ['Z4', 'Z5', 'Z6', 'Z8', 'Z10']
  Z10: neighbours ['Z5', 'Z6', 'Z7', 'Z9', 'Z11']
  Z11: neighbours ['Z6', 'Z7', 'Z10']


In [ ]:
# ============================================================
# CELL 18 — ASSIGN PEOPLE TO ZONES (foot point) + remove border jitter
# ============================================================

def smooth_runs(z):
    """Replace zone visits shorter than MIN_RUN frames by the neighbouring visit."""
    change = np.r_[True, z[1:] != z[:-1]]
    starts = np.flatnonzero(change)
    lens = np.diff(np.r_[starts, len(z)])
    vals = z[starts].copy()
    if len(vals) > 1:
        for i in range(len(vals)):
            if lens[i] < MIN_RUN:
                vals[i] = vals[i - 1] if i > 0 else vals[i + 1]
    return np.repeat(vals, lens)

def assign_zones(df):
    df = df.copy()
    df["zone_raw"] = ZONE_MASK[df.foot_y.round().astype(int).values, df.foot_x.round().astype(int).values]
    df = df[df.zone_raw >= 0].sort_values(["track_id", "frame"]).reset_index(drop=True)
    zones = df.zone_raw.values
    bounds = np.flatnonzero(np.r_[True, df.track_id.values[1:] != df.track_id.values[:-1], True])
    smoothed = np.empty_like(zones)
    for a, b in zip(bounds[:-1], bounds[1:]):
        smoothed[a:b] = smooth_runs(zones[a:b])
    df["zone"] = smoothed
    return df

t0 = time.time()
TRK_Z = assign_zones(TRK)
GT_ALL_Z = assign_zones(GT_ALL)
GT_VIS_Z = assign_zones(GT_VIS)

def raw_changes(df):
    same = df.track_id.values[1:] == df.track_id.values[:-1]
    raw = (same & (df.zone_raw.values[1:] != df.zone_raw.values[:-1])).sum()
    smooth = (same & (df.zone.values[1:] != df.zone.values[:-1])).sum()
    return raw, smooth

r, s = raw_changes(TRK_Z)
print(f"✓ Zones assigned in {time.time() - t0:.0f}s")
print(f"Tracker zone changes: {r:,} raw → {s:,} after removing border jitter ({100 * (1 - s / max(r, 1)):.0f}% were jitter)")
# Check: nobody lost (with a full grid, every foot point lies in a zone)
print("Check — people per frame preserved:", len(TRK_Z) == len(TRK) if not CUSTOM_ZONES else "custom zones (people outside zones are dropped)")

✓ Zones assigned in 0s
Tracker zone changes: 4,172 raw → 2,579 after removing border jitter (38% were jitter)
Check — people per frame preserved: True


In [ ]:
# ============================================================
# CELL 19 — PER-ZONE, PER-SECOND STATISTICS (tracker and ground truth)
# ============================================================

STEPS = range(N_STEPS)

def zone_statistics(df):
    df = df.sort_values(["track_id", "frame"]).reset_index(drop=True)
    df["step"] = (df.frame - 1) // STEP
    same_track = np.r_[False, df.track_id.values[1:] == df.track_id.values[:-1]]

    # ---- counts (per frame, then per step) ----
    per_frame = (df.groupby(["frame", "zone"]).size().unstack(fill_value=0)
                 .reindex(index=FRAME_IDX, columns=ZONE_IDS, fill_value=0))
    step_of_frame = (np.asarray(per_frame.index) - 1) // STEP
    count = per_frame.groupby(step_of_frame).mean()
    max_count = per_frame.groupby(step_of_frame).max()

    # ---- level calibration: learned on VAL frames only ----
    val = np.asarray(per_frame.index) <= SPLIT_FRAME
    own_lo = per_frame[val].quantile(LEVEL_Q[0])                       # per zone
    own_hi = per_frame[val].quantile(LEVEL_Q[1]).clip(lower=1.0)
    person_area = (df[df.frame <= SPLIT_FRAME].groupby("zone").box_area.median()
                   .reindex(ZONE_IDS))
    person_area = person_area.fillna(person_area.median())             # zones nobody visited in VAL
    zone_area = pd.Series(AREA).reindex(ZONE_IDS)
    coverage_pf = per_frame * (person_area / zone_area)               # per frame, per zone
    floor_lo, floor_hi = np.quantile(coverage_pf[val].values.ravel(), LEVEL_Q)   # pooled over ALL zones
    calib = pd.DataFrame({"zone": ZONE_IDS, "name": [ZONE_NAME[z] for z in ZONE_IDS],
                          "own_low_max": own_lo.values, "own_high_min": own_hi.values,
                          "person_area_px": person_area.values, "zone_area_px": zone_area.values,
                          "floor_low_max": floor_lo, "floor_high_min": floor_hi})

    # ---- transitions (zone changes inside a track) ----
    moved = same_track & np.r_[False, df.zone.values[1:] != df.zone.values[:-1]]
    idx = np.flatnonzero(moved)
    trans = pd.DataFrame({"step": df.step.values[idx], "from_zone": df.zone.values[idx - 1],
                          "to_zone": df.zone.values[idx]})
    trans_table = trans.groupby(["step", "from_zone", "to_zone"]).size().rename("n").reset_index()
    inflow = trans.groupby(["step", "to_zone"]).size().unstack(fill_value=0)
    outflow = trans.groupby(["step", "from_zone"]).size().unstack(fill_value=0)

    # ---- tracks starting / ending inside a zone ----
    first, last = df.groupby("track_id").head(1), df.groupby("track_id").tail(1)
    appear = first.groupby(["step", "zone"]).size().unstack(fill_value=0)
    disappear = last.groupby(["step", "zone"]).size().unstack(fill_value=0)

    # ---- speed and direction (foot displacement over SPEED_LAG frames) ----
    g = df.groupby("track_id")
    dt = (df.frame - g.frame.shift(SPEED_LAG)) / FPS
    ok = dt.notna() & (dt > 0) & (dt <= 2 * SPEED_LAG / FPS)
    df["vx"] = np.where(ok, (df.foot_x - g.foot_x.shift(SPEED_LAG)) / dt, np.nan)
    df["vy"] = np.where(ok, (df.foot_y - g.foot_y.shift(SPEED_LAG)) / dt, np.nan)
    df["speed"] = np.hypot(df.vx, df.vy)

    # ---- waiting time: seconds since the person entered their current zone ----
    new_visit = ~same_track | np.r_[True, df.zone.values[1:] != df.zone.values[:-1]]
    df["visit"] = np.cumsum(new_visit)
    df["wait_s"] = (df.frame - df.groupby("visit").frame.transform("min")) / FPS

    motion = df.groupby(["step", "zone"]).agg(speed=("speed", "mean"), vx=("vx", "mean"),
                                              vy=("vy", "mean"), wait_s=("wait_s", "mean"))

    def long(m, name):
        m = m.reindex(index=STEPS, columns=ZONE_IDS, fill_value=0)
        m.index.name, m.columns.name = "step", "zone"
        return m.stack().rename(name)

    out = pd.concat([long(count, "count"), long(max_count, "max_count"), long(inflow, "inflow"),
                     long(outflow, "outflow"), long(appear, "appear"), long(disappear, "disappear")], axis=1)
    out = out.join(motion)
    out = out.reset_index()
    out["zone_name"] = out.zone.map(ZONE_NAME)
    out["t_start_s"] = out.step * STEP / FPS
    out['split'] = np.where((out.step + 1) * STEP <= SPLIT_FRAME, 'VAL',
                            np.where(out.step * STEP + 1 > SPLIT_FRAME, 'TEST', 'BOUNDARY'))
    out["density_per_100x100px"] = out["count"] / out.zone.map(AREA) * 1e4
    out["level_own"] = level3(out["count"], out.zone.map(own_lo), out.zone.map(own_hi))
    out["floor_coverage"] = out["count"] * out.zone.map(person_area) / out.zone.map(zone_area)
    out["level_floor"] = level3(out["floor_coverage"], floor_lo, floor_hi)
    out["dir_deg"] = np.degrees(np.arctan2(-out.vy, out.vx))          # 0 = right, 90 = up (image)
    for c in ["speed", "vx", "vy", "wait_s", "dir_deg"]:
        out[c] = out[c].round(2)
    return out, trans_table, per_frame, calib, df

t0 = time.time()
STATS, TRANS, PF, CALIB, TRK_M = zone_statistics(TRK_Z)
STATS_GT, TRANS_GT, PF_GT, CALIB_GT, _ = zone_statistics(GT_ALL_Z)
STATS_GV, TRANS_GV, PF_GV, CALIB_GV, _ = zone_statistics(GT_VIS_Z)

STATS.round(4).to_csv(f"{ZONES_DIR}/zone_stats_{SEQ}.csv", index=False)
STATS_GT.round(4).to_csv(f"{ZONES_DIR}/zone_stats_GT_{SEQ}.csv", index=False)
STATS_GV.round(4).to_csv(f"{ZONES_DIR}/zone_stats_GT_visible_{SEQ}.csv", index=False)
TRANS.to_csv(f"{ZONES_DIR}/zone_transitions_{SEQ}.csv", index=False)
TRANS_GT.to_csv(f"{ZONES_DIR}/zone_transitions_GT_{SEQ}.csv", index=False)
CALIB.round(4).to_csv(f"{ZONES_DIR}/zone_level_calibration_{SEQ}.csv", index=False)

print("LEVEL THRESHOLDS (learned on VAL frames 1-%d only)" % SPLIT_FRAME)
print(CALIB.set_index("name")[["own_low_max", "own_high_min", "person_area_px"]].round(1).to_string())
print(f"Floor coverage (all zones): LOW <= {CALIB.floor_low_max[0]:.3f} < MEDIUM < {CALIB.floor_high_min[0]:.3f} <= HIGH\n")

# ---- Consistency checks ----
print(f"✓ Statistics computed in {time.time() - t0:.0f}s: {len(STATS):,} rows ({N_STEPS} steps × {len(ZONE_IDS)} zones)")
frame_total = PF.sum(axis=1).values
tracked = TRK.groupby("frame").size().reindex(FRAME_IDX, fill_value=0).values   # original trajectory rows
print("Check 1 — zone counts add up to people tracked in every frame:", bool((frame_total == tracked).all()))
print("Check 2 — every move counted once as outflow and once as inflow:",
      int(STATS.inflow.sum()) == int(STATS.outflow.sum()) == int(TRANS.n.sum()))

summary = (STATS.groupby("zone_name", sort=False)
           .agg(mean_count=("count", "mean"), max_count=("max_count", "max"),
                inflow_total=("inflow", "sum"), outflow_total=("outflow", "sum"),
                mean_speed_px_s=("speed", "mean"), mean_wait_s=("wait_s", "mean"),
                mean_floor_coverage=("floor_coverage", "mean"),
                high_own_pct=("level_own", lambda s: 100 * (s == "HIGH").mean()),
                high_floor_pct=("level_floor", lambda s: 100 * (s == "HIGH").mean())))
summary.round(4).to_csv(f"{ZONES_DIR}/zone_summary_{SEQ}.csv")
print("\nPER-ZONE SUMMARY (tracker, whole video)")
print(summary.round(1).to_string())

LEVEL THRESHOLDS (learned on VAL frames 1-1657 only)
      own_low_max  own_high_min  person_area_px
name                                           
Z0           16.0          23.0          8319.4
Z1           16.0          25.0          7593.5
Z2           21.0          30.0          7428.6
Z3           20.0          25.0          7705.5
Z4           19.0          30.0         10162.7
Z5           19.0          23.0          9977.9
Z6           18.0          28.0         10564.0
Z7           19.0          28.0          9926.7
Z8           12.0          18.0         12987.3
Z9           15.0          19.0         12975.1
Z10          15.0          20.0         12635.0
Z11          13.0          17.0         12907.5
Floor coverage (all zones): LOW <= 1.134 < MEDIUM < 1.658 <= HIGH

✓ Statistics computed in 5s: 1,596 rows (133 steps × 12 zones)
Check 1 — zone counts add up to people tracked in every frame: True
Check 2 — every move counted once as outflow and once as inflow: True

PER-ZO

## How accurate are the zone numbers?
The ground truth (GT) gives the true position of every person, so the same zone statistics computed from GT are the "correct answer".
We compare tracker vs GT for every zone and every second, on the **TEST** half (frames 1658+) and on the full video.

- **GT (all):** every annotated pedestrian, including people almost completely hidden. No camera-based system can see those, so this is a strict reference.
- **GT (visible ≥ 25%):** only people at least a quarter visible. This is a fairer reference for a camera system.

In [ ]:
# ============================================================
# CELL 21 — ACCURACY CHECK vs GROUND TRUTH
# ============================================================

def compare(trk, ref, ref_name):
    m = trk.merge(ref, on=["step", "zone"], suffixes=("_trk", "_gt"))
    rows = []
    for split_name, sel in [("TEST", m.split_trk == "TEST"), ("Full video", np.ones(len(m), bool))]:
        mm = m[sel]
        for var in ["count", "inflow", "outflow"]:
            a, b = mm[f"{var}_trk"].values, mm[f"{var}_gt"].values
            mae = np.abs(a - b).mean()
            rows.append({"reference": ref_name, "split": split_name, "quantity": var,
                         "GT mean": b.mean(), "tracker mean": a.mean(), "MAE": mae,
                         "relative MAE %": 100 * mae / max(b.mean(), 1e-9),
                         "bias": (a - b).mean(), "correlation r": np.corrcoef(a, b)[0, 1]})
        # Level agreement: does our system give a zone-second the same LOW/MEDIUM/HIGH as the truth?
        # (each source uses its own thresholds, learned on its own VAL frames)
        for lv in ["level_own", "level_floor"]:
            rows.append({"reference": ref_name, "split": split_name, "quantity": lv,
                         "level agreement %": 100 * (mm[f"{lv}_trk"] == mm[f"{lv}_gt"]).mean()})
    return pd.DataFrame(rows), m

ACC_ALL, M_ALL = compare(STATS, STATS_GT, "GT (all)")
ACC_VIS, M_VIS = compare(STATS, STATS_GV, f"GT (visible >= {GT_MIN_VIS})")
ACC = pd.concat([ACC_ALL, ACC_VIS], ignore_index=True)
ACC.round(3).to_csv(f"{ZONES_DIR}/zone_accuracy_{SEQ}.csv", index=False)

pd.set_option("display.width", 220)
print("ZONE ACCURACY — tracker vs ground truth (per zone, per second)")
print(ACC.round(2).to_string(index=False))

per_zone = (M_VIS[M_VIS.split_trk == "TEST"]
            .assign(err=lambda d: (d.count_trk - d.count_gt).abs())
            .groupby("zone").agg(gt_mean=("count_gt", "mean"), mae=("err", "mean")))
per_zone.index = per_zone.index.map(ZONE_NAME)
per_zone.to_csv(f'{ZONES_DIR}/zone_count_error_by_zone_TEST.csv')
print("\nCount error per zone on TEST (vs visible GT):")
print(per_zone.round(2).to_string())

ZONE ACCURACY — tracker vs ground truth (per zone, per second)
           reference      split    quantity  GT mean  tracker mean  MAE  relative MAE %  bias  correlation r  level agreement %
            GT (all)       TEST       count    16.03         16.75 1.84           11.47  0.72           0.93                NaN
            GT (all)       TEST      inflow     1.49          1.60 0.55           36.93  0.11           0.82                NaN
            GT (all)       TEST     outflow     1.49          1.60 0.55           36.93  0.11           0.82                NaN
            GT (all)       TEST   level_own      NaN           NaN  NaN             NaN   NaN            NaN              78.54
            GT (all)       TEST level_floor      NaN           NaN  NaN             NaN   NaN            NaN              81.44
            GT (all) Full video       count    16.25         16.96 1.91           11.73  0.71           0.93                NaN
            GT (all) Full video      infl

In [ ]:
# ============================================================
# CELL 22 — FIGURES
# ============================================================

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

if saved("zones_figures", ZONES_ZIP):
    print("Zone figures:", FIG_DIR)
else:
    C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
    INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"
    plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 200, "font.size": 10, "axes.edgecolor": "#c3c2b7",
                         "axes.labelcolor": INK, "text.color": INK, "xtick.color": MUTED, "ytick.color": MUTED,
                         "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
                         "axes.spines.top": False, "axes.spines.right": False})
    BLUES = LinearSegmentedColormap.from_list("blues", ["#f7fbff", "#9ec5f4", "#2a78d6", "#0d366b"])
    HEAT = LinearSegmentedColormap.from_list("heat", [(0.92, 0.41, 0.20, 0.0), (0.92, 0.41, 0.20, 0.55),
                                                      (1.0, 0.85, 0.40, 0.95)])
    frame0 = cv2.cvtColor(cv2.imread(os.path.join(IMG_DIR, FRAMES[SPLIT_FRAME])), cv2.COLOR_BGR2RGB)

    def save(fig, name):
        fig.tight_layout()
        fig.savefig(f"{FIG_DIR}/{name}", bbox_inches="tight")
        plt.close(fig)
        print("✓", name)

    def draw_grid(ax, color="white"):
        for z in ZONES:
            poly = np.array(z["polygon"] + [z["polygon"][0]])
            ax.plot(poly[:, 0], poly[:, 1], color=color, linewidth=0.8, alpha=0.8)
            cx, cy = CENTROID[z["id"]]
            ax.text(cx, cy, z["name"], color=color, fontsize=8, ha="center", va="center", alpha=0.9)

    # ---- 6a. Density heatmap: where people spend time ----
    acc = np.zeros((H, W), np.float32)
    np.add.at(acc, (TRK.foot_y.round().astype(int).values, TRK.foot_x.round().astype(int).values), 1)
    heat = cv2.GaussianBlur(acc, (0, 0), sigmaX=18)
    heat = heat / np.percentile(heat[heat > 0], 99.5)
    fig, ax = plt.subplots(figsize=(10, 6.5))
    ax.imshow((frame0 * 0.45).astype(np.uint8))
    im = ax.imshow(np.clip(heat, 0, 1), cmap=HEAT, vmin=0, vmax=1)
    draw_grid(ax)
    ax.set_axis_off()
    cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.01)
    cb.set_label("Relative time spent by people (foot points)")
    ax.set_title(f"Crowd density heatmap — {SEQ}, whole video", loc="left")
    save(fig, "density_heatmap.png")

    # ---- 6b. Temporal congestion map: zone × time ----
    mat = STATS.pivot(index="zone", columns="step", values="count").reindex(ZONE_IDS)
    fig, ax = plt.subplots(figsize=(11, 4.2))
    im = ax.imshow(mat.values, aspect="auto", cmap=BLUES, interpolation="nearest")
    ax.set_yticks(range(len(ZONE_IDS)), [ZONE_NAME[z] for z in ZONE_IDS])
    ax.set_xlabel("Time (seconds)")
    ax.axvline(FIRST_TEST_STEP - 0.5, color=INK, linestyle="--", linewidth=1)
    ax.text(FIRST_TEST_STEP, -0.8, " TEST half →", fontsize=8, va="bottom")
    ax.grid(False)
    fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01).set_label("People in zone (mean per second)")
    ax.set_title("Temporal congestion map: people per zone over time", loc="left")
    save(fig, "temporal_congestion_map.png")

    # ---- 6c. Flow map: mean walking direction and speed per zone ----
    flow = TRK_M.groupby("zone").agg(vx=("vx", "mean"), vy=("vy", "mean"), speed=("speed", "mean"))
    fig, ax = plt.subplots(figsize=(10, 6.5))
    ax.imshow((frame0 * 0.55).astype(np.uint8))
    draw_grid(ax)
    for zid, r in flow.iterrows():
        cx, cy = CENTROID[zid]
        ax.annotate("", xy=(cx + r.vx * 3, cy + r.vy * 3), xytext=(cx, cy),
                    arrowprops=dict(arrowstyle="-|>", color="#eda100", lw=2.5, mutation_scale=18))
        ax.text(cx, cy + 45, f"{r.speed:.0f} px/s", color="white", fontsize=8, ha="center")
    ax.set_axis_off()
    ax.set_title("Flow map: average movement per zone (arrow = 3 s of average motion)", loc="left")
    save(fig, "flow_map.png")

    # ---- 6d. Zone-to-zone transition matrix ----
    tm = (TRANS.groupby(["from_zone", "to_zone"]).n.sum().unstack(fill_value=0)
          .reindex(index=ZONE_IDS, columns=ZONE_IDS, fill_value=0))
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(tm.values, cmap=BLUES)
    names = [ZONE_NAME[z] for z in ZONE_IDS]
    ax.set_xticks(range(len(names)), names, rotation=45)
    ax.set_yticks(range(len(names)), names)
    ax.set_xlabel("To zone")
    ax.set_ylabel("From zone")
    ax.grid(False)
    vmax = tm.values.max()
    for i in range(len(names)):
        for j in range(len(names)):
            v = tm.values[i, j]
            if v:
                ax.text(j, i, str(v), ha="center", va="center", fontsize=7,
                        color="white" if v > vmax * 0.5 else INK)
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02).set_label("People moving (whole video)")
    ax.set_title("Movements between zones", loc="left")
    save(fig, "transition_matrix.png")

    # ---- 6e. Busiest zone: tracker vs ground truth over time ----
    busiest = STATS.groupby("zone")["count"].mean().idxmax()
    sel = lambda d: d[d.zone == busiest].sort_values("step")
    fig, ax = plt.subplots(figsize=(10, 3.8))
    ax.plot(sel(STATS_GT).t_start_s, sel(STATS_GT)["count"], color=C[0], lw=2, label="GT (all people)")
    ax.plot(sel(STATS_GV).t_start_s, sel(STATS_GV)["count"], color=C[2], lw=2, label=f"GT (visible >= {GT_MIN_VIS})")
    ax.plot(sel(STATS).t_start_s, sel(STATS)["count"], color=C[1], lw=2, label=f"Tracker ({BEST_TRACKER})")
    ax.axvline(FIRST_TEST_STEP * STEP / FPS, color=MUTED, linestyle="--", lw=1)
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("People in zone")
    ax.set_title(f"Busiest zone ({ZONE_NAME[busiest]}): tracker count vs ground truth", loc="left")
    ax.legend(frameon=False, ncols=3, loc="upper center", bbox_to_anchor=(0.5, -0.18))
    save(fig, "busiest_zone_vs_gt.png")

    # ---- 6f. Accuracy per zone on TEST ----
    fig, ax = plt.subplots(figsize=(9, 3.6))
    bars = ax.bar(per_zone.index, per_zone["mae"], color=C[0], width=0.6)
    ax.bar_label(bars, labels=[f"{v:.1f}" for v in per_zone["mae"]], fontsize=8, padding=2)
    ax.set_ylabel("Count error (people, MAE)")
    ax.set_title(f"Zone count error on TEST vs visible GT (lower is better)", loc="left")
    save(fig, "zone_count_error_test.png")

    # ---- 6g. How often each zone is HIGH: own level vs floor level ----
    x = np.arange(len(summary))
    fig, ax = plt.subplots(figsize=(10, 3.8))
    for i, (col, lab) in enumerate([("high_own_pct", "own level (vs the zone's own history)"),
                                    ("high_floor_pct", "floor level (physical crowding, comparable)")]):
        bars = ax.bar(x + (i - 0.5) * 0.4, summary[col], 0.37, color=C[i], label=lab)
        ax.bar_label(bars, labels=[f"{v:.0f}" for v in summary[col]], fontsize=7, padding=2)
    ax.set_xticks(x, summary.index)
    ax.set_ylabel("% of seconds at HIGH")
    ax.set_title("How often each zone is HIGH: own level vs floor level", loc="left")
    ax.legend(frameon=False, ncols=2, loc="upper center", bbox_to_anchor=(0.5, -0.12))
    save(fig, "levels_own_vs_floor.png")

✓ density_heatmap.png
✓ temporal_congestion_map.png
✓ flow_map.png
✓ transition_matrix.png
✓ busiest_zone_vs_gt.png
✓ zone_count_error_test.png
✓ levels_own_vs_floor.png


In [ ]:
# ============================================================
# CELL 23 — ZONE VIDEO: crowding level per zone, live count, flow arrow
# Colour = VIDEO_LEVEL (config); the label shows both levels: own (vs zone history) and floor (physical)
# ============================================================

if saved("zones_video", ZONES_VIDEO):
    print("Zone video:", ZONES_VIDEO)
else:
    LEVEL_BGR = {"LOW": (12, 163, 12), "MEDIUM": (25, 178, 250), "HIGH": (59, 59, 208)}   # green / amber / red
    SHORT = {"LOW": "LOW", "MEDIUM": "MED", "HIGH": "HIGH"}
    step_motion = STATS.set_index(["step", "zone"])[["vx", "vy"]]
    by_frame = {f: g for f, g in TRK_Z.groupby("frame")}

    # Per-frame levels for every zone (same VAL-learned thresholds as the statistics)
    cal = CALIB.set_index("zone")
    OWN_PF = pd.DataFrame({z: level3(PF[z], cal.at[z, "own_low_max"], cal.at[z, "own_high_min"]) for z in ZONE_IDS},
                          index=PF.index)
    COV_PF = PF * (cal.person_area_px / cal.zone_area_px)
    FLOOR_PF = pd.DataFrame(level3(COV_PF, cal.floor_low_max.iloc[0], cal.floor_high_min.iloc[0]),
                            index=PF.index, columns=PF.columns)
    COLOUR_PF = FLOOR_PF if VIDEO_LEVEL == "floor" else OWN_PF

    RAW = "/content/zones_raw.mp4"
    LOCAL_H264 = "/content/zones_h264.mp4"
    writer = cv2.VideoWriter(RAW, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (W, H))

    for f, name in enumerate(FRAMES, 1):
        img = cv2.imread(os.path.join(IMG_DIR, name))
        tint = img.copy()
        step = (f - 1) // STEP
        for z in ZONES:
            zid = z["id"]
            cv2.fillPoly(tint, [np.array(z["polygon"], np.int32)], LEVEL_BGR[COLOUR_PF.at[f, zid]])
        img = cv2.addWeighted(tint, 0.22, img, 0.78, 0)
        if f in by_frame:
            for x, y in by_frame[f][["foot_x", "foot_y"]].values.astype(int):
                cv2.circle(img, (x, y), 3, (255, 255, 255), -1)
        for z in ZONES:
            zid = z["id"]
            cv2.polylines(img, [np.array(z["polygon"], np.int32)], True, (255, 255, 255), 1)
            n = int(PF.at[f, zid])
            x0, y0 = z["polygon"][0]
            line1 = f"{z['name']}  {n} people"
            line2 = f"own {SHORT[OWN_PF.at[f, zid]]} | floor {SHORT[FLOOR_PF.at[f, zid]]}"
            cv2.rectangle(img, (x0 + 4, y0 + 4), (x0 + 12 + 10 * max(len(line1), len(line2)), y0 + 52), (0, 0, 0), -1)
            cv2.putText(img, line1, (x0 + 8, y0 + 23), cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                        LEVEL_BGR[COLOUR_PF.at[f, zid]], 2, cv2.LINE_AA)
            cv2.putText(img, line2, (x0 + 8, y0 + 45), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
            vx, vy = step_motion.loc[(step, zid)] if (step, zid) in step_motion.index else (np.nan, np.nan)
            if np.isfinite(vx) and np.isfinite(vy):
                cx, cy = CENTROID[zid]
                cv2.arrowedLine(img, (cx, cy), (int(cx + vx * 2), int(cy + vy * 2)), (0, 215, 255), 3, tipLength=0.3)
        cv2.putText(img, f"t = {f / FPS:6.1f} s", (W - 210, H - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
        writer.write(img)
        if f % 500 == 0:
            print(f"{f}/{N_FRAMES}")
    writer.release()

    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", RAW, "-c:v", "libx264", "-pix_fmt", "yuv420p",
                    "-movflags", "+faststart", LOCAL_H264], check=True)
    shutil.copy2(LOCAL_H264, ZONES_VIDEO)

    # Stills for the report
    cap = cv2.VideoCapture(LOCAL_H264)
    for fno in [500, 1700, 3000]:
        cap.set(cv2.CAP_PROP_POS_FRAMES, fno - 1)
        ok, still = cap.read()
        if ok:
            cv2.imwrite(f"{FIG_DIR}/zones_video_f{fno}.jpg", still, [cv2.IMWRITE_JPEG_QUALITY, 90])
    cap.release()
    print("✓ Zone video:", ZONES_VIDEO)

500/3315
1000/3315
1500/3315
2000/3315
2500/3315
3000/3315
✓ Zone video: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/videos/zones_MOT20-05.mp4


In [ ]:
# ============================================================
# CELL 24 — SUMMARY + ZIP FOR THE REPORT
# ============================================================

print("=" * 90)
print(f"ZONE ANALYSIS — {SEQ} | {len(ZONES)} zones ({GRID_ROWS}x{GRID_COLS} grid) | {N_STEPS} one-second steps")
print("=" * 90)
print("\nPER-ZONE SUMMARY (tracker, whole video)")
print(summary.round(1).to_string())
print("\nACCURACY vs GROUND TRUTH")
print(ACC.round(2).to_string(index=False))
print("\nLevel distribution (tracker, % of zone-seconds):")
print(pd.DataFrame({lv: STATS[lv].value_counts(normalize=True) * 100 for lv in ["level_own", "level_floor"]})
      .reindex(["LOW", "MEDIUM", "HIGH"]).round(1).to_string())
print("\nLevel thresholds (learned on VAL only):")
print(CALIB.round(3).to_string(index=False))

for fname in os.listdir(ZONES_DIR):
    if fname.endswith(".csv") or fname.endswith(".json"):
        shutil.copy2(f"{ZONES_DIR}/{fname}", f"{FIG_DIR}/{fname}")
shutil.make_archive(ZONES_ZIP[:-4], "zip", FIG_DIR)
print(f"\n✓ Everything saved in {ZONES_DIR}")
print(f"✓ Download for the report: {ZONES_DIR}/zones_figures.zip")

ZONE ANALYSIS — MOT20-05 | 12 zones (3x4 grid) | 133 one-second steps

PER-ZONE SUMMARY (tracker, whole video)
           mean_count  max_count  inflow_total  outflow_total  mean_speed_px_s  mean_wait_s  mean_floor_coverage  high_own_pct  high_floor_pct
zone_name                                                                                                                                     
Z0               14.0         27           182            104             50.8          4.0                  0.8           6.0             0.0
Z1               15.4         29           246            129             45.3          9.5                  0.8           6.0             0.0
Z2               21.9         34           383            115             44.6          4.6                  1.1           7.5             0.8
Z3               18.8         37           303            112             42.6          4.7                  1.0           9.8             3.0
Z4               16.7         3

## Optional corrected ground-truth count diagnostic (optimization integration)
MOT20 has class-7 *static person* annotations. Those were excluded in the initial class-1-only counting reference. This optional diagnostic reports both counts for the first 1657 frames and writes a new validation CSV. It does **not** rewrite or replace official MOTChallenge tracking metrics or TEST labels.

In [ ]:
# CELL 25 — OPTIONAL CLASS-7 STATIC-PERSON GROUND-TRUTH COUNT CHECK (VAL)
import pandas as pd
if 'gt_raw' in globals():
    raw_gt=gt_raw.copy()
else:
    raw_gt=load_gt(EVAL_SEQUENCE).copy()
valid=raw_gt[(raw_gt['mark']==1)&(raw_gt['frame'].between(1,SPLIT_FRAME))]
rows=[]
for cls,title in [(1,'pedestrian'),(7,'static_person')]:
    g=valid[valid['cls']==cls]
    rows.append({'reference_class':title,'class_id':cls,
                 'annotations_val':len(g),'unique_tracks_val':g['id'].nunique(),
                 'frames_with_annotations':g['frame'].nunique()})
count_diag=pd.DataFrame(rows)
count_diag.to_csv(f'{ZONES_DIR}/ground_truth_class_counts_VAL.csv',index=False)
print(count_diag.to_string(index=False))
print('Class-7 diagnostic only; official TrackEval GT is unchanged.')

reference_class  class_id  annotations_val  unique_tracks_val  frames_with_annotations
     pedestrian         1           327068                726                     1657
  static_person         7                0                  0                        0
Class-7 diagnostic only; official TrackEval GT is unchanged.


## Drive output structure and completion audit
Everything is written to `MyDrive/Crowd_AI_Project/runs_fresh/`:

- `models/<detector>/weights/best.pt` — newly fine-tuned weights, train logs / plots.
- `detection/` — VAL/TEST comparison, measured inference speed, new cached detections.
- `tracking/` — VAL/TEST official metrics and tracker MOT text files under `trackeval/`.
- `trajectories/` — per-frame full trajectories CSV.
- `videos/` — tracking, trajectory and zone videos.
- `zones/` — GT comparisons, calibration, zone statistics, zone transitions, graph edges, figures, ZIP.
- `figures/` — detector/tracker comparison plots and still frames.
- `metadata/` — training settings, split counts, checks, model and tracker selection JSON.

`datasets/` is never written to or deleted by this notebook. The `.ipynb` notebook itself is a separate source file, not a training result. Local `/content` extractions may disappear, but final deliverables live on Drive.


In [ ]:
# CELL 26 — FINAL GOOGLE DRIVE OUTPUT AUDIT
from pathlib import Path
import pandas as pd
required_paths = [
    *[Path(best_weights(name)) for name in DETECTORS],
    Path(DET_VAL_CSV), Path(DET_COMPARE_CSV), Path(FPS_CSV),
    Path(SELECTION_JSON), Path(TRACK_VAL_CSV), Path(BEST_TRACK_JSON),
    Path(FINAL_CSV), Path(TRAJ_CSV), Path(TRACK_VIDEO), Path(TRAJ_VIDEO),
    Path(ZONES_VIDEO), Path(ZONES_DIR) / f'zone_stats_{EVAL_SEQUENCE}.csv',
    Path(ZONES_DIR) / f'zone_accuracy_{EVAL_SEQUENCE}.csv',
    Path(ZONES_DIR) / 'zone_graph_edges.csv', Path(REPORT_ZIP), Path(ZONES_ZIP),
]
missing = []
for path in required_paths:
    exists = path.is_file() and path.stat().st_size > 0
    print(('OK     ' if exists else 'MISSING'), path.relative_to(PROJECT_DRIVE))
    if not exists:
        missing.append(str(path))
print('\n' + ('ALL EXPECTED OUTPUTS FOUND IN GOOGLE DRIVE' if not missing else
             f'{len(missing)} expected outputs missing; check earlier cells.'))
print('Preserved datasets:', sorted(p.name for p in Path(DATASETS_DRIVE).iterdir()))
assert not missing, 'The pipeline has incomplete saved results; see missing paths above.'


OK      runs_fresh/models/yolov8n_MOT20_1280/weights/best.pt
OK      runs_fresh/models/yolov8s_MOT20_1280/weights/best.pt
OK      runs_fresh/models/yolov8m_MOT20_1280/weights/best.pt
OK      runs_fresh/models/yolo11n_MOT20_1280/weights/best.pt
OK      runs_fresh/detection/detector_validation.csv
OK      runs_fresh/detection/detector_comparison_VAL_TEST.csv
OK      runs_fresh/detection/detector_fps_validation.csv
OK      runs_fresh/metadata/selection.json
OK      runs_fresh/tracking/tracking_VAL_comparison.csv
OK      runs_fresh/metadata/best_tracker.json
OK      runs_fresh/tracking/tracking_TEST_trackeval.csv
OK      runs_fresh/trajectories/trajectories_MOT20-05.csv
OK      runs_fresh/videos/tracking_MOT20-05.mp4
OK      runs_fresh/videos/trajectories_MOT20-05.mp4
OK      runs_fresh/videos/zones_MOT20-05.mp4
OK      runs_fresh/zones/zone_stats_MOT20-05.csv
OK      runs_fresh/zones/zone_accuracy_MOT20-05.csv
OK      runs_fresh/zones/zone_graph_edges.csv
OK      runs_fresh/figures/report

In [ ]:
# CELL 27 — MOUNT GOOGLE DRIVE

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

meta = Path(
    "/content/drive/MyDrive/Crowd_AI_Project/runs_fresh/metadata"
)

print("Metadata folder:", meta.exists())
print("Leakage report:", (meta / "leakage_check.json").exists())
print("Run config:", (meta / "run_config.json").exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Metadata folder: True
Leakage report: True
Run config: True


In [ ]:
# CELL 27 — DATASET & GROUND TRUTH AUDIT (NO RETRAINING)

from pathlib import Path
import json
import zipfile
import pandas as pd
from io import BytesIO

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project")
META = ROOT / "runs_fresh" / "metadata"
LABELS = ROOT / "datasets" / "MOT20Labels.zip"

# 1. Verify the original leakage-check result
with open(META / "leakage_check.json") as f:
    leakage = json.load(f)

print("=== DATA LEAKAGE CHECK ===")
print("Problems:", leakage["problems"] or "None")
assert not leakage["problems"], "Previously reported leakage problems!"

# 2. Verify dataset configuration
with open(META / "run_config.json") as f:
    config = json.load(f)

print("\n=== DATASET SPLIT ===")
print("Training sequences:", config["train_sequences"])
print("Evaluation sequence:", config["evaluation_sequence"])
print("Validation frames:", config["val_frames"])
print("Test starts:", config["test_start_frame"])

assert set(config["train_sequences"]) == {
    "MOT20-01", "MOT20-02", "MOT20-03"
}
assert config["evaluation_sequence"] == "MOT20-05"
assert config["val_frames"] == [1, 1657]
assert config["test_start_frame"] == 1658

# 3. Check original Ground Truth integrity
results = []
problems = []

with zipfile.ZipFile(LABELS) as z:
    for seq in ["MOT20-01", "MOT20-02", "MOT20-03", "MOT20-05"]:
        matches = [
            name for name in z.namelist()
            if name.replace("\\", "/").endswith(f"{seq}/gt/gt.txt")
        ]
        assert len(matches) == 1, f"Cannot uniquely locate GT for {seq}"

        gt = pd.read_csv(
            BytesIO(z.read(matches[0])),
            header=None,
            names=["frame", "id", "x", "y", "w", "h",
                   "mark", "cls", "vis"]
        )

        people = gt[(gt["mark"] == 1) & (gt["cls"] == 1)]

        issues = {
            "duplicate_frame_id": int(
                people.duplicated(["frame", "id"]).sum()
            ),
            "invalid_dimensions": int(
                ((people["w"] <= 0) | (people["h"] <= 0)).sum()
            ),
            "invalid_visibility": int(
                ((people["vis"] < 0) | (people["vis"] > 1)).sum()
            ),
            "missing_values": int(people.isna().sum().sum())
        }

        if any(issues.values()):
            problems.append({"sequence": seq, **issues})

        results.append({
            "sequence": seq,
            "GT_pedestrian_rows": len(people),
            "unique_ids": people["id"].nunique(),
            "last_annotated_frame": int(gt["frame"].max()),
            **issues
        })

summary = pd.DataFrame(results)

print("\n=== GROUND TRUTH INTEGRITY ===")
print(summary.to_string(index=False))

report = {
    "split_config": config,
    "previous_leakage_check": leakage,
    "ground_truth_checks": results,
    "ground_truth_problems": problems,
    "note": "VAL and TEST are non-overlapping temporal segments of the same sequence."
}

output = META / "dataset_ground_truth_audit.json"
with open(output, "w") as f:
    json.dump(report, f, indent=2)

print("\nSaved:", output)
print("\nSTATUS:", "PASS" if not problems else "REVIEW REQUIRED")

=== DATA LEAKAGE CHECK ===
Problems: None

=== DATASET SPLIT ===
Training sequences: ['MOT20-01', 'MOT20-02', 'MOT20-03']
Evaluation sequence: MOT20-05
Validation frames: [1, 1657]
Test starts: 1658

=== GROUND TRUTH INTEGRITY ===
sequence  GT_pedestrian_rows  unique_ids  last_annotated_frame  duplicate_frame_id  invalid_dimensions  invalid_visibility  missing_values
MOT20-01               19870          74                   429                   0                   0                   0               0
MOT20-02              154742         270                  2782                   0                   0                   0               0
MOT20-03              313658         702                  2405                   0                   0                   0               0
MOT20-05              646344        1169                  3315                   0                   0                   0               0

Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/metadata/datase

In [ ]:
# CELL 28 — YOLO DETECTOR BASELINE ANALYSIS (VAL ONLY)

from pathlib import Path
import pandas as pd
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
DET = ROOT / "detection"
META = ROOT / "metadata"

# Load existing results — no inference or retraining
val = pd.read_csv(
    DET / "detector_validation.csv",
    index_col=0
)

speed = pd.read_csv(
    DET / "detector_fps_validation.csv",
    index_col=0
)

with open(META / "selection.json") as f:
    selection = json.load(f)

# Combine accuracy and speed
report = val.join(speed, how="inner")

required = [
    "Params (M)", "VAL P", "VAL R",
    "VAL mAP50", "VAL mAP50-95",
    "Mean latency (ms)", "P95 latency (ms)", "FPS"
]

assert all(c in report.columns for c in required), (
    f"Unexpected columns: {report.columns.tolist()}"
)

assert len(report) == 4, "Expected four YOLO models"

# Rank detectors using Validation only
report["Rank mAP50"] = report["VAL mAP50"].rank(
    ascending=False, method="min"
).astype(int)

report["Rank mAP50-95"] = report["VAL mAP50-95"].rank(
    ascending=False, method="min"
).astype(int)

report["Rank Recall"] = report["VAL R"].rank(
    ascending=False, method="min"
).astype(int)

report["Rank Speed"] = report["FPS"].rank(
    ascending=False, method="min"
).astype(int)

selected = selection["track_detector"]

print("=== CURRENT SELECTED DETECTOR ===")
print(selected)

print("\n=== ACCURACY AND SPEED COMPARISON ===")
print(report.round(4).to_string())

print("\n=== BEST MODEL BY METRIC ===")

for metric in [
    "VAL P", "VAL R",
    "VAL mAP50", "VAL mAP50-95", "FPS"
]:
    best = report[metric].idxmax()
    print(f"{metric}: {best} ({report.loc[best, metric]:.4f})")

print("\n=== SELECTED DETECTOR TRADE-OFF ===")
print(report.loc[selected, required].round(4).to_string())

# Save analysis separately without changing original results
out_dir = DET / "optimization_analysis"
out_dir.mkdir(parents=True, exist_ok=True)

output = out_dir / "detector_baseline_VAL.csv"
report.to_csv(output)

print("\nSaved:", output)
print("Original detector selection unchanged.")
print("No TEST data used.")

=== CURRENT SELECTED DETECTOR ===
yolov8m

=== ACCURACY AND SPEED COMPARISON ===
         Params (M)   VAL P   VAL R  VAL mAP50  VAL mAP50-95  Mean latency (ms)  P95 latency (ms)    FPS  Rank mAP50  Rank mAP50-95  Rank Recall  Rank Speed
Model                                                                                                                                                       
yolov8n        3.14  0.8307  0.8489     0.8716        0.5316              26.04             26.98  38.40           3              1            4           2
yolov8s       11.14  0.8415  0.8647     0.8710        0.5134              25.41             26.01  39.35           4              4            2           1
yolov8m       25.86  0.8396  0.8849     0.8755        0.5164              27.50             28.02  36.37           1              3            1           3
yolo11n        2.60  0.8442  0.8505     0.8731        0.5264              27.71             28.38  36.09           2              2   

In [ ]:
# CELL 29 — YOLOv8m CONFIDENCE ANALYSIS (VAL ONLY)

from pathlib import Path
from io import BytesIO
import zipfile
import numpy as np
import pandas as pd

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project")
DET = ROOT / "runs_fresh" / "detection"
OUT = DET / "optimization_analysis"
OUT.mkdir(parents=True, exist_ok=True)

SEQ = "MOT20-05"
VAL_END = 1657
IOU_THRESHOLD = 0.50
THRESHOLDS = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]

# Load existing YOLOv8m detections
cache = DET / f"detections_yolov8m_{SEQ}.parquet"
assert cache.is_file(), f"Missing detection cache: {cache}"

detections = pd.read_parquet(cache)
detections = detections[detections["frame"].between(1, VAL_END)]

required = {"frame", "x1", "y1", "x2", "y2", "conf"}
assert required.issubset(detections.columns)

# Read original MOT20 Ground Truth directly from ZIP
labels_zip = ROOT / "datasets" / "MOT20Labels.zip"
assert labels_zip.is_file(), f"Missing labels ZIP: {labels_zip}"

with zipfile.ZipFile(labels_zip) as z:
    matches = [
        name for name in z.namelist()
        if name.replace("\\", "/").endswith(
            f"{SEQ}/gt/gt.txt"
        )
    ]
    assert len(matches) == 1, f"GT not uniquely found: {matches}"

    gt = pd.read_csv(
        BytesIO(z.read(matches[0])),
        header=None,
        names=[
            "frame", "id", "x", "y", "w", "h",
            "mark", "cls", "vis"
        ]
    )

# Evaluate annotated pedestrians
gt = gt[
    (gt["frame"].between(1, VAL_END)) &
    (gt["mark"] == 1) &
    (gt["cls"] == 1)
].copy()

gt["x2"] = gt["x"] + gt["w"]
gt["y2"] = gt["y"] + gt["h"]

gt_by_frame = {
    int(f): g[["x", "y", "x2", "y2"]].to_numpy(
        dtype=np.float32
    )
    for f, g in gt.groupby("frame")
}

det_by_frame = {
    int(f): g[
        ["x1", "y1", "x2", "y2", "conf"]
    ].to_numpy(dtype=np.float32)
    for f, g in detections.groupby("frame")
}

EMPTY_GT = np.empty((0, 4), dtype=np.float32)
EMPTY_DET = np.empty((0, 5), dtype=np.float32)


def match_frame(pred, truth, threshold):
    """Confidence-ordered, one-to-one IoU matching."""

    pred = pred[pred[:, 4] >= threshold]
    pred = pred[np.argsort(-pred[:, 4])]

    used_gt = np.zeros(len(truth), dtype=bool)
    tp = 0
    fp = 0

    for box in pred[:, :4]:

        if len(truth) == 0:
            fp += 1
            continue

        x1 = np.maximum(box[0], truth[:, 0])
        y1 = np.maximum(box[1], truth[:, 1])
        x2 = np.minimum(box[2], truth[:, 2])
        y2 = np.minimum(box[3], truth[:, 3])

        intersection = (
            np.maximum(0, x2 - x1) *
            np.maximum(0, y2 - y1)
        )

        area_pred = max(
            0, (box[2] - box[0]) * (box[3] - box[1])
        )

        area_gt = (
            np.maximum(0, truth[:, 2] - truth[:, 0]) *
            np.maximum(0, truth[:, 3] - truth[:, 1])
        )

        union = area_pred + area_gt - intersection
        ious = intersection / np.maximum(union, 1e-9)

        ious[used_gt] = -1
        best = int(np.argmax(ious))

        if ious[best] >= IOU_THRESHOLD:
            tp += 1
            used_gt[best] = True
        else:
            fp += 1

    fn = len(truth) - tp
    return tp, fp, fn


results = []

for threshold in THRESHOLDS:
    tp = fp = fn = 0

    for frame in range(1, VAL_END + 1):
        pred = det_by_frame.get(frame, EMPTY_DET)
        truth = gt_by_frame.get(frame, EMPTY_GT)

        a, b, c = match_frame(pred, truth, threshold)
        tp += a
        fp += b
        fn += c

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = (
        2 * precision * recall /
        max(precision + recall, 1e-9)
    )

    results.append({
        "threshold": threshold,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "F1": f1
    })

report = pd.DataFrame(results)

print("=== YOLOv8m CONFIDENCE ANALYSIS — VAL ===")
print(report.round(4).to_string(index=False))

best = report.loc[report["F1"].idxmax()]

print("\n=== BEST THRESHOLD BY APPROXIMATE F1 ===")
print(f"Threshold: {best['threshold']:.2f}")
print(f"Precision: {best['precision']:.4f}")
print(f"Recall:    {best['recall']:.4f}")
print(f"F1:        {best['F1']:.4f}")

output = OUT / "yolov8m_confidence_analysis_VAL.csv"
report.to_csv(output, index=False)

print("\nSaved:", output)
print("No model or tracker configuration changed.")
print("TEST data was not used.")

=== YOLOv8m CONFIDENCE ANALYSIS — VAL ===
 threshold     TP     FP    FN  precision  recall     F1
      0.05 314247 256593 12821     0.5505  0.9608 0.7000
      0.10 311039 172019 16029     0.6439  0.9510 0.7679
      0.20 306547 111395 20521     0.7335  0.9373 0.8229
      0.30 302522  85045 24546     0.7806  0.9250 0.8466
      0.40 297847  68641 29221     0.8127  0.9107 0.8589
      0.50 291226  55771 35842     0.8393  0.8904 0.8641
      0.60 279156  44114 47912     0.8635  0.8535 0.8585
      0.70 247528  31200 79540     0.8881  0.7568 0.8172

=== BEST THRESHOLD BY APPROXIMATE F1 ===
Threshold: 0.50
Precision: 0.8393
Recall:    0.8904
F1:        0.8641

Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/detection/optimization_analysis/yolov8m_confidence_analysis_VAL.csv
No model or tracker configuration changed.
TEST data was not used.


In [ ]:
# CELL 30 — CHECK DETECTOR CACHES AND COMPARISON READINESS

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
DET = ROOT / "detection"

MODELS = ["yolov8n", "yolov8s", "yolov8m", "yolo11n"]
SEQ = "MOT20-05"
VAL_END = 1657

results = []

for model in MODELS:
    path = DET / f"detections_{model}_{SEQ}.parquet"

    row = {
        "Model": model,
        "Cache": "FOUND" if path.is_file() else "MISSING",
        "VAL frames": 0,
        "Detections": 0,
        "Ready": False
    }

    if path.is_file():
        df = pd.read_parquet(path)

        required = {"frame", "x1", "y1", "x2", "y2", "conf"}

        if required.issubset(df.columns):
            val = df[df["frame"].between(1, VAL_END)]

            row["VAL frames"] = val["frame"].nunique()
            row["Detections"] = len(val)
            row["Ready"] = True

    results.append(row)

report = pd.DataFrame(results)

print("=== DETECTOR CACHE AVAILABILITY ===")
print(report.to_string(index=False))

print("\n=== NEXT STEP ===")

ready = report[report["Ready"]]["Model"].tolist()
missing = report[~report["Ready"]]["Model"].tolist()

print("Ready for confidence analysis:", ready)
print("Need VAL inference:", missing)

if len(ready) == 4:
    print("\nAll four models can be compared without inference.")
else:
    print(
        "\nSome models need detection caches. "
        "We can generate only their VAL detections, "
        "using existing trained weights."
    )

print("\nNo training, inference, or files modified.")

=== DETECTOR CACHE AVAILABILITY ===
  Model Cache  VAL frames  Detections  Ready
yolov8n FOUND        1657     1428284   True
yolov8s FOUND        1657      971371   True
yolov8m FOUND        1657      857246   True
yolo11n FOUND        1657     1163677   True

=== NEXT STEP ===
Ready for confidence analysis: ['yolov8n', 'yolov8s', 'yolov8m', 'yolo11n']
Need VAL inference: []

All four models can be compared without inference.

No training, inference, or files modified.


In [ ]:
# CELL 31 — GENERATE MISSING YOLO DETECTION CACHES (VAL ONLY)

from pathlib import Path
import zipfile
import shutil
import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

PROJECT = Path("/content/drive/MyDrive/Crowd_AI_Project")
RUNS = PROJECT / "runs_fresh"
DET = RUNS / "detection"
MODELS = RUNS / "models"

SEQ = "MOT20-05"
VAL_END = 1657
IMG_SIZE = 1280
MAX_DET = 1000

TARGET_MODELS = ["yolov8n", "yolov8s", "yolo11n"]

assert torch.cuda.is_available(), (
    "Enable GPU in Runtime > Change runtime type."
)

# --------------------------------------------------
# 1. Locate validation images
# --------------------------------------------------

def find_images(root):
    for p in root.rglob("img1"):
        if p.parent.name == SEQ:
            return p
    return None

img_dir = find_images(Path("/content/MOT20"))

if img_dir is None:
    print("Extracted MOT20 images not found.")
    print("Restoring dataset from Google Drive...")

    zip_path = PROJECT / "datasets" / "MOT20.zip"
    assert zip_path.is_file(), f"Missing: {zip_path}"

    local_zip = Path("/content/MOT20.zip")

    if (
        not local_zip.exists()
        or local_zip.stat().st_size != zip_path.stat().st_size
    ):
        shutil.copy2(zip_path, local_zip)

    with zipfile.ZipFile(local_zip) as z:
        members = [
            name for name in z.namelist()
            if f"{SEQ}/img1/" in name.replace("\\", "/")
            and name.lower().endswith((".jpg", ".jpeg", ".png"))
        ]

        assert members, f"No {SEQ} images found in ZIP"

        z.extractall("/content/MOT20")

    img_dir = find_images(Path("/content/MOT20"))

assert img_dir is not None, "Validation image directory not found"

frames = sorted(
    [
        p for p in img_dir.iterdir()
        if p.suffix.lower() in (".jpg", ".jpeg", ".png")
        and p.stem.isdigit()
        and 1 <= int(p.stem) <= VAL_END
    ],
    key=lambda p: int(p.stem)
)

assert len(frames) == VAL_END, (
    f"Expected {VAL_END} validation frames, found {len(frames)}"
)

print(f"Validation images: {len(frames)}")
print("Image directory:", img_dir)

# --------------------------------------------------
# 2. Generate only missing caches
# --------------------------------------------------

columns = ["frame", "x1", "y1", "x2", "y2", "conf"]

for model_name in TARGET_MODELS:

    cache = DET / f"detections_{model_name}_{SEQ}.parquet"

    if cache.exists():
        print(f"\nSKIPPED: {model_name} — cache already exists")
        continue

    weights = (
        MODELS
        / f"{model_name}_MOT20_{IMG_SIZE}"
        / "weights"
        / "best.pt"
    )

    assert weights.is_file(), f"Missing trained weights: {weights}"

    print(f"\n=== GENERATING {model_name} VAL CACHE ===")

    model = YOLO(str(weights))
    chunks = []

    for i, path in enumerate(frames, 1):

        image = cv2.imread(str(path))
        assert image is not None, f"Cannot read: {path}"

        result = model.predict(
            image,
            imgsz=IMG_SIZE,
            conf=0.01,
            iou=0.7,
            max_det=MAX_DET,
            classes=[0],
            device=0,
            verbose=False
        )[0]

        boxes = result.boxes.xyxy.cpu().numpy()
        scores = result.boxes.conf.cpu().numpy()

        chunks.append(
            pd.DataFrame({
                "frame": np.full(len(scores), int(path.stem)),
                "x1": boxes[:, 0],
                "y1": boxes[:, 1],
                "x2": boxes[:, 2],
                "y2": boxes[:, 3],
                "conf": scores
            })
        )

        if i % 250 == 0 or i == len(frames):
            print(f"{model_name}: {i}/{len(frames)} frames")

    detections = pd.concat(chunks, ignore_index=True)

    # Save atomically to avoid treating interrupted writes
    # as completed caches.
    temp = cache.with_name(cache.stem + ".tmp.parquet")

    detections.to_parquet(temp, index=False)
    temp.replace(cache)

    print(f"SAVED: {cache.name}")
    print(f"Detections: {len(detections):,}")

    del model
    torch.cuda.empty_cache()

# --------------------------------------------------
# 3. Final availability check
# --------------------------------------------------

print("\n=== FINAL CACHE STATUS ===")

for name in ["yolov8n", "yolov8s", "yolov8m", "yolo11n"]:

    cache = DET / f"detections_{name}_{SEQ}.parquet"

    if cache.exists():
        print(f"{name}: READY")
    else:
        print(f"{name}: MISSING")

print("\nCELL 31 COMPLETE")
print("Existing YOLOv8m cache unchanged.")
print("No TEST inference or retraining performed.")

Validation images: 1657
Image directory: /content/MOT20/MOT20/train/MOT20-05/img1

SKIPPED: yolov8n — cache already exists

SKIPPED: yolov8s — cache already exists

SKIPPED: yolo11n — cache already exists

=== FINAL CACHE STATUS ===
yolov8n: READY
yolov8s: READY
yolov8m: READY
yolo11n: READY

CELL 31 COMPLETE
Existing YOLOv8m cache unchanged.
No TEST inference or retraining performed.


In [ ]:
# CELL 32 — COMPARE ALL YOLO DETECTORS (VALIDATION ONLY)

from pathlib import Path
from io import BytesIO
import zipfile
import numpy as np
import pandas as pd

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project")
DET = ROOT / "runs_fresh" / "detection"
OUT = DET / "optimization_analysis"
OUT.mkdir(parents=True, exist_ok=True)

MODELS = ["yolov8n", "yolov8s", "yolov8m", "yolo11n"]
SEQ = "MOT20-05"
VAL_END = 1657

THRESHOLDS = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
IOU_THRESHOLD = 0.50

# --------------------------------------------------
# 1. LOAD GROUND TRUTH
# --------------------------------------------------

zip_path = ROOT / "datasets" / "MOT20Labels.zip"
assert zip_path.is_file(), f"Missing: {zip_path}"

with zipfile.ZipFile(zip_path) as z:
    matches = [
        n for n in z.namelist()
        if n.replace("\\", "/").endswith(f"{SEQ}/gt/gt.txt")
    ]

    assert len(matches) == 1, "GT file not uniquely found"

    gt = pd.read_csv(
        BytesIO(z.read(matches[0])),
        header=None,
        names=[
            "frame", "id", "x", "y", "w", "h",
            "mark", "cls", "vis"
        ]
    )

gt = gt[
    gt["frame"].between(1, VAL_END)
    & (gt["mark"] == 1)
    & (gt["cls"] == 1)
].copy()

gt["x2"] = gt["x"] + gt["w"]
gt["y2"] = gt["y"] + gt["h"]

gt_frames = {
    int(f): group[["x", "y", "x2", "y2"]].to_numpy(
        dtype=np.float32
    )
    for f, group in gt.groupby("frame")
}

EMPTY_GT = np.empty((0, 4), dtype=np.float32)
EMPTY_DET = np.empty((0, 5), dtype=np.float32)

# --------------------------------------------------
# 2. ONE-TO-ONE DETECTION MATCHING
# --------------------------------------------------

def match_frame(pred, truth, confidence):
    pred = pred[pred[:, 4] >= confidence]
    pred = pred[np.argsort(-pred[:, 4])]

    used = np.zeros(len(truth), dtype=bool)
    tp = fp = 0

    if len(truth) == 0:
        return 0, len(pred), 0

    gt_area = (
        np.maximum(0, truth[:, 2] - truth[:, 0])
        * np.maximum(0, truth[:, 3] - truth[:, 1])
    )

    for box in pred[:, :4]:
        x1 = np.maximum(box[0], truth[:, 0])
        y1 = np.maximum(box[1], truth[:, 1])
        x2 = np.minimum(box[2], truth[:, 2])
        y2 = np.minimum(box[3], truth[:, 3])

        intersection = (
            np.maximum(0, x2 - x1)
            * np.maximum(0, y2 - y1)
        )

        pred_area = (
            max(0, box[2] - box[0])
            * max(0, box[3] - box[1])
        )

        union = pred_area + gt_area - intersection
        iou = intersection / np.maximum(union, 1e-9)

        iou[used] = -1
        best = int(np.argmax(iou))

        if iou[best] >= IOU_THRESHOLD:
            used[best] = True
            tp += 1
        else:
            fp += 1

    fn = len(truth) - tp
    return tp, fp, fn

# --------------------------------------------------
# 3. EVALUATE FOUR MODELS
# --------------------------------------------------

results = []

for model in MODELS:
    cache = DET / f"detections_{model}_{SEQ}.parquet"
    assert cache.is_file(), f"Missing cache: {cache}"

    df = pd.read_parquet(cache)
    required = {"frame", "x1", "y1", "x2", "y2", "conf"}
    assert required.issubset(df.columns)

    df = df[df["frame"].between(1, VAL_END)]

    assert df["frame"].nunique() == VAL_END, (
        f"{model}: incomplete VAL cache"
    )

    det_frames = {
        int(f): group[
            ["x1", "y1", "x2", "y2", "conf"]
        ].to_numpy(dtype=np.float32)
        for f, group in df.groupby("frame")
    }

    print(f"\nEvaluating {model}...")

    for threshold in THRESHOLDS:
        tp = fp = fn = 0

        for frame in range(1, VAL_END + 1):
            pred = det_frames.get(frame, EMPTY_DET)
            truth = gt_frames.get(frame, EMPTY_GT)

            a, b, c = match_frame(pred, truth, threshold)
            tp += a
            fp += b
            fn += c

        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = (
            2 * precision * recall
            / max(precision + recall, 1e-9)
        )

        results.append({
            "Model": model,
            "Threshold": threshold,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "Precision": precision,
            "Recall": recall,
            "F1": f1
        })

    print(f"{model}: COMPLETE")

# --------------------------------------------------
# 4. COMPARE BEST THRESHOLDS
# --------------------------------------------------

report = pd.DataFrame(results)

best = (
    report.sort_values(
        ["F1", "Recall"],
        ascending=False
    )
    .drop_duplicates("Model")
    .copy()
)

speed = pd.read_csv(
    DET / "detector_fps_validation.csv",
    index_col=0
)

assert "FPS" in speed.columns
best["FPS"] = best["Model"].map(speed["FPS"])

best = best.sort_values("F1", ascending=False)

print("\n=== BEST THRESHOLD PER MODEL — VAL ===")

print(
    best[
        [
            "Model", "Threshold",
            "Precision", "Recall", "F1",
            "FP", "FN", "FPS"
        ]
    ].round(4).to_string(index=False)
)

print("\n=== HIGHEST APPROXIMATE F1 ===")
winner = best.iloc[0]
print("Model:", winner["Model"])
print("Threshold:", winner["Threshold"])
print("F1:", round(winner["F1"], 4))

# --------------------------------------------------
# 5. SAVE NEW ANALYSIS
# --------------------------------------------------

all_file = OUT / "all_detectors_confidence_VAL.csv"
best_file = OUT / "best_detectors_confidence_VAL.csv"

report.to_csv(all_file, index=False)
best.to_csv(best_file, index=False)

print("\nSaved:", all_file)
print("Saved:", best_file)

print("\nNo retraining or inference performed.")
print("Original model selection unchanged.")
print("TEST data not used.")


Evaluating yolov8n...
yolov8n: COMPLETE

Evaluating yolov8s...
yolov8s: COMPLETE

Evaluating yolov8m...
yolov8m: COMPLETE

Evaluating yolo11n...
yolo11n: COMPLETE

=== BEST THRESHOLD PER MODEL — VAL ===
  Model  Threshold  Precision  Recall     F1    FP    FN   FPS
yolov8m        0.5     0.8393  0.8904 0.8641 55771 35842 36.37
yolov8s        0.4     0.8425  0.8664 0.8543 52958 43708 39.35
yolo11n        0.4     0.8311  0.8666 0.8485 57588 43616 36.09
yolov8n        0.5     0.8410  0.8406 0.8408 51967 52141 38.40

=== HIGHEST APPROXIMATE F1 ===
Model: yolov8m
Threshold: 0.5
F1: 0.8641

Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/detection/optimization_analysis/all_detectors_confidence_VAL.csv
Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/detection/optimization_analysis/best_detectors_confidence_VAL.csv

No retraining or inference performed.
Original model selection unchanged.
TEST data not used.


In [ ]:
# CELL 33 — TRACKING BASELINE & HYPERPARAMETER SETUP

from pathlib import Path
import pandas as pd
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
TRACKING = ROOT / "tracking"
META = ROOT / "metadata"

OUT = TRACKING / "optimization_v2"
OUT.mkdir(parents=True, exist_ok=True)

# 1. Load saved Validation tracking results
scores_path = TRACKING / "tracking_VAL_comparison.csv"
best_path = META / "best_tracker.json"

assert scores_path.is_file(), f"Missing: {scores_path}"
assert best_path.is_file(), f"Missing: {best_path}"

scores = pd.read_csv(scores_path, index_col=0)

with open(best_path) as f:
    selected = json.load(f)

print("=== CURRENT TRACKING BASELINE — VAL ===")

columns = [
    c for c in
    ["HOTA", "DetA", "AssA", "MOTA",
     "IDF1", "IDSW", "FP", "FN",
     "tracker_ms", "tracker_fps"]
    if c in scores.columns
]

print(scores[columns].round(3).to_string())

print("\n=== CURRENT SELECTED TRACKER ===")
print("Tracker:", selected["tracker"])
print("Type:", selected["kind"])
print("Hyperparameters:")
print(json.dumps(selected["config"], indent=2))

# 2. Define the tuning search
# Do not run these configurations yet.

BASE = {
    "high": 0.60,
    "low": 0.20,
    "new": 0.70,
    "buffer": 15,
    "match": 0.70
}

SEARCH = {
    "high":   [0.40, 0.50, 0.60],
    "low":    [0.10, 0.20],
    "new":    [0.50, 0.60, 0.70],
    "buffer": [15, 30],
    "match":  [0.70, 0.80]
}

# Controlled one-parameter-at-a-time candidates
candidates = []

for kind in ["byte", "bot"]:

    candidates.append({
        "kind": kind,
        "changed": "baseline",
        **BASE
    })

    for parameter, values in SEARCH.items():
        for value in values:
            if value == BASE[parameter]:
                continue

            config = BASE.copy()
            config[parameter] = value

            # Ensure valid confidence threshold ordering
            if not (
                config["low"] <= config["high"]
                <= config["new"]
            ):
                continue

            candidates.append({
                "kind": kind,
                "changed": parameter,
                **config
            })

plan = pd.DataFrame(candidates)
plan.insert(
    0, "experiment",
    [f"EXP_{i:03d}" for i in range(1, len(plan) + 1)]
)

print("\n=== TRACKER TUNING PLAN ===")
print(plan.to_string(index=False))

print("\nTotal experiments:", len(plan))
print("Tracker types:", plan["kind"].unique().tolist())

# 3. Save baseline and plan without altering old results
scores.to_csv(OUT / "tracking_baseline_VAL.csv")
plan.to_csv(OUT / "tracker_tuning_plan.csv", index=False)

with open(OUT / "original_best_tracker.json", "w") as f:
    json.dump(selected, f, indent=2)

print("\n=== SAVED ===")
print(OUT / "tracking_baseline_VAL.csv")
print(OUT / "tracker_tuning_plan.csv")
print(OUT / "original_best_tracker.json")

print("\nNo tracking or training performed.")
print("Original tracker selection unchanged.")
print("TEST data not used.")

=== CURRENT TRACKING BASELINE — VAL ===
                       HOTA    DetA    AssA    MOTA    IDF1  IDSW     FP     FN  tracker_ms  tracker_fps
tracker                                                                                                 
BoTSORT_candidate    61.272  66.831  56.318  83.082  75.540   791  17266  37275      20.952       47.728
ByteTrack_candidate  60.684  66.299  55.679  82.908  75.135   808  17135  37959      21.207       47.155
ByteTrack_default    56.883  62.898  51.630  76.149  68.207  2000  53504  22505      27.482       36.388

=== CURRENT SELECTED TRACKER ===
Tracker: BoTSORT_candidate
Type: bot
Hyperparameters:
{
  "high": 0.6,
  "low": 0.2,
  "new": 0.7,
  "buffer": 15,
  "match": 0.7
}

=== TRACKER TUNING PLAN ===
experiment kind  changed  high  low  new  buffer  match
   EXP_001 byte baseline   0.6  0.2  0.7      15    0.7
   EXP_002 byte     high   0.4  0.2  0.7      15    0.7
   EXP_003 byte     high   0.5  0.2  0.7      15    0.7
   EXP_004 byte 

In [ ]:
# CELL 34 — TRACKER HYPERPARAMETER TUNING (VAL ONLY)

from pathlib import Path
import pandas as pd
import numpy as np
import json
import time
import gc

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "tracking" / "optimization_v2"

PLAN_FILE = OUT / "tracker_tuning_plan.csv"
RESULTS_FILE = OUT / "tracker_tuning_results_VAL.csv"

assert PLAN_FILE.is_file(), "Run CELL 33 first."

# Reuse the original notebook's tracking and TrackEval helpers
required_helpers = [
    "run_tracker",
    "write_tracker",
    "run_trackeval"
]

missing = [
    name for name in required_helpers
    if name not in globals() or not callable(globals()[name])
]

assert not missing, (
    f"Missing helpers: {missing}. "
    "Run the original CELL 9 (tracking helpers), not training."
)

assert globals().get("DET_NAME") == "yolov8m", (
    "Tracking helpers must use YOLOv8m detections."
)

plan = pd.read_csv(PLAN_FILE)

expected_columns = {
    "experiment", "kind", "high", "low",
    "new", "buffer", "match"
}
assert expected_columns.issubset(plan.columns)

# Resume previously completed experiments
if RESULTS_FILE.exists():
    completed = pd.read_csv(RESULTS_FILE)
else:
    completed = pd.DataFrame()

done = (
    set(completed["experiment"].astype(str))
    if "experiment" in completed.columns
    else set()
)

print("=== TRACKER TUNING — VALIDATION ===")
print("Total experiments:", len(plan))
print("Already completed:", len(done))
print("Remaining:", len(plan) - len(done))

for _, exp in plan.iterrows():

    exp_id = str(exp["experiment"])

    if exp_id in done:
        print(f"\nSKIP {exp_id} — already evaluated")
        continue

    kind = str(exp["kind"])

    cfg = {
        "high": float(exp["high"]),
        "low": float(exp["low"]),
        "new": float(exp["new"]),
        "buffer": int(exp["buffer"]),
        "match": float(exp["match"])
    }

    assert kind in ("byte", "bot")
    assert 0 <= cfg["low"] <= cfg["high"] <= cfg["new"] <= 1

    # Unique name prevents overwriting original tracker predictions
    tracker_name = f"OPT_V2_{exp_id}"

    print(f"\n=== {exp_id} / {len(plan)} ===")
    print("Tracker:", kind)
    print("Config:", cfg)

    start = time.perf_counter()

    try:
        # 1. Run tracker on cached VAL detections
        rows, tracker_ms = run_tracker(
            kind, cfg, "val"
        )

        # 2. Save predictions in separate TrackEval namespace
        write_tracker(
            tracker_name, "val", rows
        )

        # 3. Evaluate this experiment using TrackEval
        scores = run_trackeval(
            "val", [tracker_name]
        )

        result = scores.loc[tracker_name].to_dict()

        result.update({
            "experiment": exp_id,
            "kind": kind,
            "changed": str(exp["changed"]),
            **cfg,
            "tracker_ms": tracker_ms,
            "tracker_fps": 1000 / tracker_ms,
            "elapsed_s": time.perf_counter() - start
        })

        # 4. Save immediately after each successful experiment
        completed = pd.concat(
            [completed, pd.DataFrame([result])],
            ignore_index=True
        )

        temp = RESULTS_FILE.with_name(
            RESULTS_FILE.stem + ".tmp.csv"
        )

        completed.to_csv(temp, index=False)
        temp.replace(RESULTS_FILE)

        done.add(exp_id)

        print(
            f"PASS | HOTA={result['HOTA']:.3f} "
            f"| IDF1={result['IDF1']:.3f} "
            f"| IDSW={int(result['IDSW'])} "
            f"| Tracker FPS={result['tracker_fps']:.2f}"
        )

    except Exception as e:
        print(f"FAILED: {exp_id}")
        print(type(e).__name__, str(e))
        print("Previous successful results are saved.")
        raise

    finally:
        gc.collect()

# --------------------------------------------------
# FINAL VALIDATION RANKING
# --------------------------------------------------

results = pd.read_csv(RESULTS_FILE)

ranking = results.sort_values(
    ["HOTA", "IDF1", "IDSW"],
    ascending=[False, False, True]
).reset_index(drop=True)

ranking_file = OUT / "tracker_tuning_ranking_VAL.csv"
ranking.to_csv(ranking_file, index=False)

print("\n=== FINAL TRACKER RANKING — VAL ===")

display_columns = [
    "experiment", "kind", "changed",
    "high", "low", "new", "buffer", "match",
    "HOTA", "IDF1", "MOTA", "IDSW",
    "tracker_fps"
]

print(
    ranking[display_columns]
    .round(3)
    .to_string(index=False)
)

winner = ranking.iloc[0]

print("\n=== BEST EXPERIMENT ===")
print("Experiment:", winner["experiment"])
print("Tracker:", winner["kind"])
print("HOTA:", round(winner["HOTA"], 3))
print("IDF1:", round(winner["IDF1"], 3))
print("ID switches:", int(winner["IDSW"]))

print("\nSaved:", RESULTS_FILE)
print("Saved:", ranking_file)

print("\nOriginal best_tracker.json unchanged.")
print("No TEST data used.")
print("No YOLO inference or training performed.")

=== TRACKER TUNING — VALIDATION ===
Total experiments: 14
Already completed: 14
Remaining: 0

SKIP EXP_001 — already evaluated

SKIP EXP_002 — already evaluated

SKIP EXP_003 — already evaluated

SKIP EXP_004 — already evaluated

SKIP EXP_005 — already evaluated

SKIP EXP_006 — already evaluated

SKIP EXP_007 — already evaluated

SKIP EXP_008 — already evaluated

SKIP EXP_009 — already evaluated

SKIP EXP_010 — already evaluated

SKIP EXP_011 — already evaluated

SKIP EXP_012 — already evaluated

SKIP EXP_013 — already evaluated

SKIP EXP_014 — already evaluated

=== FINAL TRACKER RANKING — VAL ===
experiment kind  changed  high  low  new  buffer  match   HOTA   IDF1   MOTA   IDSW  tracker_fps
   EXP_013  bot   buffer   0.6  0.2  0.7      30    0.7 61.944 76.862 83.206  756.0       47.305
   EXP_009  bot     high   0.4  0.2  0.7      15    0.7 61.795 76.522 83.329  822.0       48.448
   EXP_011  bot      low   0.6  0.1  0.7      15    0.7 61.726 75.912 83.042  800.0       46.775
   EXP

In [ ]:
# CELL 35 — BOT-SORT REFINEMENT (VALIDATION ONLY)

from pathlib import Path
import pandas as pd
import time
import gc

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "tracking" / "optimization_v2"

OUT.mkdir(parents=True, exist_ok=True)

RESULTS_FILE = OUT / "botsort_refinement_VAL.csv"
RANKING_FILE = OUT / "botsort_combined_ranking_VAL.csv"

# --------------------------------------------------
# 1. Verify tracking helpers
# --------------------------------------------------

for name in ["run_tracker", "write_tracker", "run_trackeval"]:
    assert callable(globals().get(name)), (
        f"Missing {name}. Run the original CELL 9 helpers."
    )

assert globals().get("DET_NAME") == "yolov8m", (
    "Expected YOLOv8m as the detector."
)

# --------------------------------------------------
# 2. Targeted refinement configurations
# --------------------------------------------------

configs = [
    ("REF_001", 0.40, 30),
    ("REF_002", 0.50, 30),
    ("REF_003", 0.60, 45),
    ("REF_004", 0.40, 45),
    ("REF_005", 0.50, 45),
]

# Other hyperparameters remain fixed
BASE = {
    "low": 0.20,
    "new": 0.70,
    "match": 0.70
}

# --------------------------------------------------
# 3. Resume completed experiments
# --------------------------------------------------

if RESULTS_FILE.exists():
    completed = pd.read_csv(RESULTS_FILE)
else:
    completed = pd.DataFrame()

done = (
    set(completed["experiment"].astype(str))
    if "experiment" in completed.columns
    else set()
)

print("=== BOT-SORT REFINEMENT ===")
print("Experiments:", len(configs))
print("Already completed:", len(done))

# --------------------------------------------------
# 4. Run tracking and TrackEval
# --------------------------------------------------

for exp_id, high, buffer in configs:

    if exp_id in done:
        print(f"\nSKIPPED: {exp_id}")
        continue

    cfg = {
        "high": high,
        "low": BASE["low"],
        "new": BASE["new"],
        "buffer": buffer,
        "match": BASE["match"]
    }

    tracker_name = f"OPT_V2_{exp_id}"

    print(f"\n=== {exp_id} ===")
    print("Configuration:", cfg)

    start = time.perf_counter()

    try:
        # Run BoT-SORT using saved detections
        rows, tracker_ms = run_tracker(
            "bot", cfg, "val"
        )

        # Write predictions to a unique experiment name
        write_tracker(
            tracker_name, "val", rows
        )

        # Official TrackEval metrics on our VAL split
        scores = run_trackeval(
            "val", [tracker_name]
        )

        result = scores.loc[tracker_name].to_dict()

        result.update({
            "experiment": exp_id,
            "kind": "bot",
            **cfg,
            "tracker_ms": tracker_ms,
            "tracker_fps": 1000 / tracker_ms,
            "elapsed_s": time.perf_counter() - start
        })

        completed = pd.concat(
            [completed, pd.DataFrame([result])],
            ignore_index=True
        )

        # Save immediately so Colab interruptions
        # do not lose successful experiments
        temp = RESULTS_FILE.with_name(
            RESULTS_FILE.stem + ".tmp.csv"
        )

        completed.to_csv(temp, index=False)
        temp.replace(RESULTS_FILE)

        done.add(exp_id)

        print(
            f"PASS | HOTA={result['HOTA']:.3f}"
            f" | IDF1={result['IDF1']:.3f}"
            f" | IDSW={int(result['IDSW'])}"
            f" | FPS={result['tracker_fps']:.2f}"
        )

    except Exception as e:
        print(f"FAILED: {exp_id}")
        print(type(e).__name__, str(e))
        raise

    finally:
        gc.collect()

# --------------------------------------------------
# 5. Compare against previous 14 experiments
# --------------------------------------------------

original = pd.read_csv(
    OUT / "tracker_tuning_results_VAL.csv"
)

refined = pd.read_csv(RESULTS_FILE)

combined = pd.concat(
    [original, refined],
    ignore_index=True
)

combined = combined.sort_values(
    ["HOTA", "IDF1", "IDSW"],
    ascending=[False, False, True]
).reset_index(drop=True)

combined.to_csv(RANKING_FILE, index=False)

print("\n=== TOP 10 TRACKER CONFIGURATIONS ===")

columns = [
    "experiment", "kind",
    "high", "low", "new", "buffer", "match",
    "HOTA", "IDF1", "MOTA", "IDSW", "tracker_fps"
]

print(
    combined[columns]
    .head(10)
    .round(3)
    .to_string(index=False)
)

winner = combined.iloc[0]

print("\n=== CURRENT VALIDATION WINNER ===")
print("Experiment:", winner["experiment"])
print("Tracker:", winner["kind"])
print("HOTA:", round(winner["HOTA"], 3))
print("IDF1:", round(winner["IDF1"], 3))
print("IDSW:", int(winner["IDSW"]))

print("\nSaved:", RESULTS_FILE)
print("Saved:", RANKING_FILE)

print("\nNo YOLO retraining or inference.")
print("No TEST data used.")
print("Original best_tracker.json unchanged.")

=== BOT-SORT REFINEMENT ===
Experiments: 5
Already completed: 5

SKIPPED: REF_001

SKIPPED: REF_002

SKIPPED: REF_003

SKIPPED: REF_004

SKIPPED: REF_005

=== TOP 10 TRACKER CONFIGURATIONS ===
experiment kind  high  low  new  buffer  match   HOTA   IDF1   MOTA  IDSW  tracker_fps
   REF_001  bot   0.4  0.2  0.7      30    0.7 62.171 77.574 83.290 836.0       46.714
   REF_004  bot   0.4  0.2  0.7      45    0.7 62.064 77.465 83.218 863.0       46.143
   REF_003  bot   0.6  0.2  0.7      45    0.7 62.005 77.137 83.180 765.0       47.075
   EXP_013  bot   0.6  0.2  0.7      30    0.7 61.944 76.862 83.206 756.0       47.305
   REF_002  bot   0.5  0.2  0.7      30    0.7 61.846 77.113 83.207 819.0       47.132
   REF_005  bot   0.5  0.2  0.7      45    0.7 61.833 77.205 83.229 843.0       45.962
   EXP_009  bot   0.4  0.2  0.7      15    0.7 61.795 76.522 83.329 822.0       48.448
   EXP_011  bot   0.6  0.1  0.7      15    0.7 61.726 75.912 83.042 800.0       46.775
   EXP_010  bot   0.5  0

In [ ]:
# CELL 36 — FINAL REF_001 TEST EVALUATION

from pathlib import Path
import pandas as pd
import json
import time

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "tracking" / "optimization_v2"
OUT.mkdir(parents=True, exist_ok=True)

RESULT_FILE = OUT / "REF_001_TEST_evaluation.csv"
CONFIG_FILE = OUT / "REF_001_frozen_config.json"
BASELINE_FILE = ROOT / "tracking" / "tracking_TEST_trackeval.csv"

# 1. Verify original tracking helpers
for name in ["run_tracker", "write_tracker", "run_trackeval"]:
    assert callable(globals().get(name)), (
        f"Missing {name}. Run original CELL 9 first."
    )

assert globals().get("DET_NAME") == "yolov8m", (
    "Expected YOLOv8m detections."
)

# 2. Load the Validation winner — do not tune on TEST
ranking = pd.read_csv(
    OUT / "botsort_combined_ranking_VAL.csv"
)

winner = ranking.iloc[0]

assert winner["experiment"] == "REF_001"
assert winner["kind"] == "bot"

cfg = {
    "high": float(winner["high"]),
    "low": float(winner["low"]),
    "new": float(winner["new"]),
    "buffer": int(winner["buffer"]),
    "match": float(winner["match"])
}

assert cfg == {
    "high": 0.4,
    "low": 0.2,
    "new": 0.7,
    "buffer": 30,
    "match": 0.7
}, "Unexpected configuration — stop."

tracker_name = "OPT_V2_REF_001_TEST"

# 3. Freeze configuration
if CONFIG_FILE.exists():
    with open(CONFIG_FILE) as f:
        assert json.load(f) == cfg, (
            "Frozen configuration changed."
        )
else:
    with open(CONFIG_FILE, "w") as f:
        json.dump(cfg, f, indent=2)

print("=== FROZEN TEST CONFIGURATION ===")
print("Detector: YOLOv8m")
print("Tracker: BoT-SORT")
print("Configuration:", cfg)
print("TEST frames: 1658–3315")

# 4. Reuse completed evaluation if available
if RESULT_FILE.exists():
    print("\nExisting TEST evaluation found — reusing it.")
    result = pd.read_csv(RESULT_FILE)

else:
    print("\nRunning tracker on cached TEST detections...")

    start = time.perf_counter()

    rows, tracker_ms = run_tracker(
        "bot", cfg, "test"
    )

    write_tracker(
        tracker_name, "test", rows
    )

    print("Running TrackEval...")

    scores = run_trackeval(
        "test", [tracker_name]
    )

    result = scores.reset_index()

    result["experiment"] = "REF_001"
    result["kind"] = "bot"
    result["tracker_ms"] = tracker_ms
    result["tracker_fps"] = 1000 / tracker_ms
    result["elapsed_s"] = time.perf_counter() - start

    temp = RESULT_FILE.with_name(
        RESULT_FILE.stem + ".tmp.csv"
    )
    result.to_csv(temp, index=False)
    temp.replace(RESULT_FILE)

    print("Saved:", RESULT_FILE)

# 5. Compare against original TEST baseline
assert BASELINE_FILE.is_file(), (
    f"Missing original TEST results: {BASELINE_FILE}"
)

baseline = pd.read_csv(BASELINE_FILE)

if "tracker" not in baseline.columns:
    baseline = pd.read_csv(BASELINE_FILE, index_col=0)
    baseline.index.name = "tracker"
    baseline = baseline.reset_index()

original = baseline[
    baseline["tracker"] == "BoTSORT_candidate"
]

assert len(original) == 1, (
    "Original BoT-SORT TEST baseline not found."
)

metrics = [
    "HOTA", "DetA", "AssA", "MOTA",
    "IDF1", "IDSW", "FP", "FN"
]

old = original.iloc[0]
new = result.iloc[0]

comparison = pd.DataFrame({
    "Metric": metrics,
    "Original BoT-SORT": [
        float(old[m]) for m in metrics
    ],
    "REF_001": [
        float(new[m]) for m in metrics
    ]
})

comparison["Change"] = (
    comparison["REF_001"]
    - comparison["Original BoT-SORT"]
)

print("\n=== FINAL TEST COMPARISON ===")
print(comparison.round(3).to_string(index=False))

print("\n=== TEST TRACKER PERFORMANCE ===")
print(f"Tracker latency: {float(new['tracker_ms']):.3f} ms/frame")
print(f"Tracker-only FPS: {float(new['tracker_fps']):.2f}")

comparison_file = OUT / "REF_001_vs_original_TEST.csv"
comparison.to_csv(comparison_file, index=False)

print("\nSaved:", comparison_file)
print("Original tracking results unchanged.")
print("No YOLO training or inference performed.")
print("No TEST-based tuning performed.")

=== FROZEN TEST CONFIGURATION ===
Detector: YOLOv8m
Tracker: BoT-SORT
Configuration: {'high': 0.4, 'low': 0.2, 'new': 0.7, 'buffer': 30, 'match': 0.7}
TEST frames: 1658–3315

Existing TEST evaluation found — reusing it.

=== FINAL TEST COMPARISON ===
Metric  Original BoT-SORT   REF_001    Change
  HOTA             62.479    63.053     0.574
  DetA             68.149    68.114    -0.035
  AssA             57.443    58.529     1.086
  MOTA             84.723    84.796     0.073
  IDF1             76.338    77.588     1.250
  IDSW            792.000   894.000   102.000
    FP          14282.000 17202.000  2920.000
    FN          33703.000 30447.000 -3256.000

=== TEST TRACKER PERFORMANCE ===
Tracker latency: 21.102 ms/frame
Tracker-only FPS: 47.39

Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/tracking/optimization_v2/REF_001_vs_original_TEST.csv
Original tracking results unchanged.
No YOLO training or inference performed.
No TEST-based tuning performed.


In [ ]:
# CELL 37 — TRAJECTORY QUALITY ANALYSIS (TEST) — FIXED

from pathlib import Path
import numpy as np
import pandas as pd
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "tracking" / "optimization_v2"
REPORT = OUT / "trajectory_quality"
REPORT.mkdir(parents=True, exist_ok=True)

TRACKER = "OPT_V2_REF_001_TEST"
SEQ = "MOT20-05"

# --------------------------------------------------
# 1. Load saved TEST tracking predictions
# --------------------------------------------------

track_file = (
    ROOT
    / "tracking"
    / "trackeval"
    / "trackers"
    / "test"
    / TRACKER
    / "data"
    / f"{SEQ}-test.txt"
)

assert track_file.is_file(), f"Missing: {track_file}"

print("Tracking file:", track_file)

# MOT format: frame, id, x, y, w, h, ...
df = pd.read_csv(track_file, header=None)

assert df.shape[1] >= 6, "Invalid MOT tracking file."

df = df.iloc[:, :6].copy()
df.columns = ["frame", "id", "x", "y", "w", "h"]

for c in df.columns:
    df[c] = pd.to_numeric(df[c], errors="raise")

assert not df.isna().any().any(), "Missing tracking values."
assert (df[["w", "h"]] > 0).all().all(), "Invalid bounding boxes."

df["frame"] = df["frame"].astype(int)
df["id"] = df["id"].astype(int)

assert not df.duplicated(["frame", "id"]).any(), (
    "Duplicate frame/ID combinations."
)

# --------------------------------------------------
# 2. Calculate center points and movement
# --------------------------------------------------

df["cx"] = df["x"] + df["w"] / 2
df["cy"] = df["y"] + df["h"] / 2

df = df.sort_values(["id", "frame"]).reset_index(drop=True)

g = df.groupby("id")

df["previous_frame"] = g["frame"].shift()
df["previous_cx"] = g["cx"].shift()
df["previous_cy"] = g["cy"].shift()

df["frame_gap"] = (
    df["frame"] - df["previous_frame"]
)

df["distance_px"] = np.hypot(
    df["cx"] - df["previous_cx"],
    df["cy"] - df["previous_cy"]
)

df["speed_px_frame"] = (
    df["distance_px"] / df["frame_gap"]
)

# Only consecutive frames are used to detect jumps
consecutive = df["frame_gap"] == 1

# Diagnostic threshold (not an official error metric)
JUMP_THRESHOLD = 50.0

df["large_jump"] = (
    consecutive &
    (df["distance_px"] > JUMP_THRESHOLD)
)

# --------------------------------------------------
# 3. Per-track statistics
# --------------------------------------------------

tracks = df.groupby("id").agg(
    first_frame=("frame", "min"),
    last_frame=("frame", "max"),
    observations=("frame", "count"),
    mean_speed_px_frame=("speed_px_frame", "mean"),
    max_speed_px_frame=("speed_px_frame", "max"),
    large_jumps=("large_jump", "sum")
).reset_index()

tracks["span_frames"] = (
    tracks["last_frame"] - tracks["first_frame"] + 1
)

tracks["missing_frames"] = (
    tracks["span_frames"] - tracks["observations"]
)

tracks["coverage"] = (
    tracks["observations"] / tracks["span_frames"]
)

tracks["short_track"] = (
    tracks["observations"] < 15
)

# --------------------------------------------------
# 4. Trajectory quality summary
# --------------------------------------------------

gaps = (
    df.loc[df["frame_gap"] > 1, "frame_gap"] - 1
)

summary = {
    "tracker": TRACKER,
    "sequence": SEQ,

    "total_detections": int(len(df)),

    "unique_track_ids": int(
        tracks["id"].nunique()
    ),

    "mean_track_observations": float(
        tracks["observations"].mean()
    ),

    "median_track_observations": float(
        tracks["observations"].median()
    ),

    "short_tracks_under_15": int(
        tracks["short_track"].sum()
    ),

    "short_track_percentage": float(
        100 * tracks["short_track"].mean()
    ),

    "tracks_with_missing_frames": int(
        (tracks["missing_frames"] > 0).sum()
    ),

    "total_missing_frames_inside_tracks": int(
        tracks["missing_frames"].sum()
    ),

    "gap_events": int(len(gaps)),

    "large_jumps_over_50px": int(
        df["large_jump"].sum()
    ),

    "mean_track_coverage": float(
        tracks["coverage"].mean()
    )
}

# --------------------------------------------------
# 5. Save reports
# --------------------------------------------------

tracks.to_csv(
    REPORT / "REF_001_per_track_TEST.csv",
    index=False
)

df.to_csv(
    REPORT / "REF_001_trajectory_observations_TEST.csv",
    index=False
)

with open(
    REPORT / "REF_001_trajectory_summary_TEST.json", "w"
) as f:
    json.dump(summary, f, indent=2)

# --------------------------------------------------
# 6. Display results
# --------------------------------------------------

print("\n=== TRAJECTORY QUALITY — REF_001 TEST ===")

for k, v in summary.items():
    print(f"{k}: {v}")

print("\n=== TOP 10 TRACKS WITH MOST MISSING FRAMES ===")

print(
    tracks.sort_values(
        "missing_frames",
        ascending=False
    )[
        [
            "id",
            "observations",
            "span_frames",
            "missing_frames",
            "coverage",
            "large_jumps"
        ]
    ]
    .head(10)
    .round(3)
    .to_string(index=False)
)

print("\nSaved reports:", REPORT)
print("No YOLO inference or retraining.")
print("No tracking rerun.")

Tracking file: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/tracking/trackeval/trackers/test/OPT_V2_REF_001_TEST/data/MOT20-05-test.txt

=== TRAJECTORY QUALITY — REF_001 TEST ===
tracker: OPT_V2_REF_001_TEST
sequence: MOT20-05
total_detections: 340635
unique_track_ids: 1225
mean_track_observations: 278.069387755102
median_track_observations: 159.0
short_tracks_under_15: 208
short_track_percentage: 16.979591836734695
tracks_with_missing_frames: 702
total_missing_frames_inside_tracks: 11498
gap_events: 2168
large_jumps_over_50px: 0
mean_track_coverage: 0.9293173326425446

=== TOP 10 TRACKS WITH MOST MISSING FRAMES ===
  id  observations  span_frames  missing_frames  coverage  large_jumps
 202          1526         1655             129     0.922            0
 660           655          779             124     0.841            0
1108           264          383             119     0.689            0
 407           216          317             101     0.681            0
 804           

In [ ]:
# CELL 38 — ORIGINAL VS OPTIMIZED TRAJECTORY COMPARISON (TEST)

from pathlib import Path
import numpy as np
import pandas as pd
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")

TRACKERS_DIR = (
    ROOT / "tracking" / "trackeval" / "trackers" / "test"
)

OUT = ROOT / "tracking" / "optimization_v2" / "trajectory_quality"
OUT.mkdir(parents=True, exist_ok=True)

SEQ_FILE = "MOT20-05-test.txt"
OPTIMIZED = "OPT_V2_REF_001_TEST"

# --------------------------------------------------
# 1. Find existing TEST predictions
# --------------------------------------------------

assert TRACKERS_DIR.is_dir(), f"Missing: {TRACKERS_DIR}"

available = {
    p.parent.parent.name: p
    for p in TRACKERS_DIR.glob(f"*/data/{SEQ_FILE}")
}

print("=== AVAILABLE TEST TRACKERS ===")

for name in sorted(available):
    print(name)

assert OPTIMIZED in available, (
    f"Missing optimized predictions: {OPTIMIZED}"
)

# Prefer the original baseline by its known name.
baseline_candidates = [
    "BoTSORT_candidate",
    "BoTSORT_candidate_TEST",
]

BASELINE = next(
    (name for name in baseline_candidates if name in available),
    None
)

if BASELINE is None:
    print("\nOriginal BoT-SORT predictions not found.")
    print("No tracking will be rerun.")
    print("Available tracker names are listed above.")
    print("Select the correct original baseline name if present.")

else:
    print("\nOriginal baseline:", BASELINE)
    print("Optimized tracker:", OPTIMIZED)

# --------------------------------------------------
# 2. Trajectory analysis function
# --------------------------------------------------

def analyze_trajectories(path, tracker_name):

    df = pd.read_csv(path, header=None)

    assert df.shape[1] >= 6, f"Invalid MOT file: {path}"

    df = df.iloc[:, :6].copy()
    df.columns = ["frame", "id", "x", "y", "w", "h"]

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="raise")

    assert not df.isna().any().any(), (
        f"Missing values in {tracker_name}"
    )

    assert (df[["w", "h"]] > 0).all().all(), (
        f"Invalid bounding boxes in {tracker_name}"
    )

    df["frame"] = df["frame"].astype(int)
    df["id"] = df["id"].astype(int)

    assert not df.duplicated(["frame", "id"]).any(), (
        f"Duplicate frame/ID in {tracker_name}"
    )

    df["cx"] = df["x"] + df["w"] / 2
    df["cy"] = df["y"] + df["h"] / 2

    df = df.sort_values(
        ["id", "frame"]
    ).reset_index(drop=True)

    g = df.groupby("id")

    df["previous_frame"] = g["frame"].shift()
    df["previous_cx"] = g["cx"].shift()
    df["previous_cy"] = g["cy"].shift()

    df["frame_gap"] = (
        df["frame"] - df["previous_frame"]
    )

    df["distance_px"] = np.hypot(
        df["cx"] - df["previous_cx"],
        df["cy"] - df["previous_cy"]
    )

    df["speed_px_frame"] = (
        df["distance_px"] / df["frame_gap"]
    )

    df["large_jump"] = (
        (df["frame_gap"] == 1)
        & (df["distance_px"] > 50)
    )

    tracks = df.groupby("id").agg(
        first_frame=("frame", "min"),
        last_frame=("frame", "max"),
        observations=("frame", "count"),
        large_jumps=("large_jump", "sum")
    ).reset_index()

    tracks["span_frames"] = (
        tracks["last_frame"] - tracks["first_frame"] + 1
    )

    tracks["missing_frames"] = (
        tracks["span_frames"] - tracks["observations"]
    )

    tracks["coverage"] = (
        tracks["observations"] / tracks["span_frames"]
    )

    tracks["short_track"] = (
        tracks["observations"] < 15
    )

    gaps = df.loc[df["frame_gap"] > 1, "frame_gap"] - 1

    summary = {
        "tracker": tracker_name,
        "total_detections": int(len(df)),
        "unique_track_ids": int(tracks["id"].nunique()),
        "mean_track_observations": float(
            tracks["observations"].mean()
        ),
        "median_track_observations": float(
            tracks["observations"].median()
        ),
        "short_tracks_under_15": int(
            tracks["short_track"].sum()
        ),
        "short_track_percentage": float(
            100 * tracks["short_track"].mean()
        ),
        "tracks_with_missing_frames": int(
            (tracks["missing_frames"] > 0).sum()
        ),
        "total_missing_frames_inside_tracks": int(
            tracks["missing_frames"].sum()
        ),
        "gap_events": int(len(gaps)),
        "large_jumps_over_50px": int(
            df["large_jump"].sum()
        ),
        "mean_track_coverage": float(
            tracks["coverage"].mean()
        )
    }

    return summary, tracks

# --------------------------------------------------
# 3. Compare only if original predictions exist
# --------------------------------------------------

if BASELINE is not None:

    results = []

    for tracker in [BASELINE, OPTIMIZED]:

        summary, tracks = analyze_trajectories(
            available[tracker], tracker
        )

        results.append(summary)

        tracks.to_csv(
            OUT / f"{tracker}_per_track_TEST.csv",
            index=False
        )

    comparison = pd.DataFrame(results).set_index("tracker")

    print("\n=== TRAJECTORY QUALITY COMPARISON ===")
    print(comparison.T.round(3).to_string())

    # Changes relative to the original baseline
    delta = (
        comparison.loc[OPTIMIZED].select_dtypes(include="number")
        if False else
        comparison.loc[OPTIMIZED] - comparison.loc[BASELINE]
    )

    print("\n=== REF_001 MINUS ORIGINAL ===")
    print(delta.round(3).to_string())

    comparison.to_csv(
        OUT / "original_vs_REF_001_trajectory_TEST.csv"
    )

    delta.to_csv(
        OUT / "REF_001_trajectory_changes_TEST.csv",
        header=["change"]
    )

    print("\nSaved reports:", OUT)

else:
    print("\nComparison skipped: original TEST predictions unavailable.")

print("\nNo YOLO inference or retraining.")
print("No tracking rerun.")

=== AVAILABLE TEST TRACKERS ===
BoTSORT_candidate
ByteTrack_candidate
ByteTrack_default
OPT_V2_REF_001_TEST

Original baseline: BoTSORT_candidate
Optimized tracker: OPT_V2_REF_001_TEST

=== TRAJECTORY QUALITY COMPARISON ===
tracker                             BoTSORT_candidate  OPT_V2_REF_001_TEST
total_detections                           333369.000           340635.000
unique_track_ids                             1294.000             1225.000
mean_track_observations                       257.627              278.069
median_track_observations                     143.500              159.000
short_tracks_under_15                         218.000              208.000
short_track_percentage                         16.847               16.980
tracks_with_missing_frames                    528.000              702.000
total_missing_frames_inside_tracks           5960.000            11498.000
gap_events                                   1086.000             2168.000
large_jumps_over_50px     

In [ ]:
# CELL 39 — TRAJECTORY GAP ANALYSIS (TEST)

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")

TRACKERS_DIR = (
    ROOT / "tracking" / "trackeval" / "trackers" / "test"
)

OUT = (
    ROOT / "tracking" / "optimization_v2"
    / "trajectory_quality"
)
OUT.mkdir(parents=True, exist_ok=True)

TRACKERS = {
    "Original BoT-SORT": "BoTSORT_candidate",
    "REF_001": "OPT_V2_REF_001_TEST"
}

SEQ_FILE = "MOT20-05-test.txt"

# --------------------------------------------------
# 1. Extract frame gaps
# --------------------------------------------------

def extract_gaps(path, tracker_name):

    df = pd.read_csv(path, header=None, usecols=[0, 1])
    df.columns = ["frame", "id"]

    df = df.sort_values(["id", "frame"])

    assert not df.duplicated(["id", "frame"]).any(), (
        f"Duplicate observations in {tracker_name}"
    )

    df["previous_frame"] = df.groupby("id")["frame"].shift()

    df["gap_length"] = (
        df["frame"] - df["previous_frame"] - 1
    )

    gaps = df[df["gap_length"] > 0].copy()

    gaps["gap_length"] = gaps["gap_length"].astype(int)

    gaps["category"] = pd.cut(
        gaps["gap_length"],
        bins=[0, 5, 15, np.inf],
        labels=[
            "Short (1-5)",
            "Medium (6-15)",
            "Long (>15)"
        ]
    )

    gaps["tracker"] = tracker_name

    return gaps[
        [
            "tracker",
            "id",
            "previous_frame",
            "frame",
            "gap_length",
            "category"
        ]
    ]

# --------------------------------------------------
# 2. Analyze both trackers
# --------------------------------------------------

all_gaps = []

for label, tracker in TRACKERS.items():

    path = (
        TRACKERS_DIR / tracker
        / "data" / SEQ_FILE
    )

    assert path.is_file(), f"Missing: {path}"

    gaps = extract_gaps(path, label)

    all_gaps.append(gaps)

    print(f"{label}: {len(gaps):,} gap events")

all_gaps = pd.concat(all_gaps, ignore_index=True)

# --------------------------------------------------
# 3. Compare gap categories
# --------------------------------------------------

categories = [
    "Short (1-5)",
    "Medium (6-15)",
    "Long (>15)"
]

summary = []

for label in TRACKERS:

    g = all_gaps[all_gaps["tracker"] == label]

    total_events = len(g)
    total_missing = int(g["gap_length"].sum())

    for category in categories:

        subset = g[g["category"] == category]

        count = len(subset)
        missing = int(subset["gap_length"].sum())

        summary.append({
            "tracker": label,
            "category": category,
            "gap_events": count,
            "event_percentage": (
                100 * count / total_events
                if total_events else 0
            ),
            "missing_frames": missing,
            "missing_percentage": (
                100 * missing / total_missing
                if total_missing else 0
            )
        })

summary = pd.DataFrame(summary)

# --------------------------------------------------
# 4. Display results
# --------------------------------------------------

print("\n=== GAP CATEGORY COMPARISON ===")

print(
    summary.round(2).to_string(index=False)
)

print("\n=== TOTAL GAP STATISTICS ===")

totals = all_gaps.groupby("tracker").agg(
    gap_events=("gap_length", "count"),
    total_missing_frames=("gap_length", "sum"),
    mean_gap_length=("gap_length", "mean"),
    median_gap_length=("gap_length", "median"),
    max_gap_length=("gap_length", "max")
)

print(totals.round(2).to_string())

print("\n=== LONGEST 10 GAPS ===")

print(
    all_gaps.sort_values(
        "gap_length", ascending=False
    ).head(10).to_string(index=False)
)

# --------------------------------------------------
# 5. Save reports
# --------------------------------------------------

all_gaps.to_csv(
    OUT / "trajectory_gap_events_TEST.csv",
    index=False
)

summary.to_csv(
    OUT / "trajectory_gap_categories_TEST.csv",
    index=False
)

totals.to_csv(
    OUT / "trajectory_gap_totals_TEST.csv"
)

print("\nSaved reports:", OUT)
print("No YOLO inference or retraining.")
print("No tracking rerun.")
print("No trajectory interpolation applied.")

Original BoT-SORT: 1,086 gap events
REF_001: 2,168 gap events

=== GAP CATEGORY COMPARISON ===
          tracker      category  gap_events  event_percentage  missing_frames  missing_percentage
Original BoT-SORT   Short (1-5)         645             59.39            1740               29.19
Original BoT-SORT Medium (6-15)         424             39.04            3948               66.24
Original BoT-SORT    Long (>15)          17              1.57             272                4.56
          REF_001   Short (1-5)        1532             70.66            3513               30.55
          REF_001 Medium (6-15)         470             21.68            4266               37.10
          REF_001    Long (>15)         166              7.66            3719               32.34

=== TOTAL GAP STATISTICS ===
                   gap_events  total_missing_frames  mean_gap_length  median_gap_length  max_gap_length
tracker                                                                              

In [ ]:
# CELL 40 — SAFE TRAJECTORY INTERPOLATION (TEST)

from pathlib import Path
import numpy as np
import pandas as pd
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")

TRACK_FILE = (
    ROOT / "tracking" / "trackeval" / "trackers"
    / "test" / "OPT_V2_REF_001_TEST"
    / "data" / "MOT20-05-test.txt"
)

OUT = (
    ROOT / "tracking" / "optimization_v2"
    / "trajectory_quality"
)

OUT.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUT / "REF_001_processed_trajectories_TEST.parquet"
REPORT_FILE = OUT / "REF_001_interpolation_report_TEST.json"

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

MAX_GAP = 3
MAX_SPEED_PX_FRAME = 20.0

# These are conservative diagnostic settings.
# They are not calibrated real-world speed limits.

assert TRACK_FILE.is_file(), f"Missing: {TRACK_FILE}"

# --------------------------------------------------
# 2. Load raw tracker predictions
# --------------------------------------------------

raw = pd.read_csv(TRACK_FILE, header=None)

assert raw.shape[1] >= 6, "Invalid MOT tracking file."

raw = raw.iloc[:, :6].copy()
raw.columns = ["frame", "id", "x", "y", "w", "h"]

for col in raw.columns:
    raw[col] = pd.to_numeric(raw[col], errors="raise")

assert not raw.isna().any().any()
assert (raw[["w", "h"]] > 0).all().all()

raw["frame"] = raw["frame"].astype(int)
raw["id"] = raw["id"].astype(int)

assert not raw.duplicated(["frame", "id"]).any()

raw["cx"] = raw["x"] + raw["w"] / 2
raw["cy"] = raw["y"] + raw["h"] / 2

raw["is_interpolated"] = False

raw = raw.sort_values(
    ["id", "frame"]
).reset_index(drop=True)

print("=== ORIGINAL TRAJECTORIES ===")
print("Observations:", len(raw))
print("Unique IDs:", raw["id"].nunique())

# --------------------------------------------------
# 3. Identify eligible gaps
# --------------------------------------------------

generated = []

eligible_gaps = 0
interpolated_gaps = 0
rejected_speed = 0
inserted_points = 0

for track_id, track in raw.groupby("id", sort=False):

    track = track.sort_values("frame")

    values = track[
        ["frame", "x", "y", "w", "h", "cx", "cy"]
    ].to_numpy(dtype=float)

    for i in range(1, len(values)):

        previous = values[i - 1]
        current = values[i]

        f0 = int(previous[0])
        f1 = int(current[0])

        missing = f1 - f0 - 1

        if not (1 <= missing <= MAX_GAP):
            continue

        eligible_gaps += 1

        distance = np.hypot(
            current[5] - previous[5],
            current[6] - previous[6]
        )

        speed = distance / (f1 - f0)

        if not np.isfinite(speed) or speed > MAX_SPEED_PX_FRAME:
            rejected_speed += 1
            continue

        interpolated_gaps += 1

        # Linear interpolation between observed boxes
        for step in range(1, missing + 1):

            alpha = step / (missing + 1)

            box = (
                previous[1:5]
                + alpha * (current[1:5] - previous[1:5])
            )

            x, y, w, h = box

            generated.append({
                "frame": f0 + step,
                "id": int(track_id),
                "x": float(x),
                "y": float(y),
                "w": float(w),
                "h": float(h),
                "cx": float(x + w / 2),
                "cy": float(y + h / 2),
                "is_interpolated": True
            })

            inserted_points += 1

# --------------------------------------------------
# 4. Build processed trajectories
# --------------------------------------------------

if generated:
    interpolated = pd.DataFrame(generated)

    processed = pd.concat(
        [raw, interpolated],
        ignore_index=True
    )
else:
    processed = raw.copy()

processed = processed.sort_values(
    ["frame", "id"]
).reset_index(drop=True)

# --------------------------------------------------
# 5. Validate processed data
# --------------------------------------------------

assert not processed.duplicated(
    ["frame", "id"]
).any(), "Duplicate frame/ID after interpolation."

assert (
    processed["frame"].between(1, 1658)
).all(), "Unexpected TEST frame."

assert (processed[["w", "h"]] > 0).all().all()

assert (
    len(processed) == len(raw) + inserted_points
)

assert (
    int(processed["is_interpolated"].sum())
    == inserted_points
)

# Verify every raw observation is unchanged
original_check = (
    processed.loc[
        ~processed["is_interpolated"],
        raw.columns
    ]
    .sort_values(["id", "frame"])
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    raw.reset_index(drop=True),
    original_check,
    check_dtype=False
)

# --------------------------------------------------
# 6. Save processed trajectories
# --------------------------------------------------

processed.to_parquet(
    OUTPUT_FILE,
    index=False
)

report = {
    "tracker": "OPT_V2_REF_001_TEST",
    "sequence": "MOT20-05",
    "split": "TEST",
    "max_gap_frames": MAX_GAP,
    "max_speed_px_per_frame": MAX_SPEED_PX_FRAME,
    "original_observations": int(len(raw)),
    "eligible_short_gaps": int(eligible_gaps),
    "interpolated_gaps": int(interpolated_gaps),
    "rejected_by_speed": int(rejected_speed),
    "inserted_points": int(inserted_points),
    "processed_observations": int(len(processed)),
    "original_observations_preserved": True
}

with open(REPORT_FILE, "w") as f:
    json.dump(report, f, indent=2)

# --------------------------------------------------
# 7. Display results
# --------------------------------------------------

print("\n=== SAFE INTERPOLATION RESULTS ===")

for key, value in report.items():
    print(f"{key}: {value}")

print("\n=== POINT TYPES ===")

print(
    processed["is_interpolated"]
    .value_counts()
    .rename(index={
        False: "Original",
        True: "Interpolated"
    })
    .to_string()
)

print("\nSaved:", OUTPUT_FILE)
print("Saved:", REPORT_FILE)

print("\nRaw tracking predictions unchanged.")
print("No YOLO inference or retraining.")
print("No tracking rerun.")
print("No TrackEval metrics modified.")

=== ORIGINAL TRAJECTORIES ===
Observations: 340635
Unique IDs: 1225

=== SAFE INTERPOLATION RESULTS ===
tracker: OPT_V2_REF_001_TEST
sequence: MOT20-05
split: TEST
max_gap_frames: 3
max_speed_px_per_frame: 20.0
original_observations: 340635
eligible_short_gaps: 1205
interpolated_gaps: 1203
rejected_by_speed: 2
inserted_points: 2069
processed_observations: 342704
original_observations_preserved: True

=== POINT TYPES ===
is_interpolated
Original        340635
Interpolated      2069

Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/tracking/optimization_v2/trajectory_quality/REF_001_processed_trajectories_TEST.parquet
Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/tracking/optimization_v2/trajectory_quality/REF_001_interpolation_report_TEST.json

Raw tracking predictions unchanged.
No YOLO inference or retraining.
No tracking rerun.
No TrackEval metrics modified.


In [ ]:
# CELL 41 — PROCESSED TRAJECTORY VALIDATION (TEST)

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")

OUT = (
    ROOT / "tracking" / "optimization_v2"
    / "trajectory_quality"
)

RAW_FILE = (
    ROOT / "tracking" / "trackeval" / "trackers"
    / "test" / "OPT_V2_REF_001_TEST"
    / "data" / "MOT20-05-test.txt"
)

PROCESSED_FILE = (
    OUT / "REF_001_processed_trajectories_TEST.parquet"
)

REPORT_FILE = OUT / "REF_001_validation_report_TEST.json"

MAX_GAP = 3
MAX_SPEED = 20.0
TOL = 1e-6

# --------------------------------------------------
# 1. Load raw and processed trajectories
# --------------------------------------------------

assert RAW_FILE.is_file(), f"Missing: {RAW_FILE}"
assert PROCESSED_FILE.is_file(), f"Missing: {PROCESSED_FILE}"

raw = pd.read_csv(RAW_FILE, header=None).iloc[:, :6]
raw.columns = ["frame", "id", "x", "y", "w", "h"]

processed = pd.read_parquet(PROCESSED_FILE)

cols = ["frame", "id", "x", "y", "w", "h"]

for col in cols:
    raw[col] = pd.to_numeric(raw[col], errors="raise")
    processed[col] = pd.to_numeric(
        processed[col], errors="raise"
    )

raw["frame"] = raw["frame"].astype(int)
raw["id"] = raw["id"].astype(int)

processed["frame"] = processed["frame"].astype(int)
processed["id"] = processed["id"].astype(int)

assert processed["is_interpolated"].dtype == bool

original = processed[
    ~processed["is_interpolated"]
].copy()

inserted = processed[
    processed["is_interpolated"]
].copy()

print("=== DATA LOADED ===")
print("Raw observations:", len(raw))
print("Processed observations:", len(processed))
print("Interpolated points:", len(inserted))

# --------------------------------------------------
# 2. Verify original observations are unchanged
# --------------------------------------------------

raw_sorted = raw[cols].sort_values(
    ["id", "frame"]
).reset_index(drop=True)

original_sorted = original[cols].sort_values(
    ["id", "frame"]
).reset_index(drop=True)

pd.testing.assert_frame_equal(
    raw_sorted,
    original_sorted,
    check_dtype=False,
    atol=TOL,
    rtol=0
)

print("\nPASS: All original observations preserved.")

# --------------------------------------------------
# 3. Structural validation
# --------------------------------------------------

assert not processed.duplicated(
    ["frame", "id"]
).any(), "Duplicate frame/ID."

assert processed["frame"].between(1, 1658).all()
assert (processed[["w", "h"]] > 0).all().all()

assert np.isfinite(
    processed[["x", "y", "w", "h", "cx", "cy"]]
    .to_numpy(dtype=float)
).all()

assert np.allclose(
    processed["cx"],
    processed["x"] + processed["w"] / 2,
    atol=TOL,
    rtol=0
)

assert np.allclose(
    processed["cy"],
    processed["y"] + processed["h"] / 2,
    atol=TOL,
    rtol=0
)

print("PASS: Structure, boxes, and center points valid.")

# --------------------------------------------------
# 4. Validate every interpolated point
# --------------------------------------------------

# Find neighboring ORIGINAL observations
raw_by_id = {
    tid: group.sort_values("frame").set_index("frame")
    for tid, group in raw.groupby("id")
}

validated_points = 0
validated_gaps = set()

for row in inserted.itertuples(index=False):

    tid = int(row.id)
    frame = int(row.frame)

    assert tid in raw_by_id

    track = raw_by_id[tid]
    frames = track.index.to_numpy()

    pos = np.searchsorted(frames, frame)

    assert 0 < pos < len(frames), (
        f"Missing endpoints for ID {tid}, frame {frame}"
    )

    f0 = int(frames[pos - 1])
    f1 = int(frames[pos])

    missing = f1 - f0 - 1

    assert 1 <= missing <= MAX_GAP, (
        f"Invalid gap length: ID {tid}, frame {frame}"
    )

    p0 = track.loc[f0]
    p1 = track.loc[f1]

    alpha = (frame - f0) / (f1 - f0)

    # Check interpolation against original boxes
    expected = (
        p0[["x", "y", "w", "h"]].to_numpy(dtype=float)
        + alpha * (
            p1[["x", "y", "w", "h"]].to_numpy(dtype=float)
            - p0[["x", "y", "w", "h"]].to_numpy(dtype=float)
        )
    )

    actual = np.array(
        [row.x, row.y, row.w, row.h],
        dtype=float
    )

    assert np.allclose(
        actual, expected, atol=TOL, rtol=0
    ), f"Interpolation mismatch: ID {tid}, frame {frame}"

    # Check endpoint movement speed
    cx0 = float(p0["x"] + p0["w"] / 2)
    cy0 = float(p0["y"] + p0["h"] / 2)

    cx1 = float(p1["x"] + p1["w"] / 2)
    cy1 = float(p1["y"] + p1["h"] / 2)

    speed = np.hypot(
        cx1 - cx0, cy1 - cy0
    ) / (f1 - f0)

    assert speed <= MAX_SPEED + TOL, (
        f"Speed limit exceeded: ID {tid}, frame {frame}"
    )

    validated_points += 1
    validated_gaps.add((tid, f0, f1))

print(
    f"PASS: {validated_points:,} interpolated points "
    f"across {len(validated_gaps):,} gaps validated."
)

# --------------------------------------------------
# 5. Verify inserted points form complete gaps
# --------------------------------------------------

inserted_keys = set(
    zip(inserted["id"], inserted["frame"])
)

for tid, f0, f1 in validated_gaps:
    for frame in range(f0 + 1, f1):
        assert (tid, frame) in inserted_keys, (
            f"Incomplete interpolation: ID {tid}, "
            f"frames {f0}-{f1}"
        )

print("PASS: Every interpolated gap is complete.")

# --------------------------------------------------
# 6. Remaining gaps after interpolation
# --------------------------------------------------

processed_sorted = processed.sort_values(
    ["id", "frame"]
).copy()

processed_sorted["previous_frame"] = (
    processed_sorted.groupby("id")["frame"].shift()
)

processed_sorted["missing"] = (
    processed_sorted["frame"]
    - processed_sorted["previous_frame"] - 1
)

remaining = processed_sorted[
    processed_sorted["missing"] > 0
]

remaining_events = int(len(remaining))
remaining_frames = int(remaining["missing"].sum())

print("\n=== REMAINING GAPS ===")
print("Gap events:", remaining_events)
print("Missing frames:", remaining_frames)

# --------------------------------------------------
# 7. Final report
# --------------------------------------------------

report = {
    "tracker": "OPT_V2_REF_001_TEST",
    "sequence": "MOT20-05",
    "split": "TEST",
    "validation_passed": True,
    "original_observations": int(len(raw)),
    "original_observations_preserved": True,
    "processed_observations": int(len(processed)),
    "validated_interpolated_points": validated_points,
    "validated_interpolated_gaps": len(validated_gaps),
    "remaining_gap_events": remaining_events,
    "remaining_missing_frames": remaining_frames,
    "max_gap_interpolated": MAX_GAP,
    "max_speed_px_per_frame": MAX_SPEED
}

with open(REPORT_FILE, "w") as f:
    json.dump(report, f, indent=2)

print("\n=== FINAL VALIDATION ===")
print("PASS: Processed trajectories are structurally valid.")
print("PASS: Original predictions unchanged.")
print("PASS: Interpolation calculations verified.")

print("\nSaved:", REPORT_FILE)
print("No YOLO inference or retraining.")
print("No tracking rerun.")

=== DATA LOADED ===
Raw observations: 340635
Processed observations: 342704
Interpolated points: 2069

PASS: All original observations preserved.
PASS: Structure, boxes, and center points valid.
PASS: 2,069 interpolated points across 1,203 gaps validated.
PASS: Every interpolated gap is complete.

=== REMAINING GAPS ===
Gap events: 965
Missing frames: 9429

=== FINAL VALIDATION ===
PASS: Processed trajectories are structurally valid.
PASS: Original predictions unchanged.
PASS: Interpolation calculations verified.

Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/tracking/optimization_v2/trajectory_quality/REF_001_validation_report_TEST.json
No YOLO inference or retraining.
No tracking rerun.


In [ ]:
# CELL 42 — 12-ZONE CONFIGURATION AUDIT

from pathlib import Path
import json
import re
import pandas as pd

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project")
RUNS = ROOT / "runs_fresh"

assert ROOT.is_dir(), f"Project folder not found: {ROOT}"

# --------------------------------------------------
# 1. Find potential zone configuration files
# --------------------------------------------------

EXTENSIONS = {".json", ".yaml", ".yml", ".csv", ".py", ".ipynb"}

KEYWORDS = (
    "zone", "region", "polygon",
    "occupancy", "crowd_flow", "grid"
)

EXCLUDE_DIRS = {
    ".git", "__pycache__", ".ipynb_checkpoints",
    "weights", "trackeval", "datasets"
}

candidates = []

for path in ROOT.rglob("*"):

    if not path.is_file():
        continue

    if any(part in EXCLUDE_DIRS for part in path.parts):
        continue

    if path.suffix.lower() not in EXTENSIONS:
        continue

    relative = str(path.relative_to(ROOT))
    filename = path.name.lower()

    name_match = any(k in filename for k in KEYWORDS)

    # Inspect small text configuration files only.
    content_match = False

    if not name_match and path.suffix.lower() in {
        ".json", ".yaml", ".yml", ".py"
    }:
        if path.stat().st_size <= 1_000_000:
            try:
                text = path.read_text(
                    encoding="utf-8", errors="replace"
                ).lower()

                content_match = any(
                    re.search(r"\b" + re.escape(k) + r"\b", text)
                    for k in ("zones", "zone_polygons", "zone_bounds")
                )
            except OSError:
                pass

    if name_match or content_match:
        candidates.append({
            "path": relative,
            "size_kb": round(path.stat().st_size / 1024, 1),
            "type": path.suffix.lower(),
            "match": "filename" if name_match else "content"
        })

# --------------------------------------------------
# 2. Display candidate files
# --------------------------------------------------

print("=== ZONE CONFIGURATION FILES ===")
print("Candidates found:", len(candidates))

if candidates:
    files_df = pd.DataFrame(candidates).sort_values(
        ["match", "path"]
    )
    print(files_df.to_string(index=False))
else:
    print("No obvious zone configuration files found.")

# --------------------------------------------------
# 3. Inspect JSON zone configurations
# --------------------------------------------------

print("\n=== JSON ZONE CONFIGURATION PREVIEW ===")

for item in candidates:

    if item["type"] != ".json":
        continue

    path = ROOT / item["path"]

    if path.stat().st_size > 1_000_000:
        continue

    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        continue

    if isinstance(data, dict):
        matching_keys = [
            k for k in data
            if any(word in k.lower() for word in KEYWORDS)
        ]

        if matching_keys:
            print("\nFile:", item["path"])
            print("Matching keys:", matching_keys)

            for key in matching_keys[:5]:
                value = data[key]

                if isinstance(value, (list, dict)):
                    print(
                        f"  {key}: {type(value).__name__}, "
                        f"entries={len(value)}"
                    )
                else:
                    print(f"  {key}: {str(value)[:150]}")

# --------------------------------------------------
# 4. Verify saved TEST trajectories
# --------------------------------------------------

TRAJECTORY_FILE = (
    RUNS / "tracking" / "optimization_v2"
    / "trajectory_quality"
    / "REF_001_processed_trajectories_TEST.parquet"
)

assert TRAJECTORY_FILE.is_file(), (
    f"Missing validated trajectories: {TRAJECTORY_FILE}"
)

trajectories = pd.read_parquet(
    TRAJECTORY_FILE,
    columns=["frame", "id", "cx", "cy", "is_interpolated"]
)

observed = trajectories[
    ~trajectories["is_interpolated"]
]

print("\n=== TRAJECTORY INPUT CHECK ===")
print("Processed observations:", len(trajectories))
print("Original observations:", len(observed))
print("Unique IDs:", observed["id"].nunique())
print("Frames:", observed["frame"].min(), "to", observed["frame"].max())

assert len(observed) == 340635
assert observed["frame"].between(1, 1658).all()

print("\n=== AUDIT COMPLETE ===")
print("No zone definitions modified.")
print("No occupancy or flow calculations performed.")
print("No YOLO inference or tracking rerun.")

=== ZONE CONFIGURATION FILES ===
Candidates found: 43
                                                                           path  size_kb  type    match
                            runs_fresh/zones/figures/zone_accuracy_MOT20-05.csv      1.3  .csv filename
                     runs_fresh/zones/figures/zone_count_error_by_zone_TEST.csv      0.5  .csv filename
                                  runs_fresh/zones/figures/zone_definition.json      4.0 .json filename
                                  runs_fresh/zones/figures/zone_graph_edges.csv      0.4  .csv filename
                   runs_fresh/zones/figures/zone_level_calibration_MOT20-05.csv      0.6  .csv filename
                            runs_fresh/zones/figures/zone_stats_GT_MOT20-05.csv    139.2  .csv filename
                    runs_fresh/zones/figures/zone_stats_GT_visible_MOT20-05.csv    139.1  .csv filename
                               runs_fresh/zones/figures/zone_stats_MOT20-05.csv    139.4  .csv filename
          

In [ ]:
# CELL 43 — VERIFY EXISTING ZONE GEOMETRY

import json
from pathlib import Path
from pprint import pprint

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")

ZONE_FILE = ROOT / "zones" / "zone_definition.json"

assert ZONE_FILE.exists(), "Zone configuration not found."

with open(ZONE_FILE) as f:
    config = json.load(f)

print("=== GRID CONFIGURATION ===")
pprint(config.get("grid"))

print("\n=== ZONE DEFINITIONS ===")
zones = config["zones"]

assert len(zones) == 12, "Expected exactly 12 zones."

for i, zone in enumerate(zones, 1):
    print(f"\nZone {i}:")
    pprint(zone)

print("\n=== EXISTING ZONE DATA ===")

zone_dir = ROOT / "zones"

for name in [
    "zone_stats_MOT20-05.csv",
    "zone_transitions_MOT20-05.csv",
    "zone_accuracy_MOT20-05.csv"
]:
    path = zone_dir / name

    if path.exists():
        print(f"\n{name}")

        import pandas as pd
        df = pd.read_csv(path)

        print("Rows:", len(df))
        print("Columns:", df.columns.tolist())
        print(df.head(3).to_string(index=False))

print("\nNo files modified.")

=== GRID CONFIGURATION ===
[3, 4]

=== ZONE DEFINITIONS ===

Zone 1:
{'area_px': 149040,
 'centroid': [206, 179],
 'col': 0,
 'id': 0,
 'name': 'Z0',
 'neighbours': [1, 4, 5],
 'polygon': [[0, 0], [413, 0], [413, 359], [0, 359]],
 'row': 0}

Zone 2:
{'area_px': 148680,
 'centroid': [620, 179],
 'col': 1,
 'id': 1,
 'name': 'Z1',
 'neighbours': [0, 2, 4, 5, 6],
 'polygon': [[414, 0], [826, 0], [826, 359], [414, 359]],
 'row': 0}

Zone 3:
{'area_px': 148680,
 'centroid': [1033, 179],
 'col': 2,
 'id': 2,
 'name': 'Z2',
 'neighbours': [1, 3, 5, 6, 7],
 'polygon': [[827, 0], [1239, 0], [1239, 359], [827, 359]],
 'row': 0}

Zone 4:
{'area_px': 149040,
 'centroid': [1446, 179],
 'col': 3,
 'id': 3,
 'name': 'Z3',
 'neighbours': [2, 6, 7],
 'polygon': [[1240, 0], [1653, 0], [1653, 359], [1240, 359]],
 'row': 0}

Zone 5:
{'area_px': 149040,
 'centroid': [206, 539],
 'col': 0,
 'id': 4,
 'name': 'Z4',
 'neighbours': [0, 1, 5, 8, 9],
 'polygon': [[0, 360], [413, 360], [413, 719], [0, 719]],
 'ro

In [ ]:
# CELL 44 — EXISTING ZONE ANALYSIS AUDIT

from pathlib import Path
import json
import pandas as pd

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
ZONE_DIR = ROOT / "zones"

# 1. Inspect zone configuration metadata
config = json.loads(
    (ZONE_DIR / "zone_definition.json").read_text()
)

print("=== CONFIGURATION METADATA ===")
print("Top-level keys:", list(config.keys()))

for key, value in config.items():
    if key not in ("zones", "grid"):
        print(f"{key}: {str(value)[:300]}")

# 2. Inspect existing statistics
stats = pd.read_csv(
    ZONE_DIR / "zone_stats_MOT20-05.csv"
)

transitions = pd.read_csv(
    ZONE_DIR / "zone_transitions_MOT20-05.csv"
)

print("\n=== EXISTING ZONE STATISTICS ===")

print("Splits:", stats["split"].unique().tolist())
print("Unique steps:", stats["step"].nunique())
print("Steps per split:")
print(stats.groupby("split")["step"].nunique())

print("\nZones per split:")
print(stats.groupby("split")["zone"].nunique())

print("\nTime interval examples:")
print(
    stats[["split", "step", "t_start_s"]]
    .drop_duplicates()
    .groupby("split")
    .head(5)
    .to_string(index=False)
)

print("\n=== TRANSITION DATA ===")
print("Columns:", transitions.columns.tolist())
print("Steps:", transitions["step"].nunique())
print("Total transition counts:", transitions["n"].sum())

# 3. Check available zone-related notebooks
PROJECT = ROOT.parent

print("\n=== NOTEBOOKS ===")

for path in PROJECT.rglob("*.ipynb"):
    if ".ipynb_checkpoints" in path.parts:
        continue

    print(path.relative_to(PROJECT))

print("\nAudit only. No files modified.")

=== CONFIGURATION METADATA ===
Top-level keys: ['sequence', 'image_size', 'grid', 'zones']
sequence: MOT20-05
image_size: [1654, 1080]

=== EXISTING ZONE STATISTICS ===
Splits: ['VAL', 'BOUNDARY', 'TEST']
Unique steps: 133
Steps per split:
split
BOUNDARY     1
TEST        66
VAL         66
Name: step, dtype: int64

Zones per split:
split
BOUNDARY    12
TEST        12
VAL         12
Name: zone, dtype: int64

Time interval examples:
   split  step  t_start_s
     VAL     0        0.0
     VAL     1        1.0
     VAL     2        2.0
     VAL     3        3.0
     VAL     4        4.0
BOUNDARY    66       66.0
    TEST    67       67.0
    TEST    68       68.0
    TEST    69       69.0
    TEST    70       70.0
    TEST    71       71.0

=== TRANSITION DATA ===
Columns: ['step', 'from_zone', 'to_zone', 'n']
Steps: 133
Total transition counts: 2579

=== NOTEBOOKS ===

Audit only. No files modified.


In [ ]:
# CELL 45 — ZONE DATA INTEGRITY CHECK

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
ZONE_DIR = ROOT / "zones"

stats = pd.read_csv(
    ZONE_DIR / "zone_stats_MOT20-05.csv"
)

transitions = pd.read_csv(
    ZONE_DIR / "zone_transitions_MOT20-05.csv"
)

# --------------------------------------------------
# 1. Validate zone coverage
# --------------------------------------------------

print("=== ZONE COVERAGE ===")

assert len(stats) == 1596
assert stats["zone"].nunique() == 12
assert stats["step"].nunique() == 133

coverage = stats.groupby("step")["zone"].nunique()

assert (coverage == 12).all()
assert not stats.duplicated(["step", "zone"]).any()

print("PASS: All 133 windows contain 12 unique zones.")

# --------------------------------------------------
# 2. Check split boundaries
# --------------------------------------------------

print("\n=== SPLIT BOUNDARIES ===")

split_summary = (
    stats.groupby("split")
    .agg(
        first_step=("step", "min"),
        last_step=("step", "max"),
        windows=("step", "nunique"),
        first_time=("t_start_s", "min"),
        last_time=("t_start_s", "max")
    )
)

print(split_summary.to_string())

# --------------------------------------------------
# 3. Check numerical integrity
# --------------------------------------------------

print("\n=== NUMERICAL INTEGRITY ===")

nonnegative = [
    "count", "max_count", "inflow",
    "outflow", "appear", "disappear"
]

for col in nonnegative:
    assert stats[col].notna().all()
    assert np.isfinite(stats[col]).all()
    assert (stats[col] >= 0).all()

assert (stats["count"] <= stats["max_count"] + 1e-6).all()

assert (transitions["n"] >= 0).all()
assert transitions["n"].notna().all()

assert transitions["from_zone"].between(0, 11).all()
assert transitions["to_zone"].between(0, 11).all()

print("PASS: Counts and transition values are valid.")

# --------------------------------------------------
# 4. Inspect TEST statistics
# --------------------------------------------------

test = stats[stats["split"] == "TEST"]

print("\n=== EXISTING TEST SUMMARY ===")

print("Mean occupancy per zone/window:", round(test["count"].mean(), 3))
print("Mean inflow per zone/window:", round(test["inflow"].mean(), 3))
print("Mean outflow per zone/window:", round(test["outflow"].mean(), 3))

print("Total inflow:", int(test["inflow"].sum()))
print("Total outflow:", int(test["outflow"].sum()))

print("\n=== TRANSITION COUNTS ===")

print("Total recorded transitions:", int(transitions["n"].sum()))
print("Transition rows:", len(transitions))

print("\nNo files modified.")
print("No inference or tracking rerun.")

=== ZONE COVERAGE ===
PASS: All 133 windows contain 12 unique zones.

=== SPLIT BOUNDARIES ===
          first_step  last_step  windows  first_time  last_time
split                                                          
BOUNDARY          66         66        1        66.0       66.0
TEST              67        132       66        67.0      132.0
VAL                0         65       66         0.0       65.0

=== NUMERICAL INTEGRITY ===
PASS: Counts and transition values are valid.

=== EXISTING TEST SUMMARY ===
Mean occupancy per zone/window: 16.749
Mean inflow per zone/window: 1.597
Mean outflow per zone/window: 1.597
Total inflow: 1265
Total outflow: 1265

=== TRANSITION COUNTS ===
Total recorded transitions: 2579
Transition rows: 1664

No files modified.
No inference or tracking rerun.


In [ ]:
# CELL 46 — REF_001 ZONE ASSIGNMENT (TEST)

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")

ZONE_FILE = ROOT / "zones" / "zone_definition.json"

TRACK_FILE = (
    ROOT / "tracking" / "trackeval" / "trackers"
    / "test" / "OPT_V2_REF_001_TEST"
    / "data" / "MOT20-05-test.txt"
)

OUT = ROOT / "zones" / "optimization_v2"
OUT.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUT / "REF_001_zone_assignments_TEST.parquet"
REPORT_FILE = OUT / "REF_001_zone_assignment_audit_TEST.json"

# --------------------------------------------------
# 1. Load zone configuration
# --------------------------------------------------

with open(ZONE_FILE) as f:
    config = json.load(f)

zones = config["zones"]

assert len(zones) == 12
assert config["grid"] == [3, 4]

image_w, image_h = config["image_size"]

print("=== ZONE CONFIGURATION ===")
print("Image:", image_w, "x", image_h)
print("Zones:", len(zones))

# --------------------------------------------------
# 2. Load original REF_001 observations
# --------------------------------------------------

df = pd.read_csv(TRACK_FILE, header=None).iloc[:, :6]

df.columns = ["frame", "id", "x", "y", "w", "h"]

for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="raise")

df["frame"] = df["frame"].astype(int)
df["id"] = df["id"].astype(int)

assert len(df) == 340635
assert not df.duplicated(["frame", "id"]).any()
assert df["frame"].between(1, 1658).all()
assert (df[["w", "h"]] > 0).all().all()

# --------------------------------------------------
# 3. Calculate footpoints
# --------------------------------------------------

df["foot_x"] = df["x"] + df["w"] / 2
df["foot_y"] = df["y"] + df["h"]

# Do not clip points: out-of-frame positions
# must remain detectable during the audit.

# --------------------------------------------------
# 4. Assign zones using the existing grid
# --------------------------------------------------

# The saved polygons cover integer pixel coordinates:
# X: 0..1653, Y: 0..1079.
#
# Using floor() maps continuous coordinates to
# their corresponding pixel cells.

px = np.floor(df["foot_x"].to_numpy())
py = np.floor(df["foot_y"].to_numpy())

inside = (
    np.isfinite(px)
    & np.isfinite(py)
    & (px >= 0)
    & (px < image_w)
    & (py >= 0)
    & (py < image_h)
)

df["zone"] = -1

# Derive zone IDs from the saved polygons.
# Do not assume equal-width divisions.

for zone in zones:

    polygon = np.asarray(zone["polygon"])

    xmin = polygon[:, 0].min()
    xmax = polygon[:, 0].max()
    ymin = polygon[:, 1].min()
    ymax = polygon[:, 1].max()

    match = (
        inside
        & (px >= xmin)
        & (px <= xmax)
        & (py >= ymin)
        & (py <= ymax)
    )

    assert not np.any(
        match & (df["zone"].to_numpy() != -1)
    ), "Overlapping zone assignments detected."

    df.loc[match, "zone"] = int(zone["id"])

# --------------------------------------------------
# 5. Validate assignments
# --------------------------------------------------

assigned = df["zone"] >= 0
unassigned = ~assigned

assert df.loc[assigned, "zone"].between(0, 11).all()

# Every valid image pixel must belong to one zone.
assert not np.any(
    inside & unassigned.to_numpy()
), "Uncovered pixels in zone configuration."

print("\n=== ASSIGNMENT RESULTS ===")

print("Total observations:", len(df))
print("Assigned:", int(assigned.sum()))
print("Unassigned:", int(unassigned.sum()))

print("\n=== OBSERVATIONS PER ZONE ===")

print(
    df.loc[assigned]
    .groupby("zone")
    .size()
    .reindex(range(12), fill_value=0)
    .to_string()
)

# --------------------------------------------------
# 6. Save separately
# --------------------------------------------------

df = df.sort_values(["frame", "id"]).reset_index(drop=True)

df.to_parquet(OUTPUT_FILE, index=False)

report = {
    "tracker": "OPT_V2_REF_001_TEST",
    "sequence": "MOT20-05",
    "split": "TEST",
    "zone_count": 12,
    "assignment_point": "bounding_box_bottom_center",
    "coordinate_rule": "floor_to_pixel",
    "total_observations": int(len(df)),
    "assigned_observations": int(assigned.sum()),
    "unassigned_observations": int(unassigned.sum()),
    "zone_counts": {
        str(z): int((df["zone"] == z).sum())
        for z in range(12)
    },
    "original_predictions_modified": False
}

with open(REPORT_FILE, "w") as f:
    json.dump(report, f, indent=2)

print("\nSaved:", OUTPUT_FILE)
print("Saved:", REPORT_FILE)

print("\nNo existing zone results modified.")
print("No interpolation used for zone counts.")
print("No YOLO inference or tracking rerun.")

=== ZONE CONFIGURATION ===
Image: 1654 x 1080
Zones: 12

=== ASSIGNMENT RESULTS ===
Total observations: 340635
Assigned: 335597
Unassigned: 5038

=== OBSERVATIONS PER ZONE ===
zone
0     19732
1     25319
2     39111
3     32158
4     21753
5     37628
6     30840
7     36279
8     19632
9     24419
10    26611
11    22115

Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/optimization_v2/REF_001_zone_assignments_TEST.parquet
Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/optimization_v2/REF_001_zone_assignment_audit_TEST.json

No existing zone results modified.
No interpolation used for zone counts.
No YOLO inference or tracking rerun.


In [ ]:
# CELL 47 — UNASSIGNED OBSERVATION AUDIT (TEST)

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")

INPUT = (
    ROOT / "zones" / "optimization_v2"
    / "REF_001_zone_assignments_TEST.parquet"
)

OUT = ROOT / "zones" / "optimization_v2"

df = pd.read_parquet(INPUT)

IMAGE_W = 1654
IMAGE_H = 1080

assert len(df) == 340635
assert {"frame", "id", "foot_x", "foot_y", "zone"}.issubset(df.columns)

# --------------------------------------------------
# 1. Identify unassigned observations
# --------------------------------------------------

unassigned = df[df["zone"] == -1].copy()

x = unassigned["foot_x"]
y = unassigned["foot_y"]

# Mutually exclusive categories
conditions = [
    ~np.isfinite(x) | ~np.isfinite(y),
    (x < 0) & (y < 0),
    (x >= IMAGE_W) & (y < 0),
    (x < 0) & (y >= IMAGE_H),
    (x >= IMAGE_W) & (y >= IMAGE_H),
    x < 0,
    x >= IMAGE_W,
    y < 0,
    y >= IMAGE_H
]

labels = [
    "Non-finite coordinates",
    "Top-left corner",
    "Top-right corner",
    "Bottom-left corner",
    "Bottom-right corner",
    "Left edge",
    "Right edge",
    "Top edge",
    "Bottom edge"
]

unassigned["reason"] = np.select(
    conditions,
    labels,
    default="Unexpected"
)

# --------------------------------------------------
# 2. Summarize reasons
# --------------------------------------------------

print("=== UNASSIGNED OBSERVATIONS ===")
print("Total:", len(unassigned))
print("Percentage:", round(len(unassigned) / len(df) * 100, 3), "%")

print("\n=== REASONS ===")

reason_counts = (
    unassigned["reason"]
    .value_counts()
    .rename_axis("reason")
    .reset_index(name="observations")
)

reason_counts["percentage"] = (
    reason_counts["observations"] / len(unassigned) * 100
).round(2)

print(reason_counts.to_string(index=False))

# --------------------------------------------------
# 3. Track impact
# --------------------------------------------------

print("\n=== TRACK IMPACT ===")

affected_ids = unassigned["id"].nunique()
total_ids = df["id"].nunique()

print("Affected track IDs:", affected_ids)
print("Total track IDs:", total_ids)
print("Affected percentage:", round(
    affected_ids / total_ids * 100, 2
), "%")

print("\n=== FRAME DISTRIBUTION ===")

print("Frames with unassigned observations:",
      unassigned["frame"].nunique())

print("\nTop 10 frames:")
print(
    unassigned.groupby("frame")
    .size()
    .sort_values(ascending=False)
    .head(10)
    .to_string()
)

# --------------------------------------------------
# 4. Save audit
# --------------------------------------------------

REPORT = OUT / "REF_001_unassigned_audit_TEST.json"
DETAILS = OUT / "REF_001_unassigned_details_TEST.csv"

report = {
    "total_observations": int(len(df)),
    "unassigned_observations": int(len(unassigned)),
    "unassigned_percentage": float(
        len(unassigned) / len(df) * 100
    ),
    "affected_track_ids": int(affected_ids),
    "reason_counts": {
        str(row.reason): int(row.observations)
        for row in reason_counts.itertuples(index=False)
    }
}

with open(REPORT, "w") as f:
    json.dump(report, f, indent=2)

unassigned[
    ["frame", "id", "foot_x", "foot_y", "reason"]
].to_csv(DETAILS, index=False)

print("\nSaved:", REPORT)
print("Saved:", DETAILS)

print("\nNo zone assignments modified.")
print("No YOLO inference or tracking rerun.")

=== UNASSIGNED OBSERVATIONS ===
Total: 5038
Percentage: 1.479 %

=== REASONS ===
     reason  observations  percentage
Bottom edge          5038       100.0

=== TRACK IMPACT ===
Affected track IDs: 231
Total track IDs: 1225
Affected percentage: 18.86 %

=== FRAME DISTRIBUTION ===
Frames with unassigned observations: 1614

Top 10 frames:
frame
178     9
155     9
1       9
157     8
179     8
1119    8
1120    8
1073    8
177     8
174     8

Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/optimization_v2/REF_001_unassigned_audit_TEST.json
Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/optimization_v2/REF_001_unassigned_details_TEST.csv

No zone assignments modified.
No YOLO inference or tracking rerun.


In [ ]:
# CELL 48 — REF_001 ZONE OCCUPANCY ANALYSIS (TEST)

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")

ZONE_DIR = ROOT / "zones"
OUT = ZONE_DIR / "optimization_v2"
OUT.mkdir(parents=True, exist_ok=True)

ASSIGNMENTS = OUT / "REF_001_zone_assignments_TEST.parquet"
BASELINE = ZONE_DIR / "zone_stats_MOT20-05.csv"

OUTPUT = OUT / "REF_001_zone_occupancy_TEST.csv"
COMPARISON = OUT / "REF_001_vs_existing_occupancy_TEST.csv"
REPORT = OUT / "REF_001_occupancy_summary_TEST.json"

FPS = 25
TEST_START_FRAME = 1658
TEST_END_FRAME = 3315
TOTAL_TEST_FRAMES = TEST_END_FRAME - TEST_START_FRAME + 1

# --------------------------------------------------
# 1. Load existing assignments
# --------------------------------------------------

df = pd.read_parquet(ASSIGNMENTS)

assert len(df) == 340635
assert not df.duplicated(["frame", "id"]).any()
assert df["frame"].between(1, TOTAL_TEST_FRAMES).all()

print("=== INPUT ===")
print("Observations:", len(df))
print("Assigned:", int((df["zone"] >= 0).sum()))
print("Unassigned:", int((df["zone"] == -1).sum()))

# --------------------------------------------------
# 2. Restore original MOT20-05 frame numbering
# --------------------------------------------------

# TEST local frame 1 corresponds to original frame 1658.

df["global_frame"] = (
    df["frame"] + TEST_START_FRAME - 1
)

# Original project uses 1-second windows:
# step 0 = frames 1–25
# step 1 = frames 26–50
# etc.

df["step"] = (
    (df["global_frame"] - 1) // FPS
).astype(int)

df["t_start_s"] = df["step"].astype(float)

assert df["step"].between(66, 132).all()

# --------------------------------------------------
# 3. Keep only assigned observations
# --------------------------------------------------

valid = df[df["zone"].between(0, 11)].copy()

# Each row represents one observed person in one frame.
# No interpolated observations are included.

# --------------------------------------------------
# 4. Count people per frame and zone
# --------------------------------------------------

frame_counts = (
    valid.groupby(["global_frame", "zone"])
    .size()
    .rename("count")
    .reset_index()
)

# Include zero-count zone/frame combinations.
# This is necessary for correct mean occupancy.

frame_grid = pd.MultiIndex.from_product(
    [
        range(TEST_START_FRAME, TEST_END_FRAME + 1),
        range(12)
    ],
    names=["global_frame", "zone"]
)

frame_counts = (
    frame_counts
    .set_index(["global_frame", "zone"])
    .reindex(frame_grid, fill_value=0)
    .reset_index()
)

frame_counts["step"] = (
    (frame_counts["global_frame"] - 1) // FPS
).astype(int)

# --------------------------------------------------
# 5. Aggregate occupancy per 1-second window
# --------------------------------------------------

occupancy = (
    frame_counts.groupby(["step", "zone"])
    .agg(
        count=("count", "mean"),
        max_count=("count", "max"),
        frames_in_window=("count", "size")
    )
    .reset_index()
)

occupancy["split"] = "TEST"
occupancy["zone_name"] = "Z" + occupancy["zone"].astype(str)
occupancy["t_start_s"] = occupancy["step"].astype(float)

occupancy = occupancy[
    [
        "split", "step", "t_start_s",
        "zone", "zone_name",
        "count", "max_count",
        "frames_in_window"
    ]
].sort_values(["step", "zone"]).reset_index(drop=True)

# --------------------------------------------------
# 6. Validate time-window coverage
# --------------------------------------------------

assert occupancy["zone"].nunique() == 12
assert occupancy["step"].nunique() == 67

assert len(occupancy) == 67 * 12

assert (
    occupancy.groupby("step")["zone"].nunique() == 12
).all()

assert (
    occupancy["count"] <= occupancy["max_count"]
).all()

# The TEST split begins partway through global step 66
# and ends partway through global step 132.

print("\n=== WINDOW COVERAGE ===")

print(
    occupancy.groupby("step")["frames_in_window"]
    .first()
    .head(5)
    .to_string()
)

print("...")
print(
    occupancy.groupby("step")["frames_in_window"]
    .first()
    .tail(5)
    .to_string()
)

# --------------------------------------------------
# 7. Compare against existing TEST baseline
# --------------------------------------------------

baseline = pd.read_csv(BASELINE)

baseline_test = baseline[
    baseline["split"] == "TEST"
][["step", "zone", "count", "max_count"]].copy()

baseline_test = baseline_test.rename(columns={
    "count": "baseline_count",
    "max_count": "baseline_max_count"
})

comparison = occupancy.merge(
    baseline_test,
    on=["step", "zone"],
    how="left",
    validate="one_to_one"
)

comparison["count_difference"] = (
    comparison["count"] - comparison["baseline_count"]
)

comparison["absolute_difference"] = (
    comparison["count_difference"].abs()
)

# --------------------------------------------------
# 8. Save outputs
# --------------------------------------------------

occupancy.to_csv(OUTPUT, index=False)
comparison.to_csv(COMPARISON, index=False)

report = {
    "tracker": "OPT_V2_REF_001_TEST",
    "split": "TEST",
    "fps": FPS,
    "original_observations": int(len(df)),
    "assigned_observations": int(len(valid)),
    "excluded_observations": int(len(df) - len(valid)),
    "zone_count": 12,
    "windows": int(occupancy["step"].nunique()),
    "mean_occupancy": float(occupancy["count"].mean()),
    "maximum_zone_occupancy": int(occupancy["max_count"].max()),
    "comparison_mean_absolute_difference": float(
        comparison["absolute_difference"].mean()
    ),
    "assignment_point": "bounding_box_bottom_center",
    "interpolated_points_used": False
}

with open(REPORT, "w") as f:
    json.dump(report, f, indent=2)

# --------------------------------------------------
# 9. Display results
# --------------------------------------------------

print("\n=== REF_001 OCCUPANCY SUMMARY ===")

print("Mean occupancy:", round(report["mean_occupancy"], 3))
print("Maximum occupancy:", report["maximum_zone_occupancy"])
print("Windows:", report["windows"])

print("\n=== BASELINE COMPARISON ===")

print(
    comparison[
        ["baseline_count", "count", "count_difference"]
    ].mean().round(3).to_string()
)

print("\nMean absolute difference:",
      round(report["comparison_mean_absolute_difference"], 3))

print("\nSaved:", OUTPUT)
print("Saved:", COMPARISON)
print("Saved:", REPORT)

print("\nExisting zone results unchanged.")
print("No YOLO inference or tracking rerun.")

=== INPUT ===
Observations: 340635
Assigned: 335597
Unassigned: 5038

=== WINDOW COVERAGE ===
step
66    18
67    25
68    25
69    25
70    25
...
step
128    25
129    25
130    25
131    25
132    15

=== REF_001 OCCUPANCY SUMMARY ===
Mean occupancy: 16.863
Maximum occupancy: 38
Windows: 67

=== BASELINE COMPARISON ===
baseline_count      16.749
count               16.863
count_difference     0.111

Mean absolute difference: 0.581

Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/optimization_v2/REF_001_zone_occupancy_TEST.csv
Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/optimization_v2/REF_001_vs_existing_occupancy_TEST.csv
Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/optimization_v2/REF_001_occupancy_summary_TEST.json

Existing zone results unchanged.
No YOLO inference or tracking rerun.


In [ ]:
# CELL 49 — ALIGNED TEST OCCUPANCY COMPARISON

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "zones" / "optimization_v2"

NEW_FILE = OUT / "REF_001_zone_occupancy_TEST.csv"
BASE_FILE = ROOT / "zones" / "zone_stats_MOT20-05.csv"

new = pd.read_csv(NEW_FILE)
baseline = pd.read_csv(BASE_FILE)

# Keep exactly the existing TEST windows
baseline = baseline[
    baseline["split"] == "TEST"
][["step", "zone", "count", "max_count"]].copy()

baseline = baseline.rename(columns={
    "count": "baseline_count",
    "max_count": "baseline_max_count"
})

new = new[
    new["step"].between(67, 132)
].copy()

comparison = new.merge(
    baseline,
    on=["step", "zone"],
    how="inner",
    validate="one_to_one"
)

assert len(comparison) == 66 * 12
assert comparison["step"].nunique() == 66
assert comparison["zone"].nunique() == 12
assert not comparison["baseline_count"].isna().any()

comparison["difference"] = (
    comparison["count"] - comparison["baseline_count"]
)

comparison["absolute_difference"] = (
    comparison["difference"].abs()
)

comparison["window_type"] = np.where(
    comparison["frames_in_window"] == 25,
    "FULL",
    "PARTIAL"
)

print("=== ALIGNED TEST COMPARISON ===")

print("Windows:", comparison["step"].nunique())
print("Zone/window pairs:", len(comparison))

print("\n=== ALL MATCHING TEST WINDOWS ===")

print("Baseline mean:", round(comparison["baseline_count"].mean(), 3))
print("REF_001 mean:", round(comparison["count"].mean(), 3))
print("Mean difference:", round(comparison["difference"].mean(), 3))
print("Mean absolute difference:",
      round(comparison["absolute_difference"].mean(), 3))

print("\n=== FULL VS PARTIAL WINDOWS ===")

summary = comparison.groupby("window_type").agg(
    rows=("zone", "size"),
    baseline_mean=("baseline_count", "mean"),
    ref_mean=("count", "mean"),
    mean_difference=("difference", "mean"),
    MAE_between_outputs=("absolute_difference", "mean")
).round(3)

print(summary.to_string())

print("\n=== PER-ZONE COMPARISON ===")

zone_summary = comparison.groupby("zone").agg(
    baseline_mean=("baseline_count", "mean"),
    ref_mean=("count", "mean"),
    mean_difference=("difference", "mean"),
    MAE_between_outputs=("absolute_difference", "mean")
).round(3)

print(zone_summary.to_string())

# Save corrected comparison separately
comparison_file = (
    OUT / "REF_001_vs_existing_occupancy_ALIGNED_TEST.csv"
)

summary_file = (
    OUT / "REF_001_occupancy_ALIGNED_summary_TEST.json"
)

comparison.to_csv(comparison_file, index=False)

report = {
    "tracker": "OPT_V2_REF_001_TEST",
    "steps": [67, 132],
    "compared_zone_windows": int(len(comparison)),
    "baseline_mean": float(comparison["baseline_count"].mean()),
    "ref_mean": float(comparison["count"].mean()),
    "mean_difference": float(comparison["difference"].mean()),
    "MAE_between_outputs": float(
        comparison["absolute_difference"].mean()
    ),
    "full_windows": int(
        comparison.loc[
            comparison["window_type"] == "FULL", "step"
        ].nunique()
    ),
    "partial_windows": int(
        comparison.loc[
            comparison["window_type"] == "PARTIAL", "step"
        ].nunique()
    ),
    "note": (
        "Output-to-output comparison, not Ground Truth accuracy. "
        "Zone assignment methods may differ."
    )
}

with open(summary_file, "w") as f:
    json.dump(report, f, indent=2)

print("\nSaved:", comparison_file)
print("Saved:", summary_file)

print("\nNo existing results modified.")
print("No inference or tracking rerun.")

=== ALIGNED TEST COMPARISON ===
Windows: 66
Zone/window pairs: 792

=== ALL MATCHING TEST WINDOWS ===
Baseline mean: 16.749
REF_001 mean: 16.86
Mean difference: 0.111
Mean absolute difference: 0.581

=== FULL VS PARTIAL WINDOWS ===
             rows  baseline_mean  ref_mean  mean_difference  MAE_between_outputs
window_type                                                                     
FULL          780         16.757    16.873            0.116                0.582
PARTIAL        12         16.217    16.028           -0.189                0.522

=== PER-ZONE COMPARISON ===
      baseline_mean  ref_mean  mean_difference  MAE_between_outputs
zone                                                               
0            11.346    11.886            0.540                0.562
1            14.925    15.214            0.289                0.328
2            22.925    23.536            0.611                0.667
3            18.778    19.512            0.734                0.740
4      

In [ ]:
# CELL 50 — REF_001 ZONE TRANSITIONS & INFLOW/OUTFLOW (TEST)

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "zones" / "optimization_v2"
OUT.mkdir(parents=True, exist_ok=True)

ASSIGNMENTS = OUT / "REF_001_zone_assignments_TEST.parquet"
BASE_STATS = ROOT / "zones" / "zone_stats_MOT20-05.csv"
BASE_TRANSITIONS = ROOT / "zones" / "zone_transitions_MOT20-05.csv"

TRANSITION_FILE = OUT / "REF_001_zone_transitions_TEST.csv"
FLOW_FILE = OUT / "REF_001_zone_flow_TEST.csv"
COMPARISON_FILE = OUT / "REF_001_vs_existing_flow_TEST.csv"
REPORT_FILE = OUT / "REF_001_zone_flow_summary_TEST.json"

FPS = 25
TEST_START_FRAME = 1658
TEST_END_FRAME = 3315

# --------------------------------------------------
# 1. Load saved zone assignments
# --------------------------------------------------

df = pd.read_parquet(ASSIGNMENTS)

required = {"frame", "id", "zone"}
assert required.issubset(df.columns)
assert len(df) == 340635
assert not df.duplicated(["frame", "id"]).any()
assert df["frame"].between(1, 1658).all()
assert df["zone"].between(-1, 11).all()

df = df.sort_values(["id", "frame"]).copy()

df["global_frame"] = df["frame"] + TEST_START_FRAME - 1
df["step"] = ((df["global_frame"] - 1) // FPS).astype(int)

# --------------------------------------------------
# 2. Find previous observation for each track
# --------------------------------------------------

df["previous_frame"] = df.groupby("id")["frame"].shift()
df["previous_zone"] = df.groupby("id")["zone"].shift()

# Only consecutive observed frames are eligible.
# Both zone assignments must be valid.
# A transition must involve an actual zone change.

transition_mask = (
    (df["frame"] - df["previous_frame"] == 1)
    & df["zone"].between(0, 11)
    & df["previous_zone"].between(0, 11)
    & (df["zone"] != df["previous_zone"])
)

events = df.loc[
    transition_mask,
    ["frame", "global_frame", "step", "id",
     "previous_zone", "zone"]
].copy()

events = events.rename(columns={
    "previous_zone": "from_zone",
    "zone": "to_zone"
})

events["from_zone"] = events["from_zone"].astype(int)
events["to_zone"] = events["to_zone"].astype(int)

events = events.sort_values(
    ["global_frame", "id"]
).reset_index(drop=True)

assert (events["from_zone"] != events["to_zone"]).all()
assert events["from_zone"].between(0, 11).all()
assert events["to_zone"].between(0, 11).all()

# --------------------------------------------------
# 3. Aggregate transitions by second
# --------------------------------------------------

transitions = (
    events.groupby(["step", "from_zone", "to_zone"])
    .size()
    .rename("n")
    .reset_index()
)

transitions["split"] = np.where(
    transitions["step"] == 66,
    "BOUNDARY",
    "TEST"
)

# --------------------------------------------------
# 4. Calculate inflow and outflow
# --------------------------------------------------

inflow = (
    transitions.groupby(["step", "to_zone"])["n"]
    .sum()
    .rename("inflow")
    .reset_index()
    .rename(columns={"to_zone": "zone"})
)

outflow = (
    transitions.groupby(["step", "from_zone"])["n"]
    .sum()
    .rename("outflow")
    .reset_index()
    .rename(columns={"from_zone": "zone"})
)

# Include every zone in each TEST window.
# Also retain the partial boundary window separately.

grid = pd.MultiIndex.from_product(
    [range(66, 133), range(12)],
    names=["step", "zone"]
)

flow = (
    pd.DataFrame(index=grid)
    .join(inflow.set_index(["step", "zone"]))
    .join(outflow.set_index(["step", "zone"]))
    .fillna(0)
    .reset_index()
)

flow[["inflow", "outflow"]] = (
    flow[["inflow", "outflow"]].astype(int)
)

flow["split"] = np.where(
    flow["step"] == 66,
    "BOUNDARY",
    "TEST"
)

flow["zone_name"] = "Z" + flow["zone"].astype(str)
flow["t_start_s"] = flow["step"].astype(float)

# --------------------------------------------------
# 5. Validate flow conservation
# --------------------------------------------------

assert flow["inflow"].sum() == flow["outflow"].sum()
assert flow["inflow"].sum() == len(events)
assert transitions["n"].sum() == len(events)

assert len(flow) == 67 * 12
assert flow.groupby("step")["zone"].nunique().eq(12).all()

print("=== REF_001 TRANSITION RESULTS ===")
print("Transition events:", len(events))
print("Unique transition pairs:", len(transitions))
print("Total inflow:", int(flow["inflow"].sum()))
print("Total outflow:", int(flow["outflow"].sum()))

print("\n=== TRANSITIONS BY SPLIT ===")
print(
    flow.groupby("split")[["inflow", "outflow"]]
    .sum()
    .to_string()
)

# --------------------------------------------------
# 6. Compare aligned TEST windows
# --------------------------------------------------

baseline = pd.read_csv(BASE_STATS)

baseline_test = baseline.loc[
    baseline["split"] == "TEST",
    ["step", "zone", "inflow", "outflow"]
].copy()

baseline_test = baseline_test.rename(columns={
    "inflow": "baseline_inflow",
    "outflow": "baseline_outflow"
})

ref_test = flow[flow["split"] == "TEST"].copy()

comparison = ref_test.merge(
    baseline_test,
    on=["step", "zone"],
    how="inner",
    validate="one_to_one"
)

assert len(comparison) == 66 * 12

for metric in ["inflow", "outflow"]:
    comparison[f"{metric}_difference"] = (
        comparison[metric]
        - comparison[f"baseline_{metric}"]
    )

    comparison[f"{metric}_absolute_difference"] = (
        comparison[f"{metric}_difference"].abs()
    )

print("\n=== ALIGNED TEST COMPARISON ===")

for metric in ["inflow", "outflow"]:
    print(f"\n{metric.upper()}")
    print(
        "Existing baseline mean:",
        round(comparison[f"baseline_{metric}"].mean(), 3)
    )
    print(
        "REF_001 mean:",
        round(comparison[metric].mean(), 3)
    )
    print(
        "Mean difference:",
        round(comparison[f"{metric}_difference"].mean(), 3)
    )
    print(
        "Mean absolute difference:",
        round(
            comparison[f"{metric}_absolute_difference"].mean(),
            3
        )
    )

# --------------------------------------------------
# 7. Compare transition totals
# --------------------------------------------------

base_trans = pd.read_csv(BASE_TRANSITIONS)

# Existing transition file has no split column.
# Compare matching TEST steps only.

base_test_trans = base_trans[
    base_trans["step"].between(67, 132)
]

ref_test_trans = transitions[
    transitions["step"].between(67, 132)
]

print("\n=== TEST TRANSITION TOTALS ===")
print("Existing:", int(base_test_trans["n"].sum()))
print("REF_001:", int(ref_test_trans["n"].sum()))

# --------------------------------------------------
# 8. Save outputs
# --------------------------------------------------

transitions.to_csv(TRANSITION_FILE, index=False)
flow.to_csv(FLOW_FILE, index=False)
comparison.to_csv(COMPARISON_FILE, index=False)

report = {
    "tracker": "OPT_V2_REF_001_TEST",
    "split": "TEST",
    "transition_rule": (
        "Same track ID, consecutive observed frames, "
        "two valid and different zones"
    ),
    "interpolated_points_used": False,
    "total_transition_events_including_boundary": int(len(events)),
    "test_transition_events": int(ref_test_trans["n"].sum()),
    "test_total_inflow": int(ref_test["inflow"].sum()),
    "test_total_outflow": int(ref_test["outflow"].sum()),
    "existing_test_transition_events": int(
        base_test_trans["n"].sum()
    ),
    "mean_absolute_inflow_difference": float(
        comparison["inflow_absolute_difference"].mean()
    ),
    "mean_absolute_outflow_difference": float(
        comparison["outflow_absolute_difference"].mean()
    ),
    "note": (
        "Comparison with existing outputs, not Ground Truth. "
        "Original transition rules may differ."
    )
}

with open(REPORT_FILE, "w") as f:
    json.dump(report, f, indent=2)

print("\n=== SAVED FILES ===")
print(TRANSITION_FILE)
print(FLOW_FILE)
print(COMPARISON_FILE)
print(REPORT_FILE)

print("\nExisting zone results unchanged.")
print("No YOLO inference or tracking rerun.")

=== REF_001 TRANSITION RESULTS ===
Transition events: 2020
Unique transition pairs: 1051
Total inflow: 2020
Total outflow: 2020

=== TRANSITIONS BY SPLIT ===
          inflow  outflow
split                    
BOUNDARY      19       19
TEST        2001     2001

=== ALIGNED TEST COMPARISON ===

INFLOW
Existing baseline mean: 1.597
REF_001 mean: 2.527
Mean difference: 0.929
Mean absolute difference: 0.995

OUTFLOW
Existing baseline mean: 1.597
REF_001 mean: 2.527
Mean difference: 0.929
Mean absolute difference: 0.997

=== TEST TRANSITION TOTALS ===
Existing: 1265
REF_001: 2001

=== SAVED FILES ===
/content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/optimization_v2/REF_001_zone_transitions_TEST.csv
/content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/optimization_v2/REF_001_zone_flow_TEST.csv
/content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/optimization_v2/REF_001_vs_existing_flow_TEST.csv
/content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/optimization_v2/REF_001_z

In [ ]:
# CELL 51 — ZONE TRANSITION RELIABILITY AUDIT (TEST)

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "zones" / "optimization_v2"

ASSIGNMENTS = OUT / "REF_001_zone_assignments_TEST.parquet"
ZONE_FILE = ROOT / "zones" / "zone_definition.json"

EVENTS_FILE = OUT / "REF_001_transition_reliability_events_TEST.csv"
TRACKS_FILE = OUT / "REF_001_transition_reliability_tracks_TEST.csv"
REPORT_FILE = OUT / "REF_001_transition_reliability_summary_TEST.json"

FPS = 25
TEST_START_FRAME = 1658

# Diagnostic thresholds, not validated error criteria
REVERSAL_WINDOW = 10       # Frames (0.4 seconds)
BOUNDARY_DISTANCE = 10.0   # Pixels
HIGH_TRANSITION_COUNT = 10

# --------------------------------------------------
# 1. Load data
# --------------------------------------------------

df = pd.read_parquet(ASSIGNMENTS)

with open(ZONE_FILE) as f:
    config = json.load(f)

zones = {
    int(z["id"]): z
    for z in config["zones"]
}

assert len(zones) == 12
assert len(df) == 340635

required = {
    "frame", "id", "zone", "foot_x", "foot_y"
}
assert required.issubset(df.columns)

assert not df.duplicated(["frame", "id"]).any()

df = df.sort_values(["id", "frame"]).copy()

# --------------------------------------------------
# 2. Reconstruct CELL 50 transition events
# --------------------------------------------------

df["previous_frame"] = df.groupby("id")["frame"].shift()
df["previous_zone"] = df.groupby("id")["zone"].shift()

mask = (
    (df["frame"] - df["previous_frame"] == 1)
    & df["zone"].between(0, 11)
    & df["previous_zone"].between(0, 11)
    & (df["zone"] != df["previous_zone"])
)

events = df.loc[
    mask,
    [
        "frame", "id", "zone", "previous_zone",
        "foot_x", "foot_y"
    ]
].copy()

events = events.rename(columns={
    "zone": "to_zone",
    "previous_zone": "from_zone"
})

events["from_zone"] = events["from_zone"].astype(int)
events["to_zone"] = events["to_zone"].astype(int)

events["global_frame"] = (
    events["frame"] + TEST_START_FRAME - 1
)

events["step"] = (
    (events["global_frame"] - 1) // FPS
).astype(int)

# Keep the same TEST windows used in CELL 50
events = events[
    events["step"].between(67, 132)
].copy()

events = events.sort_values(
    ["id", "frame"]
).reset_index(drop=True)

assert len(events) == 2001, (
    f"Expected 2001 TEST transitions, found {len(events)}"
)

print("=== INPUT ===")
print("TEST transition events:", len(events))
print("Unique track IDs:", events["id"].nunique())

# --------------------------------------------------
# 3. Detect rapid A -> B -> A reversals
# --------------------------------------------------

events["next_frame"] = (
    events.groupby("id")["frame"].shift(-1)
)

events["next_from_zone"] = (
    events.groupby("id")["from_zone"].shift(-1)
)

events["next_to_zone"] = (
    events.groupby("id")["to_zone"].shift(-1)
)

events["reversal_start"] = (
    (events["next_frame"] - events["frame"] <= REVERSAL_WINDOW)
    & (events["next_from_zone"] == events["to_zone"])
    & (events["next_to_zone"] == events["from_zone"])
)

# Mark both transitions participating in a reversal
events["reversal_end"] = (
    events.groupby("id")["reversal_start"]
    .shift(1, fill_value=False)
)

events["rapid_reversal"] = (
    events["reversal_start"] | events["reversal_end"]
)

# --------------------------------------------------
# 4. Check neighboring zones
# --------------------------------------------------

events["neighbor_transition"] = [
    int(to_z) in zones[int(from_z)]["neighbours"]
    for from_z, to_z in zip(
        events["from_zone"],
        events["to_zone"]
    )
]

events["non_neighbor_transition"] = (
    ~events["neighbor_transition"]
)

# --------------------------------------------------
# 5. Distance to destination zone boundary
# --------------------------------------------------

def boundary_distance(x, y, zone_id):
    polygon = np.asarray(
        zones[int(zone_id)]["polygon"],
        dtype=float
    )

    xmin = polygon[:, 0].min()
    xmax = polygon[:, 0].max()
    ymin = polygon[:, 1].min()
    ymax = polygon[:, 1].max()

    return max(
        0.0,
        min(
            abs(x - xmin),
            abs(x - xmax),
            abs(y - ymin),
            abs(y - ymax)
        )
    )

events["boundary_distance_px"] = [
    boundary_distance(x, y, z)
    for x, y, z in zip(
        events["foot_x"],
        events["foot_y"],
        events["to_zone"]
    )
]

events["near_boundary"] = (
    events["boundary_distance_px"] <= BOUNDARY_DISTANCE
)

# --------------------------------------------------
# 6. Track-level transition frequency
# --------------------------------------------------

track_summary = (
    events.groupby("id")
    .agg(
        transitions=("frame", "size"),
        rapid_reversals=("rapid_reversal", "sum"),
        near_boundary=("near_boundary", "sum"),
        non_neighbor=("non_neighbor_transition", "sum")
    )
    .reset_index()
)

track_summary["high_transition_count"] = (
    track_summary["transitions"] > HIGH_TRANSITION_COUNT
)

# --------------------------------------------------
# 7. Summary
# --------------------------------------------------

total = len(events)

report = {
    "tracker": "OPT_V2_REF_001_TEST",
    "split": "TEST",
    "total_transitions": int(total),
    "unique_transition_track_ids": int(events["id"].nunique()),
    "rapid_reversal_events": int(events["rapid_reversal"].sum()),
    "rapid_reversal_percentage": float(
        events["rapid_reversal"].mean() * 100
    ),
    "near_boundary_events": int(events["near_boundary"].sum()),
    "near_boundary_percentage": float(
        events["near_boundary"].mean() * 100
    ),
    "non_neighbor_events": int(
        events["non_neighbor_transition"].sum()
    ),
    "non_neighbor_percentage": float(
        events["non_neighbor_transition"].mean() * 100
    ),
    "tracks_over_10_transitions": int(
        track_summary["high_transition_count"].sum()
    ),
    "reversal_window_frames": REVERSAL_WINDOW,
    "boundary_threshold_px": BOUNDARY_DISTANCE,
    "note": (
        "Diagnostic flags only. A flagged event is not "
        "automatically a false transition."
    )
}

# --------------------------------------------------
# 8. Save reports
# --------------------------------------------------

events.to_csv(EVENTS_FILE, index=False)
track_summary.to_csv(TRACKS_FILE, index=False)

with open(REPORT_FILE, "w") as f:
    json.dump(report, f, indent=2)

print("\n=== RELIABILITY SUMMARY ===")

for key, value in report.items():
    print(f"{key}: {value}")

print("\n=== TOP 10 TRACKS BY TRANSITIONS ===")

print(
    track_summary.sort_values(
        "transitions", ascending=False
    ).head(10).to_string(index=False)
)

print("\n=== FLAG OVERLAP ===")

print(pd.crosstab(
    events["rapid_reversal"],
    events["near_boundary"]
).to_string())

print("\nSaved:", EVENTS_FILE)
print("Saved:", TRACKS_FILE)
print("Saved:", REPORT_FILE)

print("\nNo transitions removed or modified.")
print("No YOLO inference or tracking rerun.")

=== INPUT ===
TEST transition events: 2001
Unique track IDs: 579

=== RELIABILITY SUMMARY ===
tracker: OPT_V2_REF_001_TEST
split: TEST
total_transitions: 2001
unique_transition_track_ids: 579
rapid_reversal_events: 972
rapid_reversal_percentage: 48.57571214392804
near_boundary_events: 1972
near_boundary_percentage: 98.55072463768117
non_neighbor_events: 0
non_neighbor_percentage: 0.0
tracks_over_10_transitions: 21
reversal_window_frames: 10
boundary_threshold_px: 10.0
note: Diagnostic flags only. A flagged event is not automatically a false transition.

=== TOP 10 TRACKS BY TRANSITIONS ===
 id  transitions  rapid_reversals  near_boundary  non_neighbor  high_transition_count
 45           42               36             42             0                   True
237           37               34             37             0                   True
604           34               33             32             0                   True
523           33               27             33           

In [ ]:
# CELL 52A — INSPECT MOT20-05 GT FILES

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project")

files = sorted(
    p for p in ROOT.rglob("gt.txt")
    if "MOT20-05" in str(p)
)

print("=== GT FILES FOUND ===")
print("Count:", len(files))

for path in files:

    print("\n" + "=" * 65)
    print("FILE:", path)

    try:
        data = pd.read_csv(path, header=None)

        print("Shape:", data.shape)
        print("\nFirst 5 rows:")
        print(data.head().to_string(index=False, header=False))

        print("\nFrame range:",
              data.iloc[:, 0].min(),
              "to",
              data.iloc[:, 0].max())

        print("Unique frames:",
              data.iloc[:, 0].nunique())

        print("Unique IDs:",
              data.iloc[:, 1].nunique())

        counts = data.iloc[:, 1].value_counts()

        print("Maximum observations per ID:", counts.max())
        print("IDs observed more than once:", (counts > 1).sum())

        print("\nColumn 7 values:")
        if data.shape[1] >= 7:
            print(data.iloc[:, 6].value_counts().head(10))
        else:
            print("Not available")

        print("\nColumn 8 values:")
        if data.shape[1] >= 8:
            print(data.iloc[:, 7].value_counts().head(10))
        else:
            print("Not available")

    except Exception as e:
        print("ERROR:", e)

print("\n=== DIAGNOSTIC COMPLETE ===")
print("No files modified.")

=== GT FILES FOUND ===
Count: 2

FILE: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/tracking/trackeval/gt/MOT20-05-test/gt/gt.txt
Shape: (371490, 9)

First 5 rows:
700 385 70 1060 54 21 1 1 0.95455
701 385 66 1059 53 22 1 1 0.95652
702 385 62 1059 52 22 1 1 0.95652
703 385 58 1059 51 22 1 1 0.95652
704 385 54 1059 51 22 1 1 0.95652

Frame range: 1 to 1658
Unique frames: 1658
Unique IDs: 693
Maximum observations per ID: 1658
IDs observed more than once: 692

Column 7 values:
6
1    319276
0     52214
Name: count, dtype: int64

Column 8 values:
7
1     319276
7      44651
6       4247
11      3316
Name: count, dtype: int64

FILE: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/tracking/trackeval/gt/MOT20-05-val/gt/gt.txt
Shape: (379840, 9)

First 5 rows:
1031 1 1 140  5 149 1 1 0.68667
1032 1 1 139  7 149 1 1 0.68667
1033 1 1 138  9 149 1 1 0.68667
1034 1 1 136 11 149 1 1 0.69333
1035 1 1 135 13 149 1 1 0.69333

Frame range: 1 to 1657
Unique frames: 1657
Unique IDs: 767
Maximum 

In [ ]:
# CELL 52B — CORRECTED GT ZONE TRANSITION EVALUATION

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "zones" / "optimization_v2"

GT_FILE = ROOT / "tracking/trackeval/gt/MOT20-05-test/gt/gt.txt"
ZONE_FILE = ROOT / "zones/zone_definition.json"
PRED_FILE = OUT / "REF_001_zone_flow_TEST.csv"

FPS = 25
FRAME_OFFSET = 1657

# 1. Load GT with LOCAL frame numbering

gt = pd.read_csv(
    GT_FILE,
    header=None,
    names=[
        "frame", "id", "x", "y", "w", "h",
        "mark", "class_id", "visibility"
    ]
)

gt = gt[
    (gt["mark"] == 1) &
    (gt["class_id"] == 1) &
    (gt["frame"].between(1, 1658))
].copy()

assert not gt.empty
assert not gt.duplicated(["frame", "id"]).any()

print("=== GT INPUT ===")
print("Pedestrian observations:", len(gt))
print("Unique person IDs:", gt["id"].nunique())
print("Local frame range:", gt["frame"].min(), gt["frame"].max())

# 2. Load zone configuration

with open(ZONE_FILE) as f:
    config = json.load(f)

W, H = map(int, config["image_size"])

assert (W, H) == (1654, 1080)
assert len(config["zones"]) == 12

# 3. Apply the same bottom-center assignment as CELL 46

gt["foot_x"] = gt["x"] + gt["w"] / 2
gt["foot_y"] = gt["y"] + gt["h"]

px = np.floor(gt["foot_x"]).astype(int)
py = np.floor(gt["foot_y"]).astype(int)

valid = (
    px.between(0, W - 1) &
    py.between(0, H - 1)
)

col = np.select(
    [px <= 413, px <= 826, px <= 1239],
    [0, 1, 2],
    default=3
)

row = np.select(
    [py <= 359, py <= 719],
    [0, 1],
    default=2
)

gt["zone"] = -1
gt.loc[valid, "zone"] = (
    row[valid] * 4 + col[valid]
)

print("Assigned observations:", int(valid.sum()))
print("Outside zones:", int((~valid).sum()))

# 4. Compute GT transitions using consecutive frames only

gt = gt.sort_values(["id", "frame"]).copy()

gt["prev_frame"] = gt.groupby("id")["frame"].shift()
gt["prev_zone"] = gt.groupby("id")["zone"].shift()

mask = (
    (gt["frame"] - gt["prev_frame"] == 1) &
    gt["zone"].between(0, 11) &
    gt["prev_zone"].between(0, 11) &
    (gt["zone"] != gt["prev_zone"])
)

events = gt.loc[
    mask,
    ["frame", "id", "prev_zone", "zone"]
].copy()

events = events.rename(columns={
    "prev_zone": "from_zone",
    "zone": "to_zone"
})

events["from_zone"] = events["from_zone"].astype(int)
events["to_zone"] = events["to_zone"].astype(int)

# Local TEST -> original global timeline
events["global_frame"] = events["frame"] + FRAME_OFFSET
events["step"] = (
    (events["global_frame"] - 1) // FPS
).astype(int)

# Same aligned TEST windows as CELL 50
events = events[events["step"].between(67, 132)].copy()

# 5. Aggregate per-zone, per-second GT flow

index = pd.MultiIndex.from_product(
    [range(67, 133), range(12)],
    names=["step", "zone"]
)

inflow = events.groupby(["step", "to_zone"]).size()
inflow.index.names = ["step", "zone"]

outflow = events.groupby(["step", "from_zone"]).size()
outflow.index.names = ["step", "zone"]

gt_flow = pd.DataFrame(index=index).reset_index()

gt_flow["inflow"] = (
    inflow.reindex(index, fill_value=0).to_numpy()
)

gt_flow["outflow"] = (
    outflow.reindex(index, fill_value=0).to_numpy()
)

assert gt_flow["inflow"].sum() == len(events)
assert gt_flow["outflow"].sum() == len(events)

# 6. Compare REF_001 against GT

pred = pd.read_csv(PRED_FILE)

pred = pred[
    pred["step"].between(67, 132)
][["step", "zone", "inflow", "outflow"]].copy()

comparison = gt_flow.merge(
    pred,
    on=["step", "zone"],
    suffixes=("_gt", "_pred"),
    validate="one_to_one"
)

assert len(comparison) == 792

metrics = {}

for name in ["inflow", "outflow"]:

    error = (
        comparison[f"{name}_pred"] -
        comparison[f"{name}_gt"]
    )

    metrics[name] = {
        "MAE": float(error.abs().mean()),
        "bias": float(error.mean()),
        "RMSE": float(np.sqrt((error ** 2).mean()))
    }

# 7. Save corrected results

files = {
    "events": OUT / "GT_zone_transitions_TEST.csv",
    "flow": OUT / "GT_zone_flow_TEST.csv",
    "comparison": OUT / "REF_001_vs_GT_zone_flow_TEST.csv",
    "report": OUT / "REF_001_vs_GT_zone_flow_summary_TEST.json"
}

events.to_csv(files["events"], index=False)
gt_flow.to_csv(files["flow"], index=False)
comparison.to_csv(files["comparison"], index=False)

report = {
    "gt_source": str(GT_FILE),
    "gt_frame_numbering": "local TEST frames 1-1658",
    "global_frame_offset": FRAME_OFFSET,
    "evaluated_steps": [67, 132],
    "gt_transitions": int(len(events)),
    "ref001_transitions": int(comparison["inflow_pred"].sum()),
    "metrics": metrics
}

with open(files["report"], "w") as f:
    json.dump(report, f, indent=2)

# 8. Results

print("\n=== CORRECTED GT EVALUATION ===")
print("GT transitions:", len(events))
print("REF_001 transitions:", comparison["inflow_pred"].sum())

for name, values in metrics.items():
    print(f"\n{name.upper()}")
    for metric, value in values.items():
        print(f"{metric}: {value:.4f}")

print("\n=== SAVED FILES ===")
for path in files.values():
    print(path)

print("\nCorrected GT-derived evaluation saved.")
print("No YOLO inference or tracking rerun.")

=== GT INPUT ===
Pedestrian observations: 319276
Unique person IDs: 657
Local frame range: 1 1658
Assigned observations: 301436
Outside zones: 17840

=== CORRECTED GT EVALUATION ===
GT transitions: 1169
REF_001 transitions: 2001

INFLOW
MAE: 1.2803
bias: 1.0505
RMSE: 1.9663

OUTFLOW
MAE: 1.2828
bias: 1.0505
RMSE: 1.9313

=== SAVED FILES ===
/content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/optimization_v2/GT_zone_transitions_TEST.csv
/content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/optimization_v2/GT_zone_flow_TEST.csv
/content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/optimization_v2/REF_001_vs_GT_zone_flow_TEST.csv
/content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/optimization_v2/REF_001_vs_GT_zone_flow_summary_TEST.json

Corrected GT-derived evaluation saved.
No YOLO inference or tracking rerun.


In [ ]:
# CELL 53 — FAIR ZONE-FLOW EVALUATION: BASELINE vs REF_001

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "zones" / "optimization_v2"
OUT.mkdir(parents=True, exist_ok=True)

BASELINE_FILE = (
    ROOT / "tracking/trackeval/trackers/test/"
    "BoTSORT_candidate/data/MOT20-05-test.txt"
)

REF_FLOW_FILE = OUT / "REF_001_zone_flow_TEST.csv"
GT_FLOW_FILE = OUT / "GT_zone_flow_TEST.csv"
ZONE_FILE = ROOT / "zones/zone_definition.json"

FPS = 25
FRAME_OFFSET = 1657
STEPS = range(67, 133)
ZONE_IDS = range(12)

# --------------------------------------------------
# 1. Load and validate zone configuration
# --------------------------------------------------

with open(ZONE_FILE) as f:
    config = json.load(f)

W, H = map(int, config["image_size"])

assert (W, H) == (1654, 1080)
assert len(config["zones"]) == 12

# --------------------------------------------------
# 2. Load original tracker predictions
# --------------------------------------------------

columns = [
    "frame", "id", "x", "y", "w", "h",
    "conf", "unused1", "unused2", "unused3"
]

base = pd.read_csv(
    BASELINE_FILE,
    header=None,
    names=columns
)

base = base[base["frame"].between(1, 1658)].copy()

assert not base.empty
assert not base.duplicated(["frame", "id"]).any()

print("=== ORIGINAL TRACKER INPUT ===")
print("Observations:", len(base))
print("Unique track IDs:", base["id"].nunique())
print("Frame range:", base["frame"].min(), base["frame"].max())

# --------------------------------------------------
# 3. Assign original predictions to zones
# --------------------------------------------------

base["foot_x"] = base["x"] + base["w"] / 2
base["foot_y"] = base["y"] + base["h"]

px = np.floor(base["foot_x"]).astype(int)
py = np.floor(base["foot_y"]).astype(int)

valid = (
    px.between(0, W - 1) &
    py.between(0, H - 1)
)

col = np.select(
    [px <= 413, px <= 826, px <= 1239],
    [0, 1, 2],
    default=3
)

row = np.select(
    [py <= 359, py <= 719],
    [0, 1],
    default=2
)

base["zone"] = -1
base.loc[valid, "zone"] = (
    row[valid] * 4 + col[valid]
)

print("Assigned observations:", int(valid.sum()))
print("Outside zones:", int((~valid).sum()))

# --------------------------------------------------
# 4. Calculate original tracker transitions
# --------------------------------------------------

base = base.sort_values(["id", "frame"]).copy()

base["prev_frame"] = base.groupby("id")["frame"].shift()
base["prev_zone"] = base.groupby("id")["zone"].shift()

mask = (
    (base["frame"] - base["prev_frame"] == 1) &
    base["zone"].between(0, 11) &
    base["prev_zone"].between(0, 11) &
    (base["zone"] != base["prev_zone"])
)

events = base.loc[
    mask,
    ["frame", "id", "prev_zone", "zone"]
].copy()

events = events.rename(columns={
    "prev_zone": "from_zone",
    "zone": "to_zone"
})

events["from_zone"] = events["from_zone"].astype(int)
events["to_zone"] = events["to_zone"].astype(int)

events["global_frame"] = events["frame"] + FRAME_OFFSET
events["step"] = (
    (events["global_frame"] - 1) // FPS
).astype(int)

events = events[events["step"].between(67, 132)].copy()

# --------------------------------------------------
# 5. Aggregate baseline inflow/outflow
# --------------------------------------------------

index = pd.MultiIndex.from_product(
    [STEPS, ZONE_IDS],
    names=["step", "zone"]
)

inflow = events.groupby(["step", "to_zone"]).size()
inflow.index.names = ["step", "zone"]

outflow = events.groupby(["step", "from_zone"]).size()
outflow.index.names = ["step", "zone"]

base_flow = pd.DataFrame(index=index).reset_index()

base_flow["inflow"] = (
    inflow.reindex(index, fill_value=0).to_numpy()
)

base_flow["outflow"] = (
    outflow.reindex(index, fill_value=0).to_numpy()
)

assert base_flow["inflow"].sum() == len(events)
assert base_flow["outflow"].sum() == len(events)

# --------------------------------------------------
# 6. Load GT and REF_001 zone flows
# --------------------------------------------------

gt = pd.read_csv(GT_FLOW_FILE)
ref = pd.read_csv(REF_FLOW_FILE)

gt = gt[gt["step"].between(67, 132)][
    ["step", "zone", "inflow", "outflow"]
].copy()

ref = ref[ref["step"].between(67, 132)][
    ["step", "zone", "inflow", "outflow"]
].copy()

def check_flow(df, name):
    assert len(df) == 792, f"{name}: unexpected row count"
    assert not df.duplicated(["step", "zone"]).any()
    assert set(map(tuple, df[["step", "zone"]].to_numpy())) == set(index)

check_flow(gt, "GT")
check_flow(ref, "REF_001")
check_flow(base_flow, "Baseline")

# --------------------------------------------------
# 7. Build aligned comparison
# --------------------------------------------------

comparison = (
    gt.rename(columns={
        "inflow": "inflow_gt",
        "outflow": "outflow_gt"
    })
    .merge(
        base_flow.rename(columns={
            "inflow": "inflow_baseline",
            "outflow": "outflow_baseline"
        }),
        on=["step", "zone"],
        validate="one_to_one"
    )
    .merge(
        ref.rename(columns={
            "inflow": "inflow_ref001",
            "outflow": "outflow_ref001"
        }),
        on=["step", "zone"],
        validate="one_to_one"
    )
)

assert len(comparison) == 792

# --------------------------------------------------
# 8. Calculate metrics against GT
# --------------------------------------------------

rows = []

for tracker in ["baseline", "ref001"]:
    for direction in ["inflow", "outflow"]:

        actual = comparison[f"{direction}_gt"]
        predicted = comparison[f"{direction}_{tracker}"]

        error = predicted - actual

        rows.append({
            "tracker": tracker,
            "metric": direction,
            "GT_total": int(actual.sum()),
            "predicted_total": int(predicted.sum()),
            "MAE": float(error.abs().mean()),
            "RMSE": float(np.sqrt((error ** 2).mean())),
            "bias": float(error.mean())
        })

metrics = pd.DataFrame(rows)

# --------------------------------------------------
# 9. Save results
# --------------------------------------------------

BASE_EVENTS_FILE = OUT / "BASELINE_zone_transitions_TEST.csv"
BASE_FLOW_FILE = OUT / "BASELINE_zone_flow_TEST.csv"
COMPARISON_FILE = OUT / "BASELINE_vs_REF001_vs_GT_zone_flow_TEST.csv"
METRICS_FILE = OUT / "BASELINE_vs_REF001_GT_metrics_TEST.csv"

events.to_csv(BASE_EVENTS_FILE, index=False)
base_flow.to_csv(BASE_FLOW_FILE, index=False)
comparison.to_csv(COMPARISON_FILE, index=False)
metrics.to_csv(METRICS_FILE, index=False)

print("\n=== FAIR GT COMPARISON ===")
print(metrics.round(4).to_string(index=False))

print("\n=== MAE DIFFERENCE (REF_001 - BASELINE) ===")

for direction in ["inflow", "outflow"]:
    subset = metrics[metrics["metric"] == direction]
    base_mae = subset.loc[
        subset["tracker"] == "baseline", "MAE"
    ].iloc[0]
    ref_mae = subset.loc[
        subset["tracker"] == "ref001", "MAE"
    ].iloc[0]

    difference = ref_mae - base_mae

    print(f"{direction}: {difference:+.4f}")

print("\nNegative difference = REF_001 has lower MAE.")
print("Positive difference = REF_001 has higher MAE.")

print("\n=== SAVED FILES ===")
for path in [
    BASE_EVENTS_FILE,
    BASE_FLOW_FILE,
    COMPARISON_FILE,
    METRICS_FILE
]:
    print(path)

print("\nNo inference or tracking rerun.")
print("Original results unchanged.")

=== ORIGINAL TRACKER INPUT ===
Observations: 333369
Unique track IDs: 1294
Frame range: 1 1658
Assigned observations: 328360
Outside zones: 5009

=== FAIR GT COMPARISON ===
 tracker  metric  GT_total  predicted_total    MAE   RMSE   bias
baseline  inflow      1169             1978 1.2513 1.9250 1.0215
baseline outflow      1169             1978 1.2689 1.9152 1.0215
  ref001  inflow      1169             2001 1.2803 1.9663 1.0505
  ref001 outflow      1169             2001 1.2828 1.9313 1.0505

=== MAE DIFFERENCE (REF_001 - BASELINE) ===
inflow: +0.0290
outflow: +0.0139

Negative difference = REF_001 has lower MAE.
Positive difference = REF_001 has higher MAE.

=== SAVED FILES ===
/content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/optimization_v2/BASELINE_zone_transitions_TEST.csv
/content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/optimization_v2/BASELINE_zone_flow_TEST.csv
/content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/optimization_v2/BASELINE_vs_REF001_vs_GT_zone

In [ ]:
# CELL 54 — LOCATE SAVED VAL TRACKING FILES

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")

files = sorted(
    p for p in (ROOT / "tracking").rglob("*.txt")
    if "MOT20-05" in p.name
    and "gt" not in p.parts
    and p.stat().st_size > 0
)

print("=== SAVED TRACKING FILES ===")
print("Found:", len(files))

for p in files:
    try:
        df = pd.read_csv(p, header=None, usecols=[0, 1])
        print(
            f"\n{p.relative_to(ROOT)}"
            f"\n  Rows: {len(df)}"
            f" | Frames: {df.iloc[:,0].min()}–{df.iloc[:,0].max()}"
            f" | IDs: {df.iloc[:,1].nunique()}"
        )
    except Exception as e:
        print("SKIPPED:", p, str(e)[:100])

print("\n=== VAL GT ===")
gt = ROOT / "tracking/trackeval/gt/MOT20-05-val/gt/gt.txt"
print("Available:", gt.exists())

=== SAVED TRACKING FILES ===
Found: 26

tracking/trackeval/trackers/test/BoTSORT_candidate/data/MOT20-05-test.txt
  Rows: 333369 | Frames: 1–1658 | IDs: 1294

tracking/trackeval/trackers/test/ByteTrack_candidate/data/MOT20-05-test.txt
  Rows: 332493 | Frames: 1–1658 | IDs: 1301

tracking/trackeval/trackers/test/ByteTrack_default/data/MOT20-05-test.txt
  Rows: 397155 | Frames: 1–1658 | IDs: 3754

tracking/trackeval/trackers/test/OPT_V2_REF_001_TEST/data/MOT20-05-test.txt
  Rows: 340635 | Frames: 1–1658 | IDs: 1225

tracking/trackeval/trackers/val/BoTSORT_candidate/data/MOT20-05-val.txt
  Rows: 340931 | Frames: 1–1657 | IDs: 1405

tracking/trackeval/trackers/val/ByteTrack_candidate/data/MOT20-05-val.txt
  Rows: 340088 | Frames: 1–1657 | IDs: 1429

tracking/trackeval/trackers/val/ByteTrack_default/data/MOT20-05-val.txt
  Rows: 397172 | Frames: 1–1657 | IDs: 3582

tracking/trackeval/trackers/val/OPT_V2_EXP_001/data/MOT20-05-val.txt
  Rows: 340088 | Frames: 1–1657 | IDs: 1429

tracking/trac

In [ ]:
# CELL 55 — TUNE ZONE STABILITY ON VAL

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "zones" / "optimization_v2"
OUT.mkdir(parents=True, exist_ok=True)

TRACKERS = {
    "BoTSORT_baseline": ROOT / "tracking/trackeval/trackers/val/BoTSORT_candidate/data/MOT20-05-val.txt",
    "REF_001": ROOT / "tracking/trackeval/trackers/val/OPT_V2_REF_001/data/MOT20-05-val.txt"
}

GT_PATH = ROOT / "tracking/trackeval/gt/MOT20-05-val/gt/gt.txt"
ZONE_PATH = ROOT / "zones/zone_definition.json"

THRESHOLDS = [1, 3, 5, 7, 10]
FPS = 25
W, H = 1654, 1080
VAL_FRAMES = 1657

with open(ZONE_PATH) as f:
    config = json.load(f)

assert list(map(int, config["image_size"])) == [W, H]
assert len(config["zones"]) == 12

def load_tracks(path, is_gt=False):
    df = pd.read_csv(path, header=None)

    df = df.iloc[:, :9 if is_gt else 6].copy()
    df.columns = (
        ["frame", "id", "x", "y", "w", "h",
         "mark", "class_id", "visibility"]
        if is_gt else
        ["frame", "id", "x", "y", "w", "h"]
    )

    if is_gt:
        df = df[(df["mark"] == 1) & (df["class_id"] == 1)]

    df = df[df["frame"].between(1, VAL_FRAMES)].copy()

    assert not df.empty
    assert not df.duplicated(["frame", "id"]).any()

    px = np.floor(df["x"] + df["w"] / 2).astype(int)
    py = np.floor(df["y"] + df["h"]).astype(int)

    valid = px.between(0, W - 1) & py.between(0, H - 1)

    col = np.select(
        [px <= 413, px <= 826, px <= 1239],
        [0, 1, 2], default=3
    )
    row = np.select(
        [py <= 359, py <= 719],
        [0, 1], default=2
    )

    df["zone"] = -1
    df.loc[valid, "zone"] = row[valid] * 4 + col[valid]

    return df[["frame", "id", "zone"]].sort_values(
        ["id", "frame"]
    )

def get_events(df, threshold):
    events = []

    for person_id, group in df.groupby("id", sort=False):
        frames = group["frame"].to_numpy()
        zones = group["zone"].to_numpy()

        stable = -1
        candidate = -1
        streak = 0
        previous_frame = -999

        for frame, zone in zip(frames, zones):
            frame = int(frame)
            zone = int(zone)

            # Missing frames break candidate confirmation.
            if frame != previous_frame + 1:
                candidate = -1
                streak = 0

            previous_frame = frame

            # Outside image: do not infer movement.
            if zone == -1:
                candidate = -1
                streak = 0
                stable = -1
                continue

            if stable == -1:
                if zone == candidate:
                    streak += 1
                else:
                    candidate = zone
                    streak = 1

                if streak >= threshold:
                    stable = zone
                    candidate = -1
                    streak = 0
                continue

            if zone == stable:
                candidate = -1
                streak = 0
                continue

            if zone == candidate:
                streak += 1
            else:
                candidate = zone
                streak = 1

            if streak >= threshold:
                events.append((frame, person_id, stable, zone))
                stable = zone
                candidate = -1
                streak = 0

    return pd.DataFrame(
        events, columns=["frame", "id", "from_zone", "to_zone"]
    )

def make_flow(events):
    index = pd.MultiIndex.from_product(
        [range(67), range(12)],
        names=["step", "zone"]
    )

    if len(events):
        events = events.copy()
        events["step"] = (events["frame"] - 1) // FPS

        incoming = events.groupby(["step", "to_zone"]).size()
        outgoing = events.groupby(["step", "from_zone"]).size()

        incoming.index.names = ["step", "zone"]
        outgoing.index.names = ["step", "zone"]

        inflow = incoming.reindex(index, fill_value=0).to_numpy()
        outflow = outgoing.reindex(index, fill_value=0).to_numpy()
    else:
        inflow = np.zeros(len(index), dtype=int)
        outflow = np.zeros(len(index), dtype=int)

    flow = pd.DataFrame(index=index).reset_index()
    flow["inflow"] = inflow
    flow["outflow"] = outflow
    return flow

# GT uses the original consecutive-frame rule (threshold=1).
gt_tracks = load_tracks(GT_PATH, is_gt=True)
gt_events = get_events(gt_tracks, 1)
gt_flow = make_flow(gt_events)

print("=== VAL GT ===")
print("GT transitions:", len(gt_events))
print("GT observations:", len(gt_tracks))

results = []

for name, path in TRACKERS.items():
    tracks = load_tracks(path)

    print(f"\nProcessing {name}...")

    for threshold in THRESHOLDS:
        events = get_events(tracks, threshold)
        flow = make_flow(events)

        merged = gt_flow.merge(
            flow, on=["step", "zone"],
            suffixes=("_gt", "_pred"),
            validate="one_to_one"
        )

        inflow_error = merged["inflow_pred"] - merged["inflow_gt"]
        outflow_error = merged["outflow_pred"] - merged["outflow_gt"]

        in_mae = inflow_error.abs().mean()
        out_mae = outflow_error.abs().mean()

        results.append({
            "tracker": name,
            "threshold": threshold,
            "transitions": len(events),
            "inflow_MAE": round(float(in_mae), 4),
            "outflow_MAE": round(float(out_mae), 4),
            "average_MAE": round(float((in_mae + out_mae) / 2), 4),
            "inflow_bias": round(float(inflow_error.mean()), 4),
            "outflow_bias": round(float(outflow_error.mean()), 4)
        })

results = pd.DataFrame(results).sort_values(
    ["average_MAE", "threshold"]
)

best = results.iloc[0].to_dict()

RESULT_FILE = OUT / "zone_stability_tuning_VAL.csv"
BEST_FILE = OUT / "zone_stability_selected_VAL.json"

results.to_csv(RESULT_FILE, index=False)

with open(BEST_FILE, "w") as f:
    json.dump({
        "selected": best,
        "thresholds_tested": THRESHOLDS,
        "selection_split": "VAL",
        "note": "GT uses unsmoothed consecutive-frame transitions."
    }, f, indent=2)

print("\n=== VAL STABILITY RESULTS ===")
print(results.to_string(index=False))

print("\n=== SELECTED CONFIGURATION ===")
print(json.dumps(best, indent=2))

print("\nSaved:", RESULT_FILE)
print("Saved:", BEST_FILE)
print("No TEST tuning or inference rerun.")

=== VAL GT ===
GT transitions: 1223
GT observations: 327068

Processing BoTSORT_baseline...

Processing REF_001...

=== VAL STABILITY RESULTS ===
         tracker  threshold  transitions  inflow_MAE  outflow_MAE  average_MAE  inflow_bias  outflow_bias
BoTSORT_baseline          5         1287      0.7065       0.6816       0.6940       0.0796        0.0796
         REF_001          5         1318      0.7127       0.6754       0.6940       0.1182        0.1182
         REF_001          7         1204      0.7177       0.6729       0.6953      -0.0236       -0.0236
BoTSORT_baseline          7         1179      0.7139       0.6791       0.6965      -0.0547       -0.0547
BoTSORT_baseline          3         1430      0.7301       0.7127       0.7214       0.2575        0.2575
         REF_001          3         1485      0.7413       0.7289       0.7351       0.3259        0.3259
         REF_001         10         1135      0.8383       0.8184       0.8284      -0.1095       -0.1095
BoTSOR

In [ ]:
# CELL 56 — FINAL TEST EVALUATION (VAL-SELECTED CONFIGURATION)

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "zones" / "optimization_v2"

SELECTED = OUT / "zone_stability_selected_VAL.json"
GT_FILE = OUT / "GT_zone_flow_TEST.csv"

with open(SELECTED) as f:
    selected = json.load(f)["selected"]

assert selected["tracker"] == "BoTSORT_baseline"
assert selected["threshold"] == 5

THRESHOLD = int(selected["threshold"])

TRACK_FILE = (
    ROOT / "tracking/trackeval/trackers/test/"
    "BoTSORT_candidate/data/MOT20-05-test.txt"
)

df = pd.read_csv(TRACK_FILE, header=None)
df = df.iloc[:, :6].copy()
df.columns = ["frame", "id", "x", "y", "w", "h"]

assert not df.duplicated(["frame", "id"]).any()

# Same bottom-center zone assignment
px = np.floor(df["x"] + df["w"] / 2).astype(int)
py = np.floor(df["y"] + df["h"]).astype(int)

valid = px.between(0, 1653) & py.between(0, 1079)

col = np.select(
    [px <= 413, px <= 826, px <= 1239],
    [0, 1, 2], default=3
)
row = np.select(
    [py <= 359, py <= 719],
    [0, 1], default=2
)

df["zone"] = -1
df.loc[valid, "zone"] = row[valid] * 4 + col[valid]
df = df.sort_values(["id", "frame"])

# Apply exactly the same state machine as CELL 55
events = []

for person_id, group in df.groupby("id", sort=False):
    stable = -1
    candidate = -1
    streak = 0
    previous_frame = -999

    for frame, zone in zip(group["frame"], group["zone"]):
        frame, zone = int(frame), int(zone)

        if frame != previous_frame + 1:
            candidate = -1
            streak = 0

        previous_frame = frame

        if zone == -1:
            stable = -1
            candidate = -1
            streak = 0
            continue

        if stable == -1:
            if zone == candidate:
                streak += 1
            else:
                candidate = zone
                streak = 1

            if streak >= THRESHOLD:
                stable = zone
                candidate = -1
                streak = 0
            continue

        if zone == stable:
            candidate = -1
            streak = 0
            continue

        if zone == candidate:
            streak += 1
        else:
            candidate = zone
            streak = 1

        if streak >= THRESHOLD:
            events.append((frame, person_id, stable, zone))
            stable = zone
            candidate = -1
            streak = 0

events = pd.DataFrame(
    events,
    columns=["frame", "id", "from_zone", "to_zone"]
)

events["global_frame"] = events["frame"] + 1657
events["step"] = (events["global_frame"] - 1) // 25

# Aligned TEST windows only
events = events[events["step"].between(67, 132)].copy()

index = pd.MultiIndex.from_product(
    [range(67, 133), range(12)],
    names=["step", "zone"]
)

incoming = events.groupby(["step", "to_zone"]).size()
outgoing = events.groupby(["step", "from_zone"]).size()

incoming.index.names = ["step", "zone"]
outgoing.index.names = ["step", "zone"]

flow = pd.DataFrame(index=index).reset_index()
flow["inflow"] = incoming.reindex(index, fill_value=0).to_numpy()
flow["outflow"] = outgoing.reindex(index, fill_value=0).to_numpy()

assert flow["inflow"].sum() == len(events)
assert flow["outflow"].sum() == len(events)

gt = pd.read_csv(GT_FILE)[
    ["step", "zone", "inflow", "outflow"]
]

comparison = gt.merge(
    flow,
    on=["step", "zone"],
    suffixes=("_gt", "_pred"),
    validate="one_to_one"
)

assert len(comparison) == 792

print("=== FINAL TEST RESULTS ===")
print("Selected tracker:", selected["tracker"])
print("Stability threshold:", THRESHOLD, "frames")
print("GT transitions:", int(comparison["inflow_gt"].sum()))
print("Predicted transitions:", len(events))

metrics = {}

for direction in ["inflow", "outflow"]:
    error = comparison[f"{direction}_pred"] - comparison[f"{direction}_gt"]

    metrics[direction] = {
        "MAE": float(error.abs().mean()),
        "RMSE": float(np.sqrt((error ** 2).mean())),
        "bias": float(error.mean())
    }

    print(f"\n{direction.upper()}")
    for k, v in metrics[direction].items():
        print(f"{k}: {v:.4f}")

print("\n=== COMPARED WITH UNSMOOTHED BASELINE ===")
print("Original inflow MAE: 1.2513")
print("Original outflow MAE: 1.2689")
print("New inflow MAE:", round(metrics["inflow"]["MAE"], 4))
print("New outflow MAE:", round(metrics["outflow"]["MAE"], 4))

# Save final artifacts
events.to_csv(OUT / "VAL_SELECTED_stable_transitions_TEST.csv", index=False)
flow.to_csv(OUT / "VAL_SELECTED_stable_flow_TEST.csv", index=False)
comparison.to_csv(OUT / "VAL_SELECTED_vs_GT_flow_TEST.csv", index=False)

with open(OUT / "VAL_SELECTED_final_TEST_metrics.json", "w") as f:
    json.dump({
        "selected_on": "VAL",
        "tracker": selected["tracker"],
        "threshold": THRESHOLD,
        "gt_transitions": int(comparison["inflow_gt"].sum()),
        "predicted_transitions": len(events),
        "metrics": metrics
    }, f, indent=2)

print("\nSaved final TEST results.")
print("No inference or tracking rerun.")

=== FINAL TEST RESULTS ===
Selected tracker: BoTSORT_baseline
Stability threshold: 5 frames
GT transitions: 1169
Predicted transitions: 1234

INFLOW
MAE: 0.7008
RMSE: 1.1015
bias: 0.0821

OUTFLOW
MAE: 0.7008
RMSE: 1.0830
bias: 0.0821

=== COMPARED WITH UNSMOOTHED BASELINE ===
Original inflow MAE: 1.2513
Original outflow MAE: 1.2689
New inflow MAE: 0.7008
New outflow MAE: 0.7008

Saved final TEST results.
No inference or tracking rerun.


In [ ]:
# CELL 57 — INTEGRATE FINAL ZONE MONITORING

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
ZONES = ROOT / "zones"
OUT = ZONES / "optimization_v2"

OCC_FILE = OUT / "REF_001_zone_occupancy_TEST.csv"
FLOW_FILE = OUT / "VAL_SELECTED_stable_flow_TEST.csv"
EXISTING_FILE = ZONES / "zone_stats_MOT20-05.csv"
CONFIG_FILE = ZONES / "zone_definition.json"

FINAL_FILE = OUT / "FINAL_zone_monitoring_TEST.csv"
REPORT_FILE = OUT / "FINAL_zone_monitoring_summary_TEST.json"

# --------------------------------------------------
# 1. Load saved outputs
# --------------------------------------------------

occ = pd.read_csv(OCC_FILE)
flow = pd.read_csv(FLOW_FILE)
existing = pd.read_csv(EXISTING_FILE)

with open(CONFIG_FILE) as f:
    config = json.load(f)

print("Occupancy columns:", occ.columns.tolist())
print("Flow columns:", flow.columns.tolist())

# --------------------------------------------------
# 2. Standardize occupancy columns
# --------------------------------------------------

assert {"step", "zone", "count", "max_count"}.issubset(occ.columns)
assert {"step", "zone", "inflow", "outflow"}.issubset(flow.columns)

occ = occ[
    occ["step"].between(67, 132)
][["step", "zone", "count", "max_count"]].copy()

flow = flow[
    flow["step"].between(67, 132)
][["step", "zone", "inflow", "outflow"]].copy()

assert len(occ) == 792
assert len(flow) == 792

# --------------------------------------------------
# 3. Combine occupancy and stabilized flow
# --------------------------------------------------

final = occ.merge(
    flow,
    on=["step", "zone"],
    validate="one_to_one"
)

assert len(final) == 792

final["split"] = "TEST"
final["t_start_s"] = final["step"].astype(float)

# --------------------------------------------------
# 4. Preserve existing density/crowd-level definitions
# --------------------------------------------------

reference = existing[
    existing["step"].between(67, 132)
].copy()

# Density per 100x100 pixels for each rectangular zone.
# Image: 1654x1080, grid: 3 rows x 4 columns.

W, H = map(int, config["image_size"])
assert (W, H) == (1654, 1080)

zone_areas = {}

for z in config["zones"]:
    polygon = np.asarray(z["polygon"], dtype=float)
    xmin, xmax = polygon[:, 0].min(), polygon[:, 0].max()
    ymin, ymax = polygon[:, 1].min(), polygon[:, 1].max()

    # Pixel-inclusive rectangle
    area = (xmax - xmin + 1) * (ymax - ymin + 1)
    zone_areas[int(z["id"])] = area

final["zone_area_px"] = final["zone"].map(zone_areas)

final["density_per_100x100px"] = (
    final["count"] / final["zone_area_px"] * 10000
)

# Keep existing labels as reference ONLY.
# We cannot reuse old labels as new predictions.
if "level_own" in reference.columns:
    final = final.merge(
        reference[["step", "zone", "level_own"]].rename(
            columns={"level_own": "existing_level_reference"}
        ),
        on=["step", "zone"],
        how="left",
        validate="one_to_one"
    )

# --------------------------------------------------
# 5. Validate outputs
# --------------------------------------------------

assert not final.duplicated(["step", "zone"]).any()
assert final["count"].ge(0).all()
assert final["inflow"].ge(0).all()
assert final["outflow"].ge(0).all()
assert final["zone_area_px"].gt(0).all()

assert final["inflow"].sum() == final["outflow"].sum()
assert final["inflow"].sum() == 1234

# --------------------------------------------------
# 6. Save
# --------------------------------------------------

final.to_csv(FINAL_FILE, index=False)

report = {
    "split": "TEST",
    "steps": [67, 132],
    "zone_time_windows": len(final),
    "occupancy_source": "REF_001",
    "flow_source": "BoTSORT_baseline + 5-frame stability",
    "mean_occupancy": float(final["count"].mean()),
    "maximum_zone_occupancy": float(final["max_count"].max()),
    "total_inflow": int(final["inflow"].sum()),
    "total_outflow": int(final["outflow"].sum()),
    "mean_density_per_100x100px": float(
        final["density_per_100x100px"].mean()
    ),
    "note": (
        "Hybrid diagnostic output. Occupancy and flow originate "
        "from different trackers. Crowd-level labels have not "
        "been recalculated."
    )
}

with open(REPORT_FILE, "w") as f:
    json.dump(report, f, indent=2)

print("\n=== FINAL ZONE MONITORING ===")
for key, value in report.items():
    print(f"{key}: {value}")

print("\n=== FIRST 12 ROWS ===")
print(final.head(12).to_string(index=False))

print("\nSaved:", FINAL_FILE)
print("Saved:", REPORT_FILE)
print("Original files unchanged.")
print("No YOLO inference or tracking rerun.")

Occupancy columns: ['split', 'step', 't_start_s', 'zone', 'zone_name', 'count', 'max_count', 'frames_in_window']
Flow columns: ['step', 'zone', 'inflow', 'outflow']

=== FINAL ZONE MONITORING ===
split: TEST
steps: [67, 132]
zone_time_windows: 792
occupancy_source: REF_001
flow_source: BoTSORT_baseline + 5-frame stability
mean_occupancy: 16.860117845117845
maximum_zone_occupancy: 38.0
total_inflow: 1234
total_outflow: 1234
mean_density_per_100x100px: 1.1327474982949113
note: Hybrid diagnostic output. Occupancy and flow originate from different trackers. Crowd-level labels have not been recalculated.

=== FIRST 12 ROWS ===
 step  zone  count  max_count  inflow  outflow split  t_start_s  zone_area_px  density_per_100x100px existing_level_reference
   67     0  15.88         18       2        2  TEST       67.0      149040.0               1.065486                      LOW
   67     1  26.76         28       4        1  TEST       67.0      148680.0               1.799839                  

In [ ]:
# CELL 58 — AUDIT EXISTING CROWD-LEVEL DEFINITIONS

from pathlib import Path
import pandas as pd
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
ZONES = ROOT / "zones"

stats = pd.read_csv(ZONES / "zone_stats_MOT20-05.csv")

with open(ZONES / "zone_definition.json") as f:
    config = json.load(f)

print("=== ZONE CONFIGURATION ===")
print("Grid:", config.get("grid"))
print("Number of zones:", len(config["zones"]))

print("\n=== ZONE METADATA ===")
for z in config["zones"]:
    print(json.dumps(z, indent=2))

print("\n=== EXISTING CROWD LEVELS ===")

for level_col in ["level_own", "level_floor"]:

    if level_col not in stats.columns:
        continue

    print(f"\n{level_col}:")
    print(stats[level_col].value_counts(dropna=False))

    print("\nOccupancy by level:")
    print(
        stats.groupby(level_col)["count"]
        .agg(["min", "median", "max"])
        .round(3)
        .to_string()
    )

    if "density_per_100x100px" in stats.columns:
        print("\nDensity by level:")
        print(
            stats.groupby(level_col)["density_per_100x100px"]
            .agg(["min", "median", "max"])
            .round(3)
            .to_string()
        )

    if "floor_coverage" in stats.columns:
        print("\nFloor coverage by level:")
        print(
            stats.groupby(level_col)["floor_coverage"]
            .agg(["min", "median", "max"])
            .round(4)
            .to_string()
        )

print("\n=== LEVELS BY ZONE ===")

if "level_own" in stats.columns:
    print(
        pd.crosstab(
            stats["zone"],
            stats["level_own"]
        ).to_string()
    )

print("\nAudit complete. No files modified.")

=== ZONE CONFIGURATION ===
Grid: [3, 4]
Number of zones: 12

=== ZONE METADATA ===
{
  "id": 0,
  "name": "Z0",
  "row": 0,
  "col": 0,
  "polygon": [
    [
      0,
      0
    ],
    [
      413,
      0
    ],
    [
      413,
      359
    ],
    [
      0,
      359
    ]
  ],
  "area_px": 149040,
  "centroid": [
    206,
    179
  ],
  "neighbours": [
    1,
    4,
    5
  ]
}
{
  "id": 1,
  "name": "Z1",
  "row": 0,
  "col": 1,
  "polygon": [
    [
      414,
      0
    ],
    [
      826,
      0
    ],
    [
      826,
      359
    ],
    [
      414,
      359
    ]
  ],
  "area_px": 148680,
  "centroid": [
    620,
    179
  ],
  "neighbours": [
    0,
    2,
    4,
    5,
    6
  ]
}
{
  "id": 2,
  "name": "Z2",
  "row": 0,
  "col": 2,
  "polygon": [
    [
      827,
      0
    ],
    [
      1239,
      0
    ],
    [
      1239,
      359
    ],
    [
      827,
      359
    ]
  ],
  "area_px": 148680,
  "centroid": [
    1033,
    179
  ],
  "neighbours": [
    1,
  

In [ ]:
# CELL 59 — INVESTIGATE CROWD-LEVEL THRESHOLDS

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
df = pd.read_csv(ROOT / "zones/zone_stats_MOT20-05.csv")

FEATURES = [
    "count",
    "density_per_100x100px",
    "floor_coverage"
]

LABELS = ["level_own", "level_floor"]

# 1. Test whether each feature separates levels globally

print("=== GLOBAL THRESHOLD SEPARABILITY ===")

for label in LABELS:
    print(f"\n{label}")

    for feature in FEATURES:
        ranges = df.groupby(label)[feature].agg(["min", "max"])

        low_max = ranges.loc["LOW", "max"]
        med_min = ranges.loc["MEDIUM", "min"]
        med_max = ranges.loc["MEDIUM", "max"]
        high_min = ranges.loc["HIGH", "min"]

        separable = (
            low_max < med_min and
            med_max < high_min
        )

        print(
            f"{feature}: "
            f"{'YES' if separable else 'NO'}"
        )

# 2. Check thresholds within individual zones

print("\n=== PER-ZONE DENSITY RANGES ===")

for zone, group in df.groupby("zone"):

    print(f"\nZ{zone}")

    for label in LABELS:

        ranges = group.groupby(label)[
            "density_per_100x100px"
        ].agg(["min", "max"])

        print(f"{label}:")
        print(ranges.round(3).to_string())

# 3. Check if labels change for identical feature values

print("\n=== LABEL CONSISTENCY ===")

for label in LABELS:
    for feature in FEATURES:

        inconsistent = (
            df.groupby(["zone", feature])[label]
            .nunique()
            .gt(1)
            .sum()
        )

        print(
            f"{label} / {feature}: "
            f"{inconsistent} conflicting groups"
        )

print("\n=== AUDIT COMPLETE ===")
print("No files modified.")

=== GLOBAL THRESHOLD SEPARABILITY ===

level_own
count: NO
density_per_100x100px: NO
floor_coverage: NO

level_floor
count: NO
density_per_100x100px: NO
floor_coverage: YES

=== PER-ZONE DENSITY RANGES ===

Z0
level_own:
             min    max
level_own              
HIGH       1.581  1.645
LOW        0.204  1.060
MEDIUM     1.095  1.514
level_floor:
               min    max
level_floor              
LOW          0.204  1.361
MEDIUM       1.374  1.645

Z1
level_own:
             min    max
level_own              
HIGH       1.684  1.824
LOW        0.334  1.076
MEDIUM     1.090  1.665
level_floor:
               min    max
level_floor              
LOW          0.334  1.461
MEDIUM       1.528  1.824

Z2
level_own:
             min    max
level_own              
HIGH       2.031  2.268
LOW        0.460  1.369
MEDIUM     1.420  1.999
level_floor:
               min    max
level_floor              
HIGH         2.268  2.268
LOW          0.460  1.523
MEDIUM       1.528  2.155

Z3
level_ow

In [ ]:
# CELL 60 — RESTORE ORIGINAL VAL-ONLY CROWD CALIBRATION

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
ZONES = ROOT / "zones"
OUT = ZONES / "optimization_v2"
OUT.mkdir(parents=True, exist_ok=True)

CALIB_FILE = ZONES / "zone_level_calibration_MOT20-05.csv"
STATS_FILE = ZONES / "zone_stats_MOT20-05.csv"

calib = pd.read_csv(CALIB_FILE)
stats = pd.read_csv(STATS_FILE)

required = [
    "zone", "own_low_max", "own_high_min",
    "person_area_px", "zone_area_px",
    "floor_low_max", "floor_high_min"
]

assert set(required).issubset(calib.columns)
assert len(calib) == 12
assert set(calib["zone"]) == set(range(12))
assert calib[required].notna().all().all()

assert (calib["own_high_min"] > calib["own_low_max"]).all()
assert (calib["person_area_px"] > 0).all()
assert (calib["zone_area_px"] > 0).all()

assert calib["floor_low_max"].nunique() == 1
assert calib["floor_high_min"].nunique() == 1

floor_lo = float(calib["floor_low_max"].iloc[0])
floor_hi = float(calib["floor_high_min"].iloc[0])

assert floor_lo < floor_hi

# Original CELL 1 classification function:
# LOW <= lower threshold
# HIGH >= upper threshold
# Everything else = MEDIUM

def classify_level(values, low, high):
    return np.where(
        np.asarray(values) >= np.asarray(high),
        "HIGH",
        np.where(
            np.asarray(values) <= np.asarray(low),
            "LOW",
            "MEDIUM"
        )
    )

# Validate against original VAL labels ONLY.
val = stats[stats["split"] == "VAL"].copy()

val = val.merge(
    calib[required],
    on="zone",
    suffixes=("", "_calib"),
    validate="many_to_one"
)

pred_own = classify_level(
    val["count"],
    val["own_low_max"],
    val["own_high_min"]
)

pred_floor = classify_level(
    val["floor_coverage"],
    floor_lo,
    floor_hi
)

own_accuracy = float(np.mean(pred_own == val["level_own"]))
floor_accuracy = float(np.mean(pred_floor == val["level_floor"]))

print("=== ORIGINAL VAL-ONLY CALIBRATION ===")
print("Source:", CALIB_FILE)
print("Validation frames: 1–1657")
print("TEST data used for calibration: NO")

print("\n=== VAL REPRODUCTION ===")
print("VAL records:", len(val))
print(f"Own-level agreement: {own_accuracy:.2%}")
print(f"Floor-level agreement: {floor_accuracy:.2%}")

print("\n=== ORIGINAL THRESHOLDS ===")
print(
    calib[
        ["zone", "own_low_max", "own_high_min",
         "person_area_px"]
    ].round(4).to_string(index=False)
)

print("\nFloor LOW threshold:", floor_lo)
print("Floor HIGH threshold:", floor_hi)

report = {
    "source": str(CALIB_FILE),
    "calibration_split": "VAL",
    "calibration_frames": [1, 1657],
    "test_used_for_calibration": False,
    "own_level_feature": "count",
    "floor_level_feature": "floor_coverage",
    "val_records_checked": len(val),
    "val_own_agreement": own_accuracy,
    "val_floor_agreement": floor_accuracy,
    "note": "Uses original CELL 19 calibration without TEST-derived thresholds."
}

with open(OUT / "VAL_only_calibration_audit.json", "w") as f:
    json.dump(report, f, indent=2)

print("\nCELL 60 completed.")
print("No calibration thresholds changed.")

=== ORIGINAL VAL-ONLY CALIBRATION ===
Source: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/zone_level_calibration_MOT20-05.csv
Validation frames: 1–1657
TEST data used for calibration: NO

=== VAL REPRODUCTION ===
VAL records: 792
Own-level agreement: 100.00%
Floor-level agreement: 100.00%

=== ORIGINAL THRESHOLDS ===
 zone  own_low_max  own_high_min  person_area_px
    0         16.0          23.0       8319.3504
    1         16.0          25.0       7593.4540
    2         21.0          30.0       7428.6221
    3         20.0          25.0       7705.4643
    4         19.0          30.0      10162.6777
    5         19.0          23.0       9977.8615
    6         18.0          28.0      10564.0335
    7         19.0          28.0       9926.7160
    8         12.0          18.0      12987.3265
    9         15.0          19.0      12975.0704
   10         15.0          20.0      12635.0275
   11         13.0          17.0      12907.4920

Floor LOW threshold: 1.1345
Fl

In [ ]:
# CELL 61 — APPLY ORIGINAL VAL CALIBRATION TO TEST

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
ZONES = ROOT / "zones"
OUT = ZONES / "optimization_v2"

# Use the saved monitoring output BEFORE the previous
# TEST-derived level reconstruction.
FINAL_FILE = OUT / "FINAL_zone_monitoring_TEST.csv"
CALIB_FILE = ZONES / "zone_level_calibration_MOT20-05.csv"

final = pd.read_csv(FINAL_FILE)
calib = pd.read_csv(CALIB_FILE)

assert len(final) == 792
assert not final.duplicated(["step", "zone"]).any()
assert final["step"].between(67, 132).all()

# Remove old reference labels, not the original counts/flow.
final = final.drop(
    columns=["existing_level_reference",
             "level_own", "level_floor",
             "floor_coverage", "crowd_alert"],
    errors="ignore"
)

# Attach thresholds calculated in CELL 19 using VAL only.
calib_cols = [
    "zone", "own_low_max", "own_high_min",
    "person_area_px", "zone_area_px",
    "floor_low_max", "floor_high_min"
]

calib = calib[calib_cols].rename(
    columns={"zone_area_px": "calibrated_zone_area_px"}
)

final = final.merge(
    calib,
    on="zone",
    validate="many_to_one"
)

assert len(final) == 792

# Verify zone geometry matches original calibration.
assert np.allclose(
    final["zone_area_px"],
    final["calibrated_zone_area_px"]
), "Zone areas differ from original calibration."

# Recalculate floor coverage using original VAL person sizes.
final["floor_coverage"] = (
    final["count"]
    * final["person_area_px"]
    / final["calibrated_zone_area_px"]
)

# IMPORTANT:
# Original level_own is based on COUNT, not density.
final["level_own"] = np.where(
    final["count"] >= final["own_high_min"],
    "HIGH",
    np.where(
        final["count"] <= final["own_low_max"],
        "LOW",
        "MEDIUM"
    )
)

# Original level_floor uses VAL-calibrated coverage.
final["level_floor"] = np.where(
    final["floor_coverage"] >= final["floor_high_min"],
    "HIGH",
    np.where(
        final["floor_coverage"] <= final["floor_low_max"],
        "LOW",
        "MEDIUM"
    )
)

# Experimental alerts — not safety-certified.
final["crowd_alert"] = np.where(
    (final["level_own"] == "HIGH") |
    (final["level_floor"] == "HIGH"),
    "HIGH_CROWD",
    "NORMAL"
)

# Validation
assert final["level_own"].isin(
    ["LOW", "MEDIUM", "HIGH"]
).all()

assert final["level_floor"].isin(
    ["LOW", "MEDIUM", "HIGH"]
).all()

assert final["count"].ge(0).all()
assert final["floor_coverage"].ge(0).all()
assert final["inflow"].sum() == 1234
assert final["outflow"].sum() == 1234

# Save separately — never overwrite original experiments.
OUTPUT_FILE = OUT / "FINAL_zone_monitoring_VAL_ONLY_TEST.csv"
SUMMARY_FILE = OUT / "FINAL_crowd_alerts_VAL_ONLY_TEST.json"

final.to_csv(OUTPUT_FILE, index=False)

summary = {
    "status": "VAL_ONLY_CALIBRATION_APPLIED",
    "records": len(final),
    "calibration_source": str(CALIB_FILE),
    "threshold_selection_split": "VAL",
    "test_threshold_tuning": False,
    "own_levels": final["level_own"].value_counts().to_dict(),
    "floor_levels": final["level_floor"].value_counts().to_dict(),
    "alerts": final["crowd_alert"].value_counts().to_dict(),
    "occupancy_tracker": "REF_001",
    "flow_tracker": "BoTSORT_baseline",
    "flow_stability_frames": 5
}

with open(SUMMARY_FILE, "w") as f:
    json.dump(summary, f, indent=2)

print("=== CORRECTED CROWD MONITORING ===")
print(json.dumps(summary, indent=2))

print("\n=== SAMPLE ===")
print(
    final[
        ["step", "zone", "count", "floor_coverage",
         "level_own", "level_floor", "crowd_alert"]
    ].head(12).to_string(index=False)
)

print("\nSaved:", OUTPUT_FILE)
print("Saved:", SUMMARY_FILE)
print("Original files preserved.")
print("No training or inference rerun.")

=== CORRECTED CROWD MONITORING ===
{
  "status": "VAL_ONLY_CALIBRATION_APPLIED",
  "records": 792,
  "calibration_source": "/content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/zone_level_calibration_MOT20-05.csv",
  "threshold_selection_split": "VAL",
  "test_threshold_tuning": false,
  "own_levels": {
    "LOW": 407,
    "MEDIUM": 283,
    "HIGH": 102
  },
  "floor_levels": {
    "LOW": 410,
    "MEDIUM": 297,
    "HIGH": 85
  },
  "alerts": {
    "NORMAL": 670,
    "HIGH_CROWD": 122
  },
  "occupancy_tracker": "REF_001",
  "flow_tracker": "BoTSORT_baseline",
  "flow_stability_frames": 5
}

=== SAMPLE ===
 step  zone  count  floor_coverage level_own level_floor crowd_alert
   67     0  15.88        0.886415       LOW         LOW      NORMAL
   67     1  26.76        1.366699      HIGH      MEDIUM  HIGH_CROWD
   67     2  28.56        1.426967    MEDIUM      MEDIUM      NORMAL
   67     3   9.76        0.504598       LOW         LOW      NORMAL
   67     4  20.48        1.396482  

In [ ]:
# CELL 62 — VALIDATE CORRECTED VAL-ONLY PIPELINE

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "zones/optimization_v2"

final = pd.read_csv(OUT / "FINAL_zone_monitoring_VAL_ONLY_TEST.csv")
calib = pd.read_csv(
    ROOT / "zones/zone_level_calibration_MOT20-05.csv"
)
stable = pd.read_csv(
    OUT / "VAL_SELECTED_stable_flow_TEST.csv"
)
gt = pd.read_csv(OUT / "GT_zone_flow_TEST.csv")

checks = {}

# Structure
checks["792 records"] = len(final) == 792
checks["66 TEST windows"] = set(final.step) == set(range(67, 133))
checks["12 zones per window"] = (
    final.groupby("step").zone.nunique().eq(12).all()
)
checks["No duplicate step-zone pairs"] = (
    not final.duplicated(["step", "zone"]).any()
)

# VAL-only calibration integrity
c = calib.set_index("zone").loc[final["zone"]]

expected_own = np.where(
    final["count"].to_numpy() >= c["own_high_min"].to_numpy(),
    "HIGH",
    np.where(
        final["count"].to_numpy() <= c["own_low_max"].to_numpy(),
        "LOW", "MEDIUM"
    )
)

expected_coverage = (
    final["count"].to_numpy()
    * c["person_area_px"].to_numpy()
    / c["zone_area_px"].to_numpy()
)

expected_floor = np.where(
    expected_coverage >= c["floor_high_min"].to_numpy(),
    "HIGH",
    np.where(
        expected_coverage <= c["floor_low_max"].to_numpy(),
        "LOW", "MEDIUM"
    )
)

checks["Own levels match VAL thresholds"] = (
    final["level_own"].to_numpy() == expected_own
).all()

checks["Floor coverage matches VAL formula"] = np.allclose(
    final["floor_coverage"].to_numpy(),
    expected_coverage,
    atol=1e-8
)

checks["Floor levels match VAL thresholds"] = (
    final["level_floor"].to_numpy() == expected_floor
).all()

expected_alert = np.where(
    (expected_own == "HIGH") | (expected_floor == "HIGH"),
    "HIGH_CROWD", "NORMAL"
)

checks["Alerts match calibrated levels"] = (
    final["crowd_alert"].to_numpy() == expected_alert
).all()

# Flow integrity
flow_cols = ["step", "zone", "inflow", "outflow"]

a = final[flow_cols].sort_values(["step", "zone"]).reset_index(drop=True)
b = stable[flow_cols].sort_values(["step", "zone"]).reset_index(drop=True)

checks["Stabilized flow unchanged"] = a.equals(b)
checks["Inflow total 1234"] = final["inflow"].sum() == 1234
checks["Outflow total 1234"] = final["outflow"].sum() == 1234

# TEST flow metrics
compare = gt[flow_cols].merge(
    final[flow_cols],
    on=["step", "zone"],
    suffixes=("_gt", "_pred"),
    validate="one_to_one"
)

checks["GT alignment complete"] = len(compare) == 792

inflow_mae = (
    compare["inflow_pred"] - compare["inflow_gt"]
).abs().mean()

outflow_mae = (
    compare["outflow_pred"] - compare["outflow_gt"]
).abs().mean()

checks["Inflow MAE 0.7008"] = abs(inflow_mae - 0.7008) < 1e-4
checks["Outflow MAE 0.7008"] = abs(outflow_mae - 0.7008) < 1e-4

print("=== CORRECTED PIPELINE VALIDATION ===")

for name, passed in checks.items():
    print(("PASS" if passed else "FAIL"), "|", name)

print(f"\nPassed: {sum(checks.values())}/{len(checks)}")

assert all(checks.values()), "Validation failed."

report = {
    "status": "PASSED",
    "validation_checks": len(checks),
    "calibration_split": "VAL",
    "test_used_for_calibration": False,
    "inflow_MAE": float(inflow_mae),
    "outflow_MAE": float(outflow_mae),
    "high_crowd_alerts": int(
        (final["crowd_alert"] == "HIGH_CROWD").sum()
    ),
    "occupancy_tracker": "REF_001",
    "flow_tracker": "BoTSORT_baseline",
    "limitations": [
        "Hybrid tracker configuration",
        "TEST inspected during earlier development",
        "Single-camera temporal TEST split",
        "Experimental crowd thresholds"
    ]
}

path = OUT / "FINAL_pipeline_validation_VAL_ONLY_TEST.json"

with open(path, "w") as f:
    json.dump(report, f, indent=2)

print("\nSaved:", path)
print("Original files unchanged.")

=== CORRECTED PIPELINE VALIDATION ===
PASS | 792 records
PASS | 66 TEST windows
PASS | 12 zones per window
PASS | No duplicate step-zone pairs
PASS | Own levels match VAL thresholds
PASS | Floor coverage matches VAL formula
PASS | Floor levels match VAL thresholds
PASS | Alerts match calibrated levels
PASS | Stabilized flow unchanged
PASS | Inflow total 1234
PASS | Outflow total 1234
PASS | GT alignment complete
PASS | Inflow MAE 0.7008
PASS | Outflow MAE 0.7008

Passed: 14/14

Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/optimization_v2/FINAL_pipeline_validation_VAL_ONLY_TEST.json
Original files unchanged.


In [ ]:
# CELL 63 — CORRECTED Z5 / STATIC-PERSON AUDIT

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "zones/optimization_v2"

GT_FILE = ROOT / "tracking/trackeval/gt/MOT20-05-test/gt/gt.txt"
PRED_FILE = OUT / "REF_001_zone_occupancy_TEST.csv"

cols = [
    "frame", "id", "x", "y", "w", "h",
    "mark", "class_id", "visibility"
]

gt = pd.read_csv(GT_FILE, header=None, names=cols)
pred = pd.read_csv(PRED_FILE)

# 1. Inspect ALL GT classes before filtering
print("=== COMPLETE GT CLASS AUDIT ===")
print(pd.crosstab(gt["class_id"], gt["mark"]))

static_raw = gt[gt["class_id"] == 7]
static_marked = static_raw[static_raw["mark"] == 1]

print("\nClass-7 observations (all marks):", len(static_raw))
print("Class-7 observations (mark=1):", len(static_marked))

# 2. Assign zones using bounding-box bottom-center
def assign_zones(data):
    data = data.copy()

    px = np.floor(data["x"] + data["w"] / 2).astype(int)
    py = np.floor(data["y"] + data["h"]).astype(int)

    valid = px.between(0, 1653) & py.between(0, 1079)

    col = np.select(
        [px <= 413, px <= 826, px <= 1239],
        [0, 1, 2],
        default=3
    )

    row = np.select(
        [py <= 359, py <= 719],
        [0, 1],
        default=2
    )

    data["zone"] = -1
    data.loc[valid, "zone"] = (row * 4 + col)[valid]

    return data

ped = assign_zones(
    gt[(gt["class_id"] == 1) & (gt["mark"] == 1)]
)

visible = ped[ped["visibility"] >= 0.25].copy()
static = assign_zones(static_marked)

# 3. Align local TEST frames with global time windows
frame_map = pd.DataFrame({"frame": range(1, 1659)})

frame_map["step"] = (
    frame_map["frame"] + 1656
) // 25

frame_map = frame_map[
    frame_map["step"].between(67, 132)
]

idx = pd.MultiIndex.from_product(
    [frame_map["frame"], range(12)],
    names=["frame", "zone"]
)

def zone_counts(data, name):
    counts = (
        data[data["zone"].between(0, 11)]
        .groupby(["frame", "zone"])
        .size()
        .reindex(idx, fill_value=0)
        .rename("count")
        .reset_index()
        .merge(frame_map, on="frame", validate="many_to_one")
    )

    return (
        counts.groupby(["step", "zone"])["count"]
        .mean()
        .rename(name)
        .reset_index()
    )

# 4. Compare against different GT references
result = pred[
    pred["step"].between(67, 132)
][["step", "zone", "count"]].copy()

for data, name in [
    (ped, "ped_gt"),
    (visible, "visible_gt"),
    (static, "marked_static_gt")
]:
    result = result.merge(
        zone_counts(data, name),
        on=["step", "zone"],
        validate="one_to_one"
    )

assert len(result) == 792

result["ped_plus_static_gt"] = (
    result["ped_gt"] + result["marked_static_gt"]
)

# 5. Calculate counting errors
for ref in [
    "ped_gt",
    "visible_gt",
    "ped_plus_static_gt"
]:
    result[f"MAE_{ref}"] = (
        result["count"] - result[ref]
    ).abs()

    result[f"bias_{ref}"] = (
        result["count"] - result[ref]
    )

summary = result.groupby("zone").agg(
    predicted_mean=("count", "mean"),
    ped_MAE=("MAE_ped_gt", "mean"),
    visible_MAE=("MAE_visible_gt", "mean"),
    ped_static_MAE=("MAE_ped_plus_static_gt", "mean"),
    ped_bias=("bias_ped_gt", "mean"),
    marked_static_mean=("marked_static_gt", "mean")
)

print("\n=== Z5 RESULTS ===")
print(summary.loc[[5]].round(3).to_string())

print("\n=== ALL ZONES ===")
print(summary.round(3).to_string())

if len(static_marked) == 0:
    print("\nWARNING: No marked class-7 GT observations.")
    print("Static-person hypothesis not confirmed.")

# 6. Save diagnostic results
summary.to_csv(
    OUT / "Z5_corrected_class_audit_TEST.csv"
)

result.to_csv(
    OUT / "Z5_corrected_count_comparison_TEST.csv",
    index=False
)

print("\nSaved diagnostic results.")
print("No inference or retraining.")
print("Original files unchanged.")

=== COMPLETE GT CLASS AUDIT ===
mark          0       1
class_id               
1             0  319276
6          4247       0
7         44651       0
11         3316       0

Class-7 observations (all marks): 44651
Class-7 observations (mark=1): 0

=== Z5 RESULTS ===
      predicted_mean  ped_MAE  visible_MAE  ped_static_MAE  ped_bias  marked_static_mean
zone                                                                                    
5             22.723    4.527        9.165           4.527     4.435                 0.0

=== ALL ZONES ===
      predicted_mean  ped_MAE  visible_MAE  ped_static_MAE  ped_bias  marked_static_mean
zone                                                                                    
0             11.886    1.317        1.929           1.317    -0.494                 0.0
1             15.214    1.411        2.463           1.411    -0.703                 0.0
2             23.536    1.904        7.147           1.904     0.392                 0.0

In [ ]:
# CELL 64 — TEST STATIC-PERSON HYPOTHESIS (DIAGNOSTIC ONLY)

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "zones/optimization_v2"

GT_FILE = ROOT / "tracking/trackeval/gt/MOT20-05-test/gt/gt.txt"
PRED_FILE = OUT / "REF_001_zone_occupancy_TEST.csv"

cols = [
    "frame", "id", "x", "y", "w", "h",
    "mark", "class_id", "visibility"
]

gt = pd.read_csv(GT_FILE, header=None, names=cols)
pred = pd.read_csv(PRED_FILE)

# Class 1 = official pedestrian GT
# Class 7 = static-person diagnostic annotations
ped = gt[(gt.class_id == 1) & (gt.mark == 1)].copy()
static = gt[gt.class_id == 7].copy()

def assign_zones(data):
    data = data.copy()

    px = np.floor(data.x + data.w / 2).astype(int)
    py = np.floor(data.y + data.h).astype(int)

    valid = px.between(0, 1653) & py.between(0, 1079)

    col = np.select(
        [px <= 413, px <= 826, px <= 1239],
        [0, 1, 2],
        default=3
    )

    row = np.select(
        [py <= 359, py <= 719],
        [0, 1],
        default=2
    )

    data["zone"] = -1
    data.loc[valid, "zone"] = (row * 4 + col)[valid]

    return data

ped = assign_zones(ped)
static = assign_zones(static)

# Align TEST local frames with global steps
frame_map = pd.DataFrame({"frame": range(1, 1659)})
frame_map["step"] = (frame_map.frame + 1656) // 25
frame_map = frame_map[frame_map.step.between(67, 132)]

idx = pd.MultiIndex.from_product(
    [frame_map.frame, range(12)],
    names=["frame", "zone"]
)

def aggregate_counts(data, name):
    counts = (
        data[data.zone.between(0, 11)]
        .groupby(["frame", "zone"])
        .size()
        .reindex(idx, fill_value=0)
        .rename("count")
        .reset_index()
        .merge(frame_map, on="frame")
    )

    return (
        counts.groupby(["step", "zone"])["count"]
        .mean()
        .rename(name)
        .reset_index()
    )

result = pred[
    pred.step.between(67, 132)
][["step", "zone", "count"]].copy()

result = result.merge(
    aggregate_counts(ped, "ped_gt"),
    on=["step", "zone"]
)

result = result.merge(
    aggregate_counts(static, "static_gt"),
    on=["step", "zone"]
)

assert len(result) == 792

result["ped_plus_static"] = (
    result["ped_gt"] + result["static_gt"]
)

result["error_official"] = (
    result["count"] - result["ped_gt"]
).abs()

result["error_diagnostic"] = (
    result["count"] - result["ped_plus_static"]
).abs()

summary = result.groupby("zone").agg(
    predicted_mean=("count", "mean"),
    pedestrian_mean=("ped_gt", "mean"),
    static_mean=("static_gt", "mean"),
    official_MAE=("error_official", "mean"),
    diagnostic_MAE=("error_diagnostic", "mean")
)

summary["MAE_change"] = (
    summary["diagnostic_MAE"] - summary["official_MAE"]
)

print("=== Z5 STATIC-PERSON HYPOTHESIS ===")
print(summary.loc[[5]].round(3).to_string())

print("\n=== ALL ZONES ===")
print(summary.round(3).to_string())

print("\nInterpretation:")
print("Negative MAE_change = diagnostic reference reduces error.")
print("Positive MAE_change = diagnostic reference increases error.")

summary.to_csv(OUT / "Z5_static_hypothesis_TEST.csv")
result.to_csv(
    OUT / "Z5_static_hypothesis_details_TEST.csv",
    index=False
)

print("\nSaved diagnostic results.")
print("Official GT unchanged.")
print("No retraining or inference.")

=== Z5 STATIC-PERSON HYPOTHESIS ===
      predicted_mean  pedestrian_mean  static_mean  official_MAE  diagnostic_MAE  MAE_change
zone                                                                                        
5             22.723           18.287        6.304         4.527           2.005      -2.523

=== ALL ZONES ===
      predicted_mean  pedestrian_mean  static_mean  official_MAE  diagnostic_MAE  MAE_change
zone                                                                                        
0             11.886           12.380        1.444         1.317           2.004       0.687
1             15.214           15.917        0.146         1.411           1.557       0.146
2             23.536           23.144        2.117         1.904           2.331       0.427
3             19.512           16.541        3.872         3.124           1.620      -1.505
4             13.051           10.778        3.763         2.273           1.613      -0.660
5             2

In [ ]:
# CELL 65 — MATCH REF_001 TRACKS TO CLASS-7 GT
# Diagnostic only; no model changes or inference.

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "zones/optimization_v2"

GT_FILE = ROOT / "tracking/trackeval/gt/MOT20-05-test/gt/gt.txt"
TRACK_FILE = (
    ROOT / "tracking/trackeval/trackers/test/"
    "OPT_V2_REF_001_TEST/data/MOT20-05-test.txt"
)

gt_cols = [
    "frame", "id", "x", "y", "w", "h",
    "mark", "class_id", "visibility"
]

track_cols = [
    "frame", "id", "x", "y", "w", "h",
    "conf", "a", "b", "c"
]

gt = pd.read_csv(GT_FILE, header=None, names=gt_cols)
tracks = pd.read_csv(
    TRACK_FILE, header=None, names=track_cols
)

def add_zone(df):
    df = df.copy()

    px = np.floor(df.x + df.w / 2).astype(int)
    py = np.floor(df.y + df.h).astype(int)

    valid = (
        px.between(0, 1653) &
        py.between(0, 1079)
    )

    col = np.select(
        [px <= 413, px <= 826, px <= 1239],
        [0, 1, 2], default=3
    )
    row = np.select(
        [py <= 359, py <= 719],
        [0, 1], default=2
    )

    df["zone"] = -1
    df.loc[valid, "zone"] = (row * 4 + col)[valid]

    return df

static = add_zone(gt[gt.class_id == 7])
tracks = add_zone(tracks)

def xyxy(df):
    a = df[["x", "y", "w", "h"]].to_numpy(float)
    return np.column_stack([
        a[:, 0], a[:, 1],
        a[:, 0] + a[:, 2],
        a[:, 1] + a[:, 3]
    ])

def max_iou_per_gt(gt_boxes, pred_boxes):
    if len(gt_boxes) == 0:
        return np.array([])

    if len(pred_boxes) == 0:
        return np.zeros(len(gt_boxes))

    x1 = np.maximum(gt_boxes[:, None, 0],
                    pred_boxes[None, :, 0])
    y1 = np.maximum(gt_boxes[:, None, 1],
                    pred_boxes[None, :, 1])
    x2 = np.minimum(gt_boxes[:, None, 2],
                    pred_boxes[None, :, 2])
    y2 = np.minimum(gt_boxes[:, None, 3],
                    pred_boxes[None, :, 3])

    intersection = (
        np.maximum(0, x2 - x1) *
        np.maximum(0, y2 - y1)
    )

    area_gt = (
        (gt_boxes[:, 2] - gt_boxes[:, 0]) *
        (gt_boxes[:, 3] - gt_boxes[:, 1])
    )

    area_pred = (
        (pred_boxes[:, 2] - pred_boxes[:, 0]) *
        (pred_boxes[:, 3] - pred_boxes[:, 1])
    )

    union = (
        area_gt[:, None] +
        area_pred[None, :] -
        intersection
    )

    iou = intersection / np.maximum(union, 1e-9)
    return iou.max(axis=1)

# Group by local TEST frame
pred_groups = {
    int(f): xyxy(g)
    for f, g in tracks.groupby("frame")
}

results = []

for frame, group in static.groupby("frame"):
    boxes = xyxy(group)
    pred_boxes = pred_groups.get(
        int(frame), np.empty((0, 4))
    )

    scores = max_iou_per_gt(boxes, pred_boxes)

    temp = group[["frame", "id", "zone"]].copy()
    temp["max_iou"] = scores
    temp["matched_03"] = scores >= 0.3
    temp["matched_05"] = scores >= 0.5

    results.append(temp)

matches = pd.concat(results, ignore_index=True)

summary = matches[
    matches.zone.between(0, 11)
].groupby("zone").agg(
    static_annotations=("id", "size"),
    matched_IoU03=("matched_03", "sum"),
    matched_IoU05=("matched_05", "sum"),
    mean_max_IoU=("max_iou", "mean")
)

summary["match_rate_03"] = (
    100 * summary.matched_IoU03 /
    summary.static_annotations
)

summary["match_rate_05"] = (
    100 * summary.matched_IoU05 /
    summary.static_annotations
)

print("=== Z5 CLASS-7 DETECTION MATCHING ===")
print(summary.loc[[5]].round(2).to_string())

print("\n=== ALL ZONES ===")
print(summary.round(2).to_string())

print("\n=== OVERALL ===")
print("Class-7 observations:", len(matches))
print("IoU >= 0.3:", int(matches.matched_03.sum()))
print("IoU >= 0.5:", int(matches.matched_05.sum()))

summary.to_csv(
    OUT / "class7_detection_matching_by_zone_TEST.csv"
)

matches.to_csv(
    OUT / "class7_detection_matching_details_TEST.csv",
    index=False
)

print("\nSaved diagnostic results.")
print("No retraining or inference.")
print("Official GT unchanged.")

=== Z5 CLASS-7 DETECTION MATCHING ===
      static_annotations  matched_IoU03  matched_IoU05  mean_max_IoU  match_rate_03  match_rate_05
zone                                                                                              
5                  10421          10069           9088          0.69          96.62          87.21

=== ALL ZONES ===
      static_annotations  matched_IoU03  matched_IoU05  mean_max_IoU  match_rate_03  match_rate_05
zone                                                                                              
0                   2419           1036            744          0.37          42.83          30.76
1                    231             27              4          0.18          11.69           1.73
2                   3555           2656           2174          0.56          74.71          61.15
3                   6395           6298           5805          0.70          98.48          90.77
4                   6241           5648           54

In [ ]:
# CELL 66 — Z5 ONE-TO-ONE GT ATTRIBUTION
# Diagnostic only. No inference, training, or GT changes.

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "zones/optimization_v2"

GT_FILE = ROOT / "tracking/trackeval/gt/MOT20-05-test/gt/gt.txt"
TRACK_FILE = (
    ROOT / "tracking/trackeval/trackers/test/"
    "OPT_V2_REF_001_TEST/data/MOT20-05-test.txt"
)

gt = pd.read_csv(
    GT_FILE, header=None,
    names=["frame","id","x","y","w","h",
           "mark","class_id","visibility"]
)

pred = pd.read_csv(
    TRACK_FILE, header=None,
    names=["frame","id","x","y","w","h",
           "conf","a","b","c"]
)

def add_zone(df):
    df = df.copy()
    px = np.floor(df.x + df.w / 2).astype(int)
    py = np.floor(df.y + df.h).astype(int)

    valid = px.between(0,1653) & py.between(0,1079)

    col = np.select(
        [px <= 413, px <= 826, px <= 1239],
        [0,1,2], default=3
    )
    row = np.select(
        [py <= 359, py <= 719],
        [0,1], default=2
    )

    df["zone"] = -1
    df.loc[valid,"zone"] = (row*4+col)[valid]
    return df

gt = add_zone(gt)
pred = add_zone(pred)

# Only final aligned TEST windows and Z5
valid_frames = np.arange(1,1659)
valid_frames = valid_frames[
    ((valid_frames + 1656)//25 >= 67)
]

ped = gt[
    (gt.zone == 5) &
    (gt.class_id == 1) &
    (gt.mark == 1)
]

static = gt[
    (gt.zone == 5) &
    (gt.class_id == 7)
]

pred = pred[pred.zone == 5]

def boxes(df):
    a = df[["x","y","w","h"]].to_numpy(float)
    return np.column_stack([
        a[:,0], a[:,1],
        a[:,0]+a[:,2], a[:,1]+a[:,3]
    ])

def iou_matrix(a,b):
    if len(a)==0 or len(b)==0:
        return np.zeros((len(a),len(b)))

    x1=np.maximum(a[:,None,0],b[None,:,0])
    y1=np.maximum(a[:,None,1],b[None,:,1])
    x2=np.minimum(a[:,None,2],b[None,:,2])
    y2=np.minimum(a[:,None,3],b[None,:,3])

    inter=np.maximum(0,x2-x1)*np.maximum(0,y2-y1)
    aa=(a[:,2]-a[:,0])*(a[:,3]-a[:,1])
    bb=(b[:,2]-b[:,0])*(b[:,3]-b[:,1])

    return inter/np.maximum(aa[:,None]+bb[None,:]-inter,1e-9)

def match(gt_boxes,pred_boxes,threshold=0.5):
    matrix=iou_matrix(gt_boxes,pred_boxes)
    if matrix.size==0:
        return [],[]

    score=np.where(matrix>=threshold,matrix,0)
    rows,cols=linear_sum_assignment(-score)

    valid=score[rows,cols]>=threshold
    return rows[valid].tolist(),cols[valid].tolist()

ped_groups={f:g for f,g in ped.groupby("frame")}
static_groups={f:g for f,g in static.groupby("frame")}
pred_groups={f:g for f,g in pred.groupby("frame")}

records=[]

for frame in valid_frames:
    p=pred_groups.get(frame,pred.iloc[:0])
    g=ped_groups.get(frame,ped.iloc[:0])
    s=static_groups.get(frame,static.iloc[:0])

    pb=boxes(p)
    gb=boxes(g)
    sb=boxes(s)

    # First match official pedestrian GT
    _,ped_matches=match(gb,pb)

    remaining=[
        i for i in range(len(pb))
        if i not in set(ped_matches)
    ]

    # Then match excluded static-person annotations
    _,static_matches=match(sb,pb[remaining])
    static_pred_indices=[
        remaining[i] for i in static_matches
    ]

    records.append({
        "frame":int(frame),
        "step":int((frame+1656)//25),
        "predictions":len(pb),
        "ped_matches":len(ped_matches),
        "static_matches":len(static_pred_indices),
        "unmatched":len(pb)-len(ped_matches)-len(static_pred_indices),
        "ped_gt":len(gb),
        "static_gt":len(sb)
    })

result=pd.DataFrame(records)

assert (
    result.ped_matches+
    result.static_matches+
    result.unmatched == result.predictions
).all()

print("=== Z5 ONE-TO-ONE ATTRIBUTION ===")
print(result[
    ["predictions","ped_matches","static_matches",
     "unmatched","ped_gt","static_gt"]
].sum().to_string())

total=result.predictions.sum()

print("\n=== PREDICTION DISTRIBUTION ===")
for name in ["ped_matches","static_matches","unmatched"]:
    print(f"{name}: {100*result[name].sum()/total:.2f}%")

print("\n=== PER-FRAME AVERAGES ===")
print(result[
    ["predictions","ped_matches","static_matches",
     "unmatched"]
].mean().round(3).to_string())

result.to_csv(
    OUT/"Z5_one_to_one_attribution_TEST.csv",
    index=False
)

print("\nSaved diagnostic results.")
print("Official GT unchanged.")
print("No inference or training.")

=== Z5 ONE-TO-ONE ATTRIBUTION ===
predictions       37277
ped_matches       26679
static_matches     8411
unmatched          2187
ped_gt            30020
static_gt         10331

=== PREDICTION DISTRIBUTION ===
ped_matches: 71.57%
static_matches: 22.56%
unmatched: 5.87%

=== PER-FRAME AVERAGES ===
predictions       22.730
ped_matches       16.268
static_matches     5.129
unmatched          1.334

Saved diagnostic results.
Official GT unchanged.
No inference or training.


In [ ]:
# CELL 67 — FIVE-SECOND ZONE FLOW EVALUATION

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "zones/optimization_v2"

PRED_FILE = OUT / "VAL_SELECTED_stable_flow_TEST.csv"
GT_FILE = OUT / "GT_zone_flow_TEST.csv"

pred = pd.read_csv(PRED_FILE)
gt = pd.read_csv(GT_FILE)

cols = ["step", "zone", "inflow", "outflow"]

pred = pred[cols].copy()
gt = gt[cols].copy()

# 67–131: 65 complete one-second windows.
# Exclude step 132 because it contains only 15 frames.
steps = range(67, 132)

pred = pred[pred.step.isin(steps)]
gt = gt[gt.step.isin(steps)]

assert len(pred) == len(gt) == 65 * 12
assert not pred.duplicated(["step", "zone"]).any()
assert not gt.duplicated(["step", "zone"]).any()

merged = pred.merge(
    gt,
    on=["step", "zone"],
    suffixes=("_pred", "_gt"),
    validate="one_to_one"
)

assert len(merged) == 780

# Group 5 consecutive seconds into one window.
merged["window_5s"] = (merged["step"] - 67) // 5

aggregated = (
    merged.groupby(["window_5s", "zone"], as_index=False)
    [["inflow_pred", "inflow_gt",
      "outflow_pred", "outflow_gt"]]
    .sum()
)

assert len(aggregated) == 13 * 12

def metrics(data):
    result = {}
    for direction in ["inflow", "outflow"]:
        error = (
            data[f"{direction}_pred"] -
            data[f"{direction}_gt"]
        )
        result[direction] = {
            "MAE": float(error.abs().mean()),
            "RMSE": float(np.sqrt((error ** 2).mean())),
            "bias": float(error.mean())
        }
    return result

one_second = metrics(merged)
five_second = metrics(aggregated)

print("=== ONE-SECOND FLOW ===")
print(json.dumps(one_second, indent=2))

print("\n=== FIVE-SECOND FLOW ===")
print(json.dumps(five_second, indent=2))

print("\n=== NORMALIZED MAE PER SECOND ===")
for direction in ["inflow", "outflow"]:
    a = one_second[direction]["MAE"]
    b = five_second[direction]["MAE"] / 5
    print(
        f"{direction}: "
        f"1s={a:.4f}, "
        f"5s_normalized={b:.4f}, "
        f"change={(b/a-1)*100:+.2f}%"
    )

summary = {
    "split": "TEST",
    "selected_stability_frames": 5,
    "complete_steps": [67, 131],
    "excluded_partial_step": 132,
    "one_second": one_second,
    "five_second": five_second,
    "note": (
        "5-second counts are sums of five 1-second "
        "windows. MAE/5 is a normalized comparison, "
        "not a replacement for event-level accuracy."
    )
}

aggregated.to_csv(
    OUT / "FINAL_5second_zone_flow_TEST.csv",
    index=False
)

with open(
    OUT / "FINAL_5second_zone_flow_summary_TEST.json",
    "w"
) as f:
    json.dump(summary, f, indent=2)

print("\nSaved 5-second evaluation.")
print("Original files unchanged.")
print("No TEST tuning or inference rerun.")

=== ONE-SECOND FLOW ===
{
  "inflow": {
    "MAE": 0.7038461538461539,
    "RMSE": 1.1053483033844453,
    "bias": 0.08846153846153847
  },
  "outflow": {
    "MAE": 0.7012820512820512,
    "RMSE": 1.0854515780103358,
    "bias": 0.08846153846153847
  }
}

=== FIVE-SECOND FLOW ===
{
  "inflow": {
    "MAE": 1.6217948717948718,
    "RMSE": 2.167652635621757,
    "bias": 0.4423076923076923
  },
  "outflow": {
    "MAE": 1.5320512820512822,
    "RMSE": 2.0048019276803304,
    "bias": 0.4423076923076923
  }
}

=== NORMALIZED MAE PER SECOND ===
inflow: 1s=0.7038, 5s_normalized=0.3244, change=-53.92%
outflow: 1s=0.7013, 5s_normalized=0.3064, change=-56.31%

Saved 5-second evaluation.
Original files unchanged.
No TEST tuning or inference rerun.


In [ ]:
# CELL 68 — VAL OCCUPANCY: BASELINE VS REF_001
# Reuse saved predictions. No inference or training.

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "zones/optimization_v2"

BASE = ROOT / "tracking/trackeval/trackers/val"

FILES = {
    "GT": ROOT / "tracking/trackeval/gt/MOT20-05-val/gt/gt.txt",
    "BoTSORT_baseline": BASE / "BoTSORT_candidate/data/MOT20-05-val.txt",
    "REF_001": BASE / "OPT_V2_REF_001/data/MOT20-05-val.txt"
}

def load_boxes(path, is_gt=False):
    assert path.exists(), f"Missing: {path}"
    df = pd.read_csv(path, header=None)

    df = df.iloc[:, :6].copy()
    df.columns = ["frame", "id", "x", "y", "w", "h"]

    if is_gt:
        raw = pd.read_csv(path, header=None)
        df = df[
            (raw.iloc[:, 6].to_numpy() == 1) &
            (raw.iloc[:, 7].to_numpy() == 1)
        ].copy()

    # VAL steps 0–65: complete 25-frame windows
    df = df[df.frame.between(1, 1650)].copy()

    px = np.floor(df.x + df.w / 2).astype(int)
    py = np.floor(df.y + df.h).astype(int)

    valid = px.between(0, 1653) & py.between(0, 1079)

    col = np.select(
        [px <= 413, px <= 826, px <= 1239],
        [0, 1, 2], default=3
    )
    row = np.select(
        [py <= 359, py <= 719],
        [0, 1], default=2
    )

    df["zone"] = -1
    df.loc[valid, "zone"] = (row * 4 + col)[valid]

    return df

idx = pd.MultiIndex.from_product(
    [range(1, 1651), range(12)],
    names=["frame", "zone"]
)

def occupancy(df, name):
    counts = (
        df[df.zone.between(0, 11)]
        .groupby(["frame", "zone"])
        .size()
        .reindex(idx, fill_value=0)
        .rename("count")
        .reset_index()
    )

    counts["step"] = (counts.frame - 1) // 25

    return (
        counts.groupby(["step", "zone"])["count"]
        .mean()
        .rename(name)
        .reset_index()
    )

combined = None

for name, path in FILES.items():
    counts = occupancy(
        load_boxes(path, is_gt=(name == "GT")),
        name
    )

    combined = (
        counts if combined is None
        else combined.merge(
            counts,
            on=["step", "zone"],
            validate="one_to_one"
        )
    )

assert len(combined) == 66 * 12

print("=== VAL OCCUPANCY ACCURACY ===")

results = []

for tracker in ["BoTSORT_baseline", "REF_001"]:
    error = combined[tracker] - combined["GT"]

    results.append({
        "tracker": tracker,
        "MAE": error.abs().mean(),
        "RMSE": np.sqrt((error ** 2).mean()),
        "bias": error.mean()
    })

metrics = pd.DataFrame(results)
print(metrics.round(4).to_string(index=False))

print("\n=== PER-ZONE MAE ===")
for tracker in ["BoTSORT_baseline", "REF_001"]:
    combined[f"{tracker}_AE"] = (
        combined[tracker] - combined.GT
    ).abs()

print(
    combined.groupby("zone")[
        ["BoTSORT_baseline_AE", "REF_001_AE"]
    ].mean().round(3).to_string()
)

metrics.to_csv(
    OUT / "VAL_tracker_occupancy_comparison.csv",
    index=False
)

combined.to_csv(
    OUT / "VAL_tracker_occupancy_details.csv",
    index=False
)

print("\nSaved VAL occupancy comparison.")
print("TEST not used for tracker selection.")
print("Original files unchanged.")

=== VAL OCCUPANCY ACCURACY ===
         tracker    MAE   RMSE   bias
BoTSORT_baseline 2.2432 2.7477 1.2038
         REF_001 2.3571 2.8939 1.5958

=== PER-ZONE MAE ===
      BoTSORT_baseline_AE  REF_001_AE
zone                                 
0                   2.095       2.053
1                   1.226       1.133
2                   2.805       3.148
3                   2.912       2.925
4                   1.858       1.953
5                   2.973       3.661
6                   1.155       0.979
7                   2.077       2.067
8                   3.695       3.890
9                   2.974       3.279
10                  1.898       1.873
11                  1.252       1.322

Saved VAL occupancy comparison.
TEST not used for tracker selection.
Original files unchanged.


In [ ]:
# CELL 69 — UNIFIED BoT-SORT MONITORING
# Configuration selected using VAL only.

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "zones/optimization_v2"
OUT.mkdir(parents=True, exist_ok=True)

TRACK_FILE = (
    ROOT / "tracking/trackeval/trackers/test/"
    "BoTSORT_candidate/data/MOT20-05-test.txt"
)
FLOW_FILE = OUT / "VAL_SELECTED_stable_flow_TEST.csv"
CALIB_FILE = ROOT / "zones/zone_level_calibration_MOT20-05.csv"

for p in [TRACK_FILE, FLOW_FILE, CALIB_FILE]:
    assert p.exists(), f"Missing file: {p}"

tracks = pd.read_csv(TRACK_FILE, header=None).iloc[:, :6]
tracks.columns = ["frame", "id", "x", "y", "w", "h"]

flow = pd.read_csv(FLOW_FILE)
calib = pd.read_csv(CALIB_FILE)

assert tracks["frame"].between(1, 1658).all()
assert not tracks.duplicated(["frame", "id"]).any()
assert set(calib.zone) == set(range(12))

# 1. Assign zones using bottom-center footpoint
px = np.floor(tracks.x + tracks.w / 2).astype(int)
py = np.floor(tracks.y + tracks.h).astype(int)

valid = px.between(0, 1653) & py.between(0, 1079)

col = np.select(
    [px <= 413, px <= 826, px <= 1239],
    [0, 1, 2],
    default=3
)

row = np.select(
    [py <= 359, py <= 719],
    [0, 1],
    default=2
)

tracks["zone"] = -1
tracks.loc[valid, "zone"] = (row * 4 + col)[valid]

# 2. Align local TEST frames to global steps 67–132
frames = pd.DataFrame({"frame": np.arange(1, 1659)})
frames["step"] = (frames.frame + 1656) // 25
frames = frames[frames.step.between(67, 132)]

idx = pd.MultiIndex.from_product(
    [frames.frame, range(12)],
    names=["frame", "zone"]
)

counts = (
    tracks[tracks.zone.between(0, 11)]
    .groupby(["frame", "zone"])
    .size()
    .reindex(idx, fill_value=0)
    .rename("count")
    .reset_index()
    .merge(frames, on="frame", validate="many_to_one")
)

occupancy = (
    counts.groupby(["step", "zone"], as_index=False)
    .agg(
        count=("count", "mean"),
        max_count=("count", "max"),
        frames_in_window=("frame", "nunique")
    )
)

# 3. Add VAL-selected stabilized flow
flow = flow[
    ["step", "zone", "inflow", "outflow"]
].copy()

flow = flow[flow.step.between(67, 132)]

assert len(flow) == 792
assert not flow.duplicated(["step", "zone"]).any()

final = occupancy.merge(
    flow,
    on=["step", "zone"],
    validate="one_to_one"
)

# 4. Apply original VAL-only calibration
calib_cols = [
    "zone",
    "own_low_max",
    "own_high_min",
    "person_area_px",
    "zone_area_px",
    "floor_low_max",
    "floor_high_min"
]

missing = set(calib_cols) - set(calib.columns)
assert not missing, f"Missing calibration columns: {missing}"

final = final.merge(
    calib[calib_cols],
    on="zone",
    validate="many_to_one"
)

final["density_per_100x100px"] = (
    final["count"] * 10000 / final["zone_area_px"]
)

final["floor_coverage"] = (
    final["count"] * final["person_area_px"]
    / final["zone_area_px"]
)

# 5. Crowd-level classification
final["level_own"] = np.where(
    final["count"] >= final["own_high_min"],
    "HIGH",
    np.where(
        final["count"] <= final["own_low_max"],
        "LOW",
        "MEDIUM"
    )
)

final["level_floor"] = np.where(
    final["floor_coverage"] >= final["floor_high_min"],
    "HIGH",
    np.where(
        final["floor_coverage"] <= final["floor_low_max"],
        "LOW",
        "MEDIUM"
    )
)

final["crowd_alert"] = np.where(
    (final.level_own == "HIGH") |
    (final.level_floor == "HIGH"),
    "HIGH_CROWD",
    "NORMAL"
)

# 6. Validate structure
assert len(final) == 792
assert not final.duplicated(["step", "zone"]).any()
assert final.groupby("step").zone.nunique().eq(12).all()
assert final["count"].le(final["max_count"]).all()
assert final.inflow.sum() == 1234
assert final.outflow.sum() == 1234

# 7. Save without overwriting original outputs
OUTPUT = OUT / "UNIFIED_BoTSORT_monitoring_VAL_SELECTED_TEST.csv"
final.to_csv(OUTPUT, index=False)

summary = {
    "configuration_selected_using": "VAL",
    "occupancy_tracker": "BoTSORT_baseline",
    "flow_tracker": "BoTSORT_baseline",
    "flow_stability_frames": 5,
    "calibration": "Original VAL-only calibration",
    "records": len(final),
    "mean_occupancy": float(final["count"].mean()),
    "total_inflow": int(final.inflow.sum()),
    "total_outflow": int(final.outflow.sum()),
    "alerts": final.crowd_alert.value_counts().to_dict(),
    "note": (
        "TEST output generated using VAL-selected settings. "
        "Previous TEST inspection means this remains "
        "a development-stage evaluation."
    )
}

SUMMARY_FILE = OUT / "UNIFIED_BoTSORT_summary_TEST.json"

with open(SUMMARY_FILE, "w") as f:
    json.dump(summary, f, indent=2)

print("=== UNIFIED MONITORING ===")
print(json.dumps(summary, indent=2))

print("\nSaved:", OUTPUT)
print("Saved:", SUMMARY_FILE)
print("Original files unchanged.")
print("No training or inference rerun.")

=== UNIFIED MONITORING ===
{
  "configuration_selected_using": "VAL",
  "occupancy_tracker": "BoTSORT_baseline",
  "flow_tracker": "BoTSORT_baseline",
  "flow_stability_frames": 5,
  "calibration": "Original VAL-only calibration",
  "records": 792,
  "mean_occupancy": 16.494427609427607,
  "total_inflow": 1234,
  "total_outflow": 1234,
  "alerts": {
    "NORMAL": 683,
    "HIGH_CROWD": 109
  },
  "note": "TEST output generated using VAL-selected settings. Previous TEST inspection means this remains a development-stage evaluation."
}

Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/optimization_v2/UNIFIED_BoTSORT_monitoring_VAL_SELECTED_TEST.csv
Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/zones/optimization_v2/UNIFIED_BoTSORT_summary_TEST.json
Original files unchanged.
No training or inference rerun.


In [ ]:
# CELL 70 — VALIDATE UNIFIED BoT-SORT MONITORING
# Read-only validation of existing outputs; no inference or tuning.

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "zones/optimization_v2"

FINAL_FILE = OUT / "UNIFIED_BoTSORT_monitoring_VAL_SELECTED_TEST.csv"
FLOW_FILE = OUT / "VAL_SELECTED_stable_flow_TEST.csv"
CALIB_FILE = ROOT / "zones/zone_level_calibration_MOT20-05.csv"

final = pd.read_csv(FINAL_FILE)
flow = pd.read_csv(FLOW_FILE)
calib = pd.read_csv(CALIB_FILE)

checks = {}

# 1. Structure
checks["792 zone-window records"] = len(final) == 792
checks["66 TEST windows"] = set(final.step) == set(range(67, 133))
checks["12 zones"] = set(final.zone) == set(range(12))
checks["12 zones per window"] = (
    final.groupby("step").zone.nunique().eq(12).all()
)
checks["No duplicate step-zone keys"] = (
    not final.duplicated(["step", "zone"]).any()
)
checks["No missing numeric values"] = (
    final.select_dtypes(include=np.number).notna().all().all()
)
checks["Nonnegative counts and flow"] = (
    final[["count", "max_count", "inflow", "outflow"]]
    .ge(0).all().all()
)
checks["Mean count <= maximum count"] = (
    final["count"] <= final["max_count"] + 1e-9
).all()

# 2. Compare against saved stabilized flow
keys = ["step", "zone"]
flow_cols = ["inflow", "outflow"]

a = final[keys + flow_cols].sort_values(keys).reset_index(drop=True)
b = flow[
    flow.step.between(67, 132)
][keys + flow_cols].sort_values(keys).reset_index(drop=True)

checks["Flow matches VAL-selected output"] = (
    a.equals(b)
)
checks["Flow totals consistent"] = (
    final.inflow.sum() == flow.loc[
        flow.step.between(67, 132), "inflow"
    ].sum()
    and
    final.outflow.sum() == flow.loc[
        flow.step.between(67, 132), "outflow"
    ].sum()
)

# 3. Reproduce levels using original VAL calibration
assert calib.zone.is_unique
c = calib.set_index("zone").loc[final.zone]

count = final["count"].to_numpy()
area = c["zone_area_px"].to_numpy()
person_area = c["person_area_px"].to_numpy()

expected_density = count * 10000 / area
expected_coverage = count * person_area / area

expected_own = np.where(
    count >= c["own_high_min"].to_numpy(),
    "HIGH",
    np.where(
        count <= c["own_low_max"].to_numpy(),
        "LOW", "MEDIUM"
    )
)

expected_floor = np.where(
    expected_coverage >= c["floor_high_min"].to_numpy(),
    "HIGH",
    np.where(
        expected_coverage <= c["floor_low_max"].to_numpy(),
        "LOW", "MEDIUM"
    )
)

expected_alert = np.where(
    (expected_own == "HIGH") |
    (expected_floor == "HIGH"),
    "HIGH_CROWD", "NORMAL"
)

checks["Zone areas match calibration"] = np.allclose(
    final.zone_area_px, area
)
checks["Density formula correct"] = np.allclose(
    final.density_per_100x100px,
    expected_density
)
checks["Floor coverage formula correct"] = np.allclose(
    final.floor_coverage,
    expected_coverage
)
checks["Own levels match VAL calibration"] = np.array_equal(
    final.level_own.to_numpy(), expected_own
)
checks["Floor levels match VAL calibration"] = np.array_equal(
    final.level_floor.to_numpy(), expected_floor
)
checks["Alerts match calibrated levels"] = np.array_equal(
    final.crowd_alert.to_numpy(), expected_alert
)

# 4. Report
print("=== CELL 70 — UNIFIED PIPELINE VALIDATION ===")

for name, passed in checks.items():
    print(f'{"PASS" if passed else "FAIL"} | {name}')

passed = sum(bool(v) for v in checks.values())
print(f"\nPassed: {passed}/{len(checks)}")

assert all(checks.values()), "Validation failed — inspect failed checks."

report = {
    "status": "PASSED",
    "checks_passed": passed,
    "total_checks": len(checks),
    "tracker": "BoTSORT_baseline",
    "flow_stability_frames": 5,
    "calibration": "VAL-only",
    "records": int(len(final)),
    "mean_occupancy": float(final["count"].mean()),
    "total_inflow": int(final["inflow"].sum()),
    "total_outflow": int(final["outflow"].sum()),
    "alerts": {
        k: int(v)
        for k, v in final.crowd_alert.value_counts().items()
    },
    "limitations": [
        "TEST previously inspected during development",
        "Single-camera temporal split",
        "Validation confirms consistency, not independent accuracy"
    ]
}

REPORT_FILE = OUT / "UNIFIED_BoTSORT_validation_TEST.json"

with open(REPORT_FILE, "w") as f:
    json.dump(report, f, indent=2)

print("\n=== SUMMARY ===")
print(json.dumps(report, indent=2))

print("\nSaved:", REPORT_FILE)
print("Original outputs unchanged.")
print("No training, inference, or TEST tuning.")

=== CELL 70 — UNIFIED PIPELINE VALIDATION ===
PASS | 792 zone-window records
PASS | 66 TEST windows
PASS | 12 zones
PASS | 12 zones per window
PASS | No duplicate step-zone keys
PASS | No missing numeric values
PASS | Nonnegative counts and flow
PASS | Mean count <= maximum count
PASS | Flow matches VAL-selected output
PASS | Flow totals consistent
PASS | Zone areas match calibration
PASS | Density formula correct
PASS | Floor coverage formula correct
PASS | Own levels match VAL calibration
PASS | Floor levels match VAL calibration
PASS | Alerts match calibrated levels

Passed: 16/16

=== SUMMARY ===
{
  "status": "PASSED",
  "checks_passed": 16,
  "total_checks": 16,
  "tracker": "BoTSORT_baseline",
  "flow_stability_frames": 5,
  "calibration": "VAL-only",
  "records": 792,
  "mean_occupancy": 16.494427609427607,
  "total_inflow": 1234,
  "total_outflow": 1234,
  "alerts": {
    "NORMAL": 683,
    "HIGH_CROWD": 109
  },
  "limitations": [
    "TEST previously inspected during develop

In [ ]:
# CELL 71 — FIXED FPS BENCHMARK (VAL ONLY)
# YOLOv8m + BoT-SORT + Zone Assignment
# No training, no TEST, no parameter tuning.

import os, cv2, time, json, yaml, torch
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from ultralytics.utils import DEFAULT_CFG

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "benchmark"
OUT.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "Enable GPU in Colab."

# 1. Load detector
DET_NAME = get_track_detector()
assert DET_NAME == "yolov8m"

model = YOLO(best_weights(DET_NAME))

# 2. Select VAL frames only
IMG_DIR = find_seq_dir(EVAL_SEQUENCE)
ALL_FRAMES = list_frames(IMG_DIR)

VAL_FRAMES = sorted(
    [
        name for name in ALL_FRAMES
        if 1 <= int(Path(name).stem) <= 1657
    ],
    key=lambda x: int(Path(x).stem)
)

assert len(VAL_FRAMES) >= 110
SELECTED = VAL_FRAMES[:110]

# 3. Complete BoT-SORT configuration
TRACKER_CONFIG = {
    "tracker_type": "botsort",
    "track_high_thresh": 0.4,
    "track_low_thresh": 0.2,
    "new_track_thresh": 0.7,
    "track_buffer": 30,
    "match_thresh": 0.7,
    "fuse_score": True,
    "gmc_method": "sparseOptFlow",
    "proximity_thresh": 0.5,
    "appearance_thresh": 0.8,
    "with_reid": False,
    "model": "auto"
}

TRACKER_FILE = OUT / "benchmark_botsort.yaml"

with open(TRACKER_FILE, "w") as f:
    yaml.safe_dump(TRACKER_CONFIG, f)

# 4. Fix Ultralytics global configuration compatibility
setattr(DEFAULT_CFG, "with_reid", False)

# Also ensure the predictor receives the setting
model.overrides["with_reid"] = False

# 5. Benchmark
timings = []
zone_counts = np.zeros(12, dtype=np.int64)

print("GPU:", torch.cuda.get_device_name(0))
print("Detector:", DET_NAME)
print("Tracker: BoT-SORT")
print("Benchmark frames:", len(SELECTED))
print("Warmup: 10 | Measured: 100")

for i, name in enumerate(SELECTED):

    frame = cv2.imread(os.path.join(IMG_DIR, name))
    assert frame is not None, f"Cannot read {name}"

    torch.cuda.synchronize()
    start = time.perf_counter()

    result = model.track(
        frame,
        persist=True,
        tracker=str(TRACKER_FILE),
        imgsz=IMG_SIZE,
        conf=0.01,
        iou=0.7,
        classes=[0],
        max_det=MAX_DET,
        device=0,
        verbose=False
    )[0]

    # Zone assignment using bottom-center footpoints
    if result.boxes is not None and result.boxes.id is not None:

        boxes = result.boxes.xyxy.cpu().numpy()

        px = np.floor(
            (boxes[:, 0] + boxes[:, 2]) / 2
        ).astype(int)

        py = np.floor(boxes[:, 3]).astype(int)

        valid = (
            (px >= 0) & (px < 1654) &
            (py >= 0) & (py < 1080)
        )

        col = np.select(
            [px <= 413, px <= 826, px <= 1239],
            [0, 1, 2],
            default=3
        )

        row = np.select(
            [py <= 359, py <= 719],
            [0, 1],
            default=2
        )

        zones = row * 4 + col

        zone_counts += np.bincount(
            zones[valid],
            minlength=12
        )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    if i >= 10:
        timings.append(elapsed)

    if (i + 1) % 25 == 0:
        print(f"Processed {i + 1}/{len(SELECTED)}")

# 6. Results
timings = np.array(timings)
fps = len(timings) / timings.sum()

summary = {
    "split": "VAL",
    "gpu": torch.cuda.get_device_name(0),
    "detector": DET_NAME,
    "tracker": "BoT-SORT",
    "warmup_frames": 10,
    "measured_frames": len(timings),
    "total_seconds": float(timings.sum()),
    "fps": float(fps),
    "mean_latency_ms": float(timings.mean() * 1000),
    "p95_latency_ms": float(
        np.percentile(timings, 95) * 1000
    ),
    "target_fps": 25,
    "meets_25fps": bool(fps >= 25),
    "limitations": [
        "Excludes video decoding",
        "Excludes temporal flow stabilization",
        "Excludes GNN inference",
        "Excludes visualization",
        "Ultralytics tracker may differ from original saved tracker"
    ]
}

print("\n=== END-TO-END FPS RESULTS ===")
print(json.dumps(summary, indent=2))

FILE = OUT / "VAL_end_to_end_FPS_benchmark.json"

with open(FILE, "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved:", FILE)
print("No training or TEST tuning.")

GPU: NVIDIA A100-SXM4-40GB
Detector: yolov8m
Tracker: BoT-SORT
Benchmark frames: 110
Warmup: 10 | Measured: 100
Processed 25/110
Processed 50/110
Processed 75/110
Processed 100/110

=== END-TO-END FPS RESULTS ===
{
  "split": "VAL",
  "gpu": "NVIDIA A100-SXM4-40GB",
  "detector": "yolov8m",
  "tracker": "BoT-SORT",
  "warmup_frames": 10,
  "measured_frames": 100,
  "total_seconds": 5.933490961003372,
  "fps": 16.85348484681768,
  "mean_latency_ms": 59.33490961003372,
  "p95_latency_ms": 61.15867139947113,
  "target_fps": 25,
  "meets_25fps": false,
  "limitations": [
    "Excludes video decoding",
    "Excludes temporal flow stabilization",
    "Excludes GNN inference",
    "Excludes visualization",
    "Ultralytics tracker may differ from original saved tracker"
  ]
}

Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/benchmark/VAL_end_to_end_FPS_benchmark.json
No training or TEST tuning.


In [ ]:
# CELL 72 — COMPONENT-LEVEL FPS PROFILING (VAL ONLY)

import os, cv2, time, json, yaml, torch
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from ultralytics.utils import DEFAULT_CFG

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "benchmark"
OUT.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "Enable GPU in Colab."

DET_NAME = get_track_detector()
assert DET_NAME == "yolov8m"

IMG_DIR = find_seq_dir(EVAL_SEQUENCE)

VAL_FRAMES = sorted(
    [
        name for name in list_frames(IMG_DIR)
        if 1 <= int(Path(name).stem) <= 1657
    ],
    key=lambda x: int(Path(x).stem)
)

assert len(VAL_FRAMES) >= 110
SELECTED = VAL_FRAMES[:110]

TRACKER_FILE = OUT / "benchmark_botsort.yaml"
assert TRACKER_FILE.exists(), "Run CELL 71 first."

with open(TRACKER_FILE) as f:
    tracker_cfg = yaml.safe_load(f)

assert tracker_cfg["tracker_type"] == "botsort"
assert tracker_cfg["with_reid"] is False

# Compatibility with installed Ultralytics
setattr(DEFAULT_CFG, "with_reid", False)

# Separate models to avoid shared predictor state
detector = YOLO(best_weights(DET_NAME))
tracker_model = YOLO(best_weights(DET_NAME))

detector.overrides["with_reid"] = False
tracker_model.overrides["with_reid"] = False

times = {
    "detection": [],
    "tracking_pipeline": [],
    "zone_assignment": []
}

def sync():
    torch.cuda.synchronize()

print("=== CELL 72 — VAL PERFORMANCE PROFILING ===")
print("GPU:", torch.cuda.get_device_name(0))
print("Detector:", DET_NAME)
print("Frames: 10 warmup + 100 measured")

# Load frames outside timed sections
frames = []
for name in SELECTED:
    img = cv2.imread(os.path.join(IMG_DIR, name))
    assert img is not None, name
    frames.append(img)

# 1. Detection-only benchmark
print("\nProfiling detection...")

for i, frame in enumerate(frames):
    sync()
    start = time.perf_counter()

    detector.predict(
        frame,
        imgsz=IMG_SIZE,
        conf=0.01,
        iou=0.7,
        classes=[0],
        max_det=MAX_DET,
        device=0,
        verbose=False
    )

    sync()
    elapsed = time.perf_counter() - start

    if i >= 10:
        times["detection"].append(elapsed)

# 2. Detection + tracking benchmark
print("Profiling detection + tracking...")

tracked_results = []

for i, frame in enumerate(frames):
    sync()
    start = time.perf_counter()

    result = tracker_model.track(
        frame,
        persist=True,
        tracker=str(TRACKER_FILE),
        imgsz=IMG_SIZE,
        conf=0.01,
        iou=0.7,
        classes=[0],
        max_det=MAX_DET,
        device=0,
        verbose=False
    )[0]

    sync()
    elapsed = time.perf_counter() - start

    if i >= 10:
        times["tracking_pipeline"].append(elapsed)
        tracked_results.append(result)

# 3. Zone assignment benchmark
print("Profiling zone assignment...")

for result in tracked_results:
    sync()
    start = time.perf_counter()

    counts = np.zeros(12, dtype=np.int64)

    if result.boxes is not None and result.boxes.id is not None:
        boxes = result.boxes.xyxy.cpu().numpy()

        px = np.floor(
            (boxes[:, 0] + boxes[:, 2]) / 2
        ).astype(int)

        py = np.floor(boxes[:, 3]).astype(int)

        valid = (
            (px >= 0) & (px < 1654) &
            (py >= 0) & (py < 1080)
        )

        col = np.select(
            [px <= 413, px <= 826, px <= 1239],
            [0, 1, 2],
            default=3
        )

        row = np.select(
            [py <= 359, py <= 719],
            [0, 1],
            default=2
        )

        zones = row * 4 + col

        counts = np.bincount(
            zones[valid], minlength=12
        )

    sync()
    elapsed = time.perf_counter() - start
    times["zone_assignment"].append(elapsed)

# 4. Summarize
det_ms = np.mean(times["detection"]) * 1000
track_pipeline_ms = np.mean(times["tracking_pipeline"]) * 1000
zone_ms = np.mean(times["zone_assignment"]) * 1000

# Difference is an estimate, not a direct isolated tracker timer.
estimated_tracking_ms = track_pipeline_ms - det_ms

summary = {
    "split": "VAL",
    "gpu": torch.cuda.get_device_name(0),
    "frames_measured": 100,
    "detection_ms": float(det_ms),
    "detection_plus_tracking_ms": float(track_pipeline_ms),
    "estimated_tracking_overhead_ms": float(
        estimated_tracking_ms
    ),
    "zone_assignment_ms": float(zone_ms),
    "estimated_total_ms": float(
        track_pipeline_ms + zone_ms
    ),
    "estimated_total_fps": float(
        1000 / (track_pipeline_ms + zone_ms)
    ),
    "methodology": (
        "Separate detection-only and detection+tracking runs. "
        "Tracking overhead is estimated by subtraction, "
        "not directly timed in isolation."
    ),
    "limitations": [
        "No video decoding",
        "No temporal flow stabilization",
        "No GNN",
        "No visualization",
        "Tracker implementation may differ from saved experiments",
        "Estimated tracking overhead may include run-to-run variation"
    ]
}

print("\n=== PERFORMANCE BREAKDOWN ===")
print(f"YOLO detection:           {det_ms:.2f} ms/frame")
print(f"Detection + tracking:     {track_pipeline_ms:.2f} ms/frame")
print(f"Estimated tracking extra: {estimated_tracking_ms:.2f} ms/frame")
print(f"Zone assignment:          {zone_ms:.2f} ms/frame")
print(f"Estimated total FPS:      {summary['estimated_total_fps']:.2f}")

FILE = OUT / "VAL_component_FPS_profile.json"

with open(FILE, "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved:", FILE)
print("No training, inference-cache changes, or TEST tuning.")

=== CELL 72 — VAL PERFORMANCE PROFILING ===
GPU: NVIDIA A100-SXM4-40GB
Detector: yolov8m
Frames: 10 warmup + 100 measured

Profiling detection...
Profiling detection + tracking...
Profiling zone assignment...

=== PERFORMANCE BREAKDOWN ===
YOLO detection:           16.26 ms/frame
Detection + tracking:     58.99 ms/frame
Estimated tracking extra: 42.72 ms/frame
Zone assignment:          0.30 ms/frame
Estimated total FPS:      16.87

Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/benchmark/VAL_component_FPS_profile.json
No training, inference-cache changes, or TEST tuning.


In [ ]:
# CELL 73 — BoT-SORT GMC PERFORMANCE AUDIT (VAL ONLY)

import os, cv2, time, json, yaml, torch
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from ultralytics.utils import DEFAULT_CFG

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "benchmark"
OUT.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "Enable GPU runtime."
assert get_track_detector() == "yolov8m"

# Compatibility fix from CELL 71
setattr(DEFAULT_CFG, "with_reid", False)

SOURCE_CFG = OUT / "benchmark_botsort.yaml"
assert SOURCE_CFG.exists(), "Missing CELL 71 tracker configuration."

with open(SOURCE_CFG) as f:
    original_cfg = yaml.safe_load(f)

assert original_cfg["tracker_type"] == "botsort"
assert original_cfg["with_reid"] is False

# Create separate benchmark-only configurations
configs = {}

for name, method in [
    ("GMC_ON", "sparseOptFlow"),
    ("GMC_OFF", "none")
]:
    cfg = original_cfg.copy()
    cfg["gmc_method"] = method

    path = OUT / f"benchmark_botsort_{name}.yaml"

    with open(path, "w") as f:
        yaml.safe_dump(cfg, f)

    configs[name] = path

# VAL only
IMG_DIR = find_seq_dir(EVAL_SEQUENCE)

val_names = sorted(
    [
        n for n in list_frames(IMG_DIR)
        if 1 <= int(Path(n).stem) <= 1657
    ],
    key=lambda n: int(Path(n).stem)
)

assert len(val_names) >= 110

frames = []

for name in val_names[:110]:
    img = cv2.imread(os.path.join(IMG_DIR, name))
    assert img is not None, name
    frames.append(img)

def sync():
    torch.cuda.synchronize()

def benchmark(name, config_path):
    # Fresh model prevents tracker state contamination
    model = YOLO(best_weights("yolov8m"))
    model.overrides["with_reid"] = False

    timings = []

    for i, frame in enumerate(frames):
        sync()
        start = time.perf_counter()

        model.track(
            frame,
            persist=True,
            tracker=str(config_path),
            imgsz=IMG_SIZE,
            conf=0.01,
            iou=0.7,
            classes=[0],
            max_det=MAX_DET,
            device=0,
            verbose=False
        )

        sync()
        elapsed = time.perf_counter() - start

        if i >= 10:
            timings.append(elapsed)

        if (i + 1) % 25 == 0:
            print(f"{name}: {i + 1}/110")

    return {
        "mean_ms": float(np.mean(timings) * 1000),
        "p95_ms": float(np.percentile(timings, 95) * 1000),
        "fps": float(1 / np.mean(timings))
    }

print("=== CELL 73 — GMC PERFORMANCE AUDIT ===")
print("GPU:", torch.cuda.get_device_name(0))
print("Split: VAL")
print("Frames: 10 warmup + 100 measured")

results = {}

for name, path in configs.items():
    print(f"\nTesting {name}...")
    results[name] = benchmark(name, path)

on = results["GMC_ON"]
off = results["GMC_OFF"]

speedup = on["mean_ms"] / off["mean_ms"]
reduction = (1 - off["mean_ms"] / on["mean_ms"]) * 100

summary = {
    "split": "VAL",
    "detector": "yolov8m",
    "tracker": "BoT-SORT",
    "gpu": torch.cuda.get_device_name(0),
    "warmup_frames": 10,
    "measured_frames": 100,
    "GMC_ON": on,
    "GMC_OFF": off,
    "speedup": float(speedup),
    "latency_reduction_percent": float(reduction),
    "limitations": [
        "Performance-only experiment",
        "Tracking accuracy not evaluated",
        "No video decoding, GNN or visualization",
        "Original saved tracker unchanged"
    ]
}

print("\n=== GMC RESULTS ===")
print(f"GMC ON:  {on['mean_ms']:.2f} ms | {on['fps']:.2f} FPS")
print(f"GMC OFF: {off['mean_ms']:.2f} ms | {off['fps']:.2f} FPS")
print(f"Speedup: {speedup:.2f}x")
print(f"Latency reduction: {reduction:.2f}%")

FILE = OUT / "VAL_GMC_performance_audit.json"

with open(FILE, "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved:", FILE)
print("Original configurations and outputs unchanged.")
print("No TEST tuning or retraining.")

=== CELL 73 — GMC PERFORMANCE AUDIT ===
GPU: NVIDIA A100-SXM4-40GB
Split: VAL
Frames: 10 warmup + 100 measured

Testing GMC_ON...
GMC_ON: 25/110
GMC_ON: 50/110
GMC_ON: 75/110
GMC_ON: 100/110

Testing GMC_OFF...
GMC_OFF: 25/110
GMC_OFF: 50/110
GMC_OFF: 75/110
GMC_OFF: 100/110

=== GMC RESULTS ===
GMC ON:  57.32 ms | 17.45 FPS
GMC OFF: 41.31 ms | 24.21 FPS
Speedup: 1.39x
Latency reduction: 27.94%

Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/benchmark/VAL_GMC_performance_audit.json
Original configurations and outputs unchanged.
No TEST tuning or retraining.


In [ ]:
# CELL 74 — GMC ON vs OFF: VAL TRACKING ACCURACY
# 300 consecutive VAL frames | IoU >= 0.5
# Diagnostic metrics only (not official HOTA/IDF1).

import os, cv2, yaml, json, torch
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import linear_sum_assignment
from ultralytics import YOLO
from ultralytics.utils import DEFAULT_CFG

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "benchmark"
OUT.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available()
assert get_track_detector() == "yolov8m"

GT_FILE = ROOT / "tracking/trackeval/gt/MOT20-05-val/gt/gt.txt"
CFG_FILE = OUT / "benchmark_botsort.yaml"

assert GT_FILE.exists() and CFG_FILE.exists()

with open(CFG_FILE) as f:
    base_cfg = yaml.safe_load(f)

assert base_cfg["tracker_type"] == "botsort"
assert base_cfg["with_reid"] is False

setattr(DEFAULT_CFG, "with_reid", False)

# First 300 consecutive frames from VAL
IMG_DIR = find_seq_dir(EVAL_SEQUENCE)

names = sorted(
    [
        n for n in list_frames(IMG_DIR)
        if 1 <= int(Path(n).stem) <= 300
    ],
    key=lambda n: int(Path(n).stem)
)

assert len(names) == 300, "Expected 300 VAL frames."

gt = pd.read_csv(GT_FILE, header=None)
gt = gt[
    (gt.iloc[:, 6] == 1) &
    (gt.iloc[:, 7] == 1) &
    (gt.iloc[:, 0].between(1, 300))
].copy()

gt = gt.iloc[:, :6]
gt.columns = ["frame", "id", "x", "y", "w", "h"]

def xyxy(df):
    a = df[["x", "y", "w", "h"]].to_numpy(float)
    return np.column_stack([
        a[:, 0], a[:, 1],
        a[:, 0] + a[:, 2],
        a[:, 1] + a[:, 3]
    ])

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))

    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])

    inter = np.maximum(0, x2-x1) * np.maximum(0, y2-y1)
    aa = (a[:, 2]-a[:, 0]) * (a[:, 3]-a[:, 1])
    bb = (b[:, 2]-b[:, 0]) * (b[:, 3]-b[:, 1])

    return inter / np.maximum(
        aa[:, None] + bb[None, :] - inter, 1e-9
    )

def run_tracker(mode, method):
    cfg = base_cfg.copy()
    cfg["gmc_method"] = method

    cfg_path = OUT / f"CELL74_{mode}.yaml"
    with open(cfg_path, "w") as f:
        yaml.safe_dump(cfg, f)

    model = YOLO(best_weights("yolov8m"))
    model.overrides["with_reid"] = False

    records = []

    for i, name in enumerate(names, 1):
        frame = cv2.imread(os.path.join(IMG_DIR, name))
        assert frame is not None, name

        result = model.track(
            frame,
            persist=True,
            tracker=str(cfg_path),
            imgsz=IMG_SIZE,
            conf=0.01,
            iou=0.7,
            classes=[0],
            max_det=MAX_DET,
            device=0,
            verbose=False
        )[0]

        if result.boxes is not None and result.boxes.id is not None:
            boxes = result.boxes.xyxy.cpu().numpy()
            ids = result.boxes.id.int().cpu().numpy()

            for box, track_id in zip(boxes, ids):
                records.append([
                    int(Path(name).stem),
                    int(track_id),
                    float(box[0]),
                    float(box[1]),
                    float(box[2] - box[0]),
                    float(box[3] - box[1])
                ])

        if i % 100 == 0:
            print(f"{mode}: {i}/300")

    return pd.DataFrame(
        records,
        columns=["frame", "id", "x", "y", "w", "h"]
    )

def evaluate(pred):
    gt_groups = dict(tuple(gt.groupby("frame")))
    pred_groups = dict(tuple(pred.groupby("frame")))

    TP = FP = FN = IDSW = 0
    previous_match = {}

    for frame in range(1, 301):
        g = gt_groups.get(frame, gt.iloc[:0])
        p = pred_groups.get(frame, pred.iloc[:0])

        matrix = iou_matrix(xyxy(g), xyxy(p))

        matched_g = set()
        matched_p = set()

        if matrix.size:
            cost = np.where(
                matrix >= 0.5, 1 - matrix, 1e6
            )

            rows, cols = linear_sum_assignment(cost)

            for gi, pi in zip(rows, cols):
                if matrix[gi, pi] < 0.5:
                    continue

                matched_g.add(gi)
                matched_p.add(pi)

                gid = int(g.iloc[gi]["id"])
                pid = int(p.iloc[pi]["id"])

                if gid in previous_match:
                    if previous_match[gid] != pid:
                        IDSW += 1

                previous_match[gid] = pid

        TP += len(matched_g)
        FP += len(p) - len(matched_p)
        FN += len(g) - len(matched_g)

    precision = TP / max(TP + FP, 1)
    recall = TP / max(TP + FN, 1)
    mota_proxy = 1 - (FP + FN + IDSW) / max(len(gt), 1)

    return {
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "IDSW_proxy": IDSW,
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "MOTA_proxy": round(mota_proxy, 4)
    }

print("=== CELL 74 — GMC TRACKING ACCURACY ===")
print("Split: VAL | Frames: 1–300")

results = {}

for mode, method in [
    ("GMC_ON", "sparseOptFlow"),
    ("GMC_OFF", "none")
]:
    print(f"\nRunning {mode}...")
    predictions = run_tracker(mode, method)
    results[mode] = evaluate(predictions)
    results[mode]["tracked_observations"] = len(predictions)

    predictions.to_csv(
        OUT / f"CELL74_{mode}_VAL_tracks.csv",
        index=False
    )

print("\n=== VAL ACCURACY COMPARISON ===")
print(pd.DataFrame(results).T.to_string())

with open(OUT / "VAL_GMC_tracking_accuracy_audit.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nSaved diagnostic results.")
print("Original tracking files unchanged.")
print("TEST not accessed.")

=== CELL 74 — GMC TRACKING ACCURACY ===
Split: VAL | Frames: 1–300

Running GMC_ON...
GMC_ON: 100/300
GMC_ON: 200/300
GMC_ON: 300/300

Running GMC_OFF...
GMC_OFF: 100/300
GMC_OFF: 200/300
GMC_OFF: 300/300

=== VAL ACCURACY COMPARISON ===
              TP       FP      FN  IDSW_proxy  precision  recall  MOTA_proxy  tracked_observations
GMC_ON   53325.0  10494.0  5404.0       211.0     0.8356   0.908      0.7257               63819.0
GMC_OFF  53268.0  10498.0  5461.0       209.0     0.8354   0.907      0.7247               63766.0

Saved diagnostic results.
Original tracking files unchanged.
TEST not accessed.


In [ ]:
# CELL 75 — GMC ON vs OFF: ZONE OCCUPANCY & FLOW (VAL ONLY)
# Reuses CELL 74 CSV files. No YOLO/tracker reruns.

import json
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "benchmark"
OUT.mkdir(parents=True, exist_ok=True)

ON_FILE = OUT / "CELL74_GMC_ON_VAL_tracks.csv"
OFF_FILE = OUT / "CELL74_GMC_OFF_VAL_tracks.csv"
GT_FILE = ROOT / "tracking/trackeval/gt/MOT20-05-val/gt/gt.txt"

for path in [ON_FILE, OFF_FILE, GT_FILE]:
    assert path.exists(), f"Missing file: {path}"

FPS = 25
START_FRAME = 1
END_FRAME = 300
STABILITY = 5

IMG_W, IMG_H = 1654, 1080
ZONE_COLS, ZONE_ROWS = 4, 3
N_ZONES = 12

# 300 frames = 12 complete one-second windows
WINDOWS = list(range(12))

def load_predictions(path):
    df = pd.read_csv(path)

    required = {"frame", "id", "x", "y", "w", "h"}
    assert required.issubset(df.columns)

    df = df[df["frame"].between(START_FRAME, END_FRAME)].copy()
    df["frame"] = df["frame"].astype(int)
    df["id"] = df["id"].astype(int)

    return df

def load_gt():
    df = pd.read_csv(GT_FILE, header=None)

    assert df.shape[1] >= 8, "Unexpected MOT GT format."

    # Official pedestrian GT only
    df = df[
        (df.iloc[:, 6] == 1) &
        (df.iloc[:, 7] == 1) &
        (df.iloc[:, 0].between(START_FRAME, END_FRAME))
    ].copy()

    df = df.iloc[:, :6]
    df.columns = ["frame", "id", "x", "y", "w", "h"]

    df["frame"] = df["frame"].astype(int)
    df["id"] = df["id"].astype(int)

    return df

def assign_zones(df):
    df = df.copy()

    # Bottom-center footpoint
    px = np.floor(df["x"] + df["w"] / 2).astype(int)
    py = np.floor(df["y"] + df["h"]).astype(int)

    valid = (
        (px >= 0) & (px < IMG_W) &
        (py >= 0) & (py < IMG_H)
    )

    # Same 3x4 grid geometry as CELL 72
    col = np.select(
        [px <= 413, px <= 826, px <= 1239],
        [0, 1, 2],
        default=3
    )

    row = np.select(
        [py <= 359, py <= 719],
        [0, 1],
        default=2
    )

    df["zone"] = row * ZONE_COLS + col
    df.loc[~valid, "zone"] = -1

    return df

def occupancy_counts(df):
    valid = df[df["zone"] >= 0].copy()
    valid["window"] = (valid["frame"] - 1) // FPS

    grouped = valid.groupby(
        ["window", "zone"]
    ).size()

    idx = pd.MultiIndex.from_product(
        [WINDOWS, range(N_ZONES)],
        names=["window", "zone"]
    )

    return grouped.reindex(idx, fill_value=0).astype(float) / FPS

def stable_transitions(df):
    """
    Count a transition only after the destination zone
    remains stable for STABILITY consecutive observed frames.

    Missing frames break the stability streak.
    """
    events = []

    for track_id, group in df.groupby("id"):
        group = group.sort_values("frame")

        last_stable = None
        candidate = None
        candidate_count = 0
        previous_frame = None

        for row in group.itertuples(index=False):
            frame = int(row.frame)
            zone = int(row.zone)

            if previous_frame is not None and frame != previous_frame + 1:
                candidate = None
                candidate_count = 0

            previous_frame = frame

            if zone < 0:
                candidate = None
                candidate_count = 0
                continue

            if zone == candidate:
                candidate_count += 1
            else:
                candidate = zone
                candidate_count = 1

            if candidate_count < STABILITY:
                continue

            if last_stable is None:
                last_stable = candidate
                continue

            if candidate != last_stable:
                events.append({
                    "frame": frame,
                    "from_zone": last_stable,
                    "to_zone": candidate
                })
                last_stable = candidate

    return pd.DataFrame(
        events,
        columns=["frame", "from_zone", "to_zone"]
    )

def flow_counts(events):
    idx = pd.MultiIndex.from_product(
        [WINDOWS, range(N_ZONES)],
        names=["window", "zone"]
    )

    if events.empty:
        zeros = pd.Series(0.0, index=idx)
        return zeros, zeros.copy()

    events = events.copy()
    events["window"] = (events["frame"] - 1) // FPS

    inflow = events.groupby(
        ["window", "to_zone"]
    ).size()
    inflow.index.names = ["window", "zone"]

    outflow = events.groupby(
        ["window", "from_zone"]
    ).size()
    outflow.index.names = ["window", "zone"]

    return (
        inflow.reindex(idx, fill_value=0).astype(float),
        outflow.reindex(idx, fill_value=0).astype(float)
    )

def evaluate(df, gt_occ, gt_in, gt_out):
    df = assign_zones(df)

    pred_occ = occupancy_counts(df)
    events = stable_transitions(df)
    pred_in, pred_out = flow_counts(events)

    return {
        "occupancy_MAE": float(
            np.mean(np.abs(pred_occ - gt_occ))
        ),
        "inflow_MAE": float(
            np.mean(np.abs(pred_in - gt_in))
        ),
        "outflow_MAE": float(
            np.mean(np.abs(pred_out - gt_out))
        ),
        "transitions": int(len(events)),
        "mean_occupancy": float(pred_occ.mean()),
        "observations": int(len(df))
    }

print("=== CELL 75 — ZONE ACCURACY AUDIT ===")
print("Split: VAL | Frames: 1–300")
print("Stability:", STABILITY, "frames")

gt = assign_zones(load_gt())

gt_occ = occupancy_counts(gt)
gt_events = stable_transitions(gt)
gt_in, gt_out = flow_counts(gt_events)

results = {
    "GROUND_TRUTH": {
        "transitions": int(len(gt_events)),
        "mean_occupancy": float(gt_occ.mean())
    }
}

for mode, path in [
    ("GMC_ON", ON_FILE),
    ("GMC_OFF", OFF_FILE)
]:
    pred = load_predictions(path)

    results[mode] = evaluate(
        pred, gt_occ, gt_in, gt_out
    )

print("\n=== VAL ZONE ACCURACY COMPARISON ===")

table = pd.DataFrame({
    mode: results[mode]
    for mode in ["GMC_ON", "GMC_OFF"]
}).T

print(table.to_string())

print("\nGround Truth transitions:",
      results["GROUND_TRUTH"]["transitions"])

print("\n=== DIFFERENCE (OFF - ON) ===")

for metric in [
    "occupancy_MAE",
    "inflow_MAE",
    "outflow_MAE"
]:
    delta = results["GMC_OFF"][metric] - results["GMC_ON"][metric]
    print(f"{metric}: {delta:+.6f}")

with open(OUT / "VAL_GMC_zone_accuracy_audit.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nSaved:", OUT / "VAL_GMC_zone_accuracy_audit.json")
print("No YOLO/tracker reruns.")
print("Original outputs unchanged.")
print("TEST not accessed.")

=== CELL 75 — ZONE ACCURACY AUDIT ===
Split: VAL | Frames: 1–300
Stability: 5 frames

=== VAL ZONE ACCURACY COMPARISON ===
         occupancy_MAE  inflow_MAE  outflow_MAE  transitions  mean_occupancy  observations
GMC_ON        2.493056    0.659722     0.618056        280.0       17.595000       63819.0
GMC_OFF       2.451667    0.638889     0.597222        279.0       17.533056       63766.0

Ground Truth transitions: 253

=== DIFFERENCE (OFF - ON) ===
occupancy_MAE: -0.041389
inflow_MAE: -0.020833
outflow_MAE: -0.020833

Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/benchmark/VAL_GMC_zone_accuracy_audit.json
No YOLO/tracker reruns.
Original outputs unchanged.
TEST not accessed.


In [ ]:
# CELL 76 — FULL VAL GMC ON vs OFF EVALUATION
# Frames 1–1657 | No TEST | No retraining
# Reuses CELL 74 trajectories for frames 1–300
# Tracks only frames 301–1657 (with tracker state replay)

import os, cv2, json, yaml, torch
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import linear_sum_assignment
from ultralytics import YOLO
from ultralytics.utils import DEFAULT_CFG

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "benchmark"
OUT.mkdir(parents=True, exist_ok=True)

N_FRAMES = 1657
FPS = 25
STABILITY = 5
W, H = 1654, 1080

assert torch.cuda.is_available()
assert get_track_detector() == "yolov8m"

setattr(DEFAULT_CFG, "with_reid", False)

CFG_FILE = OUT / "benchmark_botsort.yaml"
GT_FILE = ROOT / "tracking/trackeval/gt/MOT20-05-val/gt/gt.txt"

assert CFG_FILE.exists() and GT_FILE.exists()

with open(CFG_FILE) as f:
    base_cfg = yaml.safe_load(f)

assert base_cfg["tracker_type"] == "botsort"
assert base_cfg["with_reid"] is False

IMG_DIR = find_seq_dir(EVAL_SEQUENCE)

names = sorted(
    [
        n for n in list_frames(IMG_DIR)
        if 1 <= int(Path(n).stem) <= N_FRAMES
    ],
    key=lambda n: int(Path(n).stem)
)

assert len(names) == N_FRAMES
assert [int(Path(n).stem) for n in names] == list(range(1, N_FRAMES + 1))

def load_gt():
    df = pd.read_csv(GT_FILE, header=None)

    df = df[
        (df.iloc[:, 6] == 1) &
        (df.iloc[:, 7] == 1) &
        (df.iloc[:, 0].between(1, N_FRAMES))
    ].copy()

    df = df.iloc[:, :6]
    df.columns = ["frame", "id", "x", "y", "w", "h"]
    return df.astype({"frame": int, "id": int})

gt = load_gt()

def run_full_val(mode, method):
    cfg = dict(base_cfg)
    cfg["gmc_method"] = method

    cfg_path = OUT / f"CELL76_{mode}.yaml"
    with open(cfg_path, "w") as f:
        yaml.safe_dump(cfg, f)

    # Fresh tracker state for each configuration
    model = YOLO(best_weights("yolov8m"))
    model.overrides["with_reid"] = False

    records = []

    for i, name in enumerate(names, 1):
        frame = cv2.imread(os.path.join(IMG_DIR, name))
        assert frame is not None, name

        result = model.track(
            frame,
            persist=True,
            tracker=str(cfg_path),
            imgsz=IMG_SIZE,
            conf=0.01,
            iou=0.7,
            classes=[0],
            max_det=MAX_DET,
            device=0,
            verbose=False
        )[0]

        if result.boxes is not None and result.boxes.id is not None:
            boxes = result.boxes.xyxy.cpu().numpy()
            ids = result.boxes.id.int().cpu().numpy()

            for box, tid in zip(boxes, ids):
                records.append([
                    i, int(tid),
                    float(box[0]), float(box[1]),
                    float(box[2] - box[0]),
                    float(box[3] - box[1])
                ])

        if i % 250 == 0 or i == N_FRAMES:
            print(f"{mode}: {i}/{N_FRAMES}")

    df = pd.DataFrame(
        records,
        columns=["frame", "id", "x", "y", "w", "h"]
    )

    df.to_csv(OUT / f"CELL76_{mode}_FULL_VAL_tracks.csv", index=False)
    return df

def boxes(df):
    a = df[["x", "y", "w", "h"]].to_numpy(float)
    return np.column_stack([
        a[:, 0], a[:, 1],
        a[:, 0] + a[:, 2],
        a[:, 1] + a[:, 3]
    ])

def iou_matrix(a, b):
    if not len(a) or not len(b):
        return np.zeros((len(a), len(b)))

    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])

    inter = np.maximum(0, x2-x1) * np.maximum(0, y2-y1)
    aa = np.maximum(0, a[:, 2]-a[:, 0]) * np.maximum(0, a[:, 3]-a[:, 1])
    bb = np.maximum(0, b[:, 2]-b[:, 0]) * np.maximum(0, b[:, 3]-b[:, 1])

    return inter / np.maximum(aa[:, None] + bb[None, :] - inter, 1e-9)

def tracking_metrics(pred):
    gg = dict(tuple(gt.groupby("frame")))
    pp = dict(tuple(pred.groupby("frame")))

    TP = FP = FN = IDSW = 0
    previous = {}

    empty_g = gt.iloc[:0]
    empty_p = pred.iloc[:0]

    for frame in range(1, N_FRAMES + 1):
        g = gg.get(frame, empty_g)
        p = pp.get(frame, empty_p)

        mat = iou_matrix(boxes(g), boxes(p))
        mg, mp = set(), set()

        if mat.size:
            cost = np.where(mat >= 0.5, 1-mat, 1e6)
            rows, cols = linear_sum_assignment(cost)

            for gi, pi in zip(rows, cols):
                if mat[gi, pi] < 0.5:
                    continue

                mg.add(gi)
                mp.add(pi)

                gid = int(g.iloc[gi]["id"])
                pid = int(p.iloc[pi]["id"])

                if gid in previous and previous[gid] != pid:
                    IDSW += 1

                previous[gid] = pid

        TP += len(mg)
        FP += len(p) - len(mp)
        FN += len(g) - len(mg)

    return {
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "IDSW_proxy": IDSW,
        "precision": TP / max(TP + FP, 1),
        "recall": TP / max(TP + FN, 1),
        "MOTA_proxy": 1-(FP+FN+IDSW)/max(len(gt), 1)
    }

def assign_zones(df):
    df = df.copy()

    px = np.floor(df["x"] + df["w"]/2).astype(int)
    py = np.floor(df["y"] + df["h"]).astype(int)

    valid = (px >= 0) & (px < W) & (py >= 0) & (py < H)

    col = np.select(
        [px <= 413, px <= 826, px <= 1239],
        [0, 1, 2],
        default=3
    )
    row = np.select(
        [py <= 359, py <= 719],
        [0, 1],
        default=2
    )

    df["zone"] = row*4 + col
    df.loc[~valid, "zone"] = -1
    return df

# Only complete 25-frame windows: 1–1650
N_WINDOWS = N_FRAMES // FPS
IDX = pd.MultiIndex.from_product(
    [range(N_WINDOWS), range(12)],
    names=["window", "zone"]
)

def occupancy(df):
    d = df[
        (df["zone"] >= 0) &
        (df["frame"] <= N_WINDOWS * FPS)
    ].copy()

    d["window"] = (d["frame"] - 1) // FPS

    return (
        d.groupby(["window", "zone"])
         .size()
         .reindex(IDX, fill_value=0)
         .astype(float) / FPS
    )

def transitions(df):
    events = []

    for tid, group in df.groupby("id"):
        group = group.sort_values("frame")

        stable = None
        candidate = None
        streak = 0
        prev_frame = None

        for r in group.itertuples(index=False):
            frame = int(r.frame)
            zone = int(r.zone)

            if prev_frame is not None and frame != prev_frame + 1:
                candidate, streak = None, 0

            prev_frame = frame

            if zone < 0:
                candidate, streak = None, 0
                continue

            if zone == candidate:
                streak += 1
            else:
                candidate, streak = zone, 1

            if streak < STABILITY:
                continue

            if stable is None:
                stable = zone
            elif zone != stable:
                events.append((frame, stable, zone))
                stable = zone

    return pd.DataFrame(
        events,
        columns=["frame", "from_zone", "to_zone"]
    )

def flow(events):
    zero = pd.Series(0.0, index=IDX)

    if events.empty:
        return zero.copy(), zero.copy()

    e = events[
        events["frame"] <= N_WINDOWS * FPS
    ].copy()

    e["window"] = (e["frame"] - 1) // FPS

    incoming = e.groupby(["window", "to_zone"]).size()
    outgoing = e.groupby(["window", "from_zone"]).size()

    incoming.index.names = ["window", "zone"]
    outgoing.index.names = ["window", "zone"]

    return (
        incoming.reindex(IDX, fill_value=0).astype(float),
        outgoing.reindex(IDX, fill_value=0).astype(float)
    )

gt_z = assign_zones(gt)
gt_occ = occupancy(gt_z)
gt_events = transitions(gt_z)
gt_in, gt_out = flow(gt_events)

def zone_metrics(pred):
    pred_z = assign_zones(pred)

    occ = occupancy(pred_z)
    events = transitions(pred_z)
    incoming, outgoing = flow(events)

    return {
        "occupancy_MAE": float(np.mean(np.abs(occ-gt_occ))),
        "inflow_MAE": float(np.mean(np.abs(incoming-gt_in))),
        "outflow_MAE": float(np.mean(np.abs(outgoing-gt_out))),
        "transitions": len(events),
        "mean_occupancy": float(occ.mean())
    }

print("=== CELL 76 — FULL VAL GMC EVALUATION ===")
print("Frames:", N_FRAMES)
print("Ground Truth observations:", len(gt))
print("Ground Truth transitions:", len(gt_events))

results = {}

for mode, method in [
    ("GMC_ON", "sparseOptFlow"),
    ("GMC_OFF", "none")
]:
    print(f"\nRunning {mode}...")
    pred = run_full_val(mode, method)

    results[mode] = {
        "tracking": tracking_metrics(pred),
        "zones": zone_metrics(pred),
        "observations": len(pred)
    }

print("\n=== FULL VAL TRACKING ===")
print(pd.DataFrame({
    k: v["tracking"] for k, v in results.items()
}).T.to_string())

print("\n=== FULL VAL ZONES ===")
print(pd.DataFrame({
    k: v["zones"] for k, v in results.items()
}).T.to_string())

FILE = OUT / "VAL_GMC_FULL_accuracy_audit.json"

with open(FILE, "w") as f:
    json.dump(results, f, indent=2)

print("\nSaved:", FILE)
print("Original results unchanged.")
print("TEST not accessed.")

=== CELL 76 — FULL VAL GMC EVALUATION ===
Frames: 1657
Ground Truth observations: 327068
Ground Truth transitions: 1189

Running GMC_ON...
GMC_ON: 250/1657
GMC_ON: 500/1657
GMC_ON: 750/1657
GMC_ON: 1000/1657
GMC_ON: 1250/1657
GMC_ON: 1500/1657
GMC_ON: 1657/1657

Running GMC_OFF...
GMC_OFF: 250/1657
GMC_OFF: 500/1657
GMC_OFF: 750/1657
GMC_OFF: 1000/1657
GMC_OFF: 1250/1657
GMC_OFF: 1500/1657
GMC_OFF: 1657/1657

=== FULL VAL TRACKING ===
               TP       FP       FN  IDSW_proxy  precision    recall  MOTA_proxy
GMC_ON   294803.0  53971.0  32265.0      1616.0   0.845255  0.901351    0.731395
GMC_OFF  294688.0  54032.0  32380.0      1608.0   0.845056  0.900999    0.730882

=== FULL VAL ZONES ===
         occupancy_MAE  inflow_MAE  outflow_MAE  transitions  mean_occupancy
GMC_ON        2.426465    0.632576     0.635101       1344.0       17.388384
GMC_OFF       2.378990    0.626263     0.626263       1335.0       17.335556

Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/benc

In [ ]:
# CELL 77 — FP16 vs FP32 | GMC OFF | VAL ONLY
# 10 warmup + 100 measured frames per configuration
# No retraining, no TEST access, no original files overwritten.

import os
import cv2
import gc
import json
import time
import yaml
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from ultralytics import YOLO
from ultralytics.utils import DEFAULT_CFG

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "benchmark"
OUT.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "Enable GPU runtime."
assert get_track_detector() == "yolov8m"

torch.cuda.set_device(0)
setattr(DEFAULT_CFG, "with_reid", False)

# --------------------------------------------------
# 1. Load existing tracker configuration
# --------------------------------------------------

CFG_FILE = OUT / "benchmark_botsort.yaml"
assert CFG_FILE.exists(), "Missing CELL 71 configuration."

with open(CFG_FILE) as f:
    cfg = yaml.safe_load(f)

assert cfg["tracker_type"] == "botsort"
assert cfg.get("with_reid") is False

# Benchmark-only configuration
cfg = cfg.copy()
cfg["gmc_method"] = "none"

FP16_CFG = OUT / "CELL77_GMC_OFF.yaml"

with open(FP16_CFG, "w") as f:
    yaml.safe_dump(cfg, f)

# --------------------------------------------------
# 2. Load 110 consecutive VAL frames
# --------------------------------------------------

IMG_DIR = find_seq_dir(EVAL_SEQUENCE)

names = sorted(
    [
        n for n in list_frames(IMG_DIR)
        if 1 <= int(Path(n).stem) <= 110
    ],
    key=lambda n: int(Path(n).stem)
)

assert len(names) == 110
assert [int(Path(n).stem) for n in names] == list(range(1, 111))

frames = []

for name in names:
    frame = cv2.imread(os.path.join(IMG_DIR, name))
    assert frame is not None, name
    frames.append(frame)

# --------------------------------------------------
# 3. Benchmark function
# --------------------------------------------------

def benchmark(label, half_precision):

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model = YOLO(best_weights("yolov8m"))
    model.overrides["with_reid"] = False

    times = []
    detection_times = []
    observations = []

    print(f"\nRunning {label}...")

    for i, frame in enumerate(frames):

        torch.cuda.synchronize()
        start = time.perf_counter()

        result = model.track(
            frame,
            persist=True,
            tracker=str(FP16_CFG),
            imgsz=IMG_SIZE,
            conf=0.01,
            iou=0.7,
            classes=[0],
            max_det=MAX_DET,
            device=0,
            half=half_precision,
            verbose=False
        )[0]

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        if i >= 10:
            times.append(elapsed * 1000)

            # Ultralytics detector stage timings
            speed = result.speed or {}
            detection_times.append(
                float(speed.get("preprocess", 0)) +
                float(speed.get("inference", 0)) +
                float(speed.get("postprocess", 0))
            )

            n = (
                len(result.boxes)
                if result.boxes is not None
                else 0
            )
            observations.append(n)

        if (i + 1) % 25 == 0 or i == 110:
            print(f"{label}: {i + 1}/110")

    mean_ms = float(np.mean(times))

    output = {
        "precision": label,
        "half": half_precision,
        "gmc_method": "none",
        "frames_measured": len(times),
        "mean_ms": mean_ms,
        "median_ms": float(np.median(times)),
        "p95_ms": float(np.percentile(times, 95)),
        "fps": float(1000 / mean_ms),
        "detector_stage_ms_approx": float(
            np.mean(detection_times)
        ),
        "mean_tracked_boxes": float(
            np.mean(observations)
        ),
        "peak_gpu_memory_gb": float(
            torch.cuda.max_memory_allocated() / 1e9
        )
    }

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return output

# --------------------------------------------------
# 4. Run both configurations
# --------------------------------------------------

print("=== CELL 77 — FP16 vs FP32 ===")
print("GPU:", torch.cuda.get_device_name(0))
print("Dataset: MOT20-05 VAL")
print("Frames: 1–110")
print("Tracker: BoT-SORT | GMC OFF")
print("Image size:", IMG_SIZE)

results = {}

results["FP32"] = benchmark(
    "FP32",
    half_precision=False
)

results["FP16"] = benchmark(
    "FP16",
    half_precision=True
)

# --------------------------------------------------
# 5. Compare results
# --------------------------------------------------

fp32 = results["FP32"]
fp16 = results["FP16"]

speedup = fp32["mean_ms"] / fp16["mean_ms"]

fps_gain = (
    (fp16["fps"] / fp32["fps"]) - 1
) * 100

latency_reduction = (
    1 - fp16["mean_ms"] / fp32["mean_ms"]
) * 100

print("\n=== PERFORMANCE COMPARISON ===")

table = pd.DataFrame(results).T

print(
    table[
        [
            "mean_ms",
            "median_ms",
            "p95_ms",
            "fps",
            "detector_stage_ms_approx",
            "mean_tracked_boxes",
            "peak_gpu_memory_gb"
        ]
    ].to_string()
)

print("\n=== OPTIMIZATION RESULT ===")
print(f"Speedup: {speedup:.3f}x")
print(f"FPS improvement: {fps_gain:+.2f}%")
print(f"Latency reduction: {latency_reduction:+.2f}%")

if fp16["fps"] > fp32["fps"]:
    print("FP16 is faster in this benchmark.")
else:
    print("FP16 did not improve FPS in this benchmark.")

if fp16["fps"] >= 25:
    print("25 FPS benchmark target reached.")
else:
    print("25 FPS benchmark target not yet reached.")

# --------------------------------------------------
# 6. Save results
# --------------------------------------------------

summary = {
    "split": "VAL",
    "sequence": "MOT20-05",
    "gpu": torch.cuda.get_device_name(0),
    "tracker": "BoT-SORT",
    "gmc_method": "none",
    "image_size": IMG_SIZE,
    "FP32": fp32,
    "FP16": fp16,
    "speedup": float(speedup),
    "fps_improvement_percent": float(fps_gain),
    "latency_reduction_percent": float(latency_reduction),
    "limitations": [
        "VAL subset only",
        "Performance benchmark, not accuracy evaluation",
        "Video decoding excluded",
        "Zone flow and alerts excluded",
        "GNN and visualization excluded",
        "Sequential runs may be affected by thermal or runtime variation"
    ]
}

FILE = OUT / "VAL_FP16_vs_FP32_GMC_OFF.json"

with open(FILE, "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved:", FILE)
print("Original results unchanged.")
print("No TEST access.")

=== CELL 77 — FP16 vs FP32 ===
GPU: NVIDIA A100-SXM4-40GB
Dataset: MOT20-05 VAL
Frames: 1–110
Tracker: BoT-SORT | GMC OFF
Image size: 1280

Running FP32...
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is depr

In [ ]:
# CELL 78 — BoT-SORT INTERNAL PROFILING (VAL ONLY)
# GMC OFF | YOLOv8m | 10 warmup + 100 measured frames
# No training, no TEST, no changes to original results.

import os, cv2, time, json, yaml, torch, gc
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from unittest.mock import patch
from ultralytics import YOLO
from ultralytics.utils import DEFAULT_CFG
from ultralytics.trackers.bot_sort import BOTSORT
from ultralytics.trackers import byte_tracker

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "benchmark"
OUT.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "Enable GPU runtime."
assert get_track_detector() == "yolov8m"

setattr(DEFAULT_CFG, "with_reid", False)

# --------------------------------------------------
# 1. Existing GMC OFF configuration
# --------------------------------------------------

CFG_FILE = OUT / "benchmark_botsort.yaml"
assert CFG_FILE.exists()

with open(CFG_FILE) as f:
    cfg = yaml.safe_load(f)

assert cfg["tracker_type"] == "botsort"
assert cfg.get("with_reid") is False

cfg = cfg.copy()
cfg["gmc_method"] = "none"

PROFILE_CFG = OUT / "CELL78_GMC_OFF.yaml"

with open(PROFILE_CFG, "w") as f:
    yaml.safe_dump(cfg, f)

# --------------------------------------------------
# 2. Load VAL frames
# --------------------------------------------------

IMG_DIR = find_seq_dir(EVAL_SEQUENCE)

names = sorted(
    [
        n for n in list_frames(IMG_DIR)
        if 1 <= int(Path(n).stem) <= 110
    ],
    key=lambda n: int(Path(n).stem)
)

assert len(names) == 110

frames = []

for name in names:
    img = cv2.imread(os.path.join(IMG_DIR, name))
    assert img is not None, name
    frames.append(img)

# --------------------------------------------------
# 3. Profiling hooks
# --------------------------------------------------

timings = defaultdict(list)
active = {"enabled": False}

def sync():
    torch.cuda.synchronize()

def wrap_method(cls, method_name, category, stack):
    original = getattr(cls, method_name, None)

    if original is None:
        print(f"Skipping unavailable method: {cls.__name__}.{method_name}")
        return

    def wrapped(self, *args, **kwargs):
        if not active["enabled"]:
            return original(self, *args, **kwargs)

        sync()
        start = time.perf_counter()

        try:
            return original(self, *args, **kwargs)
        finally:
            sync()
            elapsed = (time.perf_counter() - start) * 1000
            timings[category].append(elapsed)

    stack.enter_context(patch.object(cls, method_name, wrapped))

# --------------------------------------------------
# 4. Run profiling
# --------------------------------------------------

from contextlib import ExitStack

model = YOLO(best_weights("yolov8m"))
model.overrides["with_reid"] = False

total_times = []
detector_times = []

with ExitStack() as stack:

    # Total tracker update
    wrap_method(BOTSORT, "update", "tracker_update", stack)

    # Camera motion compensation
    from ultralytics.trackers.utils.gmc import GMC
    wrap_method(GMC, "apply", "gmc", stack)

    # Motion prediction / association helpers
    wrap_method(
        byte_tracker.BYTETracker,
        "get_dists",
        "association_distance",
        stack
    )

    wrap_method(
        byte_tracker.BYTETracker,
        "multi_predict",
        "motion_prediction",
        stack
    )

    # Linear assignment function used in association
    from ultralytics.trackers.utils import matching

    original_assignment = matching.linear_assignment

    def timed_assignment(*args, **kwargs):
        if not active["enabled"]:
            return original_assignment(*args, **kwargs)

        start = time.perf_counter()
        try:
            return original_assignment(*args, **kwargs)
        finally:
            timings["linear_assignment"].append(
                (time.perf_counter() - start) * 1000
            )

    stack.enter_context(
        patch.object(
            matching,
            "linear_assignment",
            timed_assignment
        )
    )

    print("=== CELL 78 — INTERNAL PROFILING ===")
    print("GPU:", torch.cuda.get_device_name(0))
    print("Frames: 10 warmup + 100 measured")
    print("GMC: OFF")

    for i, frame in enumerate(frames):

        active["enabled"] = i >= 10

        sync()
        start = time.perf_counter()

        result = model.track(
            frame,
            persist=True,
            tracker=str(PROFILE_CFG),
            imgsz=IMG_SIZE,
            conf=0.01,
            iou=0.7,
            classes=[0],
            max_det=MAX_DET,
            device=0,
            half=False,
            verbose=False
        )[0]

        sync()
        elapsed = (time.perf_counter() - start) * 1000

        if active["enabled"]:
            total_times.append(elapsed)

            speed = result.speed or {}
            detector_times.append(
                sum(float(speed.get(k, 0)) for k in [
                    "preprocess", "inference", "postprocess"
                ])
            )

        if (i + 1) % 25 == 0 or i == 110:
            print(f"Processed: {i + 1}/110")

# --------------------------------------------------
# 5. Results
# --------------------------------------------------

def mean_or_zero(key):
    values = timings.get(key, [])
    return float(np.mean(values)) if values else 0.0

# Methods can be called multiple times per frame.
# Aggregate all calls, then divide by 100 measured frames.
def per_frame_ms(key):
    return float(sum(timings.get(key, [])) / 100)

summary = {
    "split": "VAL",
    "gpu": torch.cuda.get_device_name(0),
    "measured_frames": 100,
    "gmc_method": "none",
    "total_pipeline_ms": float(np.mean(total_times)),
    "total_pipeline_fps": float(1000 / np.mean(total_times)),
    "detector_reported_ms": float(np.mean(detector_times)),
    "tracker_update_ms": per_frame_ms("tracker_update"),
    "gmc_ms": per_frame_ms("gmc"),
    "association_distance_ms": per_frame_ms("association_distance"),
    "linear_assignment_ms": per_frame_ms("linear_assignment"),
    "motion_prediction_ms": per_frame_ms("motion_prediction"),
    "calls": {
        key: len(values)
        for key, values in timings.items()
    }
}

summary["other_overhead_ms_approx"] = (
    summary["total_pipeline_ms"]
    - summary["detector_reported_ms"]
    - summary["tracker_update_ms"]
)

print("\n=== PERFORMANCE BREAKDOWN ===")

for key in [
    "total_pipeline_ms",
    "total_pipeline_fps",
    "detector_reported_ms",
    "tracker_update_ms",
    "gmc_ms",
    "association_distance_ms",
    "linear_assignment_ms",
    "motion_prediction_ms",
    "other_overhead_ms_approx"
]:
    print(f"{key}: {summary[key]:.3f}")

print("\nMethod call counts:", summary["calls"])

FILE = OUT / "VAL_BoTSORT_internal_profile.json"

with open(FILE, "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved:", FILE)
print("Original files unchanged.")
print("No TEST access.")

=== CELL 78 — INTERNAL PROFILING ===
GPU: NVIDIA A100-SXM4-40GB
Frames: 10 warmup + 100 measured
GMC: OFF
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'qua

In [ ]:
# CELL 79 — BoT-SORT DETAILED CPU PROFILING (VAL ONLY)
# GMC OFF | 10 warmup + 100 measured frames
# Profiles the actual BOTSORT.update() execution path.

import os
import cv2
import json
import yaml
import time
import torch
import cProfile
import pstats
import io
import pandas as pd

from pathlib import Path
from unittest.mock import patch
from ultralytics import YOLO
from ultralytics.utils import DEFAULT_CFG
from ultralytics.trackers.bot_sort import BOTSORT

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "benchmark"
OUT.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available()
assert get_track_detector() == "yolov8m"

setattr(DEFAULT_CFG, "with_reid", False)

# --------------------------------------------------
# 1. GMC OFF configuration
# --------------------------------------------------

CFG_FILE = OUT / "benchmark_botsort.yaml"
assert CFG_FILE.exists()

with open(CFG_FILE) as f:
    cfg = yaml.safe_load(f)

assert cfg["tracker_type"] == "botsort"
assert cfg.get("with_reid") is False

cfg = dict(cfg)
cfg["gmc_method"] = "none"

PROFILE_CFG = OUT / "CELL79_GMC_OFF.yaml"

with open(PROFILE_CFG, "w") as f:
    yaml.safe_dump(cfg, f)

# --------------------------------------------------
# 2. Load VAL frames
# --------------------------------------------------

IMG_DIR = find_seq_dir(EVAL_SEQUENCE)

names = sorted(
    [
        n for n in list_frames(IMG_DIR)
        if 1 <= int(Path(n).stem) <= 110
    ],
    key=lambda n: int(Path(n).stem)
)

assert len(names) == 110
assert [int(Path(n).stem) for n in names] == list(range(1, 111))

frames = []

for name in names:
    img = cv2.imread(os.path.join(IMG_DIR, name))
    assert img is not None, name
    frames.append(img)

# --------------------------------------------------
# 3. Profile BOTSORT.update()
# --------------------------------------------------

model = YOLO(best_weights("yolov8m"))
model.overrides["with_reid"] = False

profiler = cProfile.Profile()

original_update = BOTSORT.update

stats = {
    "tracker_calls": 0,
    "tracker_wall_ms": [],
    "total_pipeline_ms": []
}

active = {"enabled": False}

def profiled_update(self, *args, **kwargs):

    if not active["enabled"]:
        return original_update(self, *args, **kwargs)

    torch.cuda.synchronize()
    start = time.perf_counter()

    profiler.enable()

    try:
        return original_update(self, *args, **kwargs)
    finally:
        profiler.disable()

        torch.cuda.synchronize()

        stats["tracker_wall_ms"].append(
            (time.perf_counter() - start) * 1000
        )

        stats["tracker_calls"] += 1

print("=== CELL 79 — BoT-SORT INTERNAL CPU PROFILE ===")
print("GPU:", torch.cuda.get_device_name(0))
print("Frames: 10 warmup + 100 measured")
print("GMC: OFF")

with patch.object(BOTSORT, "update", profiled_update):

    for i, frame in enumerate(frames):

        active["enabled"] = i >= 10

        torch.cuda.synchronize()
        start = time.perf_counter()

        model.track(
            frame,
            persist=True,
            tracker=str(PROFILE_CFG),
            imgsz=IMG_SIZE,
            conf=0.01,
            iou=0.7,
            classes=[0],
            max_det=MAX_DET,
            device=0,
            verbose=False
        )

        torch.cuda.synchronize()

        if active["enabled"]:
            stats["total_pipeline_ms"].append(
                (time.perf_counter() - start) * 1000
            )

        if (i + 1) % 25 == 0 or i == 110:
            print(f"Processed: {i + 1}/110")

# --------------------------------------------------
# 4. Detailed function-level report
# --------------------------------------------------

stream = io.StringIO()

profile_stats = pstats.Stats(
    profiler,
    stream=stream
)

profile_stats.sort_stats("cumulative")
profile_stats.print_stats(35)

report = stream.getvalue()

print("\n=== TOP 35 FUNCTIONS BY CUMULATIVE TIME ===")
print(report)

# --------------------------------------------------
# 5. Summary
# --------------------------------------------------

import numpy as np

summary = {
    "split": "VAL",
    "frames_measured": 100,
    "tracker_calls": stats["tracker_calls"],
    "tracker_wall_ms": float(
        np.mean(stats["tracker_wall_ms"])
    ),
    "pipeline_wall_ms": float(
        np.mean(stats["total_pipeline_ms"])
    ),
    "pipeline_fps_instrumented": float(
        1000 / np.mean(stats["total_pipeline_ms"])
    ),
    "gmc_method": "none",
    "profiling_note": (
        "cProfile measures Python function execution. "
        "Profiler overhead affects measured latency. "
        "Cumulative times include nested calls."
    )
}

print("\n=== SUMMARY ===")

for key, value in summary.items():
    print(f"{key}: {value}")

# --------------------------------------------------
# 6. Save results
# --------------------------------------------------

REPORT_FILE = OUT / "VAL_BoTSORT_cProfile_report.txt"
JSON_FILE = OUT / "VAL_BoTSORT_cProfile_summary.json"
PROFILE_FILE = OUT / "VAL_BoTSORT_cProfile.prof"

REPORT_FILE.write_text(report)

with open(JSON_FILE, "w") as f:
    json.dump(summary, f, indent=2)

profiler.dump_stats(str(PROFILE_FILE))

print("\nSaved:")
print(REPORT_FILE)
print(JSON_FILE)
print(PROFILE_FILE)

print("\nOriginal files unchanged.")
print("TEST not accessed.")

=== CELL 79 — BoT-SORT INTERNAL CPU PROFILE ===
GPU: NVIDIA A100-SXM4-40GB
Frames: 10 warmup + 100 measured
GMC: OFF
Processed: 25/110
Processed: 50/110
Processed: 75/110
Processed: 100/110

=== TOP 35 FUNCTIONS BY CUMULATIVE TIME ===
         2240550 function calls (2240234 primitive calls) in 3.803 seconds

   Ordered by: cumulative time
   List reduced from 228 to 35 due to restriction <35>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      100    0.002    0.000    2.528    0.025 /usr/local/lib/python3.13/dist-packages/ultralytics/trackers/byte_tracker.py:388(_first_association)
    22560    0.030    0.000    2.081    0.000 /usr/local/lib/python3.13/dist-packages/ultralytics/trackers/byte_tracker.py:429(_apply_match)
      199    0.021    0.000    2.051    0.010 /usr/local/lib/python3.13/dist-packages/ultralytics/trackers/byte_tracker.py:417(_apply_matches)
    22478    0.032    0.000    2.037    0.000 /usr/local/lib/python3.13/dist-packages/ultralytics/t

In [ ]:
# CELL 80 — KALMAN FILTER BATCHING FEASIBILITY AUDIT
# Inspection only | No inference | No tracker modifications | No TEST

import inspect
import json
import ultralytics
import numpy as np
from pathlib import Path

from ultralytics.trackers.utils.kalman_filter import (
    KalmanFilterXYAH,
    KalmanFilterXYWH
)
from ultralytics.trackers.bot_sort import BOTrack, BOTSORT
from ultralytics.trackers.byte_tracker import STrack, BYTETracker

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "benchmark"
OUT.mkdir(parents=True, exist_ok=True)

print("=== CELL 80 — KALMAN FILTER FEASIBILITY ===")
print("Ultralytics version:", ultralytics.__version__)
print("NumPy version:", np.__version__)

def inspect_method(cls, method):
    fn = getattr(cls, method, None)

    if fn is None:
        print(f"\n{cls.__name__}.{method}: NOT AVAILABLE")
        return None

    source = inspect.getsource(fn)

    print(f"\n{'=' * 65}")
    print(f"{cls.__name__}.{method}")
    print(f"{'=' * 65}")
    print(source)

    return source

sources = {}

targets = [
    (KalmanFilterXYWH, "update"),
    (KalmanFilterXYWH, "project"),
    (KalmanFilterXYWH, "multi_predict"),
    (KalmanFilterXYAH, "update"),
    (BOTrack, "update"),
    (BOTrack, "multi_predict"),
    (BOTSORT, "update"),
]

for cls, method in targets:
    key = f"{cls.__name__}.{method}"
    sources[key] = inspect_method(cls, method)

# Check existing batch APIs
print("\n=== AVAILABLE BATCH-RELATED METHODS ===")

for cls in [
    KalmanFilterXYWH,
    KalmanFilterXYAH,
    BOTrack,
    STrack,
    BOTSORT,
    BYTETracker
]:
    methods = [
        name for name in dir(cls)
        if any(
            word in name.lower()
            for word in ["multi", "batch", "update", "predict"]
        )
        and callable(getattr(cls, name, None))
    ]

    print(f"{cls.__name__}: {methods}")

# Inspect tracker Kalman configuration
print("\n=== TRACKER KALMAN CONFIGURATION ===")

for cls in [BOTrack, BOTSORT]:
    try:
        print(f"{cls.__name__} source file:",
              inspect.getfile(cls))
    except Exception as e:
        print("Inspection warning:", e)

# Previous profiling evidence
profile_summary = OUT / "VAL_BoTSORT_cProfile_summary.json"

if profile_summary.exists():
    with open(profile_summary) as f:
        previous = json.load(f)

    print("\n=== PREVIOUS PROFILE ===")
    print("Tracker calls:",
          previous.get("tracker_calls"))
    print("Tracker wall time:",
          previous.get("tracker_wall_ms"))
else:
    print("\nPrevious profile summary not found.")

# Save inspection results
report = {
    "ultralytics_version": ultralytics.__version__,
    "numpy_version": np.__version__,
    "methods": sources,
    "note": (
        "Static inspection only. No batching implementation "
        "or performance optimization has been applied."
    )
}

FILE = OUT / "CELL80_Kalman_batching_feasibility.json"

with open(FILE, "w") as f:
    json.dump(report, f, indent=2)

print("\nSaved:", FILE)
print("No model inference performed.")
print("No original code or results changed.")
print("No TEST access.")

=== CELL 80 — KALMAN FILTER FEASIBILITY ===
Ultralytics version: 8.4.183
NumPy version: 2.1.3

KalmanFilterXYWH.update
    def update(
        self, mean: np.ndarray, covariance: np.ndarray, measurement: np.ndarray, confidence: float | None = None
    ):
        """Run Kalman filter correction step.

        Args:
            mean (np.ndarray): The predicted state's mean vector (8 dimensional).
            covariance (np.ndarray): The state's covariance matrix (8x8 dimensional).
            measurement (np.ndarray): The 4 dimensional measurement vector (x, y, w, h), where (x, y) is the center
                position, w the width, and h the height of the bounding box.
            confidence (float, optional): Detection confidence; when set, scales measurement noise by max(1 -
                confidence, 0.05) (NSA-Kalman).

        Returns:
            new_mean (np.ndarray): Measurement-corrected state mean.
            new_covariance (np.ndarray): Measurement-corrected state covarianc

In [ ]:
# CELL 81 — BATCHED KALMAN FILTER EXPERIMENT
# Synthetic data only | No YOLO | No TEST | No tracker modifications

import time
import json
import numpy as np
from pathlib import Path

from ultralytics.trackers.utils.kalman_filter import KalmanFilterXYWH

OUT = Path(
    "/content/drive/MyDrive/Crowd_AI_Project/runs_fresh/benchmark"
)
OUT.mkdir(parents=True, exist_ok=True)

kf = KalmanFilterXYWH()
rng = np.random.default_rng(42)

def original_update_batch(means, covariances, measurements, confidences):
    new_means = []
    new_covariances = []

    for m, p, z, c in zip(
        means, covariances, measurements, confidences
    ):
        nm, np_cov = kf.update(m, p, z, confidence=float(c))
        new_means.append(nm)
        new_covariances.append(np_cov)

    return np.stack(new_means), np.stack(new_covariances)


def vectorized_update(means, covariances, measurements, confidences):
    # Same XYWH projection and confidence-dependent measurement noise
    n = len(means)

    std = np.stack([
        kf._std_weight_position * means[:, 2],
        kf._std_weight_position * means[:, 3],
        kf._std_weight_position * means[:, 2],
        kf._std_weight_position * means[:, 3],
    ], axis=1)

    noise = np.zeros((n, 4, 4), dtype=np.float64)
    idx = np.arange(4)
    noise[:, idx, idx] = std ** 2

    scale = np.maximum(1.0 - confidences, 0.05)
    noise *= scale[:, None, None]

    projected_means = means[:, :4].copy()
    projected_covs = covariances[:, :4, :4] + noise

    # Batched linear solve
    cross_cov = covariances[:, :, :4]

    gains = np.linalg.solve(
        projected_covs,
        cross_cov.transpose(0, 2, 1)
    ).transpose(0, 2, 1)

    innovation = measurements - projected_means

    new_means = means + np.einsum(
        "ni,nji->nj", innovation, gains
    )

    correction = np.matmul(
        np.matmul(gains, projected_covs),
        gains.transpose(0, 2, 1)
    )

    new_covs = covariances - correction

    return new_means, new_covs


def make_data(n):
    means = np.zeros((n, 8), dtype=np.float64)
    means[:, :2] = rng.uniform(100, 1000, (n, 2))
    means[:, 2:4] = rng.uniform(20, 150, (n, 2))
    means[:, 4:] = rng.normal(0, 1, (n, 4))

    a = rng.normal(size=(n, 8, 8))
    covariances = (
        np.matmul(a, a.transpose(0, 2, 1))
        + np.eye(8)[None] * 0.1
    )

    measurements = means[:, :4] + rng.normal(
        0, 2, (n, 4)
    )

    confidences = rng.uniform(0.2, 0.99, n)

    return means, covariances, measurements, confidences


results = []

print("=== CELL 81 — BATCHED KALMAN TEST ===")

for n in [50, 100, 200, 400]:
    data = make_data(n)

    # Verify numerical correctness
    original_m, original_p = original_update_batch(*data)
    batched_m, batched_p = vectorized_update(*data)

    mean_error = float(np.max(np.abs(original_m - batched_m)))
    cov_error = float(np.max(np.abs(original_p - batched_p)))

    equivalent = (
        np.allclose(original_m, batched_m, rtol=1e-9, atol=1e-8)
        and np.allclose(original_p, batched_p, rtol=1e-9, atol=1e-8)
    )

    # Warm up both implementations
    for _ in range(3):
        original_update_batch(*data)
        vectorized_update(*data)

    repeats = 10
    original_times = []
    batched_times = []

    for _ in range(repeats):
        t0 = time.perf_counter()
        original_update_batch(*data)
        original_times.append((time.perf_counter() - t0) * 1000)

        t0 = time.perf_counter()
        vectorized_update(*data)
        batched_times.append((time.perf_counter() - t0) * 1000)

    original_ms = float(np.median(original_times))
    batched_ms = float(np.median(batched_times))

    speedup = original_ms / batched_ms

    result = {
        "tracks": n,
        "original_ms": round(original_ms, 4),
        "batched_ms": round(batched_ms, 4),
        "speedup": round(speedup, 3),
        "max_mean_error": mean_error,
        "max_covariance_error": cov_error,
        "equivalent": bool(equivalent)
    }

    results.append(result)

    print(
        f"{n:3d} tracks | "
        f"Original: {original_ms:.3f} ms | "
        f"Batched: {batched_ms:.3f} ms | "
        f"Speedup: {speedup:.2f}x | "
        f"Equivalent: {equivalent}"
    )

    assert equivalent, f"Numerical mismatch at {n} tracks"

path = OUT / "CELL81_batched_kalman_experiment.json"

with open(path, "w") as f:
    json.dump(results, f, indent=2)

print("\nALL NUMERICAL TESTS PASSED")
print("Saved:", path)
print("Original BoT-SORT unchanged.")
print("TEST not accessed.")

=== CELL 81 — BATCHED KALMAN TEST ===
 50 tracks | Original: 2.389 ms | Batched: 0.189 ms | Speedup: 12.62x | Equivalent: True
100 tracks | Original: 4.415 ms | Batched: 0.267 ms | Speedup: 16.51x | Equivalent: True
200 tracks | Original: 9.189 ms | Batched: 0.432 ms | Speedup: 21.28x | Equivalent: True
400 tracks | Original: 18.326 ms | Batched: 0.817 ms | Speedup: 22.43x | Equivalent: True

ALL NUMERICAL TESTS PASSED
Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/benchmark/CELL81_batched_kalman_experiment.json
Original BoT-SORT unchanged.
TEST not accessed.


In [ ]:
# CELL 82 — SAFE BATCHED KALMAN INTEGRATION AUDIT
# No inference | No TEST | No original library modifications

import inspect
import json
import textwrap
from pathlib import Path

from ultralytics.trackers.byte_tracker import BYTETracker, STrack
from ultralytics.trackers.bot_sort import BOTSORT, BOTrack
from ultralytics.trackers.utils.kalman_filter import KalmanFilterXYWH

OUT = Path(
    "/content/drive/MyDrive/Crowd_AI_Project/runs_fresh/benchmark"
)
OUT.mkdir(parents=True, exist_ok=True)

targets = [
    (BYTETracker, "_apply_match"),
    (BYTETracker, "_apply_matches"),
    (BYTETracker, "_first_association"),
    (BYTETracker, "_second_association"),
    (BYTETracker, "_unconfirmed_association"),
    (STrack, "update"),
    (STrack, "re_activate"),
    (BOTrack, "update"),
    (BOTrack, "re_activate"),
    (KalmanFilterXYWH, "update"),
]

report = {}

print("=== CELL 82 — BATCHED KALMAN INTEGRATION AUDIT ===")

for cls, method in targets:

    key = f"{cls.__name__}.{method}"

    print("\n" + "=" * 70)
    print(key)
    print("=" * 70)

    try:
        fn = getattr(cls, method)
        source = inspect.getsource(fn)
        owner = next(
            (
                parent.__name__
                for parent in cls.__mro__
                if method in parent.__dict__
            ),
            "Unknown"
        )

        print("Actual implementation owner:", owner)
        print(textwrap.dedent(source))

        report[key] = {
            "owner": owner,
            "source": source
        }

    except Exception as e:
        print("Unavailable:", str(e))
        report[key] = {
            "error": str(e)
        }

print("\n=== CLASS INHERITANCE ===")

for cls in [BOTSORT, BOTrack, STrack]:
    names = [c.__name__ for c in cls.__mro__]
    print(cls.__name__, "→", " → ".join(names))

    report[f"{cls.__name__}_mro"] = names

print("\n=== CELL 81 RESULT CHECK ===")

cell81 = OUT / "CELL81_batched_kalman_experiment.json"

if cell81.exists():
    with open(cell81) as f:
        results = json.load(f)

    for item in results:
        print(
            f"{item['tracks']} tracks | "
            f"{item['speedup']}x | "
            f"Equivalent: {item['equivalent']}"
        )

    report["cell81_results"] = results

else:
    print("CELL 81 result file not found.")

file = OUT / "CELL82_batch_integration_audit.json"

with open(file, "w") as f:
    json.dump(report, f, indent=2)

print("\nSaved:", file)
print("No inference performed.")
print("Original tracker unchanged.")
print("TEST not accessed.")

=== CELL 82 — BATCHED KALMAN INTEGRATION AUDIT ===

BYTETracker._apply_match
Actual implementation owner: BYTETracker
def _apply_match(self, track: STrack, det: STrack, activated: list[STrack], refind: list[STrack]) -> None:
    """Update or re-activate a single track with its matched detection."""
    if track.state == TrackState.Tracked:
        track.update(det, self.frame_id)
        activated.append(track)
    else:
        track.re_activate(det, self.frame_id, new_id=False)
        refind.append(track)


BYTETracker._apply_matches
Actual implementation owner: BYTETracker
def _apply_matches(
    self,
    matches: list[list[int]] | np.ndarray,
    pool: list[STrack],
    detections: list[STrack],
    activated: list[STrack],
    refind: list[STrack],
) -> None:
    """Apply a list of matched (track, detection) pairs from an association stage."""
    for itracked, idet in matches:
        self._apply_match(pool[itracked], detections[idet], activated, refind)


BYTETracker._first_as

In [ ]:
# CELL 83 — TRACK UPDATE INTEGRATION REVIEW
# No inference | No TEST | No tracker modifications

import json
from pathlib import Path

OUT = Path(
    "/content/drive/MyDrive/Crowd_AI_Project/runs_fresh/benchmark"
)

audit_path = OUT / "CELL82_batch_integration_audit.json"

assert audit_path.exists(), "CELL 82 audit file not found."

with open(audit_path) as f:
    audit = json.load(f)

methods = [
    "BYTETracker._apply_match",
    "BYTETracker._apply_matches",
    "BYTETracker._first_association",
    "BYTETracker._second_association",
    "BYTETracker._unconfirmed_association",
    "STrack.update",
    "STrack.re_activate",
]

print("=== CELL 83 — TRACK UPDATE INTEGRATION REVIEW ===")

for name in methods:
    entry = audit.get(name)

    print("\n" + "=" * 65)
    print(name)
    print("=" * 65)

    if not entry:
        print("NOT FOUND")
        continue

    if "error" in entry:
        print("ERROR:", entry["error"])
        continue

    print("Implementation owner:", entry.get("owner"))
    print(entry.get("source", "SOURCE NOT AVAILABLE"))

print("\nNo inference performed.")
print("Original tracker unchanged.")
print("TEST not accessed.")

=== CELL 83 — TRACK UPDATE INTEGRATION REVIEW ===

BYTETracker._apply_match
Implementation owner: BYTETracker
    def _apply_match(self, track: STrack, det: STrack, activated: list[STrack], refind: list[STrack]) -> None:
        """Update or re-activate a single track with its matched detection."""
        if track.state == TrackState.Tracked:
            track.update(det, self.frame_id)
            activated.append(track)
        else:
            track.re_activate(det, self.frame_id, new_id=False)
            refind.append(track)


BYTETracker._apply_matches
Implementation owner: BYTETracker
    def _apply_matches(
        self,
        matches: list[list[int]] | np.ndarray,
        pool: list[STrack],
        detections: list[STrack],
        activated: list[STrack],
        refind: list[STrack],
    ) -> None:
        """Apply a list of matched (track, detection) pairs from an association stage."""
        for itracked, idet in matches:
            self._apply_match(pool[itracked],

In [ ]:
# CELL 84 — BATCHED KALMAN TRACK UPDATE PROTOTYPE
# Synthetic tracks only | No YOLO | No TEST | No library modifications

import json
import time
import numpy as np
from pathlib import Path

from ultralytics.trackers.utils.kalman_filter import KalmanFilterXYWH

OUT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh/benchmark")
OUT.mkdir(parents=True, exist_ok=True)

kf = KalmanFilterXYWH()
rng = np.random.default_rng(84)

def batched_correction(means, covariances, measurements):
    """Vectorized equivalent of KalmanFilterXYWH.update, confidence=None."""

    means = np.asarray(means, dtype=np.float64)
    covariances = np.asarray(covariances, dtype=np.float64)
    measurements = np.asarray(measurements, dtype=np.float64)

    n = len(means)

    std = np.stack([
        kf._std_weight_position * means[:, 2],
        kf._std_weight_position * means[:, 3],
        kf._std_weight_position * means[:, 2],
        kf._std_weight_position * means[:, 3]
    ], axis=1)

    noise = np.zeros((n, 4, 4), dtype=np.float64)
    idx = np.arange(4)
    noise[:, idx, idx] = std ** 2

    projected_cov = covariances[:, :4, :4] + noise

    gains = np.linalg.solve(
        projected_cov,
        covariances[:, :, :4].transpose(0, 2, 1)
    ).transpose(0, 2, 1)

    innovations = measurements - means[:, :4]

    corrected_means = means + np.einsum(
        "ni,nji->nj", innovations, gains
    )

    corrected_covariances = covariances - (
        gains @ projected_cov @ gains.transpose(0, 2, 1)
    )

    return corrected_means, corrected_covariances


def make_tracks(n):
    means = np.zeros((n, 8), dtype=np.float64)
    means[:, :2] = rng.uniform(100, 1200, (n, 2))
    means[:, 2:4] = rng.uniform(20, 140, (n, 2))
    means[:, 4:] = rng.normal(size=(n, 4))

    a = rng.normal(size=(n, 8, 8))
    covariances = a @ a.transpose(0, 2, 1)
    covariances += np.eye(8)[None] * 0.1

    measurements = means[:, :4] + rng.normal(
        0, 2, (n, 4)
    )

    return means, covariances, measurements


print("=== CELL 84 — BATCHED TRACK UPDATE PROTOTYPE ===")

results = []

for n in [50, 100, 200, 400]:
    means, covs, measurements = make_tracks(n)

    # Original sequential updates
    original_means = []
    original_covs = []

    for i in range(n):
        m, c = kf.update(
            means[i],
            covs[i],
            measurements[i]
        )
        original_means.append(m)
        original_covs.append(c)

    original_means = np.stack(original_means)
    original_covs = np.stack(original_covs)

    # Experimental batch
    batch_means, batch_covs = batched_correction(
        means, covs, measurements
    )

    mean_error = float(
        np.max(np.abs(original_means - batch_means))
    )
    cov_error = float(
        np.max(np.abs(original_covs - batch_covs))
    )

    passed = (
        np.allclose(original_means, batch_means, rtol=1e-9, atol=1e-8)
        and np.allclose(original_covs, batch_covs, rtol=1e-9, atol=1e-8)
    )

    print(
        f"{n:3d} tracks | "
        f"Mean error: {mean_error:.2e} | "
        f"Cov error: {cov_error:.2e} | "
        f"Equivalent: {passed}"
    )

    results.append({
        "tracks": n,
        "mean_error": mean_error,
        "covariance_error": cov_error,
        "equivalent": bool(passed)
    })

    assert passed, f"Numerical mismatch at {n} tracks"

file = OUT / "CELL84_batch_track_update_prototype.json"

with open(file, "w") as f:
    json.dump(results, f, indent=2)

print("\nALL TESTS PASSED")
print("Saved:", file)
print("Original BoT-SORT unchanged.")
print("No inference. No TEST.")

=== CELL 84 — BATCHED TRACK UPDATE PROTOTYPE ===
 50 tracks | Mean error: 4.44e-16 | Cov error: 1.78e-15 | Equivalent: True
100 tracks | Mean error: 4.44e-16 | Cov error: 1.78e-15 | Equivalent: True
200 tracks | Mean error: 1.42e-14 | Cov error: 1.78e-15 | Equivalent: True
400 tracks | Mean error: 3.55e-15 | Cov error: 1.78e-15 | Equivalent: True

ALL TESTS PASSED
Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/benchmark/CELL84_batch_track_update_prototype.json
Original BoT-SORT unchanged.
No inference. No TEST.


In [ ]:
# CELL 85 — FINAL BATCH INTEGRATION REVIEW
# No inference | No TEST | No tracker modifications

import json
import inspect
from pathlib import Path

from ultralytics.trackers.byte_tracker import BYTETracker
from ultralytics.trackers.bot_sort import BOTSORT

OUT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh/benchmark")
OUT.mkdir(parents=True, exist_ok=True)

methods = [
    "_apply_match",
    "_apply_matches",
    "_first_association",
    "_second_association",
    "_unconfirmed_association",
]

report = {}

print("=== CELL 85 — MATCHING AND UPDATE FLOW ===")

for name in methods:
    fn = getattr(BOTSORT, name)
    source = inspect.getsource(fn)

    owner = next(
        cls.__name__
        for cls in BOTSORT.__mro__
        if name in cls.__dict__
    )

    report[name] = {
        "owner": owner,
        "source": source,
        "signature": str(inspect.signature(fn))
    }

    print("\n" + "=" * 65)
    print(name, "| Owner:", owner)
    print("=" * 65)
    print(source)

file = OUT / "CELL85_matching_update_flow.json"

with open(file, "w") as f:
    json.dump(report, f, indent=2)

print("\nSaved:", file)
print("Original tracker unchanged.")
print("No inference. No TEST.")

=== CELL 85 — MATCHING AND UPDATE FLOW ===

_apply_match | Owner: BYTETracker
    def _apply_match(self, track: STrack, det: STrack, activated: list[STrack], refind: list[STrack]) -> None:
        """Update or re-activate a single track with its matched detection."""
        if track.state == TrackState.Tracked:
            track.update(det, self.frame_id)
            activated.append(track)
        else:
            track.re_activate(det, self.frame_id, new_id=False)
            refind.append(track)


_apply_matches | Owner: BYTETracker
    def _apply_matches(
        self,
        matches: list[list[int]] | np.ndarray,
        pool: list[STrack],
        detections: list[STrack],
        activated: list[STrack],
        refind: list[STrack],
    ) -> None:
        """Apply a list of matched (track, detection) pairs from an association stage."""
        for itracked, idet in matches:
            self._apply_match(pool[itracked], detections[idet], activated, refind)


_first_associatio

In [ ]:
# CELL 86 — EXPERIMENTAL BATCHED BoT-SORT
# Synthetic integration test | No YOLO | No TEST
# Original Ultralytics library remains unchanged.

import json
import numpy as np
from pathlib import Path

from ultralytics.trackers.bot_sort import BOTSORT, BOTrack
from ultralytics.trackers.byte_tracker import STrack
from ultralytics.trackers.basetrack import TrackState

OUT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh/benchmark")
OUT.mkdir(parents=True, exist_ok=True)


def batch_kalman_update(kf, means, covariances, measurements):
    """Vectorized Kalman XYWH correction, confidence=None."""

    means = np.asarray(means, dtype=np.float64)
    covariances = np.asarray(covariances, dtype=np.float64)
    measurements = np.asarray(measurements, dtype=np.float64)

    n = len(means)

    std = np.stack([
        kf._std_weight_position * means[:, 2],
        kf._std_weight_position * means[:, 3],
        kf._std_weight_position * means[:, 2],
        kf._std_weight_position * means[:, 3]
    ], axis=1)

    noise = np.zeros((n, 4, 4), dtype=np.float64)
    idx = np.arange(4)
    noise[:, idx, idx] = std ** 2

    projected_cov = covariances[:, :4, :4] + noise

    gains = np.linalg.solve(
        projected_cov,
        covariances[:, :, :4].transpose(0, 2, 1)
    ).transpose(0, 2, 1)

    innovation = measurements - means[:, :4]

    new_means = means + np.einsum(
        "ni,nji->nj", innovation, gains
    )

    new_covariances = covariances - (
        gains @ projected_cov @ gains.transpose(0, 2, 1)
    )

    return new_means, new_covariances


class BatchedBOTSORT(BOTSORT):
    """Experimental BoT-SORT with batched matched-track corrections."""

    def _apply_matches(
        self, matches, pool, detections, activated, refind
    ):
        if len(matches) == 0:
            return

        pairs = [
            (pool[int(i)], detections[int(j)])
            for i, j in matches
        ]

        # Preserve original behavior for small batches.
        if len(pairs) < 2:
            for track, det in pairs:
                self._apply_match(track, det, activated, refind)
            return

        # All active tracks in a BoT-SORT instance share the XYWH filter.
        tracks = [track for track, _ in pairs]

        means = np.stack([track.mean for track in tracks])
        covs = np.stack([track.covariance for track in tracks])

        measurements = np.stack([
            track.convert_coords(det.tlwh)
            for track, det in pairs
        ])

        corrected_means, corrected_covs = batch_kalman_update(
            tracks[0].kalman_filter,
            means,
            covs,
            measurements
        )

        # Apply original metadata updates in the same match order.
        # Temporarily replace each track's correction with its precomputed result.
        for index, (track, det) in enumerate(pairs):

            class PrecomputedKalman:
                def __init__(self, mean, covariance):
                    self.mean = mean
                    self.covariance = covariance

                def update(self, *args, **kwargs):
                    return self.mean, self.covariance

            original_kf = track.kalman_filter

            try:
                track.kalman_filter = PrecomputedKalman(
                    corrected_means[index],
                    corrected_covs[index]
                )
                self._apply_match(track, det, activated, refind)
            finally:
                track.kalman_filter = original_kf


print("=== CELL 86 — EXPERIMENTAL TRACKER CREATED ===")
print("Class:", BatchedBOTSORT.__name__)
print("Parent:", BatchedBOTSORT.__mro__[1].__name__)
print("Original BOTSORT class unchanged.")

# Verify batch mathematics using actual Kalman initialization.
from ultralytics.trackers.utils.kalman_filter import KalmanFilterXYWH

kf = KalmanFilterXYWH()
rng = np.random.default_rng(86)

means = []
covariances = []
measurements = []

for _ in range(100):
    xywh = np.array([
        rng.uniform(100, 1000),
        rng.uniform(100, 700),
        rng.uniform(20, 100),
        rng.uniform(40, 150)
    ])

    mean, cov = kf.initiate(xywh)

    means.append(mean)
    covariances.append(cov)
    measurements.append(
        xywh + rng.normal(0, 1, size=4)
    )

means = np.stack(means)
covariances = np.stack(covariances)
measurements = np.stack(measurements)

original = [
    kf.update(means[i], covariances[i], measurements[i])
    for i in range(100)
]

batch_mean, batch_cov = batch_kalman_update(
    kf, means, covariances, measurements
)

mean_error = float(np.max(np.abs(
    np.stack([x[0] for x in original]) - batch_mean
)))

cov_error = float(np.max(np.abs(
    np.stack([x[1] for x in original]) - batch_cov
)))

equivalent = (
    mean_error < 1e-8
    and cov_error < 1e-8
)

print("\n=== NUMERICAL INTEGRATION TEST ===")
print("Tracks:", 100)
print("Mean error:", mean_error)
print("Covariance error:", cov_error)
print("Equivalent:", equivalent)

assert equivalent, "Batched correction failed numerical equivalence."

report = {
    "cell": 86,
    "tracks_tested": 100,
    "mean_error": mean_error,
    "covariance_error": cov_error,
    "equivalent": equivalent,
    "tracker_class": "BatchedBOTSORT",
    "original_library_modified": False,
    "test_data_accessed": False
}

path = OUT / "CELL86_batched_tracker_prototype.json"

with open(path, "w") as f:
    json.dump(report, f, indent=2)

print("\nPASS — Experimental tracker prototype ready.")
print("Saved:", path)
print("No inference. No TEST.")

=== CELL 86 — EXPERIMENTAL TRACKER CREATED ===
Class: BatchedBOTSORT
Parent: BOTSORT
Original BOTSORT class unchanged.

=== NUMERICAL INTEGRATION TEST ===
Tracks: 100
Mean error: 0.0
Covariance error: 0.0
Equivalent: True

PASS — Experimental tracker prototype ready.
Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/benchmark/CELL86_batched_tracker_prototype.json
No inference. No TEST.


In [ ]:
# CELL 87 — ORIGINAL VS BATCHED BoT-SORT
# VAL 1–300 | Cached YOLO detections | GMC OFF
# No YOLO inference | No TEST | No library modifications

import json
import time
import yaml
import inspect
import numpy as np
import pandas as pd

from pathlib import Path
from types import SimpleNamespace

from ultralytics.trackers.bot_sort import BOTSORT
from ultralytics.trackers.basetrack import BaseTrack

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "benchmark"
OUT.mkdir(parents=True, exist_ok=True)

assert "BatchedBOTSORT" in globals(), (
    "BatchedBOTSORT is missing. Run CELL 86 first."
)

# --------------------------------------------------
# 1. Tracker configuration
# --------------------------------------------------

with open(OUT / "benchmark_botsort.yaml") as f:
    cfg = yaml.safe_load(f)

assert cfg["tracker_type"] == "botsort"

cfg["gmc_method"] = "none"
cfg["with_reid"] = False

print("=== CELL 87 — TRACKER COMPARISON ===")
print("BOTSORT constructor:", inspect.signature(BOTSORT))

# --------------------------------------------------
# 2. Load cached detections
# --------------------------------------------------

det_path = (
    ROOT / "detection" /
    "detections_yolov8m_MOT20-05.parquet"
)

df = pd.read_parquet(det_path)

required = ["frame", "x1", "y1", "x2", "y2", "conf"]

assert set(required).issubset(df.columns)

val = df.loc[
    df["frame"].between(1, 300),
    required
].copy()

val = val.sort_values("frame", kind="stable")

valid = (
    np.isfinite(
        val[["x1", "y1", "x2", "y2", "conf"]]
    ).all(axis=1)
    & (val["x2"] > val["x1"])
    & (val["y2"] > val["y1"])
)

excluded = int((~valid).sum())
val = val.loc[valid].copy()

assert excluded == 7, (
    f"Expected 7 invalid boxes, found {excluded}"
)

print("VAL frames: 1–300")
print("Valid detections:", len(val))
print("Excluded invalid boxes:", excluded)
print("GMC: OFF")
print("ReID: OFF")

# --------------------------------------------------
# 3. Detection adapter
# --------------------------------------------------

class DetectionResults:

    def __init__(self, rows):

        coords = rows[
            ["x1", "y1", "x2", "y2"]
        ].to_numpy(dtype=np.float64)

        x1, y1, x2, y2 = coords.T

        self.xywh = np.column_stack([
            (x1 + x2) / 2,
            (y1 + y2) / 2,
            x2 - x1,
            y2 - y1
        ])

        self.conf = rows["conf"].to_numpy(
            dtype=np.float64
        )

        self.cls = np.zeros(
            len(rows), dtype=np.float64
        )

    def __len__(self):
        return len(self.conf)

    def __getitem__(self, index):

        result = DetectionResults.__new__(
            DetectionResults
        )

        result.xywh = self.xywh[index]
        result.conf = self.conf[index]
        result.cls = self.cls[index]

        return result


groups = {
    int(frame): group
    for frame, group in val.groupby(
        "frame", sort=False
    )
}

empty = val.iloc[:0]

# --------------------------------------------------
# 4. Initialize trackers correctly
# --------------------------------------------------

def create_tracker(tracker_class):

    signature = inspect.signature(tracker_class)

    kwargs = {
        "args": SimpleNamespace(**cfg)
    }

    # Pass frame_rate only if supported by this version.
    if "frame_rate" in signature.parameters:
        kwargs["frame_rate"] = 25

    if "reid_weights" in signature.parameters:
        kwargs["reid_weights"] = None

    return tracker_class(**kwargs)


def run_tracker(tracker_class):

    BaseTrack.reset_id()

    tracker = create_tracker(tracker_class)

    records = []
    timings = []

    for frame in range(1, 301):

        rows = groups.get(frame, empty)
        detections = DetectionResults(rows)

        start = time.perf_counter()

        output = tracker.update(detections)

        timings.append(
            (time.perf_counter() - start) * 1000
        )

        for item in output:

            records.append([
                frame,
                int(item[4]),
                float(item[0]),
                float(item[1]),
                float(item[2]),
                float(item[3]),
                float(item[5])
            ])

        if frame % 100 == 0:
            print(
                f"{tracker_class.__name__}: "
                f"{frame}/300 frames"
            )

    result = pd.DataFrame(
        records,
        columns=[
            "frame", "id",
            "x1", "y1", "x2", "y2", "score"
        ]
    )

    return result, np.asarray(timings)


# --------------------------------------------------
# 5. Run both trackers
# --------------------------------------------------

print("\nRunning original BoT-SORT...")
original, original_times = run_tracker(BOTSORT)

print("\nRunning experimental Batched BoT-SORT...")
batched, batched_times = run_tracker(
    BatchedBOTSORT
)

# --------------------------------------------------
# 6. Compare outputs
# --------------------------------------------------

keys = ["frame", "id"]
values = ["x1", "y1", "x2", "y2", "score"]

a = original.sort_values(
    keys
).reset_index(drop=True)

b = batched.sort_values(
    keys
).reset_index(drop=True)

same_shape = a.shape == b.shape

same_ids = (
    same_shape
    and a[keys].equals(b[keys])
)

same_boxes = (
    same_ids
    and np.allclose(
        a[values].to_numpy(),
        b[values].to_numpy(),
        rtol=1e-8,
        atol=1e-6
    )
)

max_box_error = None

if same_ids and len(a):
    max_box_error = float(
        np.max(np.abs(
            a[values].to_numpy()
            - b[values].to_numpy()
        ))
    )

original_ms = float(np.median(original_times))
batched_ms = float(np.median(batched_times))

report = {
    "split": "VAL",
    "frames": 300,
    "excluded_invalid_detections": excluded,
    "original_rows": int(len(a)),
    "batched_rows": int(len(b)),
    "same_shape": bool(same_shape),
    "same_ids": bool(same_ids),
    "same_boxes": bool(same_boxes),
    "max_box_error": max_box_error,
    "original_median_ms": original_ms,
    "batched_median_ms": batched_ms,
    "tracker_only_speedup": (
        original_ms / batched_ms
        if batched_ms > 0 else None
    )
}

print("\n=== RESULTS ===")

for key, value in report.items():
    print(f"{key}: {value}")

# --------------------------------------------------
# 7. Save results
# --------------------------------------------------

original.to_csv(
    OUT / "CELL87_original_VAL_tracks.csv",
    index=False
)

batched.to_csv(
    OUT / "CELL87_batched_VAL_tracks.csv",
    index=False
)

with open(
    OUT / "CELL87_tracker_equivalence.json",
    "w"
) as f:
    json.dump(report, f, indent=2)

print("\nOriginal detection cache unchanged.")
print("Original BoT-SORT unchanged.")
print("No YOLO inference.")
print("No TEST access.")

if same_ids and same_boxes:
    print("\nPASS — Tracking outputs equivalent.")
else:
    print("\nWARNING — Tracking outputs differ.")
    print("Experimental tracker not approved.")

=== CELL 87 — TRACKER COMPARISON ===
BOTSORT constructor: (args: 'Any')
VAL frames: 1–300
Valid detections: 165005
Excluded invalid boxes: 7
GMC: OFF
ReID: OFF

Running original BoT-SORT...
BOTSORT: 100/300 frames
BOTSORT: 200/300 frames
BOTSORT: 300/300 frames

Running experimental Batched BoT-SORT...
BatchedBOTSORT: 100/300 frames
BatchedBOTSORT: 200/300 frames
BatchedBOTSORT: 300/300 frames

=== RESULTS ===
split: VAL
frames: 300
excluded_invalid_detections: 7
original_rows: 63766
batched_rows: 63766
same_shape: True
same_ids: True
same_boxes: True
max_box_error: 0.0
original_median_ms: 21.653818000231695
batched_median_ms: 17.04895300008502
tracker_only_speedup: 1.2700966446516517

Original detection cache unchanged.
Original BoT-SORT unchanged.
No YOLO inference.
No TEST access.

PASS — Tracking outputs equivalent.


In [ ]:
# CELL 88 — FULL VAL: ORIGINAL VS BATCHED BoT-SORT
# 1657 VAL frames | Cached detections | GMC OFF
# No YOLO inference | No TEST | No library modifications

import json
import time
import yaml
import inspect
import numpy as np
import pandas as pd

from pathlib import Path
from types import SimpleNamespace

from ultralytics.trackers.bot_sort import BOTSORT
from ultralytics.trackers.basetrack import BaseTrack

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "benchmark"
OUT.mkdir(parents=True, exist_ok=True)

N_FRAMES = 1657

assert "BatchedBOTSORT" in globals(), (
    "BatchedBOTSORT is missing. Run CELL 86 first."
)

# --------------------------------------------------
# 1. Load tracker configuration
# --------------------------------------------------

with open(OUT / "benchmark_botsort.yaml") as f:
    cfg = yaml.safe_load(f)

assert cfg["tracker_type"] == "botsort"

cfg["gmc_method"] = "none"
cfg["with_reid"] = False

# --------------------------------------------------
# 2. Load cached YOLO detections
# --------------------------------------------------

det_path = (
    ROOT / "detection" /
    "detections_yolov8m_MOT20-05.parquet"
)

df = pd.read_parquet(det_path)

required = ["frame", "x1", "y1", "x2", "y2", "conf"]
assert set(required).issubset(df.columns)

val = df.loc[
    df["frame"].between(1, N_FRAMES),
    required
].copy()

val = val.sort_values("frame", kind="stable")

valid = (
    np.isfinite(
        val[["x1", "y1", "x2", "y2", "conf"]]
    ).all(axis=1)
    & (val["x2"] > val["x1"])
    & (val["y2"] > val["y1"])
)

excluded = int((~valid).sum())
val = val.loc[valid].copy()

assert not val.empty

print("=== CELL 88 — FULL VAL VERIFICATION ===")
print("Frames:", N_FRAMES)
print("Valid detections:", len(val))
print("Excluded invalid boxes:", excluded)
print("GMC: OFF")
print("ReID: OFF")

# --------------------------------------------------
# 3. Detection adapter
# --------------------------------------------------

class DetectionResults:

    def __init__(self, rows):

        coords = rows[
            ["x1", "y1", "x2", "y2"]
        ].to_numpy(dtype=np.float64)

        x1, y1, x2, y2 = coords.T

        self.xywh = np.column_stack([
            (x1 + x2) / 2,
            (y1 + y2) / 2,
            x2 - x1,
            y2 - y1
        ])

        self.conf = rows["conf"].to_numpy(
            dtype=np.float64
        )

        self.cls = np.zeros(
            len(rows), dtype=np.float64
        )

    def __len__(self):
        return len(self.conf)

    def __getitem__(self, index):

        result = DetectionResults.__new__(
            DetectionResults
        )

        result.xywh = self.xywh[index]
        result.conf = self.conf[index]
        result.cls = self.cls[index]

        return result


groups = {
    int(frame): group
    for frame, group in val.groupby(
        "frame", sort=False
    )
}

empty = val.iloc[:0]

# --------------------------------------------------
# 4. Tracker initialization
# --------------------------------------------------

def create_tracker(tracker_class):

    signature = inspect.signature(tracker_class)

    kwargs = {
        "args": SimpleNamespace(**cfg)
    }

    if "frame_rate" in signature.parameters:
        kwargs["frame_rate"] = 25

    return tracker_class(**kwargs)


# --------------------------------------------------
# 5. Run tracker across full VAL
# --------------------------------------------------

def run_tracker(tracker_class):

    BaseTrack.reset_id()

    tracker = create_tracker(tracker_class)

    records = []
    timings = []

    for frame in range(1, N_FRAMES + 1):

        detections = DetectionResults(
            groups.get(frame, empty)
        )

        start = time.perf_counter()

        output = tracker.update(detections)

        timings.append(
            (time.perf_counter() - start) * 1000
        )

        for item in output:

            records.append([
                frame,
                int(item[4]),
                float(item[0]),
                float(item[1]),
                float(item[2]),
                float(item[3]),
                float(item[5])
            ])

        if frame % 250 == 0 or frame == N_FRAMES:
            print(
                f"{tracker_class.__name__}: "
                f"{frame}/{N_FRAMES}"
            )

    result = pd.DataFrame(
        records,
        columns=[
            "frame", "id",
            "x1", "y1", "x2", "y2", "score"
        ]
    )

    return result, np.asarray(timings)


# --------------------------------------------------
# 6. Run original and experimental trackers
# --------------------------------------------------

print("\nRunning original BoT-SORT...")
original, original_times = run_tracker(BOTSORT)

print("\nRunning Batched BoT-SORT...")
batched, batched_times = run_tracker(BatchedBOTSORT)

# --------------------------------------------------
# 7. Compare full VAL tracking outputs
# --------------------------------------------------

keys = ["frame", "id"]
values = ["x1", "y1", "x2", "y2", "score"]

a = original.sort_values(keys).reset_index(drop=True)
b = batched.sort_values(keys).reset_index(drop=True)

same_shape = a.shape == b.shape

same_ids = (
    same_shape
    and a[keys].equals(b[keys])
)

same_boxes = (
    same_ids
    and np.allclose(
        a[values].to_numpy(),
        b[values].to_numpy(),
        rtol=1e-8,
        atol=1e-6
    )
)

max_box_error = None

if same_ids and len(a):
    max_box_error = float(
        np.max(np.abs(
            a[values].to_numpy()
            - b[values].to_numpy()
        ))
    )

# --------------------------------------------------
# 8. Speed statistics
# --------------------------------------------------

def timing_stats(times):
    return {
        "mean_ms": float(np.mean(times)),
        "median_ms": float(np.median(times)),
        "p95_ms": float(np.percentile(times, 95)),
        "total_seconds": float(np.sum(times) / 1000)
    }

original_stats = timing_stats(original_times)
batched_stats = timing_stats(batched_times)

speedup = (
    original_stats["mean_ms"] /
    batched_stats["mean_ms"]
)

report = {
    "split": "VAL",
    "frames": N_FRAMES,
    "valid_detections": int(len(val)),
    "excluded_invalid_detections": excluded,
    "original_rows": int(len(a)),
    "batched_rows": int(len(b)),
    "same_shape": bool(same_shape),
    "same_ids": bool(same_ids),
    "same_boxes": bool(same_boxes),
    "max_box_error": max_box_error,
    "original_timing": original_stats,
    "batched_timing": batched_stats,
    "tracker_only_speedup_mean": float(speedup),
    "equivalent": bool(same_ids and same_boxes)
}

# --------------------------------------------------
# 9. Print results
# --------------------------------------------------

print("\n=== FULL VAL EQUIVALENCE ===")
print("Original rows:", len(a))
print("Batched rows:", len(b))
print("Same IDs:", same_ids)
print("Same boxes:", same_boxes)
print("Max box error:", max_box_error)

print("\n=== TRACKER SPEED ===")

print(
    f"Original: "
    f"{original_stats['mean_ms']:.3f} ms/frame"
)

print(
    f"Batched:  "
    f"{batched_stats['mean_ms']:.3f} ms/frame"
)

print(f"Speedup: {speedup:.3f}x")

print(
    f"Time saved: "
    f"{original_stats['total_seconds'] - batched_stats['total_seconds']:.2f} seconds"
)

# --------------------------------------------------
# 10. Save results
# --------------------------------------------------

original.to_csv(
    OUT / "CELL88_original_FULL_VAL_tracks.csv",
    index=False
)

batched.to_csv(
    OUT / "CELL88_batched_FULL_VAL_tracks.csv",
    index=False
)

with open(
    OUT / "CELL88_full_VAL_verification.json",
    "w"
) as f:
    json.dump(report, f, indent=2)

print("\nSaved full VAL comparison.")
print("Original tracker unchanged.")
print("Cached detections unchanged.")
print("No YOLO inference.")
print("No TEST access.")

if report["equivalent"]:
    print("\nPASS — Full VAL tracking outputs equivalent.")
else:
    print("\nWARNING — Tracking outputs differ.")
    print("Do not adopt experimental tracker.")

=== CELL 88 — FULL VAL VERIFICATION ===
Frames: 1657
Valid detections: 857219
Excluded invalid boxes: 27
GMC: OFF
ReID: OFF

Running original BoT-SORT...
BOTSORT: 250/1657
BOTSORT: 500/1657
BOTSORT: 750/1657
BOTSORT: 1000/1657
BOTSORT: 1250/1657
BOTSORT: 1500/1657
BOTSORT: 1657/1657

Running Batched BoT-SORT...
BatchedBOTSORT: 250/1657
BatchedBOTSORT: 500/1657
BatchedBOTSORT: 750/1657
BatchedBOTSORT: 1000/1657
BatchedBOTSORT: 1250/1657
BatchedBOTSORT: 1500/1657
BatchedBOTSORT: 1657/1657

=== FULL VAL EQUIVALENCE ===
Original rows: 348720
Batched rows: 348720
Same IDs: True
Same boxes: True
Max box error: 0.0

=== TRACKER SPEED ===
Original: 21.492 ms/frame
Batched:  17.159 ms/frame
Speedup: 1.253x
Time saved: 7.18 seconds

Saved full VAL comparison.
Original tracker unchanged.
Cached detections unchanged.
No YOLO inference.
No TEST access.

PASS — Full VAL tracking outputs equivalent.


In [ ]:
# CELL 89 — PIPELINE SPEED BENCHMARK (CORRECTED)
# VAL only | YOLO once | Original vs Batched BoT-SORT
# No TEST | No training | No original library modifications

import json
import time
import yaml
import gc
import cv2
import numpy as np
import torch

from pathlib import Path
from types import SimpleNamespace
from ultralytics import YOLO
from ultralytics.trackers.bot_sort import BOTSORT
from ultralytics.trackers.basetrack import BaseTrack

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "benchmark"
OUT.mkdir(parents=True, exist_ok=True)

N_FRAMES = 1657
W, H = 1654, 1080
ZONE_COLS, ZONE_ROWS = 3, 4

assert "BatchedBOTSORT" in globals(), "Run CELL 86 first."
assert "best_weights" in globals()
assert "find_seq_dir" in globals()
assert "list_frames" in globals()

# --------------------------------------------------
# 1. Tracker configuration
# --------------------------------------------------

with open(OUT / "benchmark_botsort.yaml") as f:
    cfg = yaml.safe_load(f)

assert cfg["tracker_type"] == "botsort"

cfg["gmc_method"] = "none"
cfg["with_reid"] = False

# --------------------------------------------------
# 2. Resolve VAL image paths correctly
# --------------------------------------------------

seq_dir = Path(find_seq_dir(EVAL_SEQUENCE))
img_dir = seq_dir / "img1"

if not img_dir.is_dir():
    img_dir = seq_dir

raw_frames = list_frames(img_dir)
assert len(raw_frames) >= N_FRAMES

frames = []

for item in raw_frames[:N_FRAMES]:
    p = Path(item)

    if not p.is_file():
        p = img_dir / p.name

    assert p.is_file(), f"Frame not found: {p}"
    frames.append(p)

# Verify correct sequence ordering
assert frames[0].stem == "000001", (
    f"Unexpected first frame: {frames[0]}"
)

test_image = cv2.imread(str(frames[0]))
assert test_image is not None, (
    f"Cannot decode image: {frames[0]}"
)

assert test_image.shape[:2] == (H, W), (
    f"Unexpected image dimensions: {test_image.shape}"
)

print("=== CELL 89 — PIPELINE BENCHMARK ===")
print("Sequence:", EVAL_SEQUENCE)
print("VAL frames:", N_FRAMES)
print("First image:", frames[0])
print("Last image:", frames[-1])
print("GMC: OFF | ReID: OFF")

# --------------------------------------------------
# 3. Run YOLO once
# --------------------------------------------------

model = YOLO(str(best_weights("yolov8m")))
device = 0 if torch.cuda.is_available() else "cpu"

detections_by_frame = []
detector_times = []
decode_times = []

print("\nRunning YOLO once...")

for i, path in enumerate(frames, start=1):

    t_decode = time.perf_counter()
    image = cv2.imread(str(path))
    assert image is not None, f"Cannot read: {path}"
    decode_times.append(
        (time.perf_counter() - t_decode) * 1000
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    t0 = time.perf_counter()

    result = model.predict(
        image,
        imgsz=IMG_SIZE,
        conf=0.01,
        iou=0.7,
        classes=[0],
        max_det=MAX_DET,
        device=device,
        verbose=False
    )[0]

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    detector_times.append(
        (time.perf_counter() - t0) * 1000
    )

    boxes = result.boxes

    if boxes is None or len(boxes) == 0:
        arr = np.empty((0, 6), dtype=np.float64)
    else:
        arr = np.column_stack([
            boxes.xyxy.cpu().numpy(),
            boxes.conf.cpu().numpy(),
            boxes.cls.cpu().numpy()
        ]).astype(np.float64)

        valid = (
            np.isfinite(arr).all(axis=1)
            & (arr[:, 2] > arr[:, 0])
            & (arr[:, 3] > arr[:, 1])
        )
        arr = arr[valid]

    detections_by_frame.append(arr)

    if i % 100 == 0:
        print(f"YOLO: {i}/{N_FRAMES}")

del model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# --------------------------------------------------
# 4. Detection adapter
# --------------------------------------------------

class DetectionResults:

    def __init__(self, arr):
        self.xywh = np.column_stack([
            (arr[:, 0] + arr[:, 2]) / 2,
            (arr[:, 1] + arr[:, 3]) / 2,
            arr[:, 2] - arr[:, 0],
            arr[:, 3] - arr[:, 1]
        ])

        self.conf = arr[:, 4].copy()
        self.cls = arr[:, 5].copy()

    def __len__(self):
        return len(self.conf)

    def __getitem__(self, index):
        obj = DetectionResults.__new__(DetectionResults)
        obj.xywh = self.xywh[index]
        obj.conf = self.conf[index]
        obj.cls = self.cls[index]
        return obj

# --------------------------------------------------
# 5. Zone occupancy assignment
# --------------------------------------------------

def zone_monitoring(tracks):

    counts = np.zeros(
        (ZONE_ROWS, ZONE_COLS),
        dtype=np.int32
    )

    for track in tracks:

        x1, y1, x2, y2 = track[:4]

        foot_x = (x1 + x2) / 2
        foot_y = y2

        if not (0 <= foot_x < W and 0 <= foot_y < H):
            continue

        col = min(
            int(foot_x / (W / ZONE_COLS)),
            ZONE_COLS - 1
        )

        row = min(
            int(foot_y / (H / ZONE_ROWS)),
            ZONE_ROWS - 1
        )

        counts[row, col] += 1

    return counts

# --------------------------------------------------
# 6. Run both trackers
# --------------------------------------------------

def benchmark_tracker(tracker_class):

    BaseTrack.reset_id()

    tracker = tracker_class(
        args=SimpleNamespace(**cfg)
    )

    tracking_times = []
    zone_times = []
    zone_counts = []
    records = []

    for frame, arr in enumerate(
        detections_by_frame, start=1
    ):

        det = DetectionResults(arr)

        t0 = time.perf_counter()
        tracks = tracker.update(det)
        tracking_times.append(
            (time.perf_counter() - t0) * 1000
        )

        t1 = time.perf_counter()
        counts = zone_monitoring(tracks)
        zone_times.append(
            (time.perf_counter() - t1) * 1000
        )

        zone_counts.append(counts)

        for track in tracks:
            records.append([
                frame,
                int(track[4]),
                *map(float, track[:4]),
                float(track[5])
            ])

        if frame % 100 == 0:
            print(
                f"{tracker_class.__name__}: "
                f"{frame}/{N_FRAMES}"
            )

    return {
        "tracking_ms": np.asarray(tracking_times),
        "zone_ms": np.asarray(zone_times),
        "zones": np.stack(zone_counts),
        "records": np.asarray(records).reshape(-1, 7)
    }


print("\nRunning original BoT-SORT...")
original = benchmark_tracker(BOTSORT)

print("\nRunning Batched BoT-SORT...")
batched = benchmark_tracker(BatchedBOTSORT)

# --------------------------------------------------
# 7. Calculate performance
# --------------------------------------------------

detector_ms = np.asarray(detector_times)
decode_ms = np.asarray(decode_times)

def summarize(result):

    pipeline_ms = (
        detector_ms
        + result["tracking_ms"]
        + result["zone_ms"]
    )

    return {
        "detector_mean_ms": float(detector_ms.mean()),
        "tracker_mean_ms": float(
            result["tracking_ms"].mean()
        ),
        "zone_mean_ms": float(
            result["zone_ms"].mean()
        ),
        "pipeline_mean_ms": float(
            pipeline_ms.mean()
        ),
        "pipeline_fps": float(
            1000 / pipeline_ms.mean()
        ),
        "tracker_p95_ms": float(
            np.percentile(result["tracking_ms"], 95)
        ),
        "pipeline_p95_ms": float(
            np.percentile(pipeline_ms, 95)
        ),
        "tracking_rows": int(len(result["records"]))
    }


a = summarize(original)
b = summarize(batched)

same_zones = np.array_equal(
    original["zones"],
    batched["zones"]
)

same_tracking = (
    original["records"].shape == batched["records"].shape
    and np.allclose(
        original["records"],
        batched["records"],
        rtol=1e-8,
        atol=1e-6
    )
)

fps_gain = (
    b["pipeline_fps"] / a["pipeline_fps"] - 1
) * 100

report = {
    "split": "VAL",
    "frames": N_FRAMES,
    "gmc": "OFF",
    "original": a,
    "batched": b,
    "same_tracking": bool(same_tracking),
    "same_zone_counts": bool(same_zones),
    "pipeline_fps_gain_percent": float(fps_gain),
    "decode_mean_ms_excluded": float(decode_ms.mean()),
    "scope": (
        "YOLO predict + tracker update + instantaneous "
        "zone occupancy. Excludes image decoding, "
        "temporal flow, alerts, GNN and visualization."
    )
}

print("\n=== CELL 89 — RESULTS ===")
print(f"Original pipeline: {a['pipeline_fps']:.2f} FPS")
print(f"Batched pipeline:  {b['pipeline_fps']:.2f} FPS")
print(f"Pipeline FPS gain: {fps_gain:+.2f}%")
print(f"Same tracking:     {same_tracking}")
print(f"Same zone counts:  {same_zones}")

print("\nMean time per frame:")
print(f"YOLO:              {detector_ms.mean():.2f} ms")
print(f"Original tracker:  {a['tracker_mean_ms']:.2f} ms")
print(f"Batched tracker:   {b['tracker_mean_ms']:.2f} ms")
print(f"Original zones:    {a['zone_mean_ms']:.2f} ms")
print(f"Batched zones:     {b['zone_mean_ms']:.2f} ms")
print(f"Image decoding:    {decode_ms.mean():.2f} ms (excluded)")

with open(
    OUT / "CELL89_pipeline_speed_comparison.json", "w"
) as f:
    json.dump(report, f, indent=2)

print("\nSaved:", OUT / "CELL89_pipeline_speed_comparison.json")
print("No TEST data accessed.")
print("Original BoT-SORT unchanged.")

if same_tracking and same_zones and fps_gain > 0:
    print("\nPASS — Batched BoT-SORT is the faster candidate.")
else:
    print("\nReview results before selecting a tracker.")

=== CELL 89 — PIPELINE BENCHMARK ===
Sequence: MOT20-05
VAL frames: 1657
First image: /content/MOT20/MOT20/train/MOT20-05/img1/000001.jpg
Last image: /content/MOT20/MOT20/train/MOT20-05/img1/001657.jpg
GMC: OFF | ReID: OFF

Running YOLO once...
YOLO: 100/1657
YOLO: 200/1657
YOLO: 300/1657
YOLO: 400/1657
YOLO: 500/1657
YOLO: 600/1657
YOLO: 700/1657
YOLO: 800/1657
YOLO: 900/1657
YOLO: 1000/1657
YOLO: 1100/1657
YOLO: 1200/1657
YOLO: 1300/1657
YOLO: 1400/1657
YOLO: 1500/1657
YOLO: 1600/1657

Running original BoT-SORT...
BOTSORT: 100/1657
BOTSORT: 200/1657
BOTSORT: 300/1657
BOTSORT: 400/1657
BOTSORT: 500/1657
BOTSORT: 600/1657
BOTSORT: 700/1657
BOTSORT: 800/1657
BOTSORT: 900/1657
BOTSORT: 1000/1657
BOTSORT: 1100/1657
BOTSORT: 1200/1657
BOTSORT: 1300/1657
BOTSORT: 1400/1657
BOTSORT: 1500/1657
BOTSORT: 1600/1657

Running Batched BoT-SORT...
BatchedBOTSORT: 100/1657
BatchedBOTSORT: 200/1657
BatchedBOTSORT: 300/1657
BatchedBOTSORT: 400/1657
BatchedBOTSORT: 500/1657
BatchedBOTSORT: 600/1657
Batc

In [ ]:
# CELL 90 — SAVE BATCHED BoT-SORT AS A REUSABLE MODULE
# No YOLO | No training | No TEST

import ast
import importlib.util
import json
import sys
import textwrap
from pathlib import Path

import numpy as np

from ultralytics.trackers.bot_sort import BOTSORT
from ultralytics.trackers.byte_tracker import BYTETracker

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
MODULE_DIR = ROOT / "optimized_tracker"
MODULE_DIR.mkdir(parents=True, exist_ok=True)

MODULE_PATH = MODULE_DIR / "batched_botsort.py"
REPORT_PATH = MODULE_DIR / "CELL90_module_verification.json"

assert "BatchedBOTSORT" in globals(), (
    "Run CELL 86 first."
)

# --------------------------------------------------
# 1. Write standalone implementation
# --------------------------------------------------

module_code = r'''
import numpy as np
from ultralytics.trackers.bot_sort import BOTSORT
from ultralytics.trackers.byte_tracker import BYTETracker
from ultralytics.trackers.basetrack import TrackState


def batch_kalman_update(kf, means, covariances, measurements):
    """Batched Kalman correction with confidence=None."""

    means = np.asarray(means)
    covariances = np.asarray(covariances)
    measurements = np.asarray(measurements)

    n = len(means)

    if n == 0:
        return means.copy(), covariances.copy()

    # XYWH observation model: first four state dimensions.
    projected_means = means[:, :4]

    std = np.stack([
        kf._std_weight_position * means[:, 2],
        kf._std_weight_position * means[:, 3],
        kf._std_weight_position * means[:, 2],
        kf._std_weight_position * means[:, 3]
    ], axis=1)

    innovation_cov = np.zeros(
        (n, 4, 4), dtype=covariances.dtype
    )

    diag = np.arange(4)
    innovation_cov[:, diag, diag] = std ** 2

    projected_cov = (
        covariances[:, :4, :4] + innovation_cov
    )

    # Solve S * K.T = P[:, :, :4].T
    kalman_gain = np.linalg.solve(
        projected_cov,
        covariances[:, :, :4].transpose(0, 2, 1)
    ).transpose(0, 2, 1)

    innovation = measurements - projected_means

    new_means = means + np.einsum(
        "nij,nj->ni", kalman_gain, innovation
    )

    new_covariances = covariances - (
        kalman_gain
        @ projected_cov
        @ kalman_gain.transpose(0, 2, 1)
    )

    return new_means, new_covariances


class _PrecomputedKalman:
    """Return a precomputed correction through the original update API."""

    def __init__(self, original, mean, covariance):
        self.original = original
        self.mean = mean
        self.covariance = covariance

    def update(self, mean, covariance, measurement, *args, **kwargs):
        return self.mean, self.covariance

    def __getattr__(self, name):
        return getattr(self.original, name)


class BatchedBOTSORT(BOTSORT):
    """BoT-SORT with batched Kalman corrections for matched tracks."""

    def _apply_matches(
        self,
        matches,
        strack_pool,
        detections,
        activated_stracks,
        refind_stracks
    ):
        pairs = [
            (strack_pool[int(i)], detections[int(j)])
            for i, j in matches
        ]

        if len(pairs) < 2:
            for track, det in pairs:
                self._apply_match(
                    track, det,
                    activated_stracks,
                    refind_stracks
                )
            return

        # Some newly initialized tracks may not have a state
        # suitable for batching; preserve original behavior.
        batch_indices = [
            i for i, (track, det) in enumerate(pairs)
            if track.mean is not None
            and track.covariance is not None
        ]

        if len(batch_indices) < 2:
            for track, det in pairs:
                self._apply_match(
                    track, det,
                    activated_stracks,
                    refind_stracks
                )
            return

        selected = [pairs[i] for i in batch_indices]

        means = np.stack([
            track.mean for track, _ in selected
        ])

        covariances = np.stack([
            track.covariance for track, _ in selected
        ])

        measurements = np.stack([
            track.convert_coords(det.tlwh)
            for track, det in selected
        ])

        kf = selected[0][0].kalman_filter

        new_means, new_covariances = batch_kalman_update(
            kf, means, covariances, measurements
        )

        precomputed = {
            index: (new_means[k], new_covariances[k])
            for k, index in enumerate(batch_indices)
        }

        for i, (track, det) in enumerate(pairs):

            if i not in precomputed:
                self._apply_match(
                    track, det,
                    activated_stracks,
                    refind_stracks
                )
                continue

            original_kf = track.kalman_filter

            try:
                mean, covariance = precomputed[i]

                track.kalman_filter = _PrecomputedKalman(
                    original_kf, mean, covariance
                )

                # Preserve BoT-SORT's original metadata updates.
                self._apply_match(
                    track, det,
                    activated_stracks,
                    refind_stracks
                )

            finally:
                track.kalman_filter = original_kf
'''

# Check syntax before saving
ast.parse(module_code)

MODULE_PATH.write_text(
    textwrap.dedent(module_code),
    encoding="utf-8"
)

# --------------------------------------------------
# 2. Import module independently
# --------------------------------------------------

module_name = "crowd_batched_botsort"

spec = importlib.util.spec_from_file_location(
    module_name, MODULE_PATH
)

module = importlib.util.module_from_spec(spec)
sys.modules[module_name] = module
spec.loader.exec_module(module)

SavedBatchedBOTSORT = module.BatchedBOTSORT

assert issubclass(SavedBatchedBOTSORT, BOTSORT)
assert SavedBatchedBOTSORT is not BatchedBOTSORT

# --------------------------------------------------
# 3. Verify numerical equivalence against original KF
# --------------------------------------------------

from ultralytics.trackers.utils.kalman_filter import KalmanFilterXYWH
kf = KalmanFilterXYWH()

rng = np.random.default_rng(90)

means = []
covariances = []
measurements = []

for _ in range(100):
    measurement = np.array([
        rng.uniform(100, 1500),
        rng.uniform(100, 900),
        rng.uniform(20, 100),
        rng.uniform(40, 200)
    ])

    mean, covariance = kf.initiate(measurement)

    measurement2 = measurement + rng.normal(
        0, [2, 2, 1, 1]
    )

    means.append(mean)
    covariances.append(covariance)
    measurements.append(measurement2)

means = np.stack(means)
covariances = np.stack(covariances)
measurements = np.stack(measurements)

batch_means, batch_covs = module.batch_kalman_update(
    kf, means, covariances, measurements
)

original_results = [
    kf.update(means[i], covariances[i], measurements[i])
    for i in range(len(means))
]

original_means = np.stack([x[0] for x in original_results])
original_covs = np.stack([x[1] for x in original_results])

mean_error = float(
    np.max(np.abs(original_means - batch_means))
)

cov_error = float(
    np.max(np.abs(original_covs - batch_covs))
)

equivalent = (
    np.allclose(
        original_means, batch_means,
        rtol=1e-9, atol=1e-8
    )
    and np.allclose(
        original_covs, batch_covs,
        rtol=1e-9, atol=1e-8
    )
)

report = {
    "module_path": str(MODULE_PATH),
    "module_loaded": True,
    "independent_class": True,
    "inherits_botsort": True,
    "numerical_equivalence": bool(equivalent),
    "max_mean_error": mean_error,
    "max_covariance_error": cov_error,
    "tracking_equivalence_not_yet_tested": True
}

REPORT_PATH.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("=== CELL 90 — MODULE VERIFICATION ===")

for key, value in report.items():
    print(f"{key}: {value}")

print("\nSaved:", MODULE_PATH)

if equivalent:
    print("\nPASS — Module saved, imported, and Kalman math verified.")
    print("Next: compare the saved tracker against the tested tracker.")
else:
    print("\nWARNING — Numerical mismatch. Do not integrate yet.")

=== CELL 90 — MODULE VERIFICATION ===
module_path: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/optimized_tracker/batched_botsort.py
module_loaded: True
independent_class: True
inherits_botsort: True
numerical_equivalence: True
max_mean_error: 0.0
max_covariance_error: 0.0
tracking_equivalence_not_yet_tested: True

Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/optimized_tracker/batched_botsort.py

PASS — Module saved, imported, and Kalman math verified.
Next: compare the saved tracker against the tested tracker.


In [ ]:
# CELL 91 — VERIFY SAVED BATCHED BoT-SORT
# Full VAL | Cached detections | No YOLO | No TEST

import json
import time
import yaml
import importlib.util
import sys
import numpy as np
import pandas as pd

from pathlib import Path
from types import SimpleNamespace
from ultralytics.trackers.bot_sort import BOTSORT
from ultralytics.trackers.basetrack import BaseTrack

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "benchmark"
MODULE_PATH = ROOT / "optimized_tracker" / "batched_botsort.py"

N_FRAMES = 1657

assert MODULE_PATH.exists()
assert "BatchedBOTSORT" in globals(), "Run CELL 86 first."

# --------------------------------------------------
# 1. Import saved tracker from Drive
# --------------------------------------------------

spec = importlib.util.spec_from_file_location(
    "saved_batched_botsort_cell91",
    MODULE_PATH
)

module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
spec.loader.exec_module(module)

SavedBatchedBOTSORT = module.BatchedBOTSORT

# --------------------------------------------------
# 2. Load tracker configuration
# --------------------------------------------------

with open(OUT / "benchmark_botsort.yaml") as f:
    cfg = yaml.safe_load(f)

assert cfg["tracker_type"] == "botsort"

cfg["gmc_method"] = "none"
cfg["with_reid"] = False

# --------------------------------------------------
# 3. Load cached VAL detections
# --------------------------------------------------

path = (
    ROOT / "detection" /
    "detections_yolov8m_MOT20-05.parquet"
)

df = pd.read_parquet(path)

columns = ["frame", "x1", "y1", "x2", "y2", "conf"]

assert set(columns).issubset(df.columns)

val = df.loc[
    df["frame"].between(1, N_FRAMES),
    columns
].copy()

val = val.sort_values("frame", kind="stable")

valid = (
    np.isfinite(
        val[["x1", "y1", "x2", "y2", "conf"]]
    ).all(axis=1)
    & (val["x2"] > val["x1"])
    & (val["y2"] > val["y1"])
)

excluded = int((~valid).sum())
val = val.loc[valid].copy()

assert not val.empty

groups = {
    int(frame): group
    for frame, group in val.groupby("frame", sort=False)
}

empty = val.iloc[:0]

print("=== CELL 91 — SAVED TRACKER VERIFICATION ===")
print("VAL frames:", N_FRAMES)
print("Valid detections:", len(val))
print("Excluded invalid detections:", excluded)

# --------------------------------------------------
# 4. Detection adapter
# --------------------------------------------------

class DetectionResults:

    def __init__(self, rows):

        arr = rows[
            ["x1", "y1", "x2", "y2"]
        ].to_numpy(dtype=np.float64)

        x1, y1, x2, y2 = arr.T

        self.xywh = np.column_stack([
            (x1 + x2) / 2,
            (y1 + y2) / 2,
            x2 - x1,
            y2 - y1
        ])

        self.conf = rows["conf"].to_numpy(dtype=np.float64)
        self.cls = np.zeros(len(rows), dtype=np.float64)

    def __len__(self):
        return len(self.conf)

    def __getitem__(self, index):
        obj = DetectionResults.__new__(DetectionResults)
        obj.xywh = self.xywh[index]
        obj.conf = self.conf[index]
        obj.cls = self.cls[index]
        return obj

# --------------------------------------------------
# 5. Run each tracker independently
# --------------------------------------------------

def run_tracker(name, tracker_class):

    BaseTrack.reset_id()

    tracker = tracker_class(
        args=SimpleNamespace(**cfg)
    )

    records = []
    times = []

    for frame in range(1, N_FRAMES + 1):

        detections = DetectionResults(
            groups.get(frame, empty)
        )

        start = time.perf_counter()
        tracks = tracker.update(detections)
        times.append(
            (time.perf_counter() - start) * 1000
        )

        for track in tracks:
            records.append([
                frame,
                int(track[4]),
                float(track[0]),
                float(track[1]),
                float(track[2]),
                float(track[3]),
                float(track[5])
            ])

        if frame % 500 == 0 or frame == N_FRAMES:
            print(f"{name}: {frame}/{N_FRAMES}")

    result = pd.DataFrame(
        records,
        columns=[
            "frame", "id",
            "x1", "y1", "x2", "y2", "score"
        ]
    )

    return (
        result.sort_values(["frame", "id"]).reset_index(drop=True),
        np.asarray(times)
    )

# --------------------------------------------------
# 6. Compare all three
# --------------------------------------------------

print("\nRunning original BoT-SORT...")
original, t_original = run_tracker("Original", BOTSORT)

print("\nRunning tested Batched BoT-SORT...")
tested, t_tested = run_tracker("Tested", BatchedBOTSORT)

print("\nRunning saved Batched BoT-SORT...")
saved, t_saved = run_tracker("Saved", SavedBatchedBOTSORT)

def compare(a, b):
    keys = ["frame", "id"]
    values = ["x1", "y1", "x2", "y2", "score"]

    same_ids = (
        a.shape == b.shape
        and a[keys].equals(b[keys])
    )

    same_values = (
        same_ids
        and np.allclose(
            a[values].to_numpy(),
            b[values].to_numpy(),
            rtol=1e-8,
            atol=1e-6
        )
    )

    return {
        "same_ids": bool(same_ids),
        "same_boxes_and_scores": bool(same_values),
        "equivalent": bool(same_ids and same_values)
    }

original_vs_tested = compare(original, tested)
original_vs_saved = compare(original, saved)
tested_vs_saved = compare(tested, saved)

report = {
    "split": "VAL",
    "frames": N_FRAMES,
    "original_rows": len(original),
    "tested_rows": len(tested),
    "saved_rows": len(saved),
    "original_vs_tested": original_vs_tested,
    "original_vs_saved": original_vs_saved,
    "tested_vs_saved": tested_vs_saved,
    "mean_tracker_ms": {
        "original": float(t_original.mean()),
        "tested": float(t_tested.mean()),
        "saved": float(t_saved.mean())
    },
    "all_equivalent": bool(
        original_vs_tested["equivalent"]
        and original_vs_saved["equivalent"]
        and tested_vs_saved["equivalent"]
    )
}

print("\n=== CELL 91 — RESULTS ===")

print("Original vs Tested:", original_vs_tested)
print("Original vs Saved:", original_vs_saved)
print("Tested vs Saved:", tested_vs_saved)

print("\nMean tracker time:")
for name, value in report["mean_tracker_ms"].items():
    print(f"{name}: {value:.3f} ms/frame")

print("\nAll equivalent:", report["all_equivalent"])

# --------------------------------------------------
# 7. Save verification
# --------------------------------------------------

report_path = OUT / "CELL91_saved_tracker_verification.json"

with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print("\nSaved:", report_path)
print("No YOLO inference.")
print("No TEST accessed.")

if report["all_equivalent"]:
    print("\nPASS — Saved Batched BoT-SORT verified on full VAL.")
else:
    print("\nWARNING — Saved tracker differs.")
    print("Do not integrate the saved version yet.")

=== CELL 91 — SAVED TRACKER VERIFICATION ===
VAL frames: 1657
Valid detections: 857219
Excluded invalid detections: 27

Running original BoT-SORT...
Original: 500/1657
Original: 1000/1657
Original: 1500/1657
Original: 1657/1657

Running tested Batched BoT-SORT...
Tested: 500/1657
Tested: 1000/1657
Tested: 1500/1657
Tested: 1657/1657

Running saved Batched BoT-SORT...
Saved: 500/1657
Saved: 1000/1657
Saved: 1500/1657
Saved: 1657/1657

=== CELL 91 — RESULTS ===
Original vs Tested: {'same_ids': True, 'same_boxes_and_scores': True, 'equivalent': True}
Original vs Saved: {'same_ids': True, 'same_boxes_and_scores': True, 'equivalent': True}
Tested vs Saved: {'same_ids': True, 'same_boxes_and_scores': True, 'equivalent': True}

Mean tracker time:
original: 21.962 ms/frame
tested: 17.353 ms/frame
saved: 14.505 ms/frame

All equivalent: True

Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/benchmark/CELL91_saved_tracker_verification.json
No YOLO inference.
No TEST accessed.

PASS — Sa

In [ ]:
# CELL 92 — INTEGRATE SAVED BATCHED BoT-SORT
# Reusable tracking interface | No GNN | No TEST

import importlib.util
import json
import sys
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import yaml

from ultralytics.trackers.bot_sort import BOTSORT
from ultralytics.trackers.basetrack import BaseTrack

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")

MODULE_PATH = (
    ROOT / "optimized_tracker" / "batched_botsort.py"
)

CONFIG_PATH = (
    ROOT / "benchmark" / "benchmark_botsort.yaml"
)

REPORT_PATH = (
    ROOT / "benchmark" / "CELL92_tracker_integration.json"
)

assert MODULE_PATH.exists(), "Saved tracker module missing."
assert CONFIG_PATH.exists(), "Tracker configuration missing."

# --------------------------------------------------
# 1. Load saved optimized tracker
# --------------------------------------------------

spec = importlib.util.spec_from_file_location(
    "crowd_optimized_tracker",
    MODULE_PATH
)

module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
spec.loader.exec_module(module)

BatchedBOTSORT = module.BatchedBOTSORT

# --------------------------------------------------
# 2. Load verified configuration
# --------------------------------------------------

with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)

assert cfg["tracker_type"] == "botsort"

cfg["gmc_method"] = "none"
cfg["with_reid"] = False

# --------------------------------------------------
# 3. Reusable detection adapter
# --------------------------------------------------

class CrowdDetections:

    def __init__(self, boxes, scores):

        boxes = np.asarray(boxes, dtype=np.float64)
        scores = np.asarray(scores, dtype=np.float64)

        boxes = boxes.reshape(-1, 4)
        scores = scores.reshape(-1)

        assert len(boxes) == len(scores)

        self.xywh = np.column_stack([
            (boxes[:, 0] + boxes[:, 2]) / 2,
            (boxes[:, 1] + boxes[:, 3]) / 2,
            boxes[:, 2] - boxes[:, 0],
            boxes[:, 3] - boxes[:, 1]
        ])

        self.conf = scores
        self.cls = np.zeros(len(scores), dtype=np.float64)

    def __len__(self):
        return len(self.conf)

    def __getitem__(self, index):

        result = CrowdDetections.__new__(CrowdDetections)

        result.xywh = np.asarray(
            self.xywh[index]
        ).reshape(-1, 4)

        result.conf = np.atleast_1d(self.conf[index])
        result.cls = np.atleast_1d(self.cls[index])

        return result

# --------------------------------------------------
# 4. Reusable tracker interface
# --------------------------------------------------

class CrowdTracker:

    def __init__(self, optimized=True):

        self.optimized = optimized

        tracker_class = (
            BatchedBOTSORT if optimized else BOTSORT
        )

        BaseTrack.reset_id()

        self.tracker = tracker_class(
            args=SimpleNamespace(**cfg)
        )

        self.frame_number = 0

    def update(self, boxes, scores):

        self.frame_number += 1

        boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
        scores = np.asarray(scores, dtype=np.float64).reshape(-1)

        valid = (
            np.isfinite(boxes).all(axis=1)
            & np.isfinite(scores)
            & (boxes[:, 2] > boxes[:, 0])
            & (boxes[:, 3] > boxes[:, 1])
        )

        detections = CrowdDetections(
            boxes[valid],
            scores[valid]
        )

        tracks = self.tracker.update(detections)

        # Output:
        # x1, y1, x2, y2, track_id, confidence
        return np.asarray(tracks)[:, :6].reshape(-1, 6)

# --------------------------------------------------
# 5. Smoke test using synthetic detections
# --------------------------------------------------

tracker = CrowdTracker(optimized=True)

sample_boxes = np.array([
    [100, 100, 150, 220],
    [300, 120, 350, 250]
], dtype=np.float64)

sample_scores = np.array([0.95, 0.93])

for _ in range(5):
    tracks = tracker.update(
        sample_boxes,
        sample_scores
    )

assert tracks.ndim == 2
assert tracks.shape[1] == 6

# --------------------------------------------------
# 6. Save integration report
# --------------------------------------------------

report = {
    "status": "PASS",
    "module_path": str(MODULE_PATH),
    "tracker": "BatchedBOTSORT",
    "gmc": "OFF",
    "reid": "OFF",
    "interface_ready": True,
    "smoke_test_passed": True,
    "gnn_integrated": False,
    "test_data_accessed": False,
    "monitoring_logic_integrated": False
}

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

REPORT_PATH.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("=== CELL 92 — TRACKER INTEGRATION ===")

for key, value in report.items():
    print(f"{key}: {value}")

print("\nPASS — Saved tracker interface is ready.")
print("Next: Connect existing zone, flow, and alert logic.")

=== CELL 92 — TRACKER INTEGRATION ===
status: PASS
module_path: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/optimized_tracker/batched_botsort.py
tracker: BatchedBOTSORT
gmc: OFF
reid: OFF
interface_ready: True
smoke_test_passed: True
gnn_integrated: False
test_data_accessed: False
monitoring_logic_integrated: False

PASS — Saved tracker interface is ready.
Next: Connect existing zone, flow, and alert logic.


In [ ]:
# CELL 93 — FULL VAL MONITORING INTEGRATION
# Saved Batched BoT-SORT -> Zones -> Flow -> Alerts
# Reuses rules from CELLS 55, 69 and 70.
# No YOLO inference | No TEST | No GNN

import json
import time
import yaml
import sys
import importlib.util
import numpy as np
import pandas as pd

from pathlib import Path
from types import SimpleNamespace

from ultralytics.trackers.basetrack import BaseTrack

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
BENCH = ROOT / "benchmark"
OUT = ROOT / "zones" / "optimization_v2"
OUT.mkdir(parents=True, exist_ok=True)

N_FRAMES = 1657
FPS = 25
W, H = 1654, 1080
ZONES = 12

MODULE = ROOT / "optimized_tracker/batched_botsort.py"
CONFIG = BENCH / "benchmark_botsort.yaml"
DETECTIONS = ROOT / "detection/detections_yolov8m_MOT20-05.parquet"
REFERENCE = BENCH / "CELL88_original_FULL_VAL_tracks.csv"
CALIBRATION = ROOT / "zones/zone_level_calibration_MOT20-05.csv"
SELECTED = OUT / "zone_stability_selected_VAL.json"
ZONE_DEFINITION = ROOT / "zones/zone_definition.json"

for path in [
    MODULE, CONFIG, DETECTIONS, REFERENCE,
    CALIBRATION, SELECTED, ZONE_DEFINITION
]:
    assert path.exists(), f"Missing: {path}"

# --------------------------------------------------
# 1. Load saved tracker and VAL-selected settings
# --------------------------------------------------

spec = importlib.util.spec_from_file_location(
    "crowd_batched_botsort_cell93", MODULE
)

module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
spec.loader.exec_module(module)

SavedBatchedBOTSORT = module.BatchedBOTSORT

with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

assert cfg["tracker_type"] == "botsort"

cfg["gmc_method"] = "none"
cfg["with_reid"] = False

with open(SELECTED) as f:
    selected = json.load(f)["selected"]

assert selected["threshold"] == 5
STABILITY = int(selected["threshold"])

with open(ZONE_DEFINITION) as f:
    zone_config = json.load(f)

assert list(map(int, zone_config["image_size"])) == [W, H]
assert len(zone_config["zones"]) == ZONES

calib = pd.read_csv(CALIBRATION)

required_calib = [
    "zone",
    "own_low_max",
    "own_high_min",
    "person_area_px",
    "zone_area_px",
    "floor_low_max",
    "floor_high_min"
]

assert set(required_calib).issubset(calib.columns)
assert set(calib.zone) == set(range(ZONES))
assert calib.zone.is_unique
assert calib[required_calib].notna().all().all()

print("=== CELL 93 — MONITORING INTEGRATION ===")
print("Tracker: Saved Batched BoT-SORT")
print("Frames: 1–1657 (VAL)")
print("GMC: OFF | ReID: OFF")
print("Stable transition rule:", STABILITY, "frames")

# --------------------------------------------------
# 2. Load VAL detections only
# --------------------------------------------------

det_cols = ["frame", "x1", "y1", "x2", "y2", "conf"]

df = pd.read_parquet(
    DETECTIONS,
    columns=det_cols,
    filters=[
        ("frame", ">=", 1),
        ("frame", "<=", N_FRAMES)
    ]
)

assert df.frame.between(1, N_FRAMES).all()

df = df.sort_values("frame", kind="stable")

valid = (
    np.isfinite(df[det_cols[1:]]).all(axis=1)
    & (df.x2 > df.x1)
    & (df.y2 > df.y1)
)

excluded = int((~valid).sum())
df = df.loc[valid].copy()

groups = {
    int(frame): group
    for frame, group in df.groupby("frame", sort=False)
}

empty = df.iloc[:0]

print("Valid VAL detections:", len(df))
print("Excluded invalid boxes:", excluded)

# --------------------------------------------------
# 3. Detection adapter (same format as CELL 91)
# --------------------------------------------------

class DetectionResults:

    def __init__(self, rows):

        box = rows[
            ["x1", "y1", "x2", "y2"]
        ].to_numpy(dtype=np.float64)

        x1, y1, x2, y2 = box.T

        self.xywh = np.column_stack([
            (x1 + x2) / 2,
            (y1 + y2) / 2,
            x2 - x1,
            y2 - y1
        ])

        self.conf = rows.conf.to_numpy(dtype=np.float64)
        self.cls = np.zeros(len(rows), dtype=np.float64)

    def __len__(self):
        return len(self.conf)

    def __getitem__(self, index):

        result = DetectionResults.__new__(DetectionResults)

        result.xywh = self.xywh[index]
        result.conf = self.conf[index]
        result.cls = self.cls[index]

        return result

# --------------------------------------------------
# 4. Run saved optimized tracker on VAL
# --------------------------------------------------

BaseTrack.reset_id()

tracker = SavedBatchedBOTSORT(
    args=SimpleNamespace(**cfg)
)

records = []
tracker_times = []

for frame in range(1, N_FRAMES + 1):

    detections = DetectionResults(
        groups.get(frame, empty)
    )

    start = time.perf_counter()
    output = tracker.update(detections)
    tracker_times.append(
        (time.perf_counter() - start) * 1000
    )

    for item in output:
        records.append([
            frame,
            int(item[4]),
            float(item[0]),
            float(item[1]),
            float(item[2]),
            float(item[3]),
            float(item[5])
        ])

    if frame % 250 == 0 or frame == N_FRAMES:
        print(f"Tracking: {frame}/{N_FRAMES}")

columns = [
    "frame", "id", "x1", "y1", "x2", "y2", "score"
]

tracks = pd.DataFrame(records, columns=columns)

assert not tracks.empty
assert not tracks.duplicated(["frame", "id"]).any()

# --------------------------------------------------
# 5. Verify tracking against CELL 88 original
# --------------------------------------------------

reference = pd.read_csv(REFERENCE)

keys = ["frame", "id"]
values = ["x1", "y1", "x2", "y2", "score"]

a = tracks.sort_values(keys).reset_index(drop=True)
b = reference.sort_values(keys).reset_index(drop=True)

same_ids = (
    a.shape == b.shape
    and a[keys].equals(b[keys])
)

same_tracking = (
    same_ids
    and np.allclose(
        a[values].to_numpy(),
        b[values].to_numpy(),
        rtol=1e-8,
        atol=1e-6
    )
)

print("\nTracking identical to CELL 88:", same_tracking)

assert same_tracking, (
    "Tracking mismatch. Stop before monitoring integration."
)

# --------------------------------------------------
# 6. Assign 12 zones using bottom-center footpoint
# Exact rule from CELLS 55 and 69
# --------------------------------------------------

def assign_zones(source):

    data = source.copy()

    x = data.x1.to_numpy()
    y = data.y1.to_numpy()
    w = data.x2.to_numpy() - x
    h = data.y2.to_numpy() - y

    px = np.floor(x + w / 2).astype(int)
    py = np.floor(y + h).astype(int)

    inside = (
        (px >= 0) & (px < W)
        & (py >= 0) & (py < H)
    )

    col = np.select(
        [px <= 413, px <= 826, px <= 1239],
        [0, 1, 2],
        default=3
    )

    row = np.select(
        [py <= 359, py <= 719],
        [0, 1],
        default=2
    )

    data["zone"] = np.where(
        inside,
        row * 4 + col,
        -1
    )

    return data

# --------------------------------------------------
# 7. Stable transitions
# Exact 5-frame state machine from CELL 55
# --------------------------------------------------

def stable_transitions(zone_tracks):

    events = []

    ordered = zone_tracks.sort_values(["id", "frame"])

    for person_id, group in ordered.groupby(
        "id", sort=False
    ):

        stable = -1
        candidate = -1
        streak = 0
        previous_frame = -999

        for frame, zone in zip(
            group.frame.to_numpy(),
            group.zone.to_numpy()
        ):

            frame = int(frame)
            zone = int(zone)

            if frame != previous_frame + 1:
                candidate = -1
                streak = 0

            previous_frame = frame

            if zone == -1:
                stable = -1
                candidate = -1
                streak = 0
                continue

            if stable == -1:

                if zone == candidate:
                    streak += 1
                else:
                    candidate = zone
                    streak = 1

                if streak >= STABILITY:
                    stable = zone
                    candidate = -1
                    streak = 0

                continue

            if zone == stable:
                candidate = -1
                streak = 0
                continue

            if zone == candidate:
                streak += 1
            else:
                candidate = zone
                streak = 1

            if streak >= STABILITY:

                events.append([
                    frame,
                    int(person_id),
                    stable,
                    zone
                ])

                stable = zone
                candidate = -1
                streak = 0

    return pd.DataFrame(
        events,
        columns=["frame", "id", "from_zone", "to_zone"]
    )

# --------------------------------------------------
# 8. Build unified monitoring
# Same occupancy, flow and alert rules as CELL 69
# --------------------------------------------------

def build_monitoring(source):

    zoned = assign_zones(source)
    events = stable_transitions(zoned)

    frames = pd.DataFrame({
        "frame": np.arange(1, N_FRAMES + 1)
    })

    frames["step"] = (frames.frame - 1) // FPS

    index = pd.MultiIndex.from_product(
        [frames.frame, range(ZONES)],
        names=["frame", "zone"]
    )

    counts = (
        zoned[zoned.zone.between(0, ZONES - 1)]
        .groupby(["frame", "zone"])
        .size()
        .reindex(index, fill_value=0)
        .rename("count")
        .reset_index()
        .merge(frames, on="frame", validate="many_to_one")
    )

    occupancy = (
        counts.groupby(["step", "zone"], as_index=False)
        .agg(
            count=("count", "mean"),
            max_count=("count", "max"),
            frames_in_window=("frame", "nunique")
        )
    )

    # VAL windows 0–66.
    # The final window has only 7 VAL frames.
    flow_index = pd.MultiIndex.from_product(
        [range((N_FRAMES - 1) // FPS + 1), range(ZONES)],
        names=["step", "zone"]
    )

    if len(events):

        ev = events.copy()
        ev["step"] = (ev.frame - 1) // FPS

        incoming = ev.groupby(
            ["step", "to_zone"]
        ).size()

        outgoing = ev.groupby(
            ["step", "from_zone"]
        ).size()

        incoming.index.names = ["step", "zone"]
        outgoing.index.names = ["step", "zone"]

        in_counts = incoming.reindex(
            flow_index, fill_value=0
        ).to_numpy()

        out_counts = outgoing.reindex(
            flow_index, fill_value=0
        ).to_numpy()

    else:
        in_counts = np.zeros(len(flow_index), dtype=int)
        out_counts = np.zeros(len(flow_index), dtype=int)

    flow = flow_index.to_frame(index=False)
    flow["inflow"] = in_counts
    flow["outflow"] = out_counts

    final = occupancy.merge(
        flow,
        on=["step", "zone"],
        validate="one_to_one"
    )

    final = final.merge(
        calib[required_calib],
        on="zone",
        validate="many_to_one"
    )

    final["density_per_100x100px"] = (
        final["count"] * 10000
        / final["zone_area_px"]
    )

    final["floor_coverage"] = (
        final["count"]
        * final["person_area_px"]
        / final["zone_area_px"]
    )

    final["level_own"] = np.where(
        final["count"] >= final["own_high_min"],
        "HIGH",
        np.where(
            final["count"] <= final["own_low_max"],
            "LOW",
            "MEDIUM"
        )
    )

    final["level_floor"] = np.where(
        final["floor_coverage"] >= final["floor_high_min"],
        "HIGH",
        np.where(
            final["floor_coverage"] <= final["floor_low_max"],
            "LOW",
            "MEDIUM"
        )
    )

    final["crowd_alert"] = np.where(
        (final.level_own == "HIGH")
        | (final.level_floor == "HIGH"),
        "HIGH_CROWD",
        "NORMAL"
    )

    return (
        final.sort_values(["step", "zone"]).reset_index(drop=True),
        events,
        zoned
    )

# --------------------------------------------------
# 9. Generate and compare monitoring results
# --------------------------------------------------

print("\nBuilding optimized monitoring outputs...")

monitoring, events, zoned = build_monitoring(tracks)

print("Building reference monitoring outputs...")

reference_monitoring, reference_events, _ = (
    build_monitoring(reference)
)

compare_columns = [
    "step", "zone",
    "count", "max_count",
    "frames_in_window",
    "inflow", "outflow",
    "density_per_100x100px",
    "floor_coverage",
    "level_own", "level_floor",
    "crowd_alert"
]

m = monitoring[compare_columns]
r = reference_monitoring[compare_columns]

numeric_cols = [
    "count", "max_count",
    "density_per_100x100px",
    "floor_coverage"
]

other_cols = [
    c for c in compare_columns
    if c not in numeric_cols
]

same_monitoring = (
    m.shape == r.shape
    and m[other_cols].equals(r[other_cols])
    and np.allclose(
        m[numeric_cols].to_numpy(),
        r[numeric_cols].to_numpy(),
        rtol=1e-9,
        atol=1e-8
    )
)

# --------------------------------------------------
# 10. Validation checks
# --------------------------------------------------

checks = {
    "Tracking matches CELL 88": bool(same_tracking),

    "804 zone-window records": len(monitoring) == 804,

    "All 67 VAL windows": (
        set(monitoring.step) == set(range(67))
    ),

    "12 zones per window": (
        monitoring.groupby("step").zone.nunique().eq(12).all()
    ),

    "No duplicate step-zone records": (
        not monitoring.duplicated(["step", "zone"]).any()
    ),

    "No duplicate frame-track IDs": (
        not tracks.duplicated(["frame", "id"]).any()
    ),

    "Complete windows have 25 frames": (
        monitoring.loc[
            monitoring.step < 66, "frames_in_window"
        ].eq(25).all()
    ),

    "Final VAL window has 7 frames": (
        monitoring.loc[
            monitoring.step == 66, "frames_in_window"
        ].eq(7).all()
    ),

    "No negative occupancy or flow": (
        monitoring[
            ["count", "max_count", "inflow", "outflow"]
        ].ge(0).all().all()
    ),

    "Mean occupancy does not exceed maximum": (
        (monitoring["count"] <=
         monitoring["max_count"] + 1e-9).all()
    ),

    "Inflow equals outflow": (
        int(monitoring.inflow.sum())
        == int(monitoring.outflow.sum())
        == len(events)
    ),

    "Monitoring matches original tracker": (
        bool(same_monitoring)
    ),

    "Crowd alerts use expected labels": (
        set(monitoring.crowd_alert)
        .issubset({"NORMAL", "HIGH_CROWD"})
    )
}

print("\n=== CELL 93 — VALIDATION ===")

for name, passed in checks.items():
    print(
        ("PASS" if passed else "FAIL"),
        "|", name
    )

passed = sum(checks.values())

print(f"\nPassed: {passed}/{len(checks)}")

assert all(checks.values()), (
    "Monitoring validation failed. "
    "Do not integrate until the mismatch is resolved."
)

# --------------------------------------------------
# 11. Save VAL-only outputs
# --------------------------------------------------

MONITOR_FILE = (
    OUT / "CELL93_BatchedBOTSORT_monitoring_VAL.csv"
)

TRACK_FILE = (
    OUT / "CELL93_BatchedBOTSORT_tracks_VAL.csv"
)

EVENT_FILE = (
    OUT / "CELL93_BatchedBOTSORT_transitions_VAL.csv"
)

REPORT_FILE = (
    OUT / "CELL93_monitoring_integration_VAL.json"
)

monitoring.to_csv(MONITOR_FILE, index=False)
tracks.to_csv(TRACK_FILE, index=False)
events.to_csv(EVENT_FILE, index=False)

report = {
    "status": "PASS",
    "sequence": "MOT20-05",
    "split": "VAL",
    "frames": N_FRAMES,
    "tracker": "Saved BatchedBOTSORT",
    "gmc": "OFF",
    "reid": "OFF",
    "valid_detections": int(len(df)),
    "excluded_invalid_detections": excluded,
    "tracking_rows": int(len(tracks)),
    "stable_transition_frames": STABILITY,
    "transitions": int(len(events)),
    "zone_window_records": int(len(monitoring)),
    "mean_occupancy": float(monitoring["count"].mean()),
    "total_inflow": int(monitoring.inflow.sum()),
    "total_outflow": int(monitoring.outflow.sum()),
    "alerts": {
        str(k): int(v)
        for k, v in monitoring.crowd_alert.value_counts().items()
    },
    "same_tracking_as_original": bool(same_tracking),
    "same_monitoring_as_original": bool(same_monitoring),
    "mean_tracker_ms": float(np.mean(tracker_times)),
    "checks_passed": int(passed),
    "checks_total": int(len(checks)),
    "test_accessed": False,
    "gnn_integrated": False,
    "note": (
        "Full VAL monitoring with saved optimized tracker. "
        "Step 66 is a partial seven-frame window. "
        "Uses existing VAL-selected zone, flow and alert rules. "
        "Consistency verification is not an independent "
        "accuracy evaluation."
    )
}

REPORT_FILE.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\n=== CELL 93 — RESULTS ===")

print("Tracked observations:", len(tracks))
print("Stable transitions:", len(events))
print("Mean occupancy:", round(monitoring["count"].mean(), 3))
print("Total inflow:", int(monitoring.inflow.sum()))
print("Total outflow:", int(monitoring.outflow.sum()))

print("\nCrowd alerts:")
print(monitoring.crowd_alert.value_counts().to_string())

print("\nSame tracking:", same_tracking)
print("Same monitoring:", same_monitoring)
print("Mean tracker time:", round(np.mean(tracker_times), 3), "ms")

print("\nSaved:", MONITOR_FILE)
print("Saved:", TRACK_FILE)
print("Saved:", EVENT_FILE)
print("Saved:", REPORT_FILE)

print("\nPASS — Optimized tracker connected to VAL monitoring.")
print("No TEST. No GNN. No YOLO inference.")

=== CELL 93 — MONITORING INTEGRATION ===
Tracker: Saved Batched BoT-SORT
Frames: 1–1657 (VAL)
GMC: OFF | ReID: OFF
Stable transition rule: 5 frames
Valid VAL detections: 857219
Excluded invalid boxes: 27
Tracking: 250/1657
Tracking: 500/1657
Tracking: 750/1657
Tracking: 1000/1657
Tracking: 1250/1657
Tracking: 1500/1657
Tracking: 1657/1657

Tracking identical to CELL 88: True

Building optimized monitoring outputs...
Building reference monitoring outputs...

=== CELL 93 — VALIDATION ===
PASS | Tracking matches CELL 88
PASS | 804 zone-window records
PASS | All 67 VAL windows
PASS | 12 zones per window
PASS | No duplicate step-zone records
PASS | No duplicate frame-track IDs
PASS | Complete windows have 25 frames
PASS | Final VAL window has 7 frames
PASS | No negative occupancy or flow
PASS | Mean occupancy does not exceed maximum
PASS | Inflow equals outflow
PASS | Monitoring matches original tracker
PASS | Crowd alerts use expected labels

Passed: 13/13

=== CELL 93 — RESULTS ===
Tracke

In [ ]:
# CELL 94 — STREAMING MONITORING BENCHMARK
# Full VAL: 1657 frames
# No retraining | No TEST | No GNN | No video rendering

import cv2
import json
import time
import yaml
import sys
import importlib.util

import numpy as np
import pandas as pd
import torch

from pathlib import Path
from types import SimpleNamespace
from ultralytics import YOLO
from ultralytics.trackers.basetrack import BaseTrack

ROOT = Path("/content/drive/MyDrive/Crowd_AI_Project/runs_fresh")
OUT = ROOT / "benchmark"
OUT.mkdir(parents=True, exist_ok=True)

N_FRAMES = 1657
FPS = 25
W, H = 1654, 1080
N_ZONES = 12
STABILITY = 5

MODULE = ROOT / "optimized_tracker/batched_botsort.py"
CONFIG = OUT / "benchmark_botsort.yaml"
CALIB = ROOT / "zones/zone_level_calibration_MOT20-05.csv"
SELECTED = ROOT / "zones/optimization_v2/zone_stability_selected_VAL.json"

REF_DIR = ROOT / "zones/optimization_v2"
REF_MON = REF_DIR / "CELL93_BatchedBOTSORT_monitoring_VAL.csv"
REF_TRACK = REF_DIR / "CELL93_BatchedBOTSORT_tracks_VAL.csv"

for p in [MODULE, CONFIG, CALIB, SELECTED, REF_MON, REF_TRACK]:
    assert p.exists(), f"Missing: {p}"

assert callable(globals().get("best_weights")), (
    "best_weights() is missing. Run your project configuration cell."
)
assert "IMG_SIZE" in globals() and "MAX_DET" in globals()

with open(SELECTED) as f:
    selected = json.load(f)

assert int(selected["selected"]["threshold"]) == STABILITY

# --------------------------------------------------
# 1. Load saved tracker
# --------------------------------------------------

spec = importlib.util.spec_from_file_location(
    "batched_botsort_cell94", MODULE
)
mod = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = mod
spec.loader.exec_module(mod)

with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

assert cfg["tracker_type"] == "botsort"
cfg["gmc_method"] = "none"
cfg["with_reid"] = False

BaseTrack.reset_id()

tracker = mod.BatchedBOTSORT(
    args=SimpleNamespace(**cfg)
)

# --------------------------------------------------
# 2. Load VAL calibration
# --------------------------------------------------

calib = pd.read_csv(CALIB).sort_values("zone").reset_index(drop=True)

assert calib.zone.tolist() == list(range(N_ZONES))

calib_cols = [
    "zone", "own_low_max", "own_high_min",
    "person_area_px", "zone_area_px",
    "floor_low_max", "floor_high_min"
]

assert calib[calib_cols].notna().all().all()

own_low = calib.own_low_max.to_numpy()
own_high = calib.own_high_min.to_numpy()
person_area = calib.person_area_px.to_numpy()
zone_area = calib.zone_area_px.to_numpy()
floor_low = calib.floor_low_max.to_numpy()
floor_high = calib.floor_high_min.to_numpy()

assert (zone_area > 0).all()

# --------------------------------------------------
# 3. Resolve VAL images
# --------------------------------------------------

img_dir = Path("/content/MOT20/MOT20/train/MOT20-05/img1")

assert img_dir.is_dir(), f"Missing image folder: {img_dir}"

frames = [
    img_dir / f"{i:06d}.jpg"
    for i in range(1, N_FRAMES + 1)
]

assert all(p.is_file() for p in frames)

# --------------------------------------------------
# 4. Load YOLO
# --------------------------------------------------

device = 0 if torch.cuda.is_available() else "cpu"
model = YOLO(str(best_weights("yolov8m")))

def gpu_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

# Warm up detector without updating tracker.
warm_image = cv2.imread(str(frames[0]))
assert warm_image is not None
assert warm_image.shape[:2] == (H, W)

for _ in range(3):
    model.predict(
        warm_image,
        imgsz=IMG_SIZE,
        conf=0.01,
        iou=0.7,
        classes=[0],
        max_det=MAX_DET,
        device=device,
        verbose=False
    )

gpu_sync()
del warm_image

# --------------------------------------------------
# 5. Tracker detection adapter
# --------------------------------------------------

class DetectionResults:

    def __init__(self, arr):

        self.xywh = np.column_stack([
            (arr[:, 0] + arr[:, 2]) / 2,
            (arr[:, 1] + arr[:, 3]) / 2,
            arr[:, 2] - arr[:, 0],
            arr[:, 3] - arr[:, 1]
        ])

        self.conf = arr[:, 4].copy()
        self.cls = arr[:, 5].copy()

    def __len__(self):
        return len(self.conf)

    def __getitem__(self, index):
        obj = DetectionResults.__new__(DetectionResults)
        obj.xywh = self.xywh[index]
        obj.conf = self.conf[index]
        obj.cls = self.cls[index]
        return obj

# --------------------------------------------------
# 6. Prepare streaming monitoring state
# --------------------------------------------------

# person_id -> stable zone state
person_states = {}

track_rows = []
events = []
monitor_rows = []

window_sum = np.zeros(N_ZONES, dtype=np.float64)
window_max = np.zeros(N_ZONES, dtype=np.int32)
window_in = np.zeros(N_ZONES, dtype=np.int32)
window_out = np.zeros(N_ZONES, dtype=np.int32)
window_frames = 0

decode_ms = []
yolo_ms = []
tracker_ms = []
monitor_ms = []
other_ms = []
total_frame_ms = []

# --------------------------------------------------
# 7. Process entire VAL sequence in order
# --------------------------------------------------

print("=== CELL 94 — FULL MONITORING BENCHMARK ===")
print("Frames:", N_FRAMES)
print("Tracker: Saved Batched BoT-SORT")
print("GMC: OFF | ReID: OFF")
print("Stable transition rule:", STABILITY)
print("Starting full VAL processing...\n")

gpu_sync()
wall_start = time.perf_counter()

for frame, image_path in enumerate(frames, start=1):

    frame_start = time.perf_counter()

    # A. Image decoding
    t0 = time.perf_counter()

    image = cv2.imread(str(image_path))
    assert image is not None
    assert image.shape[:2] == (H, W)

    decode_ms.append((time.perf_counter() - t0) * 1000)

    # B. YOLO inference
    gpu_sync()
    t0 = time.perf_counter()

    result = model.predict(
        image,
        imgsz=IMG_SIZE,
        conf=0.01,
        iou=0.7,
        classes=[0],
        max_det=MAX_DET,
        device=device,
        verbose=False
    )[0]

    gpu_sync()
    yolo_ms.append((time.perf_counter() - t0) * 1000)

    # C. Convert detections
    t0 = time.perf_counter()

    boxes = result.boxes

    if boxes is None or len(boxes) == 0:
        arr = np.empty((0, 6), dtype=np.float64)
    else:
        arr = np.column_stack([
            boxes.xyxy.cpu().numpy(),
            boxes.conf.cpu().numpy(),
            boxes.cls.cpu().numpy()
        ]).astype(np.float64)

        valid = (
            np.isfinite(arr).all(axis=1)
            & (arr[:, 2] > arr[:, 0])
            & (arr[:, 3] > arr[:, 1])
        )

        arr = arr[valid]

    detections = DetectionResults(arr)

    other_ms.append((time.perf_counter() - t0) * 1000)

    # D. Optimized tracking
    t0 = time.perf_counter()

    tracks = tracker.update(detections)

    tracker_ms.append((time.perf_counter() - t0) * 1000)

    # E. Zone occupancy + stable transitions
    t0 = time.perf_counter()

    tracks = np.asarray(tracks)

    if tracks.size:
        box = tracks[:, :4]

        px = np.floor(
            (box[:, 0] + box[:, 2]) / 2
        ).astype(int)

        py = np.floor(box[:, 3]).astype(int)

        inside = (
            (px >= 0) & (px < W)
            & (py >= 0) & (py < H)
        )

        col = np.select(
            [px <= 413, px <= 826, px <= 1239],
            [0, 1, 2],
            default=3
        )

        row = np.select(
            [py <= 359, py <= 719],
            [0, 1],
            default=2
        )

        zones = np.where(
            inside, row * 4 + col, -1
        )

        counts = np.bincount(
            zones[zones >= 0],
            minlength=N_ZONES
        )

    else:
        zones = np.array([], dtype=int)
        counts = np.zeros(N_ZONES, dtype=int)

    window_sum += counts
    window_max = np.maximum(window_max, counts)
    window_frames += 1

    for idx, tr in enumerate(tracks):

        person_id = int(tr[4])
        zone = int(zones[idx])

        track_rows.append([
            frame, person_id,
            float(tr[0]), float(tr[1]),
            float(tr[2]), float(tr[3]),
            float(tr[5])
        ])

        state = person_states.setdefault(
            person_id,
            {
                "stable": -1,
                "candidate": -1,
                "streak": 0,
                "previous_frame": -999
            }
        )

        if frame != state["previous_frame"] + 1:
            state["candidate"] = -1
            state["streak"] = 0

        state["previous_frame"] = frame

        if zone == -1:
            state["stable"] = -1
            state["candidate"] = -1
            state["streak"] = 0
            continue

        if state["stable"] == -1:

            if zone == state["candidate"]:
                state["streak"] += 1
            else:
                state["candidate"] = zone
                state["streak"] = 1

            if state["streak"] >= STABILITY:
                state["stable"] = zone
                state["candidate"] = -1
                state["streak"] = 0

            continue

        if zone == state["stable"]:
            state["candidate"] = -1
            state["streak"] = 0
            continue

        if zone == state["candidate"]:
            state["streak"] += 1
        else:
            state["candidate"] = zone
            state["streak"] = 1

        if state["streak"] >= STABILITY:

            old_zone = state["stable"]

            events.append([
                frame, person_id, old_zone, zone
            ])

            window_out[old_zone] += 1
            window_in[zone] += 1

            state["stable"] = zone
            state["candidate"] = -1
            state["streak"] = 0

    # F. Finalize each 25-frame monitoring window
    if frame % FPS == 0 or frame == N_FRAMES:

        step = (frame - 1) // FPS
        mean_counts = window_sum / window_frames

        density = mean_counts * 10000 / zone_area
        coverage = mean_counts * person_area / zone_area

        for zone in range(N_ZONES):

            own_level = (
                "HIGH" if mean_counts[zone] >= own_high[zone]
                else "LOW" if mean_counts[zone] <= own_low[zone]
                else "MEDIUM"
            )

            floor_level = (
                "HIGH" if coverage[zone] >= floor_high[zone]
                else "LOW" if coverage[zone] <= floor_low[zone]
                else "MEDIUM"
            )

            alert = (
                "HIGH_CROWD"
                if own_level == "HIGH" or floor_level == "HIGH"
                else "NORMAL"
            )

            monitor_rows.append({
                "step": step,
                "zone": zone,
                "count": float(mean_counts[zone]),
                "max_count": int(window_max[zone]),
                "frames_in_window": window_frames,
                "inflow": int(window_in[zone]),
                "outflow": int(window_out[zone]),
                "density_per_100x100px": float(density[zone]),
                "floor_coverage": float(coverage[zone]),
                "level_own": own_level,
                "level_floor": floor_level,
                "crowd_alert": alert
            })

        window_sum.fill(0)
        window_max.fill(0)
        window_in.fill(0)
        window_out.fill(0)
        window_frames = 0

    monitor_ms.append((time.perf_counter() - t0) * 1000)

    gpu_sync()
    total_frame_ms.append(
        (time.perf_counter() - frame_start) * 1000
    )

    if frame % 250 == 0 or frame == N_FRAMES:
        print(f"Processed: {frame}/{N_FRAMES}")

    del image, result

gpu_sync()
wall_seconds = time.perf_counter() - wall_start

# --------------------------------------------------
# 8. Verify outputs against CELL 93
# --------------------------------------------------

tracks_df = pd.DataFrame(
    track_rows,
    columns=["frame", "id", "x1", "y1", "x2", "y2", "score"]
).sort_values(["frame", "id"]).reset_index(drop=True)

monitor_df = pd.DataFrame(
    monitor_rows
).sort_values(["step", "zone"]).reset_index(drop=True)

events_df = pd.DataFrame(
    events,
    columns=["frame", "id", "from_zone", "to_zone"]
)

ref_tracks = pd.read_csv(REF_TRACK).sort_values(
    ["frame", "id"]
).reset_index(drop=True)

ref_monitor = pd.read_csv(REF_MON).sort_values(
    ["step", "zone"]
).reset_index(drop=True)

track_keys = ["frame", "id"]
track_values = ["x1", "y1", "x2", "y2", "score"]

same_tracking = (
    len(tracks_df) == len(ref_tracks)
    and tracks_df[track_keys].equals(ref_tracks[track_keys])
    and np.allclose(
        tracks_df[track_values].to_numpy(),
        ref_tracks[track_values].to_numpy(),
        rtol=1e-8,
        atol=1e-6
    )
)

numeric = [
    "count", "max_count",
    "density_per_100x100px", "floor_coverage"
]

other = [
    "step", "zone", "frames_in_window",
    "inflow", "outflow", "level_own",
    "level_floor", "crowd_alert"
]

same_monitoring = (
    len(monitor_df) == len(ref_monitor)
    and monitor_df[other].equals(ref_monitor[other])
    and np.allclose(
        monitor_df[numeric].to_numpy(),
        ref_monitor[numeric].to_numpy(),
        rtol=1e-9,
        atol=1e-8
    )
)

checks = {
    "Full VAL processed": len(total_frame_ms) == N_FRAMES,
    "804 zone-windows": len(monitor_df) == 804,
    "12 zones per window": (
        monitor_df.groupby("step").zone.nunique().eq(12).all()
    ),
    "Final window has 7 frames": (
        monitor_df.loc[
            monitor_df.step == 66, "frames_in_window"
        ].eq(7).all()
    ),
    "No duplicate track IDs per frame": (
        not tracks_df.duplicated(["frame", "id"]).any()
    ),
    "Inflow equals outflow": (
        int(monitor_df.inflow.sum())
        == int(monitor_df.outflow.sum())
        == len(events_df)
    )
}

# --------------------------------------------------
# 9. Performance report
# --------------------------------------------------

def stats(values):
    a = np.asarray(values, dtype=float)
    return {
        "mean_ms": float(a.mean()),
        "p95_ms": float(np.percentile(a, 95))
    }

report = {
    "sequence": "MOT20-05",
    "split": "VAL",
    "frames": N_FRAMES,
    "tracker": "Saved BatchedBOTSORT",
    "gmc": "OFF",
    "reid": "OFF",
    "wall_seconds": float(wall_seconds),
    "measured_fps": float(N_FRAMES / wall_seconds),
    "mean_total_frame_ms": float(np.mean(total_frame_ms)),
    "decode": stats(decode_ms),
    "yolo": stats(yolo_ms),
    "detection_conversion": stats(other_ms),
    "tracker": stats(tracker_ms),
    "monitoring": stats(monitor_ms),
    "tracking_rows": int(len(tracks_df)),
    "transitions": int(len(events_df)),
    "zone_windows": int(len(monitor_df)),
    "same_tracking_as_cell93": bool(same_tracking),
    "same_monitoring_as_cell93": bool(same_monitoring),
    "checks": {
        k: bool(v) for k, v in checks.items()
    },
    "alerts": {
        str(k): int(v)
        for k, v in monitor_df.crowd_alert.value_counts().items()
    },
    "test_accessed": False,
    "gnn_included": False,
    "rendering_included": False,
    "scope": (
        "Sequential VAL image decoding, YOLO inference, "
        "saved batched tracking, streaming zone occupancy, "
        "five-frame stable flow and window-level alerts. "
        "Excludes model initialization, output file writing, "
        "video rendering and GNN."
    )
}

REPORT = OUT / "CELL94_full_monitoring_FPS_VAL.json"
MON_FILE = REF_DIR / "CELL94_streaming_monitoring_VAL.csv"
EVENT_FILE = REF_DIR / "CELL94_streaming_transitions_VAL.csv"

REPORT.write_text(json.dumps(report, indent=2))
monitor_df.to_csv(MON_FILE, index=False)
events_df.to_csv(EVENT_FILE, index=False)

print("\n=== CELL 94 — PERFORMANCE ===")

print(f"Image decoding:     {np.mean(decode_ms):.2f} ms")
print(f"YOLO inference:     {np.mean(yolo_ms):.2f} ms")
print(f"Tracker:            {np.mean(tracker_ms):.2f} ms")
print(f"Monitoring:         {np.mean(monitor_ms):.2f} ms")
print(f"Total wall time:    {wall_seconds:.2f} s")
print(f"Integrated FPS:     {N_FRAMES / wall_seconds:.2f}")

print("\n=== OUTPUT VALIDATION ===")

print("Same tracking as CELL 93:", same_tracking)
print("Same monitoring as CELL 93:", same_monitoring)
print("Stable transitions:", len(events_df))
print("Total zone-windows:", len(monitor_df))

print("\nAlerts:")
print(monitor_df.crowd_alert.value_counts().to_string())

print("\nValidation checks:")
for name, passed in checks.items():
    print(("PASS" if passed else "FAIL"), "|", name)

print("\nSaved:", REPORT)
print("Saved:", MON_FILE)
print("Saved:", EVENT_FILE)

if all(checks.values()) and same_tracking and same_monitoring:
    print("\nPASS — Full streaming monitoring benchmark verified.")
elif all(checks.values()):
    print("\nBENCHMARK COMPLETE — Outputs differ from CELL 93.")
    print("Check whether live YOLO detections differ from cached detections.")
else:
    print("\nWARNING — Monitoring integrity check failed.")

print("No TEST. No GNN.")

=== CELL 94 — FULL MONITORING BENCHMARK ===
Frames: 1657
Tracker: Saved Batched BoT-SORT
GMC: OFF | ReID: OFF
Stable transition rule: 5
Starting full VAL processing...

Processed: 250/1657
Processed: 500/1657
Processed: 750/1657
Processed: 1000/1657
Processed: 1250/1657
Processed: 1500/1657
Processed: 1657/1657

=== CELL 94 — PERFORMANCE ===
Image decoding:     10.64 ms
YOLO inference:     16.79 ms
Tracker:            15.61 ms
Monitoring:         0.83 ms
Total wall time:    73.66 s
Integrated FPS:     22.49

=== OUTPUT VALIDATION ===
Same tracking as CELL 93: True
Same monitoring as CELL 93: True
Stable transitions: 1319
Total zone-windows: 804

Alerts:
crowd_alert
NORMAL        674
HIGH_CROWD    130

Validation checks:
PASS | Full VAL processed
PASS | 804 zone-windows
PASS | 12 zones per window
PASS | Final window has 7 frames
PASS | No duplicate track IDs per frame
PASS | Inflow equals outflow

Saved: /content/drive/MyDrive/Crowd_AI_Project/runs_fresh/benchmark/CELL94_full_monitoring